In [ ]:
from pathlib import Path
import os
import sys
_release_root = Path(os.environ.get('AVATAR_RELEASE_ROOT', Path.cwd())).resolve()
while not (_release_root / 'avatar_paths.py').is_file():
    if _release_root == _release_root.parent:
        raise RuntimeError('Open this notebook from the code package directory.')
    _release_root = _release_root.parent
if str(_release_root) not in sys.path:
    sys.path.insert(0, str(_release_root))
from avatar_paths import project_path, get_value, get_column, get_path, get_label


In [ ]:
import os
import re
import random
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, RepeatedStratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score
import torch
import torch.nn as nn
import torch.optim as optim
import shap
RANDOM_STATE = 42
N_SPLITS = 5
N_REPEATS = 5
MAX_EPOCHS = 300
MIN_EPOCHS = 100
PATIENCE = 40
MIN_DELTA = 0.0001
INNER_VAL_SIZE = 0.2
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

INPUT_DIR = str(project_path(get_value('science.paths.directory_001', 'directory_001')))
LABEL_FILE_NAME = get_value('science.tables.outcomes_001', 'outcomes_001.csv')
label_path = os.path.join(INPUT_DIR, LABEL_FILE_NAME)
modal_files = {'Metagenomics': os.path.join(INPUT_DIR, get_value('science.tables.features_001', 'features_001.csv.gz')), 'Radiomics': os.path.join(INPUT_DIR, get_value('science.tables.features_002', 'features_002.csv.gz')), 'Blood': os.path.join(INPUT_DIR, get_value('science.tables.features_003', 'features_003.csv.gz')), 'Urine': os.path.join(INPUT_DIR, get_value('science.tables.features_004', 'features_004.csv.gz'))}
out_dir = str(project_path(get_value('science.paths.directory_002', 'directory_002')))
os.makedirs(out_dir, exist_ok=True)
single_modal_dir = os.path.join(out_dir, get_value('science.paths.directory_003', 'directory_003'))
os.makedirs(single_modal_dir, exist_ok=True)
LAMBDA_RESPONSE = 0.4
LAMBDA_PFS = 0.8
LAMBDA_OS = 1.0
modal_configs = {'Metagenomics': {'transform': 'none', 'missing_cutoff': 0.3, 'prevalence_cutoff': None, 'max_features': None, 'top_k_per_outcome': 10, 'hidden1': 64, 'hidden2': 16, 'dropout': 0.5}, 'Radiomics': {'transform': 'none', 'missing_cutoff': 0.3, 'prevalence_cutoff': None, 'max_features': None, 'top_k_per_outcome': 6, 'hidden1': 32, 'hidden2': 16, 'dropout': 0.4}, 'Blood': {'transform': 'none', 'missing_cutoff': 0.3, 'prevalence_cutoff': None, 'max_features': None, 'top_k_per_outcome': 8, 'hidden1': 64, 'hidden2': 16, 'dropout': 0.5}, 'Urine': {'transform': 'none', 'missing_cutoff': 0.3, 'prevalence_cutoff': None, 'max_features': None, 'top_k_per_outcome': 8, 'hidden1': 64, 'hidden2': 16, 'dropout': 0.55}}

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
set_seed(RANDOM_STATE)
required_paths = [label_path] + list(modal_files.values())
missing_paths = [p for p in required_paths if not os.path.exists(p)]
if missing_paths:
    print('\nThe following required files were not found:')
    for p in missing_paths:
        print(' -', p)
    raise FileNotFoundError('Please place the corrected label file and all four internal matrices in INPUT_DIR, or correct the filenames above.')


for modal_name, path in modal_files.items():
    pass


In [ ]:
def normalize_sample_id(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    if re.fullmatch('-?\\d+\\.0', s):
        s = s[:-2]
    return s.lower()

def find_column(df, candidates, required=True):
    lower_map = {str(c).strip().lower(): c for c in df.columns}
    for cand in candidates:
        key = str(cand).strip().lower()
        if key in lower_map:
            return lower_map[key]
    if required:
        raise ValueError(f'Cannot find any of columns: {candidates}')
    return None

def to_binary_response(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x, str):
        s = x.strip().lower()
        if s in ['1', 'yes', 'y', 'responder', 'response', 'r', 'cr', 'pr', 'orr']:
            return 1
        if s in ['0', 'no', 'n', 'nonresponder', 'non-responder', 'nr', 'sd', 'pd']:
            return 0
    return int(float(x))

def read_table(path):
    path_lower = path.lower()
    if path_lower.endswith(('.xlsx', '.xls')):
        return pd.read_excel(path)
    return pd.read_csv(path, encoding='utf-8-sig')

def read_labels(path):
    df = read_table(path)
    sample_col = find_column(df, ['Sample ID', 'sample_id', 'sample', 'Sample', 'ID'])
    response_col = find_column(df, ['Response', 'Response', 'response', 'ORR'])
    pfs_time_col = find_column(df, ['PFS_time', 'pfs_time', 'PFS', 'PFS_month', 'PFS_months'])
    pfs_status_col = find_column(df, ['PFS_status', 'pfs_status', 'PFS_event', 'pfs_event'])
    os_time_col = find_column(df, ['OS_time', 'os_time', 'OS', 'OS_month', 'OS_months'])
    os_status_col = find_column(df, ['OS_status', 'os_status', 'OS_event', 'os_event'])
    out = pd.DataFrame({'sample': df[sample_col].map(normalize_sample_id), 'Response': df[response_col].map(to_binary_response), 'PFS_time': pd.to_numeric(df[pfs_time_col], errors='coerce'), 'PFS_status': pd.to_numeric(df[pfs_status_col], errors='coerce'), 'OS_time': pd.to_numeric(df[os_time_col], errors='coerce'), 'OS_status': pd.to_numeric(df[os_status_col], errors='coerce')})
    out = out.dropna(subset=['sample'])
    out = out.drop_duplicates(subset='sample', keep='first')
    out = out.set_index('sample')
    required_cols = ['Response', 'PFS_time', 'PFS_status', 'OS_time', 'OS_status']
    if out[required_cols].isna().any().any():
        print('\nMissing values in label table:')
        print(out[required_cols].isna().sum())
        raise ValueError('The label table contains missing required outcomes.')
    out['Response'] = out['Response'].astype(int)
    out['PFS_status'] = out['PFS_status'].astype(int)
    out['OS_status'] = out['OS_status'].astype(int)
    n_pfs_after_os = int((out['PFS_time'] > out['OS_time'] + 1e-08).sum())
    n_os_event_pfs_censored = int(((out['OS_status'] == 1) & (out['PFS_status'] == 0)).sum())
    
    print('\nEvent summary:')
    
    print('\nTime summary:')
    
    print('\nOutcome logic audit:')
    print('PFS_time > OS_time:', n_pfs_after_os)
    print('OS event but PFS censored:', n_os_event_pfs_censored)
    if n_pfs_after_os > 0 or n_os_event_pfs_censored > 0:
        raise ValueError('Outcome logic audit failed.')
    return out

def read_matrix(path):
    df = read_table(path)
    df.columns = [str(c).strip() for c in df.columns]
    sample_col = find_column(df, ['sample', 'sample_id', 'Sample ID', 'Sample', 'ID'], required=False)
    if sample_col is None:
        sample_col = df.columns[0]
    df = df.rename(columns={sample_col: 'sample'})
    df['sample'] = df['sample'].map(normalize_sample_id)
    df = df.dropna(subset=['sample'])
    df = df.drop_duplicates(subset='sample', keep='first')
    df = df.set_index('sample')
    df = df.apply(pd.to_numeric, errors='coerce')
    return df
labels = read_labels(label_path)


In [ ]:
def preprocess_features_unsupervised(X, modal_name, config):
    X = X.copy()
    print(f'\n[{modal_name}] Raw matrix:', X.shape)
    X = X.dropna(axis=1, how='all')
    missing_cutoff = config.get('missing_cutoff', 0.3)
    miss_rate = X.isna().mean(axis=0)
    X = X.loc[:, miss_rate <= missing_cutoff]
    var = X.var(axis=0, skipna=True, ddof=0)
    X = X.loc[:, var > 0]
    print(f'[{modal_name}] After safety QC:', X.shape)
    return X
modal_matrices = {}
for modal_name, path in modal_files.items():
    X_raw = read_matrix(path)
    X_proc = preprocess_features_unsupervised(X_raw, modal_name=modal_name, config=modal_configs[modal_name])
    common = labels.index.intersection(X_proc.index)
    X_proc = X_proc.loc[common].copy()
    print(f'[{modal_name}] Common samples with labels:', X_proc.shape[0])
    if X_proc.shape[0] != labels.shape[0]:
        missing_samples = labels.index.difference(X_proc.index).tolist()
        print(f'[{modal_name}] Missing samples:', missing_samples[:20])
        raise ValueError(f'{modal_name} does not contain all labeled internal samples.')
    X_proc = X_proc.loc[labels.index]
    modal_matrices[modal_name] = X_proc
print('\nFinal internal matrices:')
for modal_name, X in modal_matrices.items():
    pass
reference_order = labels.index.tolist()
for modal_name, X in modal_matrices.items():
    if X.index.tolist() != reference_order:
        raise ValueError(f'Sample order mismatch in {modal_name}')
print('\nAll four modalities contain the same patients in the same order.')


In [ ]:
class SingleModalMultiTaskNet(nn.Module):

    def __init__(self, input_dim, hidden1=128, hidden2=32, dropout=0.4):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(input_dim, hidden1), nn.BatchNorm1d(hidden1), nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden1, hidden2), nn.BatchNorm1d(hidden2), nn.ReLU(), nn.Dropout(dropout))
        self.response_head = nn.Linear(hidden2, 1)
        self.pfs_head = nn.Linear(hidden2, 1)
        self.os_head = nn.Linear(hidden2, 1)

    def forward(self, x):
        z = self.encoder(x)
        return {'response_logit': self.response_head(z).view(-1), 'pfs_risk': self.pfs_head(z).view(-1), 'os_risk': self.os_head(z).view(-1)}

def cox_ph_loss(risk, time, event):
    pass
    risk = risk.view(-1)
    time = time.view(-1)
    event = event.view(-1)
    valid = ~torch.isnan(time) & ~torch.isnan(event)
    risk = risk[valid]
    time = time[valid]
    event = event[valid]
    if event.sum() < 1:
        return torch.tensor(0.0, device=risk.device)
    order = torch.argsort(time, descending=True)
    risk = risk[order]
    event = event[order]
    log_cumsum_exp = torch.logcumsumexp(risk, dim=0)
    loss = -torch.sum((risk - log_cumsum_exp) * event) / (torch.sum(event) + 1e-08)
    return loss

def multitask_loss(outputs, y_response, pfs_time, pfs_status, os_time, os_status, response_pos_weight=None, lambda_response=LAMBDA_RESPONSE, lambda_pfs=LAMBDA_PFS, lambda_os=LAMBDA_OS):
    if response_pos_weight is not None:
        bce = nn.BCEWithLogitsLoss(pos_weight=response_pos_weight)
    else:
        bce = nn.BCEWithLogitsLoss()
    loss_response = bce(outputs['response_logit'], y_response)
    loss_pfs = cox_ph_loss(outputs['pfs_risk'], pfs_time, pfs_status)
    loss_os = cox_ph_loss(outputs['os_risk'], os_time, os_status)
    loss = lambda_response * loss_response + lambda_pfs * loss_pfs + lambda_os * loss_os
    return (loss, {'loss_response': loss_response.item(), 'loss_pfs': loss_pfs.item(), 'loss_os': loss_os.item()})

def fit_preprocessor_for_modal(X_fit, modal_name):
    pass
    imputer = SimpleImputer(strategy='median')
    X_fit_processed = imputer.fit_transform(X_fit)
    scaler = None
    return (imputer, scaler, X_fit_processed)

def transform_with_preprocessor(X, imputer, scaler):
    return imputer.transform(X)

def make_label_tensors(y_df):
    y_response = torch.tensor(y_df['Response'].values, dtype=torch.float32).to(DEVICE)
    pfs_time = torch.tensor(y_df['PFS_time'].values, dtype=torch.float32).to(DEVICE)
    pfs_status = torch.tensor(y_df['PFS_status'].values, dtype=torch.float32).to(DEVICE)
    os_time = torch.tensor(y_df['OS_time'].values, dtype=torch.float32).to(DEVICE)
    os_status = torch.tensor(y_df['OS_status'].values, dtype=torch.float32).to(DEVICE)
    return (y_response, pfs_time, pfs_status, os_time, os_status)

def get_response_pos_weight(y_response_tensor):
    n_pos = y_response_tensor.sum().item()
    n_neg = len(y_response_tensor) - n_pos
    if n_pos > 0:
        return torch.tensor([n_neg / n_pos], dtype=torch.float32).to(DEVICE)
    else:
        return None

def determine_best_epoch_with_inner_validation(X_train, y_train, modal_name, config, fold_id):
    pass
    idx_all = np.arange(X_train.shape[0])
    stratify_y = y_train['Response'].astype(int).values
    try:
        inner_train_idx, inner_val_idx = train_test_split(idx_all, test_size=INNER_VAL_SIZE, random_state=RANDOM_STATE + fold_id, stratify=stratify_y)
    except ValueError:
        inner_train_idx, inner_val_idx = train_test_split(idx_all, test_size=INNER_VAL_SIZE, random_state=RANDOM_STATE + fold_id, stratify=None)
    X_inner_train = X_train.iloc[inner_train_idx]
    X_inner_val = X_train.iloc[inner_val_idx]
    y_inner_train = y_train.iloc[inner_train_idx]
    y_inner_val = y_train.iloc[inner_val_idx]
    imputer, scaler, X_inner_train_scaled = fit_preprocessor_for_modal(X_inner_train, modal_name)
    X_inner_val_scaled = transform_with_preprocessor(X_inner_val, imputer, scaler)
    X_inner_train_tensor = torch.tensor(X_inner_train_scaled, dtype=torch.float32).to(DEVICE)
    X_inner_val_tensor = torch.tensor(X_inner_val_scaled, dtype=torch.float32).to(DEVICE)
    y_response_train, pfs_time_train, pfs_status_train, os_time_train, os_status_train = make_label_tensors(y_inner_train)
    y_response_val, pfs_time_val, pfs_status_val, os_time_val, os_status_val = make_label_tensors(y_inner_val)
    response_pos_weight = get_response_pos_weight(y_response_train)
    model = SingleModalMultiTaskNet(input_dim=X_inner_train_scaled.shape[1], hidden1=config.get('hidden1', 64), hidden2=config.get('hidden2', 16), dropout=config.get('dropout', 0.5)).to(DEVICE)
    optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.0001)
    best_val_loss = np.inf
    best_epoch = 1
    patience_counter = 0
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        optimizer.zero_grad()
        outputs = model(X_inner_train_tensor)
        loss, _ = multitask_loss(outputs, y_response_train, pfs_time_train, pfs_status_train, os_time_train, os_status_train, response_pos_weight=response_pos_weight, lambda_response=LAMBDA_RESPONSE, lambda_pfs=LAMBDA_PFS, lambda_os=LAMBDA_OS)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        model.eval()
        with torch.no_grad():
            val_outputs = model(X_inner_val_tensor)
            val_loss, _ = multitask_loss(val_outputs, y_response_val, pfs_time_val, pfs_status_val, os_time_val, os_status_val, response_pos_weight=None, lambda_response=LAMBDA_RESPONSE, lambda_pfs=LAMBDA_PFS, lambda_os=LAMBDA_OS)
        val_loss_value = val_loss.item()
        if val_loss_value < best_val_loss - MIN_DELTA:
            best_val_loss = val_loss_value
            best_epoch = epoch
            patience_counter = 0
        else:
            patience_counter += 1
        if epoch >= MIN_EPOCHS and patience_counter >= PATIENCE:
            break
    
    return best_epoch


In [ ]:
def safe_auc(y_true, y_score):
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)
    mask = ~np.isnan(y_true) & ~np.isnan(y_score)
    y_true = y_true[mask]
    y_score = y_score[mask]
    if len(np.unique(y_true)) < 2:
        return np.nan
    return roc_auc_score(y_true, y_score)

def safe_auprc(y_true, y_score):
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)
    mask = ~np.isnan(y_true) & ~np.isnan(y_score)
    y_true = y_true[mask]
    y_score = y_score[mask]
    if len(np.unique(y_true)) < 2:
        return np.nan
    return average_precision_score(y_true, y_score)

def concordance_index(time, event, risk):
    pass
    time = np.asarray(time, dtype=float)
    event = np.asarray(event, dtype=float)
    risk = np.asarray(risk, dtype=float)
    mask = ~np.isnan(time) & ~np.isnan(event) & ~np.isnan(risk)
    time = time[mask]
    event = event[mask]
    risk = risk[mask]
    n = len(time)
    if n < 2:
        return np.nan
    permissible = 0
    concordant = 0
    ties = 0
    for i in range(n):
        for j in range(n):
            if time[i] < time[j] and event[i] == 1:
                permissible += 1
                if risk[i] > risk[j]:
                    concordant += 1
                elif risk[i] == risk[j]:
                    ties += 1
    if permissible == 0:
        return np.nan
    return (concordant + 0.5 * ties) / permissible


In [ ]:
class TaskWrapper(nn.Module):

    def __init__(self, base_model, task):
        super().__init__()
        self.base_model = base_model
        self.task = task

    def forward(self, x):
        out = self.base_model(x)
        if self.task == 'Response':
            return out['response_logit'].view(-1, 1)
        elif self.task == 'PFS':
            return out['pfs_risk'].view(-1, 1)
        elif self.task == 'OS':
            return out['os_risk'].view(-1, 1)
        else:
            raise ValueError(f'Unknown task: {self.task}')

def compute_signed_shap_for_fold(model, X_train_scaled, X_test_scaled, feature_names, task, test_index, max_background=40):
    pass
    model.eval()
    n_bg = min(max_background, X_train_scaled.shape[0])
    bg_idx = np.random.choice(X_train_scaled.shape[0], size=n_bg, replace=False)
    background = torch.tensor(X_train_scaled[bg_idx], dtype=torch.float32).to(DEVICE)
    test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32).to(DEVICE)
    wrapped = TaskWrapper(model, task).to(DEVICE)
    wrapped.eval()
    try:
        explainer = shap.DeepExplainer(wrapped, background)
        shap_values = explainer.shap_values(test_tensor)
    except Exception as e:
        print(f'DeepExplainer failed for {task}; using GradientExplainer. Error: {e}')
        explainer = shap.GradientExplainer(wrapped, background)
        shap_values = explainer.shap_values(test_tensor)
    if isinstance(shap_values, list):
        shap_values = shap_values[0]
    if torch.is_tensor(shap_values):
        shap_values = shap_values.detach().cpu().numpy()
    shap_values = np.asarray(shap_values)
    if shap_values.ndim == 3:
        shap_values = shap_values[:, :, 0]
    shap_df = pd.DataFrame(shap_values, index=test_index, columns=feature_names)
    return shap_df


In [ ]:
def run_single_modal_cv_shap(modal_name, X, labels, config):
    
    
    
    common = X.index.intersection(labels.index)
    X = X.loc[common].copy()
    y_df = labels.loc[common].copy()
    required_cols = ['Response', 'PFS_time', 'PFS_status', 'OS_time', 'OS_status']
    valid = y_df[required_cols].notna().all(axis=1)
    X = X.loc[valid]
    y_df = y_df.loc[valid]
    print(f'[{modal_name}] Final modeling samples:', X.shape[0])
    print(f'[{modal_name}] Final modeling features:', X.shape[1])
    y_response = y_df['Response'].astype(int).values
    cv = RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=RANDOM_STATE)
    pred_records = []
    fold_perf_records = []
    task_shap_signed_list = {'Response': [], 'PFS': [], 'OS': []}
    feature_names = X.columns.tolist()
    for fold_id, (train_idx, test_idx) in enumerate(cv.split(X, y_response), start=1):
        print(f'\n[{modal_name}] Fold {fold_id}')
        set_seed(RANDOM_STATE + fold_id)
        X_train = X.iloc[train_idx]
        X_test = X.iloc[test_idx]
        y_train = y_df.iloc[train_idx]
        y_test = y_df.iloc[test_idx]
        idx_all = np.arange(X_train.shape[0])
        stratify_y = y_train['Response'].astype(int).values
        try:
            inner_train_idx, inner_val_idx = train_test_split(idx_all, test_size=INNER_VAL_SIZE, random_state=RANDOM_STATE + fold_id, stratify=stratify_y)
        except ValueError:
            inner_train_idx, inner_val_idx = train_test_split(idx_all, test_size=INNER_VAL_SIZE, random_state=RANDOM_STATE + fold_id, stratify=None)
        X_inner_train = X_train.iloc[inner_train_idx]
        X_inner_val = X_train.iloc[inner_val_idx]
        y_inner_train = y_train.iloc[inner_train_idx]
        y_inner_val = y_train.iloc[inner_val_idx]
        inner_imputer, inner_scaler, X_inner_train_scaled = fit_preprocessor_for_modal(X_inner_train, modal_name)
        X_inner_val_scaled = transform_with_preprocessor(X_inner_val, inner_imputer, inner_scaler)
        X_inner_train_tensor = torch.tensor(X_inner_train_scaled, dtype=torch.float32).to(DEVICE)
        X_inner_val_tensor = torch.tensor(X_inner_val_scaled, dtype=torch.float32).to(DEVICE)
        y_response_inner_train, pfs_time_inner_train, pfs_status_inner_train, os_time_inner_train, os_status_inner_train = make_label_tensors(y_inner_train)
        y_response_inner_val, pfs_time_inner_val, pfs_status_inner_val, os_time_inner_val, os_status_inner_val = make_label_tensors(y_inner_val)
        pos_weight_inner = get_response_pos_weight(y_response_inner_train)
        inner_model = SingleModalMultiTaskNet(input_dim=X_inner_train_scaled.shape[1], hidden1=config.get('hidden1', 64), hidden2=config.get('hidden2', 16), dropout=config.get('dropout', 0.5)).to(DEVICE)
        inner_optimizer = optim.AdamW(inner_model.parameters(), lr=0.001, weight_decay=0.0001)
        best_val_loss = np.inf
        best_epoch = 1
        patience_counter = 0
        for epoch in range(1, MAX_EPOCHS + 1):
            inner_model.train()
            inner_optimizer.zero_grad()
            inner_outputs = inner_model(X_inner_train_tensor)
            inner_loss, inner_loss_parts = multitask_loss(inner_outputs, y_response_inner_train, pfs_time_inner_train, pfs_status_inner_train, os_time_inner_train, os_status_inner_train, response_pos_weight=pos_weight_inner, lambda_response=LAMBDA_RESPONSE, lambda_pfs=LAMBDA_PFS, lambda_os=LAMBDA_OS)
            inner_loss.backward()
            torch.nn.utils.clip_grad_norm_(inner_model.parameters(), max_norm=5.0)
            inner_optimizer.step()
            inner_model.eval()
            with torch.no_grad():
                val_outputs = inner_model(X_inner_val_tensor)
                val_loss, val_loss_parts = multitask_loss(val_outputs, y_response_inner_val, pfs_time_inner_val, pfs_status_inner_val, os_time_inner_val, os_status_inner_val, response_pos_weight=pos_weight_inner, lambda_response=LAMBDA_RESPONSE, lambda_pfs=LAMBDA_PFS, lambda_os=LAMBDA_OS)
            val_loss_value = val_loss.item()
            if val_loss_value < best_val_loss - MIN_DELTA:
                best_val_loss = val_loss_value
                best_epoch = epoch
                patience_counter = 0
            else:
                patience_counter += 1
            if epoch % 50 == 0:
                pass
            if epoch >= MIN_EPOCHS and patience_counter >= PATIENCE:
                
                break
        
        imputer, scaler, X_train_scaled = fit_preprocessor_for_modal(X_train, modal_name)
        X_test_scaled = transform_with_preprocessor(X_test, imputer, scaler)
        X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32).to(DEVICE)
        X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32).to(DEVICE)
        y_response_train, pfs_time_train, pfs_status_train, os_time_train, os_status_train = make_label_tensors(y_train)
        pos_weight = get_response_pos_weight(y_response_train)
        model = SingleModalMultiTaskNet(input_dim=X_train_scaled.shape[1], hidden1=config.get('hidden1', 64), hidden2=config.get('hidden2', 16), dropout=config.get('dropout', 0.5)).to(DEVICE)
        optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.0001)
        model.train()
        for epoch in range(1, best_epoch + 1):
            optimizer.zero_grad()
            outputs = model(X_train_tensor)
            loss, loss_parts = multitask_loss(outputs, y_response_train, pfs_time_train, pfs_status_train, os_time_train, os_status_train, response_pos_weight=pos_weight, lambda_response=LAMBDA_RESPONSE, lambda_pfs=LAMBDA_PFS, lambda_os=LAMBDA_OS)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()
            if epoch % 100 == 0 or epoch == best_epoch:
                pass
        model.eval()
        with torch.no_grad():
            test_out = model(X_test_tensor)
            pred_response = torch.sigmoid(test_out['response_logit']).detach().cpu().numpy()
            pred_pfs_risk = test_out['pfs_risk'].detach().cpu().numpy()
            pred_os_risk = test_out['os_risk'].detach().cpu().numpy()
        fold_pred = pd.DataFrame({'sample': X_test.index, 'modal': modal_name, 'fold_id': fold_id, 'Response_true': y_test['Response'].values, 'Response_pred': pred_response, 'PFS_time': y_test['PFS_time'].values, 'PFS_status': y_test['PFS_status'].values, 'PFS_risk': pred_pfs_risk, 'OS_time': y_test['OS_time'].values, 'OS_status': y_test['OS_status'].values, 'OS_risk': pred_os_risk})
        pred_records.append(fold_pred)
        fold_auc = safe_auc(fold_pred['Response_true'], fold_pred['Response_pred'])
        fold_auprc = safe_auprc(fold_pred['Response_true'], fold_pred['Response_pred'])
        fold_pfs_c = concordance_index(fold_pred['PFS_time'], fold_pred['PFS_status'], fold_pred['PFS_risk'])
        fold_os_c = concordance_index(fold_pred['OS_time'], fold_pred['OS_status'], fold_pred['OS_risk'])
        fold_perf_records.append({'modal': modal_name, 'fold_id': fold_id, 'Response_AUC': fold_auc, 'Response_AUPRC': fold_auprc, 'PFS_Cindex': fold_pfs_c, 'OS_Cindex': fold_os_c})
        print(f'Fold {fold_id} performance | Resp AUC={fold_auc:.3f}, PFS C={fold_pfs_c:.3f}, OS C={fold_os_c:.3f}')
        for task in ['Response', 'PFS', 'OS']:
            shap_signed = compute_signed_shap_for_fold(model=model, X_train_scaled=X_train_scaled, X_test_scaled=X_test_scaled, feature_names=feature_names, task=task, test_index=X_test.index, max_background=40)
            task_shap_signed_list[task].append(shap_signed)
    pred_df = pd.concat(pred_records, axis=0)
    fold_perf_df = pd.DataFrame(fold_perf_records)
    overall_perf = {'modal': modal_name, 'n_samples': X.shape[0], 'n_features': X.shape[1], 'Response_AUC_mean': fold_perf_df['Response_AUC'].mean(), 'Response_AUC_sd': fold_perf_df['Response_AUC'].std(), 'Response_AUPRC_mean': fold_perf_df['Response_AUPRC'].mean(), 'Response_AUPRC_sd': fold_perf_df['Response_AUPRC'].std(), 'PFS_Cindex_mean': fold_perf_df['PFS_Cindex'].mean(), 'PFS_Cindex_sd': fold_perf_df['PFS_Cindex'].std(), 'OS_Cindex_mean': fold_perf_df['OS_Cindex'].mean(), 'OS_Cindex_sd': fold_perf_df['OS_Cindex'].std()}
    shap_importance_records = []
    top_feature_records = []
    shap_signed_agg_dict = {}
    shap_abs_agg_dict = {}
    for task in ['Response', 'PFS', 'OS']:
        shap_signed_all = pd.concat(task_shap_signed_list[task], axis=0)
        shap_signed_agg = shap_signed_all.groupby(level=0).mean()
        shap_abs_agg = shap_signed_all.abs().groupby(level=0).mean()
        common_shap_samples = X.index.intersection(shap_signed_agg.index)
        shap_signed_agg = shap_signed_agg.loc[common_shap_samples]
        shap_abs_agg = shap_abs_agg.loc[common_shap_samples]
        shap_signed_agg_dict[task] = shap_signed_agg
        shap_abs_agg_dict[task] = shap_abs_agg
        mean_abs_shap = shap_abs_agg.mean(axis=0).sort_values(ascending=False)
        shap_imp = mean_abs_shap.reset_index()
        shap_imp.columns = ['feature', 'mean_abs_shap']
        shap_imp['modal'] = modal_name
        shap_imp['outcome'] = task
        shap_importance_records.append(shap_imp)
        top_k = config.get('top_k_per_outcome', 10)
        top_df = shap_imp.head(top_k).copy()
        top_df['rank'] = np.arange(1, top_df.shape[0] + 1)
        top_feature_records.append(top_df)
    shap_importance_df = pd.concat(shap_importance_records, axis=0)
    top_features_df = pd.concat(top_feature_records, axis=0)
    return {'modal': modal_name, 'predictions': pred_df, 'fold_performance': fold_perf_df, 'overall_performance': overall_perf, 'shap_importance': shap_importance_df, 'top_features': top_features_df, 'shap_signed_agg': shap_signed_agg_dict, 'shap_abs_agg': shap_abs_agg_dict, 'X_stage1_input': X}


In [ ]:
all_overall_perf = []
all_fold_perf = []
all_predictions = []
all_shap_importance = []
all_top_features = []
for modal_name, X in modal_matrices.items():
    result = run_single_modal_cv_shap(modal_name=modal_name, X=X, labels=labels, config=modal_configs[modal_name])
    all_overall_perf.append(result['overall_performance'])
    all_fold_perf.append(result['fold_performance'])
    all_predictions.append(result['predictions'])
    all_shap_importance.append(result['shap_importance'])
    all_top_features.append(result['top_features'])
    modal_out_dir = os.path.join(single_modal_dir, modal_name)
    os.makedirs(modal_out_dir, exist_ok=True)
    result['predictions'].to_csv(os.path.join(modal_out_dir, f"{modal_name}{get_value('science.tables.predictions_001', 'predictions_001.csv')}"), index=False, encoding='utf-8-sig')
    result['fold_performance'].to_csv(os.path.join(modal_out_dir, f"{modal_name}{get_value('science.tables.metrics_001', 'metrics_001.csv')}"), index=False, encoding='utf-8-sig')
    result['shap_importance'].to_csv(os.path.join(modal_out_dir, f"{modal_name}{get_value('science.tables.attribution_001', 'attribution_001.csv')}"), index=False, encoding='utf-8-sig')
    result['top_features'].to_csv(os.path.join(modal_out_dir, f"{modal_name}{get_value('science.tables.result_001', 'result_001.csv')}"), index=False, encoding='utf-8-sig')
    circos_save_dir = os.path.join(modal_out_dir, get_value('science.paths.directory_004', 'directory_004'))
    os.makedirs(circos_save_dir, exist_ok=True)
    X_stage1_input = result['X_stage1_input'].copy()
    X_stage1_input.to_csv(os.path.join(circos_save_dir, f"{modal_name}{get_value('science.tables.result_002', 'result_002.csv')}"), encoding='utf-8-sig')
    imputer_vis = SimpleImputer(strategy='median')
    X_imp_vis = imputer_vis.fit_transform(X_stage1_input)
    if modal_name in ['Blood', 'Urine']:
        X_visual = pd.DataFrame(X_imp_vis, index=X_stage1_input.index, columns=X_stage1_input.columns)
    else:
        scaler_vis = StandardScaler()
        X_visual_scaled = scaler_vis.fit_transform(X_imp_vis)
        X_visual = pd.DataFrame(X_visual_scaled, index=X_stage1_input.index, columns=X_stage1_input.columns)
    X_visual.to_csv(os.path.join(circos_save_dir, f"{modal_name}{get_value('science.tables.result_003', 'result_003.csv')}"), encoding='utf-8-sig')
    pred_df = result['predictions'].copy()
    truth_cols = ['Response_true', 'PFS_time', 'PFS_status', 'OS_time', 'OS_status']
    pred_mean = pred_df.groupby('sample')[['Response_pred', 'PFS_risk', 'OS_risk']].mean()
    truth_df = pred_df.groupby('sample')[truth_cols].first()
    pred_agg = truth_df.join(pred_mean)
    pred_agg.to_csv(os.path.join(circos_save_dir, f"{modal_name}{get_value('science.tables.predictions_002', 'predictions_002.csv')}"), encoding='utf-8-sig')
    for task in ['Response', 'PFS', 'OS']:
        shap_signed = result['shap_signed_agg'][task].copy()
        shap_abs = result['shap_abs_agg'][task].copy()
        shap_signed.to_csv(os.path.join(circos_save_dir, f'{modal_name}_cv_shap_signed_{task}.csv.gz'), encoding='utf-8-sig', compression='gzip')
        shap_abs.to_csv(os.path.join(circos_save_dir, f'{modal_name}_cv_shap_abs_{task}.csv.gz'), encoding='utf-8-sig', compression='gzip')
    
overall_perf_df = pd.DataFrame(all_overall_perf)
fold_perf_df = pd.concat(all_fold_perf, axis=0)
predictions_df = pd.concat(all_predictions, axis=0)
shap_importance_df = pd.concat(all_shap_importance, axis=0)
top_features_df = pd.concat(all_top_features, axis=0)
overall_perf_df.to_csv(os.path.join(single_modal_dir, get_value('science.tables.metrics_002', 'metrics_002.csv')), index=False, encoding='utf-8-sig')
fold_perf_df.to_csv(os.path.join(single_modal_dir, get_value('science.tables.metrics_003', 'metrics_003.csv')), index=False, encoding='utf-8-sig')
predictions_df.to_csv(os.path.join(single_modal_dir, get_value('science.tables.predictions_003', 'predictions_003.csv')), index=False, encoding='utf-8-sig')
shap_importance_df.to_csv(os.path.join(single_modal_dir, get_value('science.tables.attribution_002', 'attribution_002.csv')), index=False, encoding='utf-8-sig')
top_features_df.to_csv(os.path.join(single_modal_dir, get_value('science.tables.result_004', 'result_004.csv')), index=False, encoding='utf-8-sig')
overall_perf_df


In [ ]:
golden_records = []
for modal_name in top_features_df['modal'].unique():
    sub = top_features_df[top_features_df['modal'] == modal_name].copy()
    for feature in sub['feature'].unique():
        ss = sub[sub['feature'] == feature]
        outcomes = sorted(ss['outcome'].unique().tolist())
        golden_records.append({'modal': modal_name, 'feature': feature, 'selected_outcomes': ';'.join(outcomes), 'n_outcomes': len(outcomes), 'mean_shap_across_selected': ss['mean_abs_shap'].mean(), 'max_shap_across_selected': ss['mean_abs_shap'].max()})
golden_features_df = pd.DataFrame(golden_records)
golden_features_df = golden_features_df.sort_values(['modal', 'n_outcomes', 'max_shap_across_selected'], ascending=[True, False, False])
golden_features_df.to_csv(os.path.join(single_modal_dir, get_value('science.tables.result_005', 'result_005.csv')), index=False, encoding='utf-8-sig')
golden_features_df.head()


In [ ]:
circos_feature_records = []
for modal_name in modal_matrices.keys():
    modal_out_dir = os.path.join(single_modal_dir, modal_name)
    circos_save_dir = os.path.join(modal_out_dir, get_value('science.paths.directory_004', 'directory_004'))
    shap_abs_dict = {}
    for task in ['Response', 'PFS', 'OS']:
        shap_abs_path = os.path.join(circos_save_dir, f'{modal_name}_cv_shap_abs_{task}.csv.gz')
        shap_abs_dict[task] = pd.read_csv(shap_abs_path, index_col=0, encoding='utf-8-sig', compression='gzip')
    feature_names = shap_abs_dict['Response'].columns.tolist()
    mean_abs_by_task = {task: shap_abs_dict[task].mean(axis=0) for task in ['Response', 'PFS', 'OS']}
    mean_global = pd.concat(mean_abs_by_task, axis=1)
    mean_global.columns = ['Response', 'PFS', 'OS']
    mean_global['global_mean_abs_shap'] = mean_global.mean(axis=1)
    global_features = mean_global.sort_values('global_mean_abs_shap', ascending=False).head(2).index.tolist()
    used = set(global_features)
    for rank, feat in enumerate(global_features, start=1):
        circos_feature_records.append({'modal': modal_name, 'feature': feat, 'feature_type': 'global', 'assigned_task': 'Shared', 'rank': rank, 'mean_abs_shap': mean_global.loc[feat, 'global_mean_abs_shap']})
    for task in ['Response', 'PFS', 'OS']:
        task_ranked = mean_abs_by_task[task].sort_values(ascending=False)
        selected_feat = None
        for feat in task_ranked.index:
            if feat not in used:
                selected_feat = feat
                used.add(feat)
                break
        if selected_feat is None:
            selected_feat = task_ranked.index[0]
        circos_feature_records.append({'modal': modal_name, 'feature': selected_feat, 'feature_type': 'specific', 'assigned_task': task, 'rank': 1, 'mean_abs_shap': mean_abs_by_task[task].loc[selected_feat]})
circos_feature_table = pd.DataFrame(circos_feature_records)
circos_feature_table.to_csv(os.path.join(single_modal_dir, get_value('science.tables.result_006', 'result_006.csv')), index=False, encoding='utf-8-sig')
circos_feature_table


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from pycirclize import Circos

single_modal_dir = str(project_path(get_value('science.paths.directory_005', 'directory_005')))
circos_feature_table_path = os.path.join(single_modal_dir, get_value('science.tables.result_006', 'result_006.csv'))
circos_plot_dir = os.path.join(single_modal_dir, get_value('science.paths.directory_006', 'directory_006'))
os.makedirs(circos_plot_dir, exist_ok=True)
modalities = ['Metagenomics', 'Radiomics', 'Blood', 'Urine']
task_names = ['Response', 'PFS', 'OS']
CANVAS_SIZE = (10, 10)
FONT_CENTER = 30
FONT_SECTOR = 16
FONT_LABEL_L1 = 8
FONT_LABEL_L2 = 6.5
CIRCOS_SIZE = [6, 18, 6, 14]
TRACK4_STEP = 5.5

def normalize_sample_id(x):
    return str(x).lower().strip()

def read_csv_index(path, compression=None):
    df = pd.read_csv(path, index_col=0, encoding='utf-8-sig', compression=compression)
    df.index = df.index.map(normalize_sample_id)
    return df

def minmax_01(arr, jitter_binary=False):
    arr = np.asarray(arr, dtype=float)
    if jitter_binary and len(np.unique(arr[~np.isnan(arr)])) <= 2:
        arr = arr + np.random.uniform(-0.08, 0.08, size=len(arr))
    scaler = MinMaxScaler()
    return scaler.fit_transform(arr.reshape(-1, 1)).flatten()

def clean_feature_name(feat, max_len=15):
    s = str(feat)
    if '__' in s:
        s = s.split('__')[-1]
    s = s.replace('g__', '')
    s = s.replace('s__', '')
    s = s.replace('k__', '')
    if len(s) > max_len:
        s = s[:max_len]
    return s

def display_modal_name(modal_name):
    if modal_name.lower() == 'metagenomics':
        return 'Microbiota'
    return modal_name

def safe_get_existing_features(feature_list, columns):
    return [f for f in feature_list if f in columns]
circos_feature_table = pd.read_csv(circos_feature_table_path, encoding='utf-8-sig')



def prepare_stage1_circos_data(modal_name):
    
    modal_out_dir = os.path.join(single_modal_dir, modal_name)
    circos_input_dir = os.path.join(modal_out_dir, get_value('science.paths.directory_004', 'directory_004'))
    x_visual_path = os.path.join(circos_input_dir, f"{modal_name}{get_value('science.tables.result_003', 'result_003.csv')}")
    pred_path = os.path.join(circos_input_dir, f"{modal_name}{get_value('science.tables.predictions_002', 'predictions_002.csv')}")
    X_visual = read_csv_index(x_visual_path)
    pred_df = read_csv_index(pred_path)
    shap_signed_dict = {}
    shap_abs_dict = {}
    for task in task_names:
        signed_path = os.path.join(circos_input_dir, f'{modal_name}_cv_shap_signed_{task}.csv.gz')
        abs_path = os.path.join(circos_input_dir, f'{modal_name}_cv_shap_abs_{task}.csv.gz')
        shap_signed_dict[task] = read_csv_index(signed_path, compression='gzip')
        shap_abs_dict[task] = read_csv_index(abs_path, compression='gzip')
    common_samples = X_visual.index
    common_samples = common_samples.intersection(pred_df.index)
    for task in task_names:
        common_samples = common_samples.intersection(shap_signed_dict[task].index)
        common_samples = common_samples.intersection(shap_abs_dict[task].index)
    X_visual = X_visual.loc[common_samples]
    pred_df = pred_df.loc[common_samples]
    for task in task_names:
        shap_signed_dict[task] = shap_signed_dict[task].loc[common_samples]
        shap_abs_dict[task] = shap_abs_dict[task].loc[common_samples]
    print(f'[{modal_name}] aligned samples = {len(common_samples)}')
    print(f'[{modal_name}] visual features = {X_visual.shape[1]}')
    ft = circos_feature_table[circos_feature_table['modal'].astype(str).str.lower() == modal_name.lower()].copy()
    global_features = ft[ft['feature_type'] == 'global'].sort_values('rank')['feature'].astype(str).tolist()
    specific_feature_map = {}
    for task in task_names:
        sub = ft[(ft['feature_type'] == 'specific') & (ft['assigned_task'].astype(str) == task)].copy()
        if sub.shape[0] > 0:
            specific_feature_map[task] = sub.sort_values('rank')['feature'].astype(str).iloc[0]
        else:
            specific_feature_map[task] = None
    global_features = safe_get_existing_features(global_features, X_visual.columns)
    for task in task_names:
        feat = specific_feature_map[task]
        if feat not in X_visual.columns:
            print(f'[Warning] {modal_name} {task} specific feature not found in X_visual: {feat}')
            specific_feature_map[task] = None
    print('Global features:', global_features)
    print('Specific features:', specific_feature_map)
    plot_features = []
    for feat in global_features:
        plot_features.append({'name': feat, 'type': 'global'})
    for task in task_names:
        feat = specific_feature_map.get(task)
        if feat is not None:
            plot_features.append({'name': feat, 'type': task})
    metadata_cols = []
    for col in ['Response_true', 'PFS_status', 'OS_status']:
        if col in pred_df.columns:
            metadata_cols.append(col)
    metadata = pred_df[metadata_cols].copy()
    for col in metadata.columns:
        metadata[col] = pd.to_numeric(metadata[col], errors='coerce').fillna(0).astype(int)
    metadata_label_map = {'Response_true': 'Response', 'PFS_status': 'PFS event', 'OS_status': 'OS event'}
    metadata = metadata.rename(columns=metadata_label_map)
    result_list = []
    for task in task_names:
        df_task = pd.DataFrame(index=common_samples)
        if task == 'Response':
            true_raw = pd.to_numeric(pred_df['Response_true'], errors='coerce').values
            pred_raw = pd.to_numeric(pred_df['Response_pred'], errors='coerce').values
            df_task['True_Score'] = minmax_01(true_raw, jitter_binary=True)
            df_task['Pred_Score'] = minmax_01(pred_raw, jitter_binary=False)
        elif task == 'PFS':
            true_raw = -pd.to_numeric(pred_df['PFS_time'], errors='coerce').values
            pred_raw = pd.to_numeric(pred_df['PFS_risk'], errors='coerce').values
            df_task['True_Score'] = minmax_01(true_raw, jitter_binary=False)
            df_task['Pred_Score'] = minmax_01(pred_raw, jitter_binary=False)
        elif task == 'OS':
            true_raw = -pd.to_numeric(pred_df['OS_time'], errors='coerce').values
            pred_raw = pd.to_numeric(pred_df['OS_risk'], errors='coerce').values
            df_task['True_Score'] = minmax_01(true_raw, jitter_binary=False)
            df_task['Pred_Score'] = minmax_01(pred_raw, jitter_binary=False)
        shared_features = safe_get_existing_features(global_features, shap_signed_dict[task].columns)
        if len(shared_features) > 0:
            df_task['shared_shap_sum'] = shap_signed_dict[task][shared_features].sum(axis=1)
        else:
            df_task['shared_shap_sum'] = 0.0
        specific_feat = specific_feature_map.get(task)
        if specific_feat is not None and specific_feat in shap_signed_dict[task].columns:
            df_task['specific_shap_sum'] = shap_signed_dict[task][specific_feat]
        else:
            df_task['specific_shap_sum'] = 0.0
        for f_info in plot_features:
            feat = f_info['name']
            if feat in X_visual.columns:
                df_task[feat] = X_visual[feat].values
        df_task = df_task.sort_values(by='Pred_Score')
        result_list.append(df_task)
    return {'modal_name': modal_name, 'X_visual': X_visual, 'pred_df': pred_df, 'metadata': metadata, 'global_features': global_features, 'specific_feature_map': specific_feature_map, 'plot_features': plot_features, 'result_list': result_list}

def render_stage1_nm_circos(circos_data, output_prefix):
    modal_name = circos_data['modal_name']
    metadata = circos_data['metadata']
    plot_features = circos_data['plot_features']
    result_list = circos_data['result_list']
    sector_width = 10
    r_pad_ratio = 0.05
    sectors = {task_names[i]: sector_width for i in range(3)}
    circos = Circos(sectors, space=1.5)
    center_name = display_modal_name(modal_name)
    circos.text(center_name, size=FONT_CENTER, fontweight='bold', color='#2c3e50')
    for index, sector in enumerate(circos.sectors):
        title_text = sector.name
        sector.text(title_text, r=101, size=FONT_SECTOR, fontweight='bold', color='#2c3e50')
        df_plot = result_list[index]
        num_patients = df_plot.shape[0]
        x = np.linspace(sector.start + r_pad_ratio, sector.end - r_pad_ratio, num_patients)
        out_pos = 100
        inner_pos = out_pos - CIRCOS_SIZE[0]
        track0 = sector.add_track((inner_pos, out_pos), r_pad_ratio=r_pad_ratio)
        track0.axis(fc='none', ec='black', lw=0.5)
        hm_raw = df_plot[['specific_shap_sum', 'shared_shap_sum']].abs().values.T
        hm_data = (hm_raw - hm_raw.min(axis=1, keepdims=True)) / (hm_raw.max(axis=1, keepdims=True) - hm_raw.min(axis=1, keepdims=True) + 1e-08)
        hm_data = np.power(hm_data, 0.4)
        track0.heatmap(hm_data, cmap='Blues')
        if index == 0:
            track0.yticks([inner_pos + 1.5, inner_pos + 4.5], ['Specific SHAP', 'Shared SHAP'], side='left', label_size=FONT_LABEL_L2)
        out_pos = inner_pos
        inner_pos = out_pos - CIRCOS_SIZE[1]
        track1 = sector.add_track((inner_pos, out_pos), r_pad_ratio=r_pad_ratio)
        track1.axis(fc='none', ec='black', lw=0.8)
        track1.grid(lw=0.3, ls=':')
        bar_data = df_plot[['specific_shap_sum', 'shared_shap_sum']].values
        colors = ['#A74AC7', '#95a5a6']
        pos_sums = np.sum(np.where(bar_data > 0, bar_data, 0), axis=1)
        neg_sums = np.sum(np.where(bar_data < 0, bar_data, 0), axis=1)
        max_abs_stack = np.max([np.max(pos_sums), np.abs(np.min(neg_sums))])
        if max_abs_stack == 0 or np.isnan(max_abs_stack):
            max_abs_stack = 1e-06
        vmin = -max_abs_stack * 1.05
        vmax = max_abs_stack * 1.05
        for i in range(2):
            bottoms_pos = [sum((bar_data[g][j] for j in range(i) if bar_data[g][j] > 0)) for g in range(num_patients)]
            bottoms_neg = [sum((bar_data[g][j] for j in range(i) if bar_data[g][j] < 0)) for g in range(num_patients)]
            values = [bar_data[g][i] for g in range(num_patients)]
            bottoms = [bottoms_pos[g] if values[g] > 0 else bottoms_neg[g] for g in range(num_patients)]
            track1.bar(x, values, color=colors[i], width=(sector.end - sector.start) / num_patients, bottom=bottoms, vmin=vmin, vmax=vmax)
        if index == 0:
            track1.yticks([out_pos - CIRCOS_SIZE[1] / 2], ['Aggregated\nSHAP'], side='left', label_size=FONT_LABEL_L1, line_kws=dict(lw=0))
        out_pos = inner_pos
        inner_pos = out_pos - CIRCOS_SIZE[2]
        track2 = sector.add_track((inner_pos, out_pos), r_pad_ratio=r_pad_ratio)
        track2.axis(fc='none', ec='black', lw=0.8)
        hm_meta = metadata.loc[df_plot.index].values.T
        track2.heatmap(hm_meta, cmap='binary')
        if index == 0:
            ytick_positions = np.linspace(inner_pos + 1, out_pos - 1, metadata.shape[1])
            track2.yticks(ytick_positions, metadata.columns[::-1], side='left', label_size=FONT_LABEL_L2)
        out_pos = inner_pos
        inner_pos = out_pos - CIRCOS_SIZE[3]
        track3 = sector.add_track((inner_pos, out_pos), r_pad_ratio=r_pad_ratio)
        track3.axis(fc='none', ec='black', lw=0.8)
        track3.scatter(x, df_plot['True_Score'].values, vmin=-0.1, vmax=1.1, color='#00539C', s=5, alpha=0.8)
        track3.scatter(x, df_plot['Pred_Score'].values, vmin=-0.1, vmax=1.1, color='#EEA47F', s=5, alpha=0.9)
        if index == 0:
            track3.yticks([out_pos - CIRCOS_SIZE[3] / 2], ['True vs\nPred'], side='left', label_size=FONT_LABEL_L1, line_kws=dict(lw=0))
        out_pos = inner_pos
        colors_lines = sns.color_palette('tab10', len(plot_features)).as_hex()
        start = out_pos
        for i, f_info in enumerate(plot_features):
            feat = f_info['name']
            f_type = f_info['type']
            if f_type == 'global' or f_type == title_text:
                if feat not in df_plot.columns:
                    start -= TRACK4_STEP
                    continue
                track4 = sector.add_track((start - TRACK4_STEP + 0.5, start), r_pad_ratio=r_pad_ratio)
                y_feat = df_plot[feat].values.astype(float)
                y_min = np.nanmin(y_feat)
                y_max = np.nanmax(y_feat)
                if y_min == y_max:
                    y_max += 1e-05
                y_vmax = y_max + (y_max - y_min) * 0.15
                track4.line(x, [y_min] * len(x), vmin=y_min, vmax=y_vmax, color='grey', lw=0.7, ls='--', alpha=0.6)
                track4.line(x, y_feat, vmin=y_min, vmax=y_vmax, color=colors_lines[i], lw=1.3)
                if f_type == 'global' and index == 0 or f_type == title_text:
                    track4.yticks([start - TRACK4_STEP / 2], [clean_feature_name(feat, max_len=15)], side='left', label_size=FONT_LABEL_L2, line_kws=dict(lw=0))
            start -= TRACK4_STEP
    fig = circos.plotfig()
    fig.set_size_inches(CANVAS_SIZE)
    out_pdf = output_prefix + '.pdf'
    out_png = output_prefix + '.png'
    circos.savefig(out_pdf, dpi=600)
    circos.savefig(out_png, dpi=600)
    plt.close(fig)
    
    
for modal_name in modalities:
    
    
    
    circos_data = prepare_stage1_circos_data(modal_name)
    output_prefix = os.path.join(circos_plot_dir, f"{get_value('science.paths.directory_007', 'directory_007')}{modal_name}_Response_PFS_OS")
    render_stage1_nm_circos(circos_data=circos_data, output_prefix=output_prefix)



In [ ]:
import os
import re
import random
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
from sklearn.model_selection import RepeatedStratifiedKFold, train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
RANDOM_STATE = 42
N_SPLITS = 5
N_REPEATS = 5
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

LAMBDA_RESPONSE = 0.4
LAMBDA_PFS = 0.8
LAMBDA_OS = 1.0
PROJECT_DIR = str(project_path(get_value('science.paths.directory_008', 'directory_008')))
INPUT_DIR = os.path.join(PROJECT_DIR, get_value('science.paths.directory_009', 'directory_009'))
LABEL_FILE_NAME = get_value('science.tables.outcomes_001', 'outcomes_001.csv')
label_path = os.path.join(INPUT_DIR, LABEL_FILE_NAME)
modal_files = {'Metagenomics': os.path.join(INPUT_DIR, get_value('science.tables.features_001', 'features_001.csv.gz')), 'Radiomics': os.path.join(INPUT_DIR, get_value('science.tables.features_002', 'features_002.csv.gz')), 'Blood': os.path.join(INPUT_DIR, get_value('science.tables.features_003', 'features_003.csv.gz')), 'Urine': os.path.join(INPUT_DIR, get_value('science.tables.features_004', 'features_004.csv.gz'))}
stage1_dir = os.path.join(PROJECT_DIR, get_value('science.paths.directory_010', 'directory_010'), get_value('science.paths.directory_003', 'directory_003'))
golden_feature_path = os.path.join(stage1_dir, get_value('science.tables.result_005', 'result_005.csv'))
top_feature_path = os.path.join(stage1_dir, get_value('science.tables.result_004', 'result_004.csv'))
out_dir = os.path.join(PROJECT_DIR, get_value('science.paths.directory_010', 'directory_010'), get_value('science.paths.directory_011', 'directory_011'))
os.makedirs(out_dir, exist_ok=True)
USE_EARLY_STOPPING = True
MAX_EPOCHS = 500
MIN_EPOCHS = 50
PATIENCE = 40
MIN_DELTA = 0.0001
INNER_VAL_SIZE = 0.2
FIXED_EPOCHS = 250

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
set_seed(RANDOM_STATE)
required_paths = [label_path, golden_feature_path, top_feature_path] + list(modal_files.values())
missing_paths = [p for p in required_paths if not os.path.exists(p)]
if missing_paths:
    print('\nThe following required files were not found:')
    for p in missing_paths:
        print(' -', p)
    raise FileNotFoundError('Please check the Stage 1 output directory and the input filenames.')




for modal_name, path in modal_files.items():
    pass




In [ ]:
def normalize_sample_id(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    if re.fullmatch('-?\\d+\\.0', s):
        s = s[:-2]
    return s.lower()

def find_column(df, candidates, required=True):
    lower_map = {str(c).strip().lower(): c for c in df.columns}
    for cand in candidates:
        key = str(cand).strip().lower()
        if key in lower_map:
            return lower_map[key]
    if required:
        raise ValueError(f'Cannot find any of columns: {candidates}')
    return None

def to_binary_response(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x, str):
        s = x.strip().lower()
        if s in ['1', 'yes', 'y', 'responder', 'response', 'r', 'cr', 'pr', 'orr']:
            return 1
        if s in ['0', 'no', 'n', 'nonresponder', 'non-responder', 'nr', 'sd', 'pd']:
            return 0
    return int(float(x))

def read_table(path):
    path_lower = path.lower()
    if path_lower.endswith(('.xlsx', '.xls')):
        return pd.read_excel(path)
    return pd.read_csv(path, encoding='utf-8-sig')

def read_labels(path):
    df = read_table(path)
    sample_col = find_column(df, ['Sample ID', 'sample_id', 'sample', 'Sample', 'ID'])
    response_col = find_column(df, ['Response', 'Response', 'response', 'ORR'])
    pfs_time_col = find_column(df, ['PFS_time', 'pfs_time', 'PFS', 'PFS_month', 'PFS_months'])
    pfs_status_col = find_column(df, ['PFS_status', 'pfs_status', 'PFS_event', 'pfs_event'])
    os_time_col = find_column(df, ['OS_time', 'os_time', 'OS', 'OS_month', 'OS_months'])
    os_status_col = find_column(df, ['OS_status', 'os_status', 'OS_event', 'os_event'])
    out = pd.DataFrame({'sample': df[sample_col].map(normalize_sample_id), 'Response': df[response_col].map(to_binary_response), 'PFS_time': pd.to_numeric(df[pfs_time_col], errors='coerce'), 'PFS_status': pd.to_numeric(df[pfs_status_col], errors='coerce'), 'OS_time': pd.to_numeric(df[os_time_col], errors='coerce'), 'OS_status': pd.to_numeric(df[os_status_col], errors='coerce')})
    out = out.dropna(subset=['sample'])
    out = out.drop_duplicates(subset='sample', keep='first')
    out = out.set_index('sample')
    required = ['Response', 'PFS_time', 'PFS_status', 'OS_time', 'OS_status']
    if out[required].isna().any().any():
        print('\nMissing values in required outcomes:')
        print(out[required].isna().sum())
        raise ValueError('The label table contains missing required outcomes.')
    out['Response'] = out['Response'].astype(int)
    out['PFS_status'] = out['PFS_status'].astype(int)
    out['OS_status'] = out['OS_status'].astype(int)
    n_pfs_after_os = int((out['PFS_time'] > out['OS_time'] + 1e-08).sum())
    n_os_event_pfs_censored = int(((out['OS_status'] == 1) & (out['PFS_status'] == 0)).sum())
    
    print('Response positive:', int(out['Response'].sum()), '/', out.shape[0])
    print('PFS events:', int(out['PFS_status'].sum()), '/', out.shape[0])
    print('OS events:', int(out['OS_status'].sum()), '/', out.shape[0])
    print('\nOutcome logic audit:')
    print('PFS_time > OS_time:', n_pfs_after_os)
    print('OS event but PFS censored:', n_os_event_pfs_censored)
    if n_pfs_after_os > 0 or n_os_event_pfs_censored > 0:
        raise ValueError('Outcome logic audit failed.')
    return out

def read_matrix(path):
    df = read_table(path)
    df.columns = [str(c).strip() for c in df.columns]
    sample_col = find_column(df, ['sample', 'sample_id', 'Sample ID', 'Sample', 'ID'], required=False)
    if sample_col is None:
        sample_col = df.columns[0]
    df = df.rename(columns={sample_col: 'sample'})
    df['sample'] = df['sample'].map(normalize_sample_id)
    df = df.dropna(subset=['sample'])
    df = df.drop_duplicates(subset='sample', keep='first')
    df = df.set_index('sample')
    df = df.apply(pd.to_numeric, errors='coerce')
    return df
labels = read_labels(label_path)
golden_features = pd.read_csv(golden_feature_path, encoding='utf-8-sig')
top_features = pd.read_csv(top_feature_path, encoding='utf-8-sig')
golden_features['modal'] = golden_features['modal'].astype(str).str.strip()
golden_features['feature'] = golden_features['feature'].astype(str).str.strip()
print('\nGolden feature table:')
print(golden_features.head())
print('\nGolden feature counts:')
print(golden_features.groupby('modal')['feature'].nunique())


In [ ]:
def apply_basic_transform_before_selection(X, modal_name):
    return X.copy()

def build_reduced_matrix_for_modal(modal_name, path, golden_features):
    X = read_matrix(path)
    X = apply_basic_transform_before_selection(X, modal_name)
    selected = golden_features.loc[golden_features['modal'] == modal_name, 'feature'].dropna().astype(str).unique().tolist()
    available = [f for f in selected if f in X.columns]
    missing = [f for f in selected if f not in X.columns]
    if len(selected) == 0:
        raise ValueError(f'No Stage 1 golden features were found for {modal_name}.')
    if len(missing) > 0:
        print(f'[{modal_name}] Warning: {len(missing)} selected features were not found.')
        print('First missing features:', missing[:10])
    if len(available) == 0:
        raise ValueError(f'No selected features are available for {modal_name}.')
    X_sel = X[available].copy()
    print(f'[{modal_name}] selected: {len(selected)} | available: {len(available)}')
    return X_sel
single_modal_matrices = {}
for modal_name, path in modal_files.items():
    X_sel = build_reduced_matrix_for_modal(modal_name=modal_name, path=path, golden_features=golden_features)
    missing_samples = labels.index.difference(X_sel.index).tolist()
    if missing_samples:
        print(f'[{modal_name}] Missing labeled samples:', missing_samples[:20])
        raise ValueError(f'{modal_name} does not contain all labeled samples.')
    X_sel = X_sel.loc[labels.index].copy()
    single_modal_matrices[modal_name] = X_sel
    X_sel.to_csv(os.path.join(out_dir, f"{modal_name}{get_value('science.tables.attribution_003', 'attribution_003.csv')}"), encoding='utf-8-sig')

def build_multimodal_matrix(single_modal_matrices):
    mats = []
    for modal_name in modal_files.keys():
        X = single_modal_matrices[modal_name].loc[labels.index].copy()
        X.columns = [f'{modal_name}__{c}' for c in X.columns]
        mats.append(X)
    return pd.concat(mats, axis=1)
X_multimodal = build_multimodal_matrix(single_modal_matrices)
X_multimodal.to_csv(os.path.join(out_dir, get_value('science.tables.attribution_004', 'attribution_004.csv')), encoding='utf-8-sig')
print('\nReduced matrix sizes:')
for modal_name, X in single_modal_matrices.items():
    print(modal_name, X.shape)
print('Multimodal:', X_multimodal.shape)
if X_multimodal.index.tolist() != labels.index.tolist():
    raise ValueError('Multimodal matrix sample order does not match labels.')
print('\nStage 2 reduced matrices are ready.')


In [ ]:
class ReducedMultiTaskNet(nn.Module):

    def __init__(self, input_dim, hidden1=64, hidden2=16, dropout=0.4):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(input_dim, hidden1), nn.BatchNorm1d(hidden1), nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden1, hidden2), nn.BatchNorm1d(hidden2), nn.ReLU(), nn.Dropout(dropout))
        self.response_head = nn.Linear(hidden2, 1)
        self.pfs_head = nn.Linear(hidden2, 1)
        self.os_head = nn.Linear(hidden2, 1)

    def forward(self, x):
        z = self.encoder(x)
        return {'response_logit': self.response_head(z).view(-1), 'pfs_risk': self.pfs_head(z).view(-1), 'os_risk': self.os_head(z).view(-1)}

def cox_ph_loss(risk, time, event):
    risk = risk.view(-1)
    time = time.view(-1)
    event = event.view(-1)
    valid = ~torch.isnan(time) & ~torch.isnan(event)
    risk = risk[valid]
    time = time[valid]
    event = event[valid]
    if len(risk) == 0 or event.sum() < 1:
        return torch.tensor(0.0, device=risk.device)
    order = torch.argsort(time, descending=True)
    risk = risk[order]
    event = event[order]
    log_cumsum_exp = torch.logcumsumexp(risk, dim=0)
    loss = -torch.sum((risk - log_cumsum_exp) * event) / (torch.sum(event) + 1e-08)
    return loss

def multitask_loss(outputs, y_response, pfs_time, pfs_status, os_time, os_status, response_pos_weight=None, lambda_response=LAMBDA_RESPONSE, lambda_pfs=LAMBDA_PFS, lambda_os=LAMBDA_OS):
    if response_pos_weight is not None:
        bce = nn.BCEWithLogitsLoss(pos_weight=response_pos_weight)
    else:
        bce = nn.BCEWithLogitsLoss()
    loss_response = bce(outputs['response_logit'], y_response)
    loss_pfs = cox_ph_loss(outputs['pfs_risk'], pfs_time, pfs_status)
    loss_os = cox_ph_loss(outputs['os_risk'], os_time, os_status)
    loss = lambda_response * loss_response + lambda_pfs * loss_pfs + lambda_os * loss_os
    return (loss, {'loss_response': loss_response.item(), 'loss_pfs': loss_pfs.item(), 'loss_os': loss_os.item(), 'weighted_total': loss.item()})

def safe_auc(y_true, y_score):
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)
    mask = ~np.isnan(y_true) & ~np.isnan(y_score)
    y_true = y_true[mask]
    y_score = y_score[mask]
    if len(np.unique(y_true)) < 2:
        return np.nan
    return roc_auc_score(y_true, y_score)

def safe_auprc(y_true, y_score):
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)
    mask = ~np.isnan(y_true) & ~np.isnan(y_score)
    y_true = y_true[mask]
    y_score = y_score[mask]
    if len(np.unique(y_true)) < 2:
        return np.nan
    return average_precision_score(y_true, y_score)

def concordance_index(time, event, risk):
    time = np.asarray(time, dtype=float)
    event = np.asarray(event, dtype=float)
    risk = np.asarray(risk, dtype=float)
    mask = ~np.isnan(time) & ~np.isnan(event) & ~np.isnan(risk)
    time = time[mask]
    event = event[mask]
    risk = risk[mask]
    n = len(time)
    if n < 2:
        return np.nan
    permissible = 0
    concordant = 0
    ties = 0
    for i in range(n):
        for j in range(n):
            if time[i] < time[j] and event[i] == 1:
                permissible += 1
                if risk[i] > risk[j]:
                    concordant += 1
                elif risk[i] == risk[j]:
                    ties += 1
    if permissible == 0:
        return np.nan
    return (concordant + 0.5 * ties) / permissible


In [ ]:
def fit_preprocessor_for_model(X_fit, model_name):
    pass
    imputer = SimpleImputer(strategy='median')
    X_fit_processed = imputer.fit_transform(X_fit)
    scaler = None
    return (imputer, scaler, X_fit_processed)

def transform_with_preprocessor(X, imputer, scaler):
    return imputer.transform(X)

def make_label_tensors(y_df):
    y_response = torch.tensor(y_df['Response'].values, dtype=torch.float32).to(DEVICE)
    pfs_time = torch.tensor(y_df['PFS_time'].values, dtype=torch.float32).to(DEVICE)
    pfs_status = torch.tensor(y_df['PFS_status'].values, dtype=torch.float32).to(DEVICE)
    os_time = torch.tensor(y_df['OS_time'].values, dtype=torch.float32).to(DEVICE)
    os_status = torch.tensor(y_df['OS_status'].values, dtype=torch.float32).to(DEVICE)
    return (y_response, pfs_time, pfs_status, os_time, os_status)

def get_response_pos_weight(y_response_tensor):
    n_pos = y_response_tensor.sum().item()
    n_neg = len(y_response_tensor) - n_pos
    if n_pos > 0:
        return torch.tensor([n_neg / n_pos], dtype=torch.float32).to(DEVICE)
    else:
        return None

def get_model_hyperparams(model_name, input_dim):
    pass
    if model_name == 'Multimodal':
        return {'hidden1': min(64, max(16, input_dim // 2)), 'hidden2': 16, 'dropout': 0.45}
    elif model_name == 'Radiomics':
        return {'hidden1': min(64, max(16, input_dim // 2)), 'hidden2': 16, 'dropout': 0.4}
    elif model_name in ['Blood', 'Urine']:
        return {'hidden1': min(64, max(16, input_dim // 2)), 'hidden2': 16, 'dropout': 0.45}
    else:
        return {'hidden1': min(64, max(16, input_dim // 2)), 'hidden2': 16, 'dropout': 0.45}

def determine_best_epoch(X_train, y_train, model_name, fold_id, input_dim):
    pass
    idx_all = np.arange(X_train.shape[0])
    stratify_y = y_train['Response'].astype(int).values
    try:
        inner_train_idx, inner_val_idx = train_test_split(idx_all, test_size=INNER_VAL_SIZE, random_state=RANDOM_STATE + fold_id, stratify=stratify_y)
    except ValueError:
        inner_train_idx, inner_val_idx = train_test_split(idx_all, test_size=INNER_VAL_SIZE, random_state=RANDOM_STATE + fold_id, stratify=None)
    X_inner_train = X_train.iloc[inner_train_idx]
    X_inner_val = X_train.iloc[inner_val_idx]
    y_inner_train = y_train.iloc[inner_train_idx]
    y_inner_val = y_train.iloc[inner_val_idx]
    imputer, scaler, X_inner_train_scaled = fit_preprocessor_for_model(X_inner_train, model_name)
    X_inner_val_scaled = transform_with_preprocessor(X_inner_val, imputer, scaler)
    X_inner_train_tensor = torch.tensor(X_inner_train_scaled, dtype=torch.float32).to(DEVICE)
    X_inner_val_tensor = torch.tensor(X_inner_val_scaled, dtype=torch.float32).to(DEVICE)
    y_response_inner_train, pfs_time_inner_train, pfs_status_inner_train, os_time_inner_train, os_status_inner_train = make_label_tensors(y_inner_train)
    y_response_inner_val, pfs_time_inner_val, pfs_status_inner_val, os_time_inner_val, os_status_inner_val = make_label_tensors(y_inner_val)
    pos_weight_inner = get_response_pos_weight(y_response_inner_train)
    hp = get_model_hyperparams(model_name, input_dim)
    model = ReducedMultiTaskNet(input_dim=input_dim, hidden1=hp['hidden1'], hidden2=hp['hidden2'], dropout=hp['dropout']).to(DEVICE)
    optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.0001)
    best_val_loss = np.inf
    best_epoch = 1
    patience_counter = 0
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        optimizer.zero_grad()
        outputs = model(X_inner_train_tensor)
        loss, _ = multitask_loss(outputs, y_response_inner_train, pfs_time_inner_train, pfs_status_inner_train, os_time_inner_train, os_status_inner_train, response_pos_weight=pos_weight_inner, lambda_response=LAMBDA_RESPONSE, lambda_pfs=LAMBDA_PFS, lambda_os=LAMBDA_OS)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        model.eval()
        with torch.no_grad():
            val_outputs = model(X_inner_val_tensor)
            val_loss, _ = multitask_loss(val_outputs, y_response_inner_val, pfs_time_inner_val, pfs_status_inner_val, os_time_inner_val, os_status_inner_val, response_pos_weight=pos_weight_inner, lambda_response=LAMBDA_RESPONSE, lambda_pfs=LAMBDA_PFS, lambda_os=LAMBDA_OS)
        val_loss_value = val_loss.item()
        if val_loss_value < best_val_loss - MIN_DELTA:
            best_val_loss = val_loss_value
            best_epoch = epoch
            patience_counter = 0
        else:
            patience_counter += 1
        if epoch >= MIN_EPOCHS and patience_counter >= PATIENCE:
            break
    
    return best_epoch


In [ ]:
def evaluate_reduced_model(model_name, X, labels):
    
    
    
    common = X.index.intersection(labels.index)
    X = X.loc[common].copy()
    y_df = labels.loc[common].copy()
    required_cols = ['Response', 'PFS_time', 'PFS_status', 'OS_time', 'OS_status']
    valid = y_df[required_cols].notna().all(axis=1)
    X = X.loc[valid]
    y_df = y_df.loc[valid]
    print(f'[{model_name}] samples:', X.shape[0])
    print(f'[{model_name}] features:', X.shape[1])
    y_response = y_df['Response'].astype(int).values
    cv = RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=RANDOM_STATE)
    pred_records = []
    fold_records = []
    for fold_id, (train_idx, test_idx) in enumerate(cv.split(X, y_response), start=1):
        print(f'\n[{model_name}] Fold {fold_id}')
        set_seed(RANDOM_STATE + fold_id)
        X_train = X.iloc[train_idx]
        X_test = X.iloc[test_idx]
        y_train = y_df.iloc[train_idx]
        y_test = y_df.iloc[test_idx]
        if USE_EARLY_STOPPING:
            best_epoch = determine_best_epoch(X_train=X_train, y_train=y_train, model_name=model_name, fold_id=fold_id, input_dim=X.shape[1])
        else:
            best_epoch = FIXED_EPOCHS
        imputer, scaler, X_train_scaled = fit_preprocessor_for_model(X_train, model_name)
        X_test_scaled = transform_with_preprocessor(X_test, imputer, scaler)
        X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32).to(DEVICE)
        X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32).to(DEVICE)
        y_response_train, pfs_time_train, pfs_status_train, os_time_train, os_status_train = make_label_tensors(y_train)
        pos_weight = get_response_pos_weight(y_response_train)
        hp = get_model_hyperparams(model_name, X.shape[1])
        model = ReducedMultiTaskNet(input_dim=X.shape[1], hidden1=hp['hidden1'], hidden2=hp['hidden2'], dropout=hp['dropout']).to(DEVICE)
        optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.0001)
        model.train()
        for epoch in range(1, best_epoch + 1):
            optimizer.zero_grad()
            outputs = model(X_train_tensor)
            loss, loss_parts = multitask_loss(outputs, y_response_train, pfs_time_train, pfs_status_train, os_time_train, os_status_train, response_pos_weight=pos_weight, lambda_response=LAMBDA_RESPONSE, lambda_pfs=LAMBDA_PFS, lambda_os=LAMBDA_OS)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()
        model.eval()
        with torch.no_grad():
            test_out = model(X_test_tensor)
            pred_response = torch.sigmoid(test_out['response_logit']).detach().cpu().numpy()
            pred_pfs_risk = test_out['pfs_risk'].detach().cpu().numpy()
            pred_os_risk = test_out['os_risk'].detach().cpu().numpy()
        fold_pred = pd.DataFrame({'sample': X_test.index, 'model': model_name, 'fold_id': fold_id, 'Response_true': y_test['Response'].values, 'Response_pred': pred_response, 'PFS_time': y_test['PFS_time'].values, 'PFS_status': y_test['PFS_status'].values, 'PFS_risk': pred_pfs_risk, 'OS_time': y_test['OS_time'].values, 'OS_status': y_test['OS_status'].values, 'OS_risk': pred_os_risk})
        pred_records.append(fold_pred)
        fold_auc = safe_auc(fold_pred['Response_true'], fold_pred['Response_pred'])
        fold_auprc = safe_auprc(fold_pred['Response_true'], fold_pred['Response_pred'])
        fold_pfs_c = concordance_index(fold_pred['PFS_time'], fold_pred['PFS_status'], fold_pred['PFS_risk'])
        fold_os_c = concordance_index(fold_pred['OS_time'], fold_pred['OS_status'], fold_pred['OS_risk'])
        fold_records.append({'model': model_name, 'fold_id': fold_id, 'n_features': X.shape[1], 'Response_AUC': fold_auc, 'Response_AUPRC': fold_auprc, 'PFS_Cindex': fold_pfs_c, 'OS_Cindex': fold_os_c, 'best_epoch': best_epoch})
        print(f'[{model_name}] Fold {fold_id} | AUC={fold_auc:.3f}, PFS C={fold_pfs_c:.3f}, OS C={fold_os_c:.3f}')
    pred_df = pd.concat(pred_records, axis=0)
    fold_df = pd.DataFrame(fold_records)
    summary = {'model': model_name, 'n_samples': X.shape[0], 'n_features': X.shape[1], 'Response_AUC_mean': fold_df['Response_AUC'].mean(), 'Response_AUC_sd': fold_df['Response_AUC'].std(), 'Response_AUPRC_mean': fold_df['Response_AUPRC'].mean(), 'Response_AUPRC_sd': fold_df['Response_AUPRC'].std(), 'PFS_Cindex_mean': fold_df['PFS_Cindex'].mean(), 'PFS_Cindex_sd': fold_df['PFS_Cindex'].std(), 'OS_Cindex_mean': fold_df['OS_Cindex'].mean(), 'OS_Cindex_sd': fold_df['OS_Cindex'].std(), 'best_epoch_median': fold_df['best_epoch'].median()}
    return {'summary': summary, 'fold_performance': fold_df, 'predictions': pred_df}


In [ ]:
model_matrices = {}
for modal_name, X in single_modal_matrices.items():
    model_matrices[modal_name] = X
model_matrices['Multimodal'] = X_multimodal
all_summaries = []
all_fold_perf = []
all_predictions = []
for model_name, X in model_matrices.items():
    res = evaluate_reduced_model(model_name=model_name, X=X, labels=labels)
    all_summaries.append(res['summary'])
    all_fold_perf.append(res['fold_performance'])
    all_predictions.append(res['predictions'])
    model_out_dir = os.path.join(out_dir, model_name)
    os.makedirs(model_out_dir, exist_ok=True)
    res['fold_performance'].to_csv(os.path.join(model_out_dir, f"{model_name}{get_value('science.tables.metrics_001', 'metrics_001.csv')}"), index=False, encoding='utf-8-sig')
    res['predictions'].to_csv(os.path.join(model_out_dir, f"{model_name}{get_value('science.tables.predictions_001', 'predictions_001.csv')}"), index=False, encoding='utf-8-sig')
summary_df = pd.DataFrame(all_summaries)
fold_perf_df = pd.concat(all_fold_perf, axis=0)
predictions_df = pd.concat(all_predictions, axis=0)
summary_df.to_csv(os.path.join(out_dir, get_value('science.tables.metrics_004', 'metrics_004.csv')), index=False, encoding='utf-8-sig')
fold_perf_df.to_csv(os.path.join(out_dir, get_value('science.tables.metrics_005', 'metrics_005.csv')), index=False, encoding='utf-8-sig')
predictions_df.to_csv(os.path.join(out_dir, get_value('science.tables.predictions_004', 'predictions_004.csv')), index=False, encoding='utf-8-sig')
summary_df


In [ ]:
plot_df = summary_df.copy()
model_order = ['Metagenomics', 'Radiomics', 'Blood', 'Urine', 'Multimodal']
plot_df['model'] = pd.Categorical(plot_df['model'], categories=model_order, ordered=True)
plot_df = plot_df.sort_values('model')

def plot_metric_bar(plot_df, metric_mean, metric_sd, ylabel, title, filename):
    fig, ax = plt.subplots(figsize=(7.5, 4.8))
    x = np.arange(plot_df.shape[0])
    ax.bar(x, plot_df[metric_mean], yerr=plot_df[metric_sd], capsize=4, alpha=0.85, edgecolor='black', linewidth=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(plot_df['model'], rotation=30, ha='right')
    ax.set_ylabel(ylabel)
    ax.set_title(title, fontsize=12)
    ax.grid(axis='y', linestyle='--', alpha=0.35)
    ymin = max(0.0, plot_df[metric_mean].min() - 0.15)
    ymax = min(1.0, plot_df[metric_mean].max() + 0.15)
    ax.set_ylim(ymin, ymax)
    for i, row in plot_df.iterrows():
        ax.text(x[list(plot_df.index).index(i)], row[metric_mean] + row[metric_sd] + 0.015, f'{row[metric_mean]:.2f}', ha='center', va='bottom', fontsize=9)
    plt.tight_layout()
    fig_path = os.path.join(out_dir, filename)
    plt.savefig(fig_path, dpi=600, bbox_inches='tight')
    plt.show()
    
plot_metric_bar(plot_df, metric_mean='Response_AUC_mean', metric_sd='Response_AUC_sd', ylabel='Response AUC', title='Response prediction performance', filename=get_value('science.figures.plot_001', 'plot_001.png'))
plot_metric_bar(plot_df, metric_mean='PFS_Cindex_mean', metric_sd='PFS_Cindex_sd', ylabel='PFS C-index', title='PFS prediction performance', filename=get_value('science.figures.plot_002', 'plot_002.png'))
plot_metric_bar(plot_df, metric_mean='OS_Cindex_mean', metric_sd='OS_Cindex_sd', ylabel='OS C-index', title='OS prediction performance', filename=get_value('science.figures.plot_003', 'plot_003.png'))


In [ ]:
import os
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import pandas as pd
abbr = {'Metagenomics': 'M', 'Radiomics': 'R', 'Blood': 'B', 'Urine': 'U'}

def make_short_combo_name(modalities):
    return f'C{len(modalities)}_' + '_'.join([abbr[m] for m in modalities])

def safe_save_csv(df, path, index=False):
    folder = os.path.dirname(path)
    os.makedirs(folder, exist_ok=True)
    df.to_csv(path, index=index, encoding='utf-8-sig')
combo_out_dir = os.path.join(out_dir, get_value('science.paths.directory_012', 'directory_012'))
os.makedirs(combo_out_dir, exist_ok=True)
combo_matrix_dir = os.path.join(combo_out_dir, 'matrices')
os.makedirs(combo_matrix_dir, exist_ok=True)
combo_result_dir = os.path.join(combo_out_dir, 'results')
os.makedirs(combo_result_dir, exist_ok=True)



def build_combination_matrix(single_modal_matrices, modalities):
    pass
    common_samples = None
    for m in modalities:
        samples = set(single_modal_matrices[m].index)
        if common_samples is None:
            common_samples = samples
        else:
            common_samples = common_samples.intersection(samples)
    common_samples = sorted(list(common_samples))
    mats = []
    for m in modalities:
        X = single_modal_matrices[m].loc[common_samples].copy()
        X.columns = [f'{m}__{c}' for c in X.columns]
        mats.append(X)
    X_combo = pd.concat(mats, axis=1)
    return X_combo
base_modalities = ['Metagenomics', 'Radiomics', 'Blood', 'Urine']
combination_specs = []
for k in [2, 3]:
    for combo in itertools.combinations(base_modalities, k):
        combo_name = '+'.join(combo)
        model_name = f'Combo{k}_{combo_name}'
        combination_specs.append({'model_name': model_name, 'modalities': list(combo), 'n_modalities': k})
print('Total combinations:', len(combination_specs))
for spec in combination_specs:
    print(spec['model_name'])


In [ ]:
combo_summaries = []
combo_fold_perfs = []
combo_predictions = []
abbr = {'Metagenomics': 'M', 'Radiomics': 'R', 'Blood': 'B', 'Urine': 'U'}

def make_short_combo_name(modalities):
    return f'C{len(modalities)}_' + '_'.join([abbr[m] for m in modalities])

def safe_save_csv(df, path, index=False):
    folder = os.path.dirname(path)
    os.makedirs(folder, exist_ok=True)
    df.to_csv(path, index=index, encoding='utf-8-sig')
for spec in combination_specs:
    model_name = spec['model_name']
    modalities = spec['modalities']
    safe_model_name = make_short_combo_name(modalities)
    
    
    print('Modalities:', modalities)
    
    X_combo = build_combination_matrix(single_modal_matrices=single_modal_matrices, modalities=modalities)
    print(f'{model_name}: samples={X_combo.shape[0]}, features={X_combo.shape[1]}')
    matrix_path = os.path.join(combo_matrix_dir, f"{safe_model_name}{get_value('science.tables.features_005', 'features_005.csv')}")
    X_combo.to_csv(matrix_path, encoding='utf-8-sig')
    res = evaluate_reduced_model(model_name=model_name, X=X_combo, labels=labels)
    summary = res['summary'].copy()
    summary['n_modalities'] = spec['n_modalities']
    summary['modalities'] = '+'.join(modalities)
    fold_perf = res['fold_performance'].copy()
    fold_perf['n_modalities'] = spec['n_modalities']
    fold_perf['modalities'] = '+'.join(modalities)
    pred_df = res['predictions'].copy()
    pred_df['n_modalities'] = spec['n_modalities']
    pred_df['modalities'] = '+'.join(modalities)
    combo_summaries.append(summary)
    combo_fold_perfs.append(fold_perf)
    combo_predictions.append(pred_df)
    this_model_dir = os.path.join(combo_result_dir, safe_model_name)
    os.makedirs(this_model_dir, exist_ok=True)
    safe_save_csv(pd.DataFrame([summary]), os.path.join(this_model_dir, f"{safe_model_name}{get_value('science.tables.metrics_006', 'metrics_006.csv')}"), index=False)
    safe_save_csv(fold_perf, os.path.join(this_model_dir, f"{safe_model_name}{get_value('science.tables.metrics_001', 'metrics_001.csv')}"), index=False)
    safe_save_csv(pred_df, os.path.join(this_model_dir, f"{safe_model_name}{get_value('science.tables.predictions_001', 'predictions_001.csv')}"), index=False)
combo_summary_df = pd.DataFrame(combo_summaries)
combo_fold_perf_df = pd.concat(combo_fold_perfs, axis=0)
combo_predictions_df = pd.concat(combo_predictions, axis=0)
combo_summary_path = os.path.join(combo_out_dir, get_value('science.tables.metrics_007', 'metrics_007.csv'))
combo_fold_path = os.path.join(combo_out_dir, get_value('science.tables.metrics_008', 'metrics_008.csv'))
combo_pred_path = os.path.join(combo_out_dir, get_value('science.tables.predictions_005', 'predictions_005.csv'))
combo_summary_df.to_csv(combo_summary_path, index=False, encoding='utf-8-sig')
combo_fold_perf_df.to_csv(combo_fold_path, index=False, encoding='utf-8-sig')
combo_predictions_df.to_csv(combo_pred_path, index=False, encoding='utf-8-sig')


combo_summary_df


In [ ]:
single_and_full_df = summary_df.copy()

def infer_modalities_from_model(model):
    if model == 'Multimodal':
        return ('Metagenomics+Radiomics+Blood+Urine', 4)
    elif model in ['Metagenomics', 'Radiomics', 'Blood', 'Urine']:
        return (model, 1)
    else:
        return (model, np.nan)
modalities_info = single_and_full_df['model'].apply(infer_modalities_from_model)
single_and_full_df['modalities'] = [x[0] for x in modalities_info]
single_and_full_df['n_modalities'] = [x[1] for x in modalities_info]
all_model_summary_df = pd.concat([single_and_full_df, combo_summary_df], axis=0, ignore_index=True)
front_cols = ['model', 'modalities', 'n_modalities', 'n_samples', 'n_features']
other_cols = [c for c in all_model_summary_df.columns if c not in front_cols]
all_model_summary_df = all_model_summary_df[front_cols + other_cols]
all_model_summary_path = os.path.join(combo_out_dir, get_value('science.tables.metrics_009', 'metrics_009.csv'))
all_model_summary_df.to_csv(all_model_summary_path, index=False, encoding='utf-8-sig')

all_model_summary_df.sort_values(['n_modalities', 'model'])


In [ ]:
plot_all_df = all_model_summary_df.copy()
plot_all_df = plot_all_df.sort_values(['n_modalities', 'model']).reset_index(drop=True)
plot_all_df['display_name'] = plot_all_df['modalities']
plot_all_df['display_name'] = plot_all_df['display_name'].str.replace('+', '\n+\n', regex=False)

def plot_all_models_metric(df, metric_mean, metric_sd, ylabel, title, filename):
    fig, ax = plt.subplots(figsize=(12, 5.5))
    x = np.arange(df.shape[0])
    ax.bar(x, df[metric_mean], yerr=df[metric_sd], capsize=3, alpha=0.85, edgecolor='black', linewidth=0.7)
    ax.set_xticks(x)
    ax.set_xticklabels(df['display_name'], rotation=45, ha='right', fontsize=8)
    ax.set_ylabel(ylabel)
    ax.set_title(title, fontsize=12)
    ax.grid(axis='y', linestyle='--', alpha=0.35)
    ymin = max(0.0, df[metric_mean].min() - 0.12)
    ymax = min(1.0, df[metric_mean].max() + 0.12)
    ax.set_ylim(ymin, ymax)
    n_mods = df['n_modalities'].values
    for i in range(1, len(n_mods)):
        if n_mods[i] != n_mods[i - 1]:
            ax.axvline(i - 0.5, color='black', linewidth=1.0, linestyle='--')
    for i, row in df.iterrows():
        ax.text(i, row[metric_mean] + row[metric_sd] + 0.01, f'{row[metric_mean]:.2f}', ha='center', va='bottom', fontsize=7)
    plt.tight_layout()
    fig_path = os.path.join(combo_out_dir, filename)
    plt.savefig(fig_path, dpi=600, bbox_inches='tight')
    plt.show()
    
plot_all_models_metric(plot_all_df, metric_mean='Response_AUC_mean', metric_sd='Response_AUC_sd', ylabel='Response AUC', title='Response prediction across single-, two-, three-, and four-modality models', filename=get_value('science.figures.plot_004', 'plot_004.png'))
plot_all_models_metric(plot_all_df, metric_mean='PFS_Cindex_mean', metric_sd='PFS_Cindex_sd', ylabel='PFS C-index', title='PFS prediction across single-, two-, three-, and four-modality models', filename=get_value('science.figures.plot_005', 'plot_005.png'))
plot_all_models_metric(plot_all_df, metric_mean='OS_Cindex_mean', metric_sd='OS_Cindex_sd', ylabel='OS C-index', title='OS prediction across single-, two-, three-, and four-modality models', filename=get_value('science.figures.plot_006', 'plot_006.png'))


In [ ]:
rank_response = all_model_summary_df.sort_values('Response_AUC_mean', ascending=False)[['model', 'modalities', 'n_modalities', 'n_features', 'Response_AUC_mean', 'Response_AUC_sd']]
rank_pfs = all_model_summary_df.sort_values('PFS_Cindex_mean', ascending=False)[['model', 'modalities', 'n_modalities', 'n_features', 'PFS_Cindex_mean', 'PFS_Cindex_sd']]
rank_os = all_model_summary_df.sort_values('OS_Cindex_mean', ascending=False)[['model', 'modalities', 'n_modalities', 'n_features', 'OS_Cindex_mean', 'OS_Cindex_sd']]
rank_response.to_csv(os.path.join(combo_out_dir, get_value('science.tables.result_007', 'result_007.csv')), index=False, encoding='utf-8-sig')
rank_pfs.to_csv(os.path.join(combo_out_dir, get_value('science.tables.result_008', 'result_008.csv')), index=False, encoding='utf-8-sig')
rank_os.to_csv(os.path.join(combo_out_dir, get_value('science.tables.result_009', 'result_009.csv')), index=False, encoding='utf-8-sig')
print('Top Response models:')

print('Top PFS models:')

print('Top OS models:')



In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu

def infer_modal_count_and_label(model):
    if model == 'Multimodal':
        return (4, 'Metagenomics+Radiomics+Blood+Urine')
    elif model in ['Metagenomics', 'Radiomics', 'Blood', 'Urine']:
        return (1, model)
    else:
        return (np.nan, model)
single_full_fold_df = fold_perf_df.copy()
if 'n_modalities' not in single_full_fold_df.columns:
    info = single_full_fold_df['model'].apply(infer_modal_count_and_label)
    single_full_fold_df['n_modalities'] = [x[0] for x in info]
    single_full_fold_df['modalities'] = [x[1] for x in info]
combo_fold_df = combo_fold_perf_df.copy()
all_fold_perf_df = pd.concat([single_full_fold_df, combo_fold_df], axis=0, ignore_index=True)
all_fold_perf_df = all_fold_perf_df[all_fold_perf_df['n_modalities'].isin([1, 2, 3, 4])].copy()
all_fold_perf_df['n_modalities'] = all_fold_perf_df['n_modalities'].astype(int)
modality_group_label = {1: '1 modality', 2: '2 modalities', 3: '3 modalities', 4: '4 modalities'}
all_fold_perf_df['modality_group'] = all_fold_perf_df['n_modalities'].map(modality_group_label)
abbr = {'Metagenomics': 'M', 'Radiomics': 'R', 'Blood': 'B', 'Urine': 'U'}

def shorten_modalities(x):
    if pd.isna(x):
        return ''
    parts = str(x).split('+')
    short = []
    for p in parts:
        p = p.strip()
        short.append(abbr.get(p, p))
    if set(short) == set(['M', 'R', 'B', 'U']) and len(short) == 4:
        return 'All'
    return '+'.join(short)
all_fold_perf_df['model_short'] = all_fold_perf_df['modalities'].apply(shorten_modalities)
trend_out_dir = os.path.join(combo_out_dir, get_value('science.paths.directory_013', 'directory_013'))
os.makedirs(trend_out_dir, exist_ok=True)
all_fold_perf_path = os.path.join(trend_out_dir, get_value('science.tables.metrics_010', 'metrics_010.csv'))
all_fold_perf_df.to_csv(all_fold_perf_path, index=False, encoding='utf-8-sig')





In [ ]:
def summarize_by_modality_number(df, metric_col):
    summary = df.groupby('n_modalities')[metric_col].agg(['mean', 'std', 'count', 'median']).reset_index()
    summary['modality_group'] = summary['n_modalities'].map(modality_group_label)
    return summary

def pairwise_pvalues_by_modality_number(df, metric_col):
    pairs = []
    groups = [1, 2, 3, 4]
    for i in range(len(groups)):
        for j in range(i + 1, len(groups)):
            g1 = groups[i]
            g2 = groups[j]
            x1 = df.loc[df['n_modalities'] == g1, metric_col].dropna().values
            x2 = df.loc[df['n_modalities'] == g2, metric_col].dropna().values
            if len(x1) == 0 or len(x2) == 0:
                p = np.nan
                stat = np.nan
            else:
                stat, p = mannwhitneyu(x1, x2, alternative='two-sided')
            pairs.append({'metric': metric_col, 'group1': g1, 'group2': g2, 'group1_label': modality_group_label[g1], 'group2_label': modality_group_label[g2], 'n1': len(x1), 'n2': len(x2), 'mean1': np.mean(x1) if len(x1) else np.nan, 'mean2': np.mean(x2) if len(x2) else np.nan, 'delta_mean': np.mean(x2) - np.mean(x1) if len(x1) and len(x2) else np.nan, 'U_stat': stat, 'P_value': p})
    return pd.DataFrame(pairs)

def format_pvalue(p):
    if pd.isna(p):
        return 'P = NA'
    if p < 0.0001:
        return f'P = {p:.1e}'
    if p < 0.001:
        return f'P = {p:.2e}'
    return f'P = {p:.3f}'

def p_to_stars(p):
    if pd.isna(p):
        return ''
    if p < 0.001:
        return '***'
    if p < 0.01:
        return '**'
    if p < 0.05:
        return '*'
    return 'ns'


In [ ]:
def add_sig_bracket(ax, x1, x2, y, h, text, fontsize=8):
    ax.plot([x1, x1, x2, x2], [y, y + h, y + h, y], lw=0.8, color='red')
    ax.text((x1 + x2) / 2, y + h + 0.4, text, ha='center', va='bottom', color='red', fontsize=fontsize)

def plot_modality_number_trend_percent(df, metric_col, ylabel, title, filename, annotate_pairs=None, show_legend=True):
    pass
    plot_df = df.dropna(subset=[metric_col]).copy()
    plot_df[f'{metric_col}_percent'] = plot_df[metric_col] * 100
    metric_percent_col = f'{metric_col}_percent'
    groups = [1, 2, 3, 4]
    x_positions = np.arange(len(groups))
    summary = summarize_by_modality_number(plot_df, metric_col)
    pval_df = pairwise_pvalues_by_modality_number(plot_df, metric_col)
    summary_path = os.path.join(trend_out_dir, filename.replace('.png', get_value('science.tables.metrics_006', 'metrics_006.csv')))
    pval_path = os.path.join(trend_out_dir, filename.replace('.png', get_value('science.tables.result_010', 'result_010.csv')))
    summary.to_csv(summary_path, index=False, encoding='utf-8-sig')
    pval_df.to_csv(pval_path, index=False, encoding='utf-8-sig')
    data_by_group = [plot_df.loc[plot_df['n_modalities'] == g, metric_percent_col].values for g in groups]
    fig, ax = plt.subplots(figsize=(9.5, 6.0))
    ax.boxplot(data_by_group, positions=x_positions, widths=0.46, patch_artist=True, showfliers=False, medianprops=dict(color='black', linewidth=1.2), boxprops=dict(facecolor='white', edgecolor='black', linewidth=1.0), whiskerprops=dict(color='black', linewidth=1.0), capprops=dict(color='black', linewidth=1.0))
    unique_models = sorted(plot_df['model_short'].dropna().unique().tolist())
    cmap = plt.get_cmap('tab20')
    model_color = {m: cmap(i % 20) for i, m in enumerate(unique_models)}
    rng = np.random.default_rng(42)
    for gi, g in enumerate(groups):
        sub_g = plot_df[plot_df['n_modalities'] == g].copy()
        for model_name, sub_m in sub_g.groupby('model_short'):
            jitter = rng.normal(loc=0, scale=0.045, size=sub_m.shape[0])
            x = np.full(sub_m.shape[0], x_positions[gi]) + jitter
            ax.scatter(x, sub_m[metric_percent_col], s=18, alpha=0.55, color=model_color.get(model_name, 'gray'), edgecolor='white', linewidth=0.3, label=model_name)
    means = []
    sds = []
    for g in groups:
        vals = plot_df.loc[plot_df['n_modalities'] == g, metric_percent_col].dropna().values
        means.append(np.mean(vals))
        sds.append(np.std(vals, ddof=1))
    ax.plot(x_positions, means, color='black', linestyle='--', linewidth=1.3, marker='o', markersize=5, label='Mean performance')
    ax.errorbar(x_positions, means, yerr=sds, fmt='none', ecolor='black', elinewidth=1.0, capsize=4, alpha=0.7)
    for i, (m, sd) in enumerate(zip(means, sds)):
        ax.text(x_positions[i], min(m + sd + 2.0, 96), f'{m:.1f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
    for i in range(len(groups) - 1):
        delta = means[i + 1] - means[i]
        y_mid = (means[i] + means[i + 1]) / 2
        ax.text((x_positions[i] + x_positions[i + 1]) / 2, y_mid + 3.0, f'{delta:+.1f}', ha='center', va='bottom', fontsize=8, fontweight='bold')
    if annotate_pairs is None:
        annotate_pairs = [(1, 2), (2, 3), (3, 4), (1, 4)]
    bracket_start = 97
    bracket_step = 5.0
    bracket_h = 1.6
    for k, (g1, g2) in enumerate(annotate_pairs):
        row = pval_df[(pval_df['group1'] == g1) & (pval_df['group2'] == g2)]
        if row.shape[0] == 0:
            continue
        p = row['P_value'].iloc[0]
        text = f'{format_pvalue(p)} {p_to_stars(p)}'
        x1 = groups.index(g1)
        x2 = groups.index(g2)
        y = bracket_start - k * bracket_step
        add_sig_bracket(ax, x1=x1, x2=x2, y=y, h=bracket_h, text=text, fontsize=7)
    ax.set_xticks(x_positions)
    ax.set_xticklabels([modality_group_label[g] for g in groups], fontsize=10)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_ylim(0, 105)
    ax.set_yticks(np.arange(0, 101, 20))
    ax.grid(axis='y', linestyle='--', alpha=0.35)
    if show_legend:
        handles, labels_ = ax.get_legend_handles_labels()
        seen = set()
        new_handles = []
        new_labels = []
        for h, l in zip(handles, labels_):
            if l not in seen:
                seen.add(l)
                new_handles.append(h)
                new_labels.append(l)
        ax.legend(new_handles, new_labels, bbox_to_anchor=(1.02, 1.0), loc='upper left', frameon=False, fontsize=8, title='Model')
    plt.tight_layout()
    fig_path = os.path.join(trend_out_dir, filename)
    plt.savefig(fig_path, dpi=600, bbox_inches='tight')
    pdf_path = fig_path.replace('.png', '.pdf')
    plt.savefig(pdf_path, bbox_inches='tight')
    plt.show()
    
    
    
    
    return (summary, pval_df)


In [ ]:
response_summary, response_pvals = plot_modality_number_trend_percent(all_fold_perf_df, metric_col='Response_AUC', ylabel='Response AUC (%)', title='Response prediction by number of integrated modalities', filename=get_value('science.figures.plot_007', 'plot_007.png'), annotate_pairs=[(1, 2), (2, 3), (3, 4), (1, 4)], show_legend=True)
pfs_summary, pfs_pvals = plot_modality_number_trend_percent(all_fold_perf_df, metric_col='PFS_Cindex', ylabel='PFS C-index (%)', title='PFS prediction by number of integrated modalities', filename=get_value('science.figures.plot_008', 'plot_008.png'), annotate_pairs=[(1, 2), (2, 3), (3, 4), (1, 4)], show_legend=True)
os_summary, os_pvals = plot_modality_number_trend_percent(all_fold_perf_df, metric_col='OS_Cindex', ylabel='OS C-index (%)', title='OS prediction by number of integrated modalities', filename=get_value('science.figures.plot_009', 'plot_009.png'), annotate_pairs=[(1, 2), (2, 3), (3, 4), (1, 4)], show_legend=True)


In [ ]:
response_summary['Outcome'] = 'Response_AUC'
pfs_summary['Outcome'] = 'PFS_Cindex'
os_summary['Outcome'] = 'OS_Cindex'
trend_summary_all = pd.concat([response_summary, pfs_summary, os_summary], axis=0, ignore_index=True)
response_pvals['Outcome'] = 'Response_AUC'
pfs_pvals['Outcome'] = 'PFS_Cindex'
os_pvals['Outcome'] = 'OS_Cindex'
trend_pvalues_all = pd.concat([response_pvals, pfs_pvals, os_pvals], axis=0, ignore_index=True)
trend_summary_all_path = os.path.join(trend_out_dir, get_value('science.tables.outcomes_002', 'outcomes_002.csv'))
trend_pvalues_all_path = os.path.join(trend_out_dir, get_value('science.tables.outcomes_003', 'outcomes_003.csv'))
trend_summary_all.to_csv(trend_summary_all_path, index=False, encoding='utf-8-sig')
trend_pvalues_all.to_csv(trend_pvalues_all_path, index=False, encoding='utf-8-sig')




trend_summary_all


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
best_model_out_dir = os.path.join(combo_out_dir, get_value('science.paths.directory_014', 'directory_014'))
os.makedirs(best_model_out_dir, exist_ok=True)
abbr = {'Metagenomics': 'M', 'Radiomics': 'R', 'Blood': 'B', 'Urine': 'U'}
full_modalities = 'Metagenomics+Radiomics+Blood+Urine'

def infer_modalities_from_model_name(model):
    if model == 'Multimodal':
        return full_modalities
    elif model in ['Metagenomics', 'Radiomics', 'Blood', 'Urine']:
        return model
    else:
        return model

def infer_n_modalities(modalities):
    if pd.isna(modalities):
        return np.nan
    return len(str(modalities).split('+'))

def shorten_modalities(x):
    if pd.isna(x):
        return ''
    parts = str(x).split('+')
    short = []
    for p in parts:
        p = p.strip()
        short.append(abbr.get(p, p))
    if len(short) == 4 and set(short) == set(['M', 'R', 'B', 'U']):
        return 'All'
    return '+'.join(short)

def pretty_metric_name(metric):
    if metric == 'Response_AUC':
        return 'Response AUC'
    elif metric == 'PFS_Cindex':
        return 'PFS C-index'
    elif metric == 'OS_Cindex':
        return 'OS C-index'
    else:
        return metric
summary_for_best = all_model_summary_df.copy()
if 'modalities' not in summary_for_best.columns:
    summary_for_best['modalities'] = summary_for_best['model'].apply(infer_modalities_from_model_name)
else:
    summary_for_best['modalities'] = summary_for_best['modalities'].fillna(summary_for_best['model'].apply(infer_modalities_from_model_name))
if 'n_modalities' not in summary_for_best.columns:
    summary_for_best['n_modalities'] = summary_for_best['modalities'].apply(infer_n_modalities)
else:
    missing_n = summary_for_best['n_modalities'].isna()
    summary_for_best.loc[missing_n, 'n_modalities'] = summary_for_best.loc[missing_n, 'modalities'].apply(infer_n_modalities)
summary_for_best['n_modalities'] = summary_for_best['n_modalities'].astype(int)
summary_for_best['modalities_short'] = summary_for_best['modalities'].apply(shorten_modalities)
summary_for_best.loc[summary_for_best['model'] == 'Multimodal', 'modalities_short'] = 'All'
summary_for_best.loc[summary_for_best['model'] == 'Multimodal', 'n_modalities'] = 4
print(summary_for_best[['model', 'modalities', 'modalities_short', 'n_modalities']].head())


In [ ]:
outcome_specs = {'Response': {'metric_mean': 'Response_AUC_mean', 'metric_sd': 'Response_AUC_sd', 'metric_label': 'Response AUC'}, 'PFS': {'metric_mean': 'PFS_Cindex_mean', 'metric_sd': 'PFS_Cindex_sd', 'metric_label': 'PFS C-index'}, 'OS': {'metric_mean': 'OS_Cindex_mean', 'metric_sd': 'OS_Cindex_sd', 'metric_label': 'OS C-index'}}
category_label_map = {1: 'Best single\nmodality', 2: 'Best two-\nmodality', 3: 'Best three-\nmodality', 4: 'Full\nmultimodal'}
best_rows = []
for outcome, spec in outcome_specs.items():
    metric_mean = spec['metric_mean']
    metric_sd = spec['metric_sd']
    for n_mod in [1, 2, 3, 4]:
        sub = summary_for_best[summary_for_best['n_modalities'] == n_mod].copy()
        if sub.shape[0] == 0:
            continue
        if n_mod == 4 and 'Multimodal' in sub['model'].values:
            best = sub[sub['model'] == 'Multimodal'].iloc[0]
        else:
            best = sub.loc[sub[metric_mean].idxmax()]
        best_rows.append({'Outcome': outcome, 'Category': category_label_map[n_mod], 'n_modalities': n_mod, 'Model': best['model'], 'Modalities': best['modalities'], 'Modalities_short': best['modalities_short'], 'Metric': spec['metric_label'], 'Mean': best[metric_mean], 'SD': best[metric_sd], 'Mean_percent': best[metric_mean] * 100, 'SD_percent': best[metric_sd] * 100, 'n_samples': best.get('n_samples', np.nan), 'n_features': best.get('n_features', np.nan)})
best_model_summary_df = pd.DataFrame(best_rows)
best_model_summary_path = os.path.join(best_model_out_dir, get_value('science.tables.metrics_011', 'metrics_011.csv'))
best_model_summary_df.to_csv(best_model_summary_path, index=False, encoding='utf-8-sig')


best_model_summary_df


In [ ]:
def plot_endpoint_best_model_summary(best_df):
    outcome_order = ['Response', 'PFS', 'OS']
    category_order = ['Best single\nmodality', 'Best two-\nmodality', 'Best three-\nmodality', 'Full\nmultimodal']
    fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.8), sharey=True)
    for ax, outcome in zip(axes, outcome_order):
        sub = best_df[best_df['Outcome'] == outcome].copy()
        sub['Category'] = pd.Categorical(sub['Category'], categories=category_order, ordered=True)
        sub = sub.sort_values('Category')
        x = np.arange(sub.shape[0])
        y = sub['Mean_percent'].values
        yerr = sub['SD_percent'].values
        ax.bar(x, y, yerr=yerr, capsize=4, width=0.62, alpha=0.88, edgecolor='black', linewidth=0.8)
        for i, row in sub.iterrows():
            xpos = list(sub.index).index(i)
            value = row['Mean_percent']
            sd = row['SD_percent']
            label = row['Modalities_short']
            ax.text(xpos, min(value + sd + 3.0, 99), f'{value:.1f}\n{label}', ha='center', va='bottom', fontsize=8, fontweight='bold')
        ax.set_xticks(x)
        ax.set_xticklabels(sub['Category'], fontsize=9)
        ax.set_ylim(0, 105)
        ax.set_yticks(np.arange(0, 101, 20))
        ax.set_title(outcome, fontsize=13, fontweight='bold')
        ax.grid(axis='y', linestyle='--', alpha=0.35)
        if outcome == 'Response':
            ax.set_ylabel('Performance (%)', fontsize=11)
        metric_label = sub['Metric'].iloc[0]
        ax.text(0.5, -0.22, metric_label, transform=ax.transAxes, ha='center', va='top', fontsize=10)
    plt.suptitle('Endpoint-specific best models by modality category', fontsize=14, fontweight='bold', y=1.03)
    plt.tight_layout()
    fig_path = os.path.join(best_model_out_dir, get_value('science.figures.plot_010', 'plot_010.png'))
    pdf_path = fig_path.replace('.png', '.pdf')
    plt.savefig(fig_path, dpi=600, bbox_inches='tight')
    plt.savefig(pdf_path, bbox_inches='tight')
    plt.show()
    
    
plot_endpoint_best_model_summary(best_model_summary_df)


In [ ]:
ranking_rows = []
for outcome, spec in outcome_specs.items():
    metric_mean = spec['metric_mean']
    metric_sd = spec['metric_sd']
    tmp = summary_for_best.copy()
    tmp = tmp.sort_values(metric_mean, ascending=False).reset_index(drop=True)
    tmp['Outcome'] = outcome
    tmp['Rank'] = np.arange(1, tmp.shape[0] + 1)
    tmp['Metric'] = spec['metric_label']
    tmp['Mean'] = tmp[metric_mean]
    tmp['SD'] = tmp[metric_sd]
    tmp['Mean_percent'] = tmp['Mean'] * 100
    tmp['SD_percent'] = tmp['SD'] * 100
    ranking_rows.append(tmp[['Outcome', 'Rank', 'model', 'modalities', 'modalities_short', 'n_modalities', 'n_features', 'Metric', 'Mean', 'SD', 'Mean_percent', 'SD_percent']])
endpoint_ranking_df = pd.concat(ranking_rows, axis=0)
endpoint_ranking_path = os.path.join(best_model_out_dir, get_value('science.tables.predictions_006', 'predictions_006.csv'))
endpoint_ranking_df.to_csv(endpoint_ranking_path, index=False, encoding='utf-8-sig')


endpoint_ranking_df.head(20)


In [ ]:
import os
import random
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import shap
import joblib
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
set_seed(RANDOM_STATE)
base_out_dir = str(project_path(get_value('science.paths.directory_002', 'directory_002')))
stage1_dir = os.path.join(base_out_dir, get_value('science.paths.directory_003', 'directory_003'))
stage2_dir = os.path.join(base_out_dir, get_value('science.paths.directory_011', 'directory_011'))
stage3_dir = os.path.join(base_out_dir, get_value('science.paths.directory_015', 'directory_015'))
os.makedirs(stage3_dir, exist_ok=True)
label_path = str(project_path(get_value('science.tables.outcomes_004', 'outcomes_004.csv')))
golden_feature_path = os.path.join(stage1_dir, get_value('science.tables.result_005', 'result_005.csv'))
modal_files = {'Metagenomics': str(project_path(get_value('science.tables.features_006', 'features_006.csv.gz'))), 'Radiomics': str(project_path(get_value('science.tables.features_007', 'features_007.csv.gz'))), 'Blood': str(project_path(get_value('science.tables.features_008', 'features_008.csv.gz'))), 'Urine': str(project_path(get_value('science.tables.features_009', 'features_009.csv.gz')))}
modalities = ['Metagenomics', 'Radiomics', 'Blood', 'Urine']
task_names = ['Response', 'PFS', 'OS']
required_paths = [label_path, golden_feature_path] + list(modal_files.values())
missing_paths = [p for p in required_paths if not os.path.exists(p)]
if missing_paths:
    print('\nStage 3 missing required files:')
    for p in missing_paths:
        print(' -', p)
    raise FileNotFoundError('Please check the new label file, four model-ready matrices, and Stage 1 golden-feature output.')



for modal_name, path in modal_files.items():
    pass
LAMBDA_RESPONSE = 0.4
LAMBDA_PFS = 0.8
LAMBDA_OS = 1.0
MAX_EPOCHS = 500
MIN_EPOCHS = 50
PATIENCE = 40
MIN_DELTA = 0.0001
INNER_VAL_SIZE = 0.2

def normalize_sample_id(x):
    return str(x).lower().strip()

def find_column(df, candidates, required=True):
    lower_map = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    if required:
        raise ValueError(f'Cannot find any of columns: {candidates}\nExisting columns: {list(df.columns)}')
    return None

def to_binary_response(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x, str):
        s = x.strip().lower()
        if s in ['1', 'yes', 'y', 'responder', 'response', 'r', 'cr', 'pr', 'orr']:
            return 1
        if s in ['0', 'no', 'n', 'nonresponder', 'non-responder', 'nr', 'sd', 'pd']:
            return 0
    return int(float(x))

def read_labels(label_path):
    if label_path.endswith('.xlsx') or label_path.endswith('.xls'):
        raw = pd.read_excel(label_path)
    else:
        raw = pd.read_csv(label_path, encoding='utf-8-sig')
    sample_col = find_column(raw, ['Sample ID', 'sample_id', 'sample', 'Sample', 'ID'])
    response_col = find_column(raw, ['Response', 'Response', 'response', 'ORR'])
    pfs_time_col = find_column(raw, ['PFS_time', 'pfs_time', 'PFS', 'PFS_month', 'PFS_months'])
    pfs_status_col = find_column(raw, ['PFS_status', 'pfs_status', 'PFS_event', 'pfs_event'])
    os_time_col = find_column(raw, ['OS_time', 'os_time', 'OS', 'OS_month', 'OS_months'])
    os_status_col = find_column(raw, ['OS_status', 'os_status', 'OS_event', 'os_event'])
    labels = pd.DataFrame()
    labels['sample'] = raw[sample_col].map(normalize_sample_id)
    labels['Response'] = raw[response_col].map(to_binary_response).astype(float)
    labels['PFS_time'] = pd.to_numeric(raw[pfs_time_col], errors='coerce')
    labels['PFS_status'] = pd.to_numeric(raw[pfs_status_col], errors='coerce')
    labels['OS_time'] = pd.to_numeric(raw[os_time_col], errors='coerce')
    labels['OS_status'] = pd.to_numeric(raw[os_status_col], errors='coerce')
    labels = labels.drop_duplicates(subset='sample', keep='first')
    labels = labels.set_index('sample')
    labels = labels.dropna(subset=['Response', 'PFS_time', 'PFS_status', 'OS_time', 'OS_status'])
    print('Labels:', labels.shape)
    print('Response positive:', int(labels['Response'].sum()), '/', labels.shape[0])
    print('PFS events:', int(labels['PFS_status'].sum()), '/', labels.shape[0])
    print('OS events:', int(labels['OS_status'].sum()), '/', labels.shape[0])
    return labels

def read_matrix(path):
    df = pd.read_csv(path, encoding='utf-8-sig')
    if 'sample' in df.columns:
        sample_col = 'sample'
    elif 'sample_id' in df.columns:
        sample_col = 'sample_id'
    else:
        sample_col = df.columns[0]
    df = df.rename(columns={sample_col: 'sample'})
    df['sample'] = df['sample'].map(normalize_sample_id)
    df = df.drop_duplicates(subset='sample', keep='first')
    df = df.set_index('sample')
    df = df.apply(pd.to_numeric, errors='coerce')
    return df

def apply_basic_transform_before_selection(X, modal_name):
    pass
    return X.copy()
labels = read_labels(label_path)
golden_features = pd.read_csv(golden_feature_path, encoding='utf-8-sig')
print('\nGolden features count:')
print(golden_features.groupby('modal')['feature'].nunique())

def build_reduced_single_modal_matrix(modal_name):
    path = modal_files[modal_name]
    X_raw = read_matrix(path)
    X_raw = apply_basic_transform_before_selection(X_raw, modal_name)
    selected_features = golden_features.loc[golden_features['modal'].astype(str).str.lower() == modal_name.lower(), 'feature'].dropna().astype(str).unique().tolist()
    available_features = [f for f in selected_features if f in X_raw.columns]
    missing_features = [f for f in selected_features if f not in X_raw.columns]
    if len(missing_features) > 0:
        print(f'[{modal_name}] Missing selected features: {len(missing_features)}')
        print('First missing:', missing_features[:5])
    X_reduced = X_raw[available_features].copy()
    common = X_reduced.index.intersection(labels.index)
    X_reduced = X_reduced.loc[common]
    print(f'[{modal_name}] raw features={X_raw.shape[1]}, selected={len(selected_features)}, available={len(available_features)}, reduced matrix={X_reduced.shape}')
    return X_reduced
single_modal_reduced = {}
for modal_name in modalities:
    single_modal_reduced[modal_name] = build_reduced_single_modal_matrix(modal_name)
common_samples = labels.index.copy()
for modal_name in modalities:
    common_samples = common_samples.intersection(single_modal_reduced[modal_name].index)
common_samples = common_samples.sort_values()
multimodal_parts = []
feature_metadata_records = []
for modal_name in modalities:
    X_part = single_modal_reduced[modal_name].loc[common_samples].copy()
    renamed_cols = [f'{modal_name}__{c}' for c in X_part.columns]
    X_part.columns = renamed_cols
    multimodal_parts.append(X_part)
    for original_feature, final_feature in zip(single_modal_reduced[modal_name].columns, renamed_cols):
        feature_metadata_records.append({'feature': final_feature, 'original_feature': original_feature, 'modal': modal_name})
X_multimodal = pd.concat(multimodal_parts, axis=1)
y_multimodal = labels.loc[common_samples].copy()
feature_metadata = pd.DataFrame(feature_metadata_records)
feature_metadata.to_csv(os.path.join(stage3_dir, get_value('science.tables.result_011', 'result_011.csv')), index=False, encoding='utf-8-sig')
X_multimodal.to_csv(os.path.join(stage3_dir, get_value('science.tables.features_010', 'features_010.csv')), encoding='utf-8-sig')
print('\nMultimodal matrix:', X_multimodal.shape)

class ReducedMultiTaskNet(nn.Module):

    def __init__(self, input_dim, hidden1=64, hidden2=16, dropout=0.4):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(input_dim, hidden1), nn.BatchNorm1d(hidden1), nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden1, hidden2), nn.BatchNorm1d(hidden2), nn.ReLU(), nn.Dropout(dropout))
        self.response_head = nn.Linear(hidden2, 1)
        self.pfs_head = nn.Linear(hidden2, 1)
        self.os_head = nn.Linear(hidden2, 1)

    def forward(self, x):
        z = self.encoder(x)
        return {'response_logit': self.response_head(z).view(-1), 'pfs_risk': self.pfs_head(z).view(-1), 'os_risk': self.os_head(z).view(-1)}

class SHAPTaskWrapper(nn.Module):

    def __init__(self, base_model, task_name):
        super().__init__()
        self.base_model = base_model
        self.task_name = task_name

    def forward(self, x):
        out = self.base_model(x)
        if self.task_name == 'Response':
            return out['response_logit'].view(-1, 1)
        elif self.task_name == 'PFS':
            return out['pfs_risk'].view(-1, 1)
        elif self.task_name == 'OS':
            return out['os_risk'].view(-1, 1)
        else:
            raise ValueError('task_name must be Response, PFS, or OS.')

def cox_ph_loss(risk, time, event):
    risk = risk.view(-1)
    time = time.view(-1)
    event = event.view(-1)
    valid = ~torch.isnan(time) & ~torch.isnan(event)
    risk = risk[valid]
    time = time[valid]
    event = event[valid]
    if len(risk) == 0 or event.sum() < 1:
        return torch.tensor(0.0, device=risk.device)
    order = torch.argsort(time, descending=True)
    risk = risk[order]
    event = event[order]
    log_cumsum_exp = torch.logcumsumexp(risk, dim=0)
    loss = -torch.sum((risk - log_cumsum_exp) * event) / (torch.sum(event) + 1e-08)
    return loss

def multitask_loss(outputs, y_response, pfs_time, pfs_status, os_time, os_status, response_pos_weight=None, lambda_response=LAMBDA_RESPONSE, lambda_pfs=LAMBDA_PFS, lambda_os=LAMBDA_OS):
    if response_pos_weight is not None:
        bce = nn.BCEWithLogitsLoss(pos_weight=response_pos_weight)
    else:
        bce = nn.BCEWithLogitsLoss()
    loss_response = bce(outputs['response_logit'], y_response)
    loss_pfs = cox_ph_loss(outputs['pfs_risk'], pfs_time, pfs_status)
    loss_os = cox_ph_loss(outputs['os_risk'], os_time, os_status)
    loss = lambda_response * loss_response + lambda_pfs * loss_pfs + lambda_os * loss_os
    return (loss, {'loss_response': float(loss_response.detach().cpu().item()), 'loss_pfs': float(loss_pfs.detach().cpu().item()), 'loss_os': float(loss_os.detach().cpu().item()), 'weighted_total': float(loss.detach().cpu().item())})

def make_label_tensors(y_df):
    y_response = torch.tensor(y_df['Response'].values, dtype=torch.float32).to(DEVICE)
    pfs_time = torch.tensor(y_df['PFS_time'].values, dtype=torch.float32).to(DEVICE)
    pfs_status = torch.tensor(y_df['PFS_status'].values, dtype=torch.float32).to(DEVICE)
    os_time = torch.tensor(y_df['OS_time'].values, dtype=torch.float32).to(DEVICE)
    os_status = torch.tensor(y_df['OS_status'].values, dtype=torch.float32).to(DEVICE)
    return (y_response, pfs_time, pfs_status, os_time, os_status)

def get_response_pos_weight(y_response_tensor):
    n_pos = y_response_tensor.sum().item()
    n_neg = len(y_response_tensor) - n_pos
    if n_pos > 0:
        return torch.tensor([n_neg / n_pos], dtype=torch.float32).to(DEVICE)
    return None

def fit_preprocessor(X_fit, model_name):
    pass
    imputer = SimpleImputer(strategy='median')
    X_processed = imputer.fit_transform(X_fit)
    scaler = None
    return (imputer, scaler, X_processed)

def transform_preprocessor(X, imputer, scaler):
    return imputer.transform(X)

def get_model_hyperparams(model_name, input_dim):
    if model_name == 'Multimodal':
        return {'hidden1': min(128, max(32, input_dim)), 'hidden2': 32, 'dropout': 0.45}
    elif model_name == 'Radiomics':
        return {'hidden1': min(64, max(16, input_dim // 2)), 'hidden2': 16, 'dropout': 0.4}
    elif model_name in ['Blood', 'Urine']:
        return {'hidden1': min(64, max(16, input_dim // 2)), 'hidden2': 16, 'dropout': 0.45}
    else:
        return {'hidden1': min(64, max(16, input_dim // 2)), 'hidden2': 16, 'dropout': 0.45}

def determine_best_epoch_for_interpretation(X_df, y_df, model_name):
    pass
    idx_all = np.arange(X_df.shape[0])
    stratify_y = y_df['Response'].astype(int).values
    try:
        inner_train_idx, inner_val_idx = train_test_split(idx_all, test_size=INNER_VAL_SIZE, random_state=RANDOM_STATE, stratify=stratify_y)
    except ValueError:
        inner_train_idx, inner_val_idx = train_test_split(idx_all, test_size=INNER_VAL_SIZE, random_state=RANDOM_STATE, stratify=None)
    X_train = X_df.iloc[inner_train_idx]
    X_val = X_df.iloc[inner_val_idx]
    y_train = y_df.iloc[inner_train_idx]
    y_val = y_df.iloc[inner_val_idx]
    imputer, scaler, X_train_scaled = fit_preprocessor(X_train, model_name)
    X_val_scaled = transform_preprocessor(X_val, imputer, scaler)
    X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32).to(DEVICE)
    X_val_tensor = torch.tensor(X_val_scaled, dtype=torch.float32).to(DEVICE)
    y_response_train, pfs_time_train, pfs_status_train, os_time_train, os_status_train = make_label_tensors(y_train)
    y_response_val, pfs_time_val, pfs_status_val, os_time_val, os_status_val = make_label_tensors(y_val)
    pos_weight = get_response_pos_weight(y_response_train)
    hp = get_model_hyperparams(model_name, X_df.shape[1])
    model = ReducedMultiTaskNet(input_dim=X_df.shape[1], hidden1=hp['hidden1'], hidden2=hp['hidden2'], dropout=hp['dropout']).to(DEVICE)
    optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.0001)
    best_val_loss = np.inf
    best_epoch = 1
    patience_counter = 0
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        optimizer.zero_grad()
        outputs = model(X_train_tensor)
        loss, _ = multitask_loss(outputs, y_response_train, pfs_time_train, pfs_status_train, os_time_train, os_status_train, response_pos_weight=pos_weight)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        model.eval()
        with torch.no_grad():
            val_outputs = model(X_val_tensor)
            val_loss, _ = multitask_loss(val_outputs, y_response_val, pfs_time_val, pfs_status_val, os_time_val, os_status_val, response_pos_weight=pos_weight)
        val_loss_value = float(val_loss.detach().cpu().item())
        if val_loss_value < best_val_loss - MIN_DELTA:
            best_val_loss = val_loss_value
            best_epoch = epoch
            patience_counter = 0
        else:
            patience_counter += 1
        if epoch >= MIN_EPOCHS and patience_counter >= PATIENCE:
            break
    
    return best_epoch

def train_final_interpretation_model(X_df, y_df, model_name):
    pass
    set_seed(RANDOM_STATE)
    best_epoch = determine_best_epoch_for_interpretation(X_df=X_df, y_df=y_df, model_name=model_name)
    imputer, scaler, X_scaled = fit_preprocessor(X_df, model_name)
    X_tensor = torch.tensor(X_scaled, dtype=torch.float32).to(DEVICE)
    y_response, pfs_time, pfs_status, os_time, os_status = make_label_tensors(y_df)
    pos_weight = get_response_pos_weight(y_response)
    hp = get_model_hyperparams(model_name, X_df.shape[1])
    model = ReducedMultiTaskNet(input_dim=X_df.shape[1], hidden1=hp['hidden1'], hidden2=hp['hidden2'], dropout=hp['dropout']).to(DEVICE)
    optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.0001)
    for epoch in range(1, best_epoch + 1):
        model.train()
        optimizer.zero_grad()
        outputs = model(X_tensor)
        loss, _ = multitask_loss(outputs, y_response, pfs_time, pfs_status, os_time, os_status, response_pos_weight=pos_weight)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
    model.eval()
    return (model, X_scaled, X_tensor, imputer, scaler, best_epoch)

def compute_sample_level_shap(model, X_scaled, X_tensor, feature_names, model_name):
    shap_signed_dict = {}
    shap_abs_dict = {}
    shap_importance_records = []
    n_bg = min(80, X_scaled.shape[0])
    bg_idx = np.random.choice(X_scaled.shape[0], size=n_bg, replace=False)
    background = torch.tensor(X_scaled[bg_idx], dtype=torch.float32).to(DEVICE)
    for task in task_names:
        
        wrapped = SHAPTaskWrapper(model, task).to(DEVICE)
        wrapped.eval()
        try:
            explainer = shap.DeepExplainer(wrapped, background)
            shap_vals = explainer.shap_values(X_tensor)
        except Exception as e:
            print(f'[{model_name}-{task}] DeepExplainer failed, using GradientExplainer. Error: {e}')
            explainer = shap.GradientExplainer(wrapped, background)
            shap_vals = explainer.shap_values(X_tensor)
        if isinstance(shap_vals, list):
            shap_vals = shap_vals[0]
        if torch.is_tensor(shap_vals):
            shap_vals = shap_vals.detach().cpu().numpy()
        shap_vals = np.asarray(shap_vals)
        if shap_vals.ndim == 3:
            shap_vals = shap_vals[:, :, 0]
        shap_signed = pd.DataFrame(shap_vals, columns=feature_names)
        shap_abs = shap_signed.abs()
        shap_signed_dict[task] = shap_signed
        shap_abs_dict[task] = shap_abs
        mean_abs = shap_abs.mean(axis=0).sort_values(ascending=False)
        imp_df = mean_abs.reset_index()
        imp_df.columns = ['feature', 'mean_abs_shap']
        imp_df['model'] = model_name
        imp_df['outcome'] = task
        imp_df['rank'] = np.arange(1, imp_df.shape[0] + 1)
        shap_importance_records.append(imp_df)
    shap_importance_df = pd.concat(shap_importance_records, axis=0)
    return (shap_signed_dict, shap_abs_dict, shap_importance_df)

def build_prediction_table(model, X_tensor, y_df):
    model.eval()
    with torch.no_grad():
        out = model(X_tensor)
        response_pred = torch.sigmoid(out['response_logit']).detach().cpu().numpy()
        pfs_risk = out['pfs_risk'].detach().cpu().numpy()
        os_risk = out['os_risk'].detach().cpu().numpy()
    pred_df = pd.DataFrame(index=y_df.index)
    pred_df['Response_true'] = y_df['Response'].values
    pred_df['Response_pred'] = response_pred
    pred_df['PFS_time'] = y_df['PFS_time'].values
    pred_df['PFS_status'] = y_df['PFS_status'].values
    pred_df['PFS_risk'] = pfs_risk
    pred_df['OS_time'] = y_df['OS_time'].values
    pred_df['OS_status'] = y_df['OS_status'].values
    pred_df['OS_risk'] = os_risk
    return pred_df

def save_final_shap_package(model_name, X_df, y_df, is_multimodal=False):
    
    print(f'>>> Final SHAP interpretation for: {model_name}')
    
    model_out_dir = os.path.join(stage3_dir, model_name)
    os.makedirs(model_out_dir, exist_ok=True)
    common = X_df.index.intersection(y_df.index)
    X_df = X_df.loc[common].copy()
    y_df = y_df.loc[common].copy()
    X_df.to_csv(os.path.join(model_out_dir, f"{model_name}{get_value('science.tables.features_011', 'features_011.csv')}"), encoding='utf-8-sig')
    model, X_scaled, X_tensor, imputer, scaler, best_epoch = train_final_interpretation_model(X_df=X_df, y_df=y_df, model_name=model_name)
    joblib.dump(imputer, os.path.join(model_out_dir, f"{model_name}{get_value('science.artifacts.transform_001', 'transform_001.joblib')}"))
    if scaler is not None:
        joblib.dump(scaler, os.path.join(model_out_dir, f"{model_name}{get_value('science.artifacts.transform_002', 'transform_002.joblib')}"))
    torch.save({'model_state_dict': model.state_dict(), 'input_dim': X_df.shape[1], 'feature_names': X_df.columns.tolist(), 'best_epoch': best_epoch, 'model_name': model_name, 'task_names': task_names, 'lambda_response': LAMBDA_RESPONSE, 'lambda_pfs': LAMBDA_PFS, 'lambda_os': LAMBDA_OS}, os.path.join(model_out_dir, f"{model_name}{get_value('science.artifacts.transform_003', 'transform_003.pt')}"))
    X_scaled_df = pd.DataFrame(X_scaled, index=X_df.index, columns=X_df.columns)
    X_scaled_df.to_csv(os.path.join(model_out_dir, f"{model_name}{get_value('science.tables.features_012', 'features_012.csv')}"), encoding='utf-8-sig')
    pred_df = build_prediction_table(model=model, X_tensor=X_tensor, y_df=y_df)
    pred_df.to_csv(os.path.join(model_out_dir, f"{model_name}{get_value('science.tables.predictions_007', 'predictions_007.csv')}"), encoding='utf-8-sig')
    shap_signed_dict, shap_abs_dict, shap_importance_df = compute_sample_level_shap(model=model, X_scaled=X_scaled, X_tensor=X_tensor, feature_names=X_df.columns.tolist(), model_name=model_name)
    for task in task_names:
        shap_signed = shap_signed_dict[task].copy()
        shap_abs = shap_abs_dict[task].copy()
        shap_signed.index = X_df.index
        shap_abs.index = X_df.index
        shap_signed.to_csv(os.path.join(model_out_dir, f'{model_name}_final_shap_signed_{task}.csv.gz'), encoding='utf-8-sig', compression='gzip')
        shap_abs.to_csv(os.path.join(model_out_dir, f'{model_name}_final_shap_abs_{task}.csv.gz'), encoding='utf-8-sig', compression='gzip')
    shap_importance_df.to_csv(os.path.join(model_out_dir, f"{model_name}{get_value('science.tables.attribution_005', 'attribution_005.csv')}"), index=False, encoding='utf-8-sig')
    if is_multimodal:
        if 'feature_metadata' not in globals():
            raise ValueError('feature_metadata is required for multimodal SHAP contribution.')
        fm = feature_metadata.copy()
        modality_contribution_records = []
        for task in task_names:
            shap_abs_task = shap_abs_dict[task].copy()
            shap_abs_task.columns = X_df.columns
            mean_abs = shap_abs_task.mean(axis=0).reset_index()
            mean_abs.columns = ['feature', 'mean_abs_shap']
            tmp = mean_abs.merge(fm, on='feature', how='left')
            contrib = tmp.groupby('modal')['mean_abs_shap'].sum().reset_index()
            total = contrib['mean_abs_shap'].sum()
            contrib['fraction'] = contrib['mean_abs_shap'] / (total + 1e-12)
            contrib['outcome'] = task
            modality_contribution_records.append(contrib)
        modality_contribution_df = pd.concat(modality_contribution_records, axis=0)
        modality_contribution_df.to_csv(os.path.join(model_out_dir, f"{model_name}{get_value('science.tables.attribution_006', 'attribution_006.csv')}"), index=False, encoding='utf-8-sig')
    
    return {'model_name': model_name, 'out_dir': model_out_dir, 'prediction': pred_df, 'shap_importance': shap_importance_df}
single_modal_final_results = {}
for modal_name in modalities:
    X_df = single_modal_reduced[modal_name].copy()
    y_df = labels.loc[X_df.index].copy()
    res = save_final_shap_package(model_name=modal_name, X_df=X_df, y_df=y_df, is_multimodal=False)
    single_modal_final_results[modal_name] = res
multimodal_result = save_final_shap_package(model_name='Multimodal', X_df=X_multimodal, y_df=y_multimodal, is_multimodal=True)



In [ ]:
import os
import re
import math
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.colors import Normalize, TwoSlopeNorm, LinearSegmentedColormap
from matplotlib.patches import Rectangle, PathPatch
from matplotlib.path import Path
import seaborn as sns

base_out_dir = str(project_path(get_value('science.paths.directory_002', 'directory_002')))
stage3_dir = os.path.join(base_out_dir, get_value('science.paths.directory_015', 'directory_015'))
multi_dir = os.path.join(stage3_dir, 'Multimodal')
fig_out_dir = os.path.join(stage3_dir, get_value('science.paths.directory_016', 'directory_016'))
os.makedirs(fig_out_dir, exist_ok=True)
single_modal_atlas_dir = os.path.join(fig_out_dir, get_value('science.paths.directory_017', 'directory_017'))
os.makedirs(single_modal_atlas_dir, exist_ok=True)
multi_fig_dir = os.path.join(fig_out_dir, get_value('science.paths.directory_018', 'directory_018'))
os.makedirs(multi_fig_dir, exist_ok=True)
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['font.family'] = 'Arial'
mpl.rcParams['axes.linewidth'] = 0.8
mpl.rcParams['axes.spines.top'] = False
mpl.rcParams['axes.spines.right'] = False
mpl.rcParams['figure.dpi'] = 120
task_names = ['Response', 'PFS', 'OS']
modalities = ['Radiomics', 'Blood', 'Urine', 'Metagenomics']
MODALITY_COLORS = {'Radiomics': '#4C78A8', 'Blood': '#D95F02', 'Urine': '#8E6BBE', 'Metagenomics': '#2A9D8F', 'Microbiota': '#2A9D8F'}
OUTCOME_COLORS = {'Response': '#2D6CDF', 'PFS': '#EE854A', 'OS': '#7A4FA3'}
DIVERGE_CMAP = plt.get_cmap('RdBu_r')

def normalize_sample_id(x):
    return str(x).lower().strip()

def read_index_csv(path, compression=None):
    df = pd.read_csv(path, index_col=0, encoding='utf-8-sig', compression=compression)
    df.index = df.index.map(normalize_sample_id)
    return df

def clean_feature_name(s, max_len=38):
    s = str(s)
    if '__' in s:
        s = s.split('__', 1)[1]
    replace_list = ['original_', 'wavelet-', 'wavelet_', 'log-sigma-', 'log_sigma_', 'glcm_', 'glrlm_', 'glszm_', 'gldm_', 'firstorder_', 'shape_', 'POS_', 'NEG_', 'pos_', 'neg_', 'k__', 'p__', 'c__', 'o__', 'f__', 'g__', 's__']
    for r in replace_list:
        s = s.replace(r, '')
    s = s.replace(';', '|')
    s = s.replace('_', ' ')
    s = re.sub('\\s+', ' ', s).strip()
    if len(s) > max_len:
        s = s[:max_len - 2] + '…'
    return s

def sanitize_filename(s):
    s = str(s)
    s = re.sub('[\\\\/:*?\\"<>|]', '_', s)
    s = s.replace(' ', '_')
    return s

def minmax_scale_series(x):
    x = np.asarray(x, dtype=float)
    xmin = np.nanmin(x)
    xmax = np.nanmax(x)
    if xmax - xmin < 1e-12:
        return np.zeros_like(x)
    return (x - xmin) / (xmax - xmin)

def load_multimodal_shap_package():
    fm_path = os.path.join(stage3_dir, get_value('science.tables.result_011', 'result_011.csv'))
    fm = pd.read_csv(fm_path, encoding='utf-8-sig')
    imp_path = os.path.join(multi_dir, get_value('science.tables.attribution_007', 'attribution_007.csv'))
    imp = pd.read_csv(imp_path, encoding='utf-8-sig')
    contrib_path = os.path.join(multi_dir, get_value('science.tables.attribution_008', 'attribution_008.csv'))
    contrib = pd.read_csv(contrib_path, encoding='utf-8-sig')
    X_path = os.path.join(multi_dir, get_value('science.tables.features_013', 'features_013.csv'))
    X_scaled = read_index_csv(X_path)
    pred_path = os.path.join(multi_dir, get_value('science.tables.predictions_008', 'predictions_008.csv'))
    pred = read_index_csv(pred_path)
    shap_signed = {}
    shap_abs = {}
    for task in task_names:
        signed_path = os.path.join(multi_dir, f'Multimodal_final_shap_signed_{task}.csv.gz')
        abs_path = os.path.join(multi_dir, f'Multimodal_final_shap_abs_{task}.csv.gz')
        shap_signed[task] = read_index_csv(signed_path, compression='gzip')
        shap_abs[task] = read_index_csv(abs_path, compression='gzip')
    common = X_scaled.index.intersection(pred.index)
    for task in task_names:
        common = common.intersection(shap_signed[task].index)
        common = common.intersection(shap_abs[task].index)
    X_scaled = X_scaled.loc[common]
    pred = pred.loc[common]
    for task in task_names:
        shap_signed[task] = shap_signed[task].loc[common]
        shap_abs[task] = shap_abs[task].loc[common]
    return (fm, imp, contrib, X_scaled, pred, shap_signed, shap_abs)

def select_top_multimodal_features(imp, fm, top_per_outcome=8, max_total=24):
    selected = []
    for task in task_names:
        sub = imp[imp['outcome'] == task].sort_values('mean_abs_shap', ascending=False).head(top_per_outcome)
        selected.extend(sub['feature'].astype(str).tolist())
    selected = list(dict.fromkeys(selected))
    pivot = imp[imp['feature'].isin(selected)].pivot_table(index='feature', columns='outcome', values='mean_abs_shap', aggfunc='mean').fillna(0)
    for task in task_names:
        if task not in pivot.columns:
            pivot[task] = 0
    pivot['total'] = pivot[task_names].sum(axis=1)
    selected = pivot.sort_values('total', ascending=False).head(max_total).index.tolist()
    meta = fm[fm['feature'].isin(selected)].drop_duplicates('feature')
    modal_order_map = {m: i for i, m in enumerate(modalities)}
    sort_df = pd.DataFrame({'feature': selected}).merge(meta[['feature', 'modal', 'original_feature']], on='feature', how='left').merge(pivot[['total']].reset_index(), on='feature', how='left')
    sort_df['modal_order'] = sort_df['modal'].map(modal_order_map).fillna(99)
    sort_df = sort_df.sort_values(['modal_order', 'total'], ascending=[True, False])
    return sort_df['feature'].tolist()

def get_feature_modal(feature, fm):
    row = fm[fm['feature'] == feature]
    if row.shape[0] == 0:
        if '__' in feature:
            return feature.split('__', 1)[0]
        return 'Unknown'
    return str(row['modal'].iloc[0])

def get_original_feature(feature, fm):
    row = fm[fm['feature'] == feature]
    if row.shape[0] == 0:
        if '__' in feature:
            return feature.split('__', 1)[1]
        return feature
    return str(row['original_feature'].iloc[0])
fm, imp_multi, contrib_multi, X_multi, pred_multi, shap_signed_multi, shap_abs_multi = load_multimodal_shap_package()
print('Multimodal feature metadata:', fm.shape)
print('Multimodal SHAP importance:', imp_multi.shape)
print('Multimodal SHAP contribution:', contrib_multi.shape)
print('Multimodal scaled matrix:', X_multi.shape)

def plot_modality_contribution(contrib_df, out_dir):
    df = contrib_df.copy()
    df['modal'] = df['modal'].replace({'Metagenomics': 'Microbiota'})
    df['outcome'] = pd.Categorical(df['outcome'], categories=task_names, ordered=True)
    df = df.sort_values('outcome')
    modal_order = ['Radiomics', 'Blood', 'Urine', 'Microbiota']
    pivot = df.pivot_table(index='outcome', columns='modal', values='fraction', aggfunc='sum').reindex(task_names).fillna(0)
    for m in modal_order:
        if m not in pivot.columns:
            pivot[m] = 0
    pivot = pivot[modal_order]
    fig, ax = plt.subplots(figsize=(7.2, 2.8))
    y = np.arange(len(task_names))
    left = np.zeros(len(task_names))
    for modal in modal_order:
        values = pivot[modal].values
        color = MODALITY_COLORS.get(modal, '#999999')
        ax.barh(y, values, left=left, height=0.58, color=color, edgecolor='white', linewidth=1.0, label=modal)
        for i, v in enumerate(values):
            if v >= 0.075:
                ax.text(left[i] + v / 2, y[i], f'{v * 100:.0f}%', ha='center', va='center', fontsize=8.5, color='white', fontweight='bold')
        left += values
    ax.set_yticks(y)
    ax.set_yticklabels(task_names, fontsize=11, fontweight='bold')
    ax.set_xlim(0, 1)
    ax.set_xlabel('Fraction of total mean |SHAP|', fontsize=10)
    ax.xaxis.set_major_formatter(mpl.ticker.PercentFormatter(1.0))
    ax.set_title('Modality-level contribution in the final multimodal model', fontsize=12, fontweight='bold', pad=12)
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.25), ncol=4, frameon=False, fontsize=9)
    ax.invert_yaxis()
    ax.grid(axis='x', ls=':', lw=0.5, alpha=0.5)
    plt.tight_layout()
    out_png = os.path.join(out_dir, get_value('science.figures.plot_011', 'plot_011.png'))
    out_pdf = os.path.join(out_dir, get_value('science.figures.plot_012', 'plot_012.pdf'))
    plt.savefig(out_png, dpi=600, bbox_inches='tight')
    plt.savefig(out_pdf, dpi=600, bbox_inches='tight')
    plt.show()
    
plot_modality_contribution(contrib_multi, multi_fig_dir)

def build_multimodal_dot_data(selected_features, imp, fm, shap_signed, shap_abs):
    records = []
    imp_map = imp.set_index(['feature', 'outcome'])['mean_abs_shap'].to_dict()
    for feat in selected_features:
        modal = get_feature_modal(feat, fm)
        original_feature = get_original_feature(feat, fm)
        for task in task_names:
            if feat in shap_abs[task].columns:
                mean_abs = shap_abs[task][feat].mean()
                mean_signed = shap_signed[task][feat].mean()
                signed_ratio = mean_signed / (mean_abs + 1e-12)
            else:
                mean_abs = imp_map.get((feat, task), 0)
                signed_ratio = 0
            records.append({'feature': feat, 'feature_label': clean_feature_name(original_feature, max_len=42), 'modal': modal, 'outcome': task, 'mean_abs_shap': mean_abs, 'signed_ratio': signed_ratio})
    df = pd.DataFrame(records)
    max_abs = df['mean_abs_shap'].max()
    df['size_scaled'] = df['mean_abs_shap'] / (max_abs + 1e-12)
    return df

def plot_top_predictor_dot_heatmap(imp, fm, shap_signed, shap_abs, out_dir, top_per_outcome=8, max_total=24):
    selected_features = select_top_multimodal_features(imp=imp, fm=fm, top_per_outcome=top_per_outcome, max_total=max_total)
    df = build_multimodal_dot_data(selected_features=selected_features, imp=imp, fm=fm, shap_signed=shap_signed, shap_abs=shap_abs)
    total_df = df.groupby(['feature', 'feature_label', 'modal'])['mean_abs_shap'].sum().reset_index()
    modal_order_map = {m: i for i, m in enumerate(modalities)}
    total_df['modal_order'] = total_df['modal'].map(modal_order_map).fillna(99)
    total_df = total_df.sort_values(['modal_order', 'mean_abs_shap'], ascending=[True, False])
    features_order = total_df['feature'].tolist()
    labels_order = total_df['feature_label'].tolist()
    y_map = {f: i for i, f in enumerate(features_order[::-1])}
    x_map = {t: i for i, t in enumerate(task_names)}
    fig, ax = plt.subplots(figsize=(7.6, max(5.8, len(features_order) * 0.28)))
    for _, row in df.iterrows():
        x = x_map[row['outcome']]
        y = y_map[row['feature']]
        size = 35 + 500 * row['size_scaled']
        ax.scatter(x, y, s=size, c=row['signed_ratio'], cmap=DIVERGE_CMAP, norm=TwoSlopeNorm(vmin=-1, vcenter=0, vmax=1), edgecolor='black', linewidth=0.25, alpha=0.92)
    labels_reversed = []
    modalities_reversed = []
    order_reversed = features_order[::-1]
    for f in order_reversed:
        labels_reversed.append(clean_feature_name(get_original_feature(f, fm), max_len=44))
        modalities_reversed.append(get_feature_modal(f, fm))
    ax.set_yticks(range(len(order_reversed)))
    ax.set_yticklabels(labels_reversed, fontsize=8)
    ax.set_xticks(range(len(task_names)))
    ax.set_xticklabels(task_names, fontsize=11, fontweight='bold')
    ax.set_xlim(-0.75, len(task_names) - 0.35)
    ax.set_ylim(-0.8, len(order_reversed) - 0.2)
    ax.set_title('Top multimodal predictors across Response, PFS and OS', fontsize=12, fontweight='bold', pad=14)
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.grid(axis='x', ls=':', lw=0.5, alpha=0.4)
    for y, modal in enumerate(modalities_reversed):
        color = MODALITY_COLORS.get(modal, '#999999')
        ax.add_patch(Rectangle((-0.68, y - 0.38), 0.1, 0.76, color=color, clip_on=False, linewidth=0))
    sm = mpl.cm.ScalarMappable(cmap=DIVERGE_CMAP, norm=TwoSlopeNorm(vmin=-1, vcenter=0, vmax=1))
    cbar = plt.colorbar(sm, ax=ax, fraction=0.035, pad=0.02)
    cbar.set_label('Mean signed SHAP / mean |SHAP|', fontsize=8)
    cbar.ax.tick_params(labelsize=7)
    size_handles = []
    size_labels = []
    for v in [0.25, 0.5, 0.9]:
        size_handles.append(ax.scatter([], [], s=35 + 500 * v, c='lightgrey', edgecolor='black', linewidth=0.25))
        size_labels.append(f'{v:.2f}')
    legend1 = ax.legend(size_handles, size_labels, title='Relative\nmean |SHAP|', loc='lower right', bbox_to_anchor=(1.22, 0.0), frameon=False, fontsize=7, title_fontsize=8)
    ax.add_artist(legend1)
    modal_handles = [mpl.patches.Patch(color=MODALITY_COLORS.get(m, '#999999'), label='Microbiota' if m == 'Metagenomics' else m) for m in modalities]
    ax.legend(handles=modal_handles, loc='upper right', bbox_to_anchor=(1.25, 1.0), frameon=False, fontsize=8, title='Modality', title_fontsize=8)
    plt.tight_layout()
    out_png = os.path.join(out_dir, get_value('science.figures.plot_013', 'plot_013.png'))
    out_pdf = os.path.join(out_dir, get_value('science.figures.plot_014', 'plot_014.pdf'))
    plt.savefig(out_png, dpi=600, bbox_inches='tight')
    plt.savefig(out_pdf, dpi=600, bbox_inches='tight')
    plt.show()
    
    dot_data_path = os.path.join(out_dir, get_value('science.tables.result_012', 'result_012.csv'))
    df.to_csv(dot_data_path, index=False, encoding='utf-8-sig')
    return (selected_features, df)
selected_multi_features, dot_data = plot_top_predictor_dot_heatmap(imp=imp_multi, fm=fm, shap_signed=shap_signed_multi, shap_abs=shap_abs_multi, out_dir=multi_fig_dir, top_per_outcome=8, max_total=24)

def draw_curve(ax, x0, y0, x1, y1, color, lw, alpha=0.35, zorder=1):
    verts = [(x0, y0), (x0 + 0.18, y0), (x1 - 0.18, y1), (x1, y1)]
    codes = [Path.MOVETO, Path.CURVE4, Path.CURVE4, Path.CURVE4]
    path = Path(verts, codes)
    patch = PathPatch(path, facecolor='none', edgecolor=color, lw=lw, alpha=alpha, capstyle='round', joinstyle='round', zorder=zorder)
    ax.add_patch(patch)

def plot_alluvial_like_flow(selected_features, fm, shap_abs, out_dir, max_features=18):
    records = []
    for feat in selected_features:
        modal = get_feature_modal(feat, fm)
        original = get_original_feature(feat, fm)
        total = 0
        task_weights = {}
        for task in task_names:
            if feat in shap_abs[task].columns:
                w = shap_abs[task][feat].mean()
            else:
                w = 0
            task_weights[task] = w
            total += w
        records.append({'feature': feat, 'original_feature': original, 'modal': modal, 'total': total, **task_weights})
    feat_df = pd.DataFrame(records)
    modal_order_map = {m: i for i, m in enumerate(modalities)}
    feat_df['modal_order'] = feat_df['modal'].map(modal_order_map).fillna(99)
    feat_df = feat_df.sort_values(['modal_order', 'total'], ascending=[True, False]).head(max_features).reset_index(drop=True)
    max_weight = max(feat_df[task_names].values.max(), feat_df['total'].max())

    def lw_scale(w):
        return 0.7 + 8.0 * (w / (max_weight + 1e-12))
    modal_labels = modalities
    endpoint_labels = task_names
    modal_y = {m: 1 - (i + 1) / (len(modal_labels) + 1) for i, m in enumerate(modal_labels)}
    endpoint_y = {t: 1 - (i + 1) / (len(endpoint_labels) + 1) for i, t in enumerate(endpoint_labels)}
    n_feat = feat_df.shape[0]
    feature_y = {row['feature']: 1 - (i + 1) / (n_feat + 1) for i, row in feat_df.iterrows()}
    fig, ax = plt.subplots(figsize=(10.5, max(5.5, n_feat * 0.32)))
    ax.set_xlim(-0.03, 1.03)
    ax.set_ylim(0, 1)
    ax.axis('off')
    for _, row in feat_df.iterrows():
        feat = row['feature']
        modal = row['modal']
        color = MODALITY_COLORS.get(modal, '#999999')
        y_m = modal_y.get(modal, 0.5)
        y_f = feature_y[feat]
        draw_curve(ax, 0.1, y_m, 0.48, y_f, color=color, lw=lw_scale(row['total']), alpha=0.28, zorder=1)
        for task in task_names:
            w = row[task]
            if w <= 0:
                continue
            draw_curve(ax, 0.52, y_f, 0.9, endpoint_y[task], color=color, lw=lw_scale(w), alpha=0.38, zorder=2)
    for modal in modal_labels:
        label = 'Microbiota' if modal == 'Metagenomics' else modal
        color = MODALITY_COLORS.get(modal, '#999999')
        y = modal_y[modal]
        ax.scatter(0.08, y, s=520, color=color, edgecolor='white', linewidth=1.0, zorder=5)
        ax.text(0.0, y, label, ha='right', va='center', fontsize=10, fontweight='bold', color='#333333')
    for _, row in feat_df.iterrows():
        feat = row['feature']
        modal = row['modal']
        y = feature_y[feat]
        color = MODALITY_COLORS.get(modal, '#999999')
        ax.scatter(0.5, y, s=120, color=color, edgecolor='white', linewidth=0.8, zorder=6)
        ax.text(0.505, y, clean_feature_name(row['original_feature'], max_len=42), ha='left', va='center', fontsize=7.7, color='#222222')
    for task in endpoint_labels:
        y = endpoint_y[task]
        color = OUTCOME_COLORS[task]
        ax.scatter(0.92, y, s=600, color=color, edgecolor='white', linewidth=1.0, zorder=5)
        ax.text(0.99, y, task, ha='left', va='center', fontsize=11, fontweight='bold', color='#333333')
    ax.text(0.08, 1.02, 'Modality', ha='center', va='bottom', fontsize=12, fontweight='bold')
    ax.text(0.5, 1.02, 'Top multimodal predictors', ha='center', va='bottom', fontsize=12, fontweight='bold')
    ax.text(0.92, 1.02, 'Endpoint', ha='center', va='bottom', fontsize=12, fontweight='bold')
    ax.set_title('Modality-to-feature-to-endpoint SHAP flow in the final multimodal model', fontsize=13, fontweight='bold', pad=20)
    plt.tight_layout()
    out_png = os.path.join(out_dir, get_value('science.figures.plot_015', 'plot_015.png'))
    out_pdf = os.path.join(out_dir, get_value('science.figures.plot_016', 'plot_016.pdf'))
    plt.savefig(out_png, dpi=600, bbox_inches='tight')
    plt.savefig(out_pdf, dpi=600, bbox_inches='tight')
    plt.show()
    source_path = os.path.join(out_dir, get_value('science.tables.result_013', 'result_013.csv'))
    feat_df.to_csv(source_path, index=False, encoding='utf-8-sig')
    
    return feat_df
flow_source_df = plot_alluvial_like_flow(selected_features=selected_multi_features, fm=fm, shap_abs=shap_abs_multi, out_dir=multi_fig_dir, max_features=18)

def select_representative_dependence_features(imp, fm, max_panels=6):
    selected_pairs = []
    used_features = set()
    used_modals = []
    for task in task_names:
        sub = imp[imp['outcome'] == task].sort_values('mean_abs_shap', ascending=False)
        for _, row in sub.iterrows():
            feat = str(row['feature'])
            if feat in used_features:
                continue
            modal = get_feature_modal(feat, fm)
            selected_pairs.append({'feature': feat, 'outcome': task, 'modal': modal, 'original_feature': get_original_feature(feat, fm), 'mean_abs_shap': row['mean_abs_shap']})
            used_features.add(feat)
            used_modals.append(modal)
            break
    sub_all = imp.sort_values('mean_abs_shap', ascending=False)
    for _, row in sub_all.iterrows():
        if len(selected_pairs) >= max_panels:
            break
        feat = str(row['feature'])
        if feat in used_features:
            continue
        selected_pairs.append({'feature': feat, 'outcome': row['outcome'], 'modal': get_feature_modal(feat, fm), 'original_feature': get_original_feature(feat, fm), 'mean_abs_shap': row['mean_abs_shap']})
        used_features.add(feat)
    return pd.DataFrame(selected_pairs)

def plot_dependence_panels(X_scaled, pred, shap_signed, imp, fm, out_dir, max_panels=6):
    dep_df = select_representative_dependence_features(imp=imp, fm=fm, max_panels=max_panels)
    n = dep_df.shape[0]
    ncols = 3
    nrows = math.ceil(n / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(12.0, 3.35 * nrows), squeeze=False)
    pred_col_map = {'Response': 'Response_pred', 'PFS': 'PFS_risk', 'OS': 'OS_risk'}
    for i, (_, row) in enumerate(dep_df.iterrows()):
        ax = axes[i // ncols][i % ncols]
        feat = row['feature']
        task = row['outcome']
        modal = row['modal']
        original = row['original_feature']
        x = X_scaled[feat].values
        y = shap_signed[task][feat].values
        c = pred[pred_col_map[task]].values
        c_scaled = minmax_scale_series(c)
        sc = ax.scatter(x, y, c=c_scaled, cmap='viridis', s=26, alpha=0.82, edgecolor='white', linewidth=0.25)
        valid = np.isfinite(x) & np.isfinite(y)
        if valid.sum() >= 5:
            coef = np.polyfit(x[valid], y[valid], deg=1)
            xx = np.linspace(np.nanmin(x), np.nanmax(x), 100)
            yy = coef[0] * xx + coef[1]
            ax.plot(xx, yy, color='#222222', lw=1.2, alpha=0.8)
        ax.axhline(0, color='#888888', lw=0.8, ls='--')
        ax.set_title(f'{task} | {modal}', fontsize=10.5, fontweight='bold', color=OUTCOME_COLORS.get(task, '#333333'))
        ax.set_xlabel(clean_feature_name(original, max_len=34), fontsize=8.5)
        if task == 'Response':
            ylabel = 'SHAP value\n(higher response probability)'
        else:
            ylabel = 'SHAP value\n(higher predicted risk)'
        ax.set_ylabel(ylabel, fontsize=8.5)
        ax.tick_params(labelsize=7.5)
        ax.grid(ls=':', lw=0.4, alpha=0.4)
    for j in range(n, nrows * ncols):
        axes[j // ncols][j % ncols].axis('off')
    cbar = fig.colorbar(sc, ax=axes.ravel().tolist(), shrink=0.78, pad=0.01)
    cbar.set_label('Scaled model prediction', fontsize=9)
    cbar.ax.tick_params(labelsize=8)
    fig.suptitle('Representative SHAP dependence patterns in the final multimodal model', fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout()
    out_png = os.path.join(out_dir, get_value('science.figures.plot_017', 'plot_017.png'))
    out_pdf = os.path.join(out_dir, get_value('science.figures.plot_018', 'plot_018.pdf'))
    plt.savefig(out_png, dpi=600, bbox_inches='tight')
    plt.savefig(out_pdf, dpi=600, bbox_inches='tight')
    plt.show()
    dep_df.to_csv(os.path.join(out_dir, get_value('science.tables.attribution_009', 'attribution_009.csv')), index=False, encoding='utf-8-sig')
    
plot_dependence_panels(X_scaled=X_multi, pred=pred_multi, shap_signed=shap_signed_multi, imp=imp_multi, fm=fm, out_dir=multi_fig_dir, max_panels=6)

def plot_patient_level_shap_heatmap(task, imp, fm, shap_signed, pred, out_dir, top_n=18):
    sub = imp[imp['outcome'] == task].sort_values('mean_abs_shap', ascending=False).head(top_n)
    features = sub['feature'].astype(str).tolist()
    features = [f for f in features if f in shap_signed[task].columns]
    pred_col_map = {'Response': 'Response_pred', 'PFS': 'PFS_risk', 'OS': 'OS_risk'}
    true_col_map = {'Response': 'Response_true', 'PFS': 'PFS_status', 'OS': 'OS_status'}
    samples_order = pred.sort_values(pred_col_map[task], ascending=True).index.tolist()
    M = shap_signed[task].loc[samples_order, features].T
    M_norm = M.copy()
    for idx in M_norm.index:
        max_abs = np.nanmax(np.abs(M_norm.loc[idx].values))
        if max_abs > 0:
            M_norm.loc[idx] = M_norm.loc[idx] / max_abs
    M_norm = M_norm.clip(-1, 1)
    feature_labels = [clean_feature_name(get_original_feature(f, fm), max_len=42) for f in features]
    feature_modals = [get_feature_modal(f, fm) for f in features]
    n_features = len(features)
    n_samples = len(samples_order)
    fig = plt.figure(figsize=(11.5, max(5.2, n_features * 0.32)))
    gs = GridSpec(nrows=4, ncols=3, height_ratios=[0.25, 0.25, 0.12, 5.0], width_ratios=[0.18, 0.2, 6.5], hspace=0.06, wspace=0.04)
    ax_true = fig.add_subplot(gs[0, 2])
    ax_pred = fig.add_subplot(gs[1, 2])
    ax_gap = fig.add_subplot(gs[2, 2])
    ax_heat = fig.add_subplot(gs[3, 2])
    ax_modal = fig.add_subplot(gs[3, 1])
    true_vals = pred.loc[samples_order, true_col_map[task]].values.astype(float).reshape(1, -1)
    ax_true.imshow(true_vals, aspect='auto', cmap='Greys', interpolation='nearest', vmin=0, vmax=1)
    ax_true.set_yticks([0])
    ax_true.set_yticklabels(['Observed'], fontsize=7)
    ax_true.set_xticks([])
    ax_true.tick_params(length=0)
    pred_vals = pred.loc[samples_order, pred_col_map[task]].values.astype(float)
    pred_vals = minmax_scale_series(pred_vals).reshape(1, -1)
    ax_pred.imshow(pred_vals, aspect='auto', cmap='viridis', interpolation='nearest', vmin=0, vmax=1)
    ax_pred.set_yticks([0])
    ax_pred.set_yticklabels(['Predicted'], fontsize=7)
    ax_pred.set_xticks([])
    ax_pred.tick_params(length=0)
    ax_gap.axis('off')
    im = ax_heat.imshow(M_norm.values, aspect='auto', cmap=DIVERGE_CMAP, interpolation='nearest', vmin=-1, vmax=1)
    ax_heat.set_yticks(np.arange(n_features))
    ax_heat.set_yticklabels(feature_labels, fontsize=7.4)
    ax_heat.set_xticks([])
    ax_heat.set_xlabel(f'Patients ordered by {task} predicted score', fontsize=9)
    ax_heat.set_title(f'{task}: patient-level signed SHAP patterns', fontsize=12, fontweight='bold', pad=10, color=OUTCOME_COLORS.get(task, '#333333'))
    modal_rgb = []
    for modal in feature_modals:
        color = MODALITY_COLORS.get(modal, '#999999')
        modal_rgb.append(mpl.colors.to_rgb(color))
    modal_rgb = np.array(modal_rgb).reshape(n_features, 1, 3)
    ax_modal.imshow(modal_rgb, aspect='auto', interpolation='nearest')
    ax_modal.set_xticks([])
    ax_modal.set_yticks([])
    ax_modal.set_ylabel('Modality', fontsize=8)
    group_df = pd.DataFrame({'modal': feature_modals, 'y': np.arange(n_features)})
    for modal, g in group_df.groupby('modal'):
        y_center = g['y'].mean()
        label = 'Microbiota' if modal == 'Metagenomics' else modal
        ax_modal.text(-0.45, y_center, label, ha='right', va='center', fontsize=7, rotation=0, color=MODALITY_COLORS.get(modal, '#333333'), fontweight='bold', transform=ax_modal.transData)
    cax = fig.add_axes([0.91, 0.18, 0.012, 0.55])
    cbar = fig.colorbar(im, cax=cax)
    cbar.set_label('Row-scaled signed SHAP', fontsize=8)
    cbar.ax.tick_params(labelsize=7)
    for ax in [ax_true, ax_pred, ax_heat, ax_modal]:
        for spine in ax.spines.values():
            spine.set_visible(False)
    out_png = os.path.join(out_dir, f'Fig6E_Patient_level_SHAP_heatmap_{task}.png')
    out_pdf = os.path.join(out_dir, f'Fig6E_Patient_level_SHAP_heatmap_{task}.pdf')
    plt.savefig(out_png, dpi=600, bbox_inches='tight')
    plt.savefig(out_pdf, dpi=600, bbox_inches='tight')
    plt.show()
    
for task in task_names:
    plot_patient_level_shap_heatmap(task=task, imp=imp_multi, fm=fm, shap_signed=shap_signed_multi, pred=pred_multi, out_dir=multi_fig_dir, top_n=18)

def load_single_modal_final_shap(modal_name):
    model_dir = os.path.join(stage3_dir, modal_name)
    imp_path = os.path.join(model_dir, f"{modal_name}{get_value('science.tables.attribution_005', 'attribution_005.csv')}")
    imp = pd.read_csv(imp_path, encoding='utf-8-sig')
    shap_signed = {}
    shap_abs = {}
    for task in task_names:
        signed_path = os.path.join(model_dir, f'{modal_name}_final_shap_signed_{task}.csv.gz')
        abs_path = os.path.join(model_dir, f'{modal_name}_final_shap_abs_{task}.csv.gz')
        shap_signed[task] = read_index_csv(signed_path, compression='gzip')
        shap_abs[task] = read_index_csv(abs_path, compression='gzip')
    return (imp, shap_signed, shap_abs)

def build_reduced_single_modal_atlas_data(top_per_outcome=5):
    records = []
    for modal in ['Metagenomics', 'Radiomics', 'Blood', 'Urine']:
        imp, shap_signed, shap_abs = load_single_modal_final_shap(modal)
        selected = []
        for task in task_names:
            sub = imp[imp['outcome'] == task].sort_values('mean_abs_shap', ascending=False).head(top_per_outcome)
            selected.extend(sub['feature'].astype(str).tolist())
        selected = list(dict.fromkeys(selected))
        for feat in selected:
            for task in task_names:
                if feat in shap_abs[task].columns:
                    mean_abs = shap_abs[task][feat].mean()
                    mean_signed = shap_signed[task][feat].mean()
                    signed_ratio = mean_signed / (mean_abs + 1e-12)
                else:
                    mean_abs = 0
                    signed_ratio = 0
                records.append({'modal': modal, 'feature': feat, 'feature_label': clean_feature_name(feat, max_len=36), 'outcome': task, 'mean_abs_shap': mean_abs, 'signed_ratio': signed_ratio})
    df = pd.DataFrame(records)
    max_abs = df['mean_abs_shap'].max()
    df['size_scaled'] = df['mean_abs_shap'] / (max_abs + 1e-12)
    return df

def plot_reduced_single_modal_atlas(out_dir, top_per_outcome=5):
    df = build_reduced_single_modal_atlas_data(top_per_outcome=top_per_outcome)
    modal_order = ['Metagenomics', 'Radiomics', 'Blood', 'Urine']
    modal_order_map = {m: i for i, m in enumerate(modal_order)}
    total_df = df.groupby(['modal', 'feature', 'feature_label'])['mean_abs_shap'].sum().reset_index()
    total_df['modal_order'] = total_df['modal'].map(modal_order_map)
    total_df = total_df.sort_values(['modal_order', 'mean_abs_shap'], ascending=[True, False])
    row_keys = [f'{row.modal}||{row.feature}' for _, row in total_df.iterrows()]
    y_map = {k: i for i, k in enumerate(row_keys[::-1])}
    x_map = {t: i for i, t in enumerate(task_names)}
    fig_height = max(7.0, len(row_keys) * 0.22)
    fig, ax = plt.subplots(figsize=(7.8, fig_height))
    for _, row in df.iterrows():
        key = f"{row['modal']}||{row['feature']}"
        if key not in y_map:
            continue
        x = x_map[row['outcome']]
        y = y_map[key]
        ax.scatter(x, y, s=25 + 420 * row['size_scaled'], c=row['signed_ratio'], cmap=DIVERGE_CMAP, norm=TwoSlopeNorm(vmin=-1, vcenter=0, vmax=1), edgecolor='black', linewidth=0.22, alpha=0.9)
    reversed_keys = row_keys[::-1]
    ylabels = []
    ymodals = []
    for key in reversed_keys:
        modal, feat = key.split('||', 1)
        ylabels.append(clean_feature_name(feat, max_len=38))
        ymodals.append(modal)
    ax.set_yticks(np.arange(len(reversed_keys)))
    ax.set_yticklabels(ylabels, fontsize=7)
    ax.set_xticks(np.arange(len(task_names)))
    ax.set_xticklabels(task_names, fontsize=11, fontweight='bold')
    ax.set_xlim(-0.72, len(task_names) - 0.35)
    ax.set_ylim(-0.8, len(reversed_keys) - 0.2)
    for y, modal in enumerate(ymodals):
        color = MODALITY_COLORS.get(modal, '#999999')
        ax.add_patch(Rectangle((-0.66, y - 0.38), 0.1, 0.76, color=color, clip_on=False, linewidth=0))
    group_df = pd.DataFrame({'modal': ymodals, 'y': np.arange(len(ymodals))})
    for modal, g in group_df.groupby('modal'):
        y_center = g['y'].mean()
        label = 'Microbiota' if modal == 'Metagenomics' else modal
        ax.text(-0.9, y_center, label, ha='right', va='center', fontsize=8.2, fontweight='bold', color=MODALITY_COLORS.get(modal, '#333333'))
    ax.grid(axis='x', ls=':', lw=0.5, alpha=0.35)
    ax.set_title('Reduced single-modal SHAP atlas after feature purification', fontsize=12.5, fontweight='bold', pad=14)
    ax.set_xlabel('')
    ax.set_ylabel('')
    sm = mpl.cm.ScalarMappable(cmap=DIVERGE_CMAP, norm=TwoSlopeNorm(vmin=-1, vcenter=0, vmax=1))
    cbar = plt.colorbar(sm, ax=ax, fraction=0.035, pad=0.02)
    cbar.set_label('Mean signed SHAP / mean |SHAP|', fontsize=8)
    cbar.ax.tick_params(labelsize=7)
    size_handles = []
    size_labels = []
    for v in [0.25, 0.5, 0.9]:
        size_handles.append(ax.scatter([], [], s=25 + 420 * v, c='lightgrey', edgecolor='black', linewidth=0.25))
        size_labels.append(f'{v:.2f}')
    ax.legend(size_handles, size_labels, title='Relative\nmean |SHAP|', loc='lower right', bbox_to_anchor=(1.24, 0.02), frameon=False, fontsize=7, title_fontsize=8)
    plt.tight_layout()
    out_png = os.path.join(out_dir, get_value('science.figures.plot_019', 'plot_019.png'))
    out_pdf = os.path.join(out_dir, get_value('science.figures.plot_020', 'plot_020.pdf'))
    plt.savefig(out_png, dpi=600, bbox_inches='tight')
    plt.savefig(out_pdf, dpi=600, bbox_inches='tight')
    plt.show()
    df.to_csv(os.path.join(out_dir, get_value('science.tables.attribution_010', 'attribution_010.csv')), index=False, encoding='utf-8-sig')
    
plot_reduced_single_modal_atlas(out_dir=single_modal_atlas_dir, top_per_outcome=5)




In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency
from sklearn.metrics import roc_auc_score, average_precision_score
from lifelines import KaplanMeierFitter
from lifelines.statistics import multivariate_logrank_test
stage2_out_dir = str(project_path(get_value('science.paths.directory_019', 'directory_019')))
label_path = str(project_path(get_value('science.tables.outcomes_004', 'outcomes_004.csv')))
clinical_baseline_path = str(project_path(get_value('science.tables.clinical_001', 'clinical_001.xlsx')))
multimodal_pred_path = os.path.join(stage2_out_dir, 'Multimodal', get_value('science.tables.predictions_009', 'predictions_009.csv'))
single_model_pred_paths = {'Metagenomics': os.path.join(stage2_out_dir, 'Metagenomics', get_value('science.tables.predictions_010', 'predictions_010.csv')), 'Radiomics': os.path.join(stage2_out_dir, 'Radiomics', get_value('science.tables.predictions_011', 'predictions_011.csv')), 'Blood': os.path.join(stage2_out_dir, 'Blood', get_value('science.tables.predictions_012', 'predictions_012.csv')), 'Urine': os.path.join(stage2_out_dir, 'Urine', get_value('science.tables.predictions_013', 'predictions_013.csv')), 'Multimodal': multimodal_pred_path}
clinical_pred_path = None
fig4_out_dir = os.path.join(stage2_out_dir, get_value('science.paths.directory_020', 'directory_020'))
os.makedirs(fig4_out_dir, exist_ok=True)




In [ ]:
def normalize_sample_id(x):
    return str(x).lower().strip()

def read_prediction_file(path, model_name=None):
    df = pd.read_csv(path, encoding='utf-8-sig')
    if 'sample' not in df.columns:
        raise ValueError(f"'sample' column not found in {path}")
    df['sample'] = df['sample'].map(normalize_sample_id)
    if model_name is not None:
        df['model'] = model_name
    required_cols = ['sample', 'Response_true', 'Response_pred', 'PFS_time', 'PFS_status', 'PFS_risk', 'OS_time', 'OS_status', 'OS_risk']
    missing = [c for c in required_cols if c not in df.columns]
    if len(missing) > 0:
        raise ValueError(f'Missing columns in {path}: {missing}')
    return df

def aggregate_cv_predictions(pred_df):
    pass
    agg = pred_df.groupby('sample').agg({'Response_true': 'first', 'Response_pred': 'mean', 'PFS_time': 'first', 'PFS_status': 'first', 'PFS_risk': 'mean', 'OS_time': 'first', 'OS_status': 'first', 'OS_risk': 'mean'}).reset_index()
    return agg
multi_pred_raw = read_prediction_file(multimodal_pred_path, model_name='Multimodal')
multi_patient = aggregate_cv_predictions(multi_pred_raw)
print('Multimodal patient-level prediction:')
print(multi_patient.shape)
multi_patient.head()


In [ ]:
WEIGHT_RESPONSE = 0.25
WEIGHT_PFS = 0.35
WEIGHT_OS = 0.4

def percentile_rank(s):
    pass
    return s.rank(method='average', pct=True)
mbp_df = multi_patient.copy()
mbp_df['response_benefit'] = percentile_rank(mbp_df['Response_pred'])
mbp_df['pfs_benefit'] = 1 - percentile_rank(mbp_df['PFS_risk'])
mbp_df['os_benefit'] = 1 - percentile_rank(mbp_df['OS_risk'])
mbp_df['multimodal_benefit_score'] = WEIGHT_RESPONSE * mbp_df['response_benefit'] + WEIGHT_PFS * mbp_df['pfs_benefit'] + WEIGHT_OS * mbp_df['os_benefit']
mbp_df['MBP_raw'] = pd.qcut(mbp_df['multimodal_benefit_score'].rank(method='first'), q=3, labels=['MBP-III', 'MBP-II', 'MBP-I'])
mbp_display_map = {'MBP-I': 'MBP-I\nHigh-benefit', 'MBP-II': 'MBP-II\nIntermediate', 'MBP-III': 'MBP-III\nLow-benefit'}
mbp_df['MBP'] = mbp_df['MBP_raw'].map(mbp_display_map)
mbp_df = mbp_df.sort_values('multimodal_benefit_score', ascending=False).reset_index(drop=True)
mbp_path = os.path.join(fig4_out_dir, get_value('science.tables.result_014', 'result_014.csv'))
mbp_df.to_csv(mbp_path, index=False, encoding='utf-8-sig')


mbp_df[['sample', 'Response_pred', 'PFS_risk', 'OS_risk', 'response_benefit', 'pfs_benefit', 'os_benefit', 'multimodal_benefit_score', 'MBP']].head()


In [ ]:
def read_clinical_baseline(path):
    if path is None or not os.path.exists(path):
        print('Clinical baseline file not found. Figure 4B will use prediction/outcome rows only.')
        return None
    if path.endswith('.xlsx') or path.endswith('.xls'):
        clin = pd.read_excel(path)
    else:
        clin = pd.read_csv(path, encoding='utf-8-sig')
    sample_candidates = ['sample', 'sample_id', 'Sample', 'ID']
    sample_col = None
    for c in sample_candidates:
        if c in clin.columns:
            sample_col = c
            break
    if sample_col is None:
        sample_col = clin.columns[0]
    clin = clin.rename(columns={sample_col: 'sample'})
    clin['sample'] = clin['sample'].map(normalize_sample_id)
    clin = clin.drop_duplicates(subset='sample', keep='first')
    clin = clin.set_index('sample')
    return clin

def to_numeric_safe(s):
    return pd.to_numeric(s, errors='coerce')

def to_binary_safe(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().lower()
    if s in ['1', 'yes', 'y', 'true', 'positive', 'present', 'pvtt', 'response', 'responder', 'r', 'cr', 'pr']:
        return 1
    if s in ['0', 'no', 'n', 'false', 'negative', 'absent', 'none', 'non-response', 'nonresponder', 'nr', 'sd', 'pd']:
        return 0
    try:
        return float(s)
    except:
        return np.nan
clinical = read_clinical_baseline(clinical_baseline_path)
selected_clinical = None
if clinical is not None:
    clinical_col_map = {'AFP': ['AFP', 'afp'], 'Tumor size': ['Size', 'Tumor_size', 'tumor_size', 'max_diameter', 'Diameter'], 'PVTT': ['PVTT', 'pvtt'], 'ALBI': ['ALBI', 'albi'], 'Albumin': ['ALB', 'Albumin', 'albumin'], 'TBIL': ['TBiL', 'TBIL', 'tbil', 'Bilirubin'], 'ALT': ['ALT', 'alt'], 'AST': ['AST', 'ast']}
    selected_clinical = pd.DataFrame(index=clinical.index)
    for display_name, candidates in clinical_col_map.items():
        found = None
        for c in candidates:
            if c in clinical.columns:
                found = c
                break
        if found is not None:
            if display_name == 'PVTT':
                selected_clinical[display_name] = clinical[found].apply(to_binary_safe)
            else:
                selected_clinical[display_name] = to_numeric_safe(clinical[found])
    selected_clinical = selected_clinical.dropna(axis=1, how='all')
    print('Selected clinical variables:')
    print(selected_clinical.columns.tolist())


In [ ]:
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec

def minmax_scale(s):
    s = pd.to_numeric(s, errors='coerce')
    if s.max() == s.min():
        return pd.Series(np.zeros(len(s)), index=s.index)
    return (s - s.min()) / (s.max() - s.min())

def percentile_rank(s):
    pass
    return pd.Series(s).rank(method='average', pct=True)

def safe_mode_fill(s, default=0):
    mode_val = s.mode(dropna=True)
    if len(mode_val) > 0:
        return s.fillna(mode_val.iloc[0])
    else:
        return s.fillna(default)
heat_df = mbp_df.set_index('sample').copy()
heat_rows = pd.DataFrame(index=heat_df.index)
heat_rows['Response probability'] = percentile_rank(heat_df['Response_pred'])
heat_rows['PFS risk percentile'] = percentile_rank(heat_df['PFS_risk'])
heat_rows['OS risk percentile'] = percentile_rank(heat_df['OS_risk'])
heat_rows['Observed response'] = heat_df['Response_true'].astype(float)
heat_rows['PFS event'] = heat_df['PFS_status'].astype(float)
heat_rows['OS event'] = heat_df['OS_status'].astype(float)
if selected_clinical is not None and selected_clinical.shape[1] > 0:
    clin_aligned = selected_clinical.reindex(heat_rows.index)
    for col in clin_aligned.columns:
        if col == 'PVTT':
            heat_rows[col] = safe_mode_fill(clin_aligned[col], default=0).astype(float)
        else:
            x = clin_aligned[col].copy()
            x = pd.to_numeric(x, errors='coerce')
            x = x.fillna(x.median())
            if col in ['AFP', 'Tumor size', 'TBIL', 'ALT', 'AST']:
                if x.min() >= 0:
                    x = np.log1p(x)
            heat_rows[col] = percentile_rank(x)
heat_mat = heat_rows.T
mbp_ordered = heat_df['MBP_raw'].values
mbp_color_map = {'MBP-I': '#2C7BB6', 'MBP-II': '#FDB863', 'MBP-III': '#D7191C'}
mbp_colors = [mbp_color_map[x] for x in mbp_ordered]
boundaries = []
for i in range(1, len(mbp_ordered)):
    if mbp_ordered[i] != mbp_ordered[i - 1]:
        boundaries.append(i - 0.5)
group_centers = []
group_labels = []
start = 0
for i in range(1, len(mbp_ordered) + 1):
    if i == len(mbp_ordered) or mbp_ordered[i] != mbp_ordered[start]:
        end = i - 1
        group_centers.append((start + end) / 2)
        group_labels.append(mbp_display_map[mbp_ordered[start]].replace('\n', ' '))
        start = i
fig = plt.figure(figsize=(15, 6.3))
gs = GridSpec(2, 1, height_ratios=[0.22, 1.0], hspace=0.04)
ax_anno = fig.add_subplot(gs[0, 0])
ax_heat = fig.add_subplot(gs[1, 0])
for i, color in enumerate(mbp_colors):
    ax_anno.add_patch(mpatches.Rectangle((i - 0.5, 0), 1, 1, facecolor=color, edgecolor=color, linewidth=0))
ax_anno.set_xlim(-0.5, len(mbp_colors) - 0.5)
ax_anno.set_ylim(0, 1)
ax_anno.set_xticks([])
ax_anno.set_yticks([])
for spine in ax_anno.spines.values():
    spine.set_visible(False)
for b in boundaries:
    ax_anno.axvline(b, color='black', linewidth=1.1)
for center, label in zip(group_centers, group_labels):
    ax_anno.text(center, 1.2, label, ha='center', va='bottom', fontsize=10, fontweight='bold')
ax_anno.text(-1.0, 0.5, 'MBP', ha='right', va='center', fontsize=10, fontweight='bold')
im = ax_heat.imshow(heat_mat.values, aspect='auto', interpolation='nearest', cmap='RdBu_r', vmin=0, vmax=1)
ax_heat.set_yticks(np.arange(heat_mat.shape[0]))
ax_heat.set_yticklabels(heat_mat.index, fontsize=9)
ax_heat.set_xticks([])
ax_heat.set_xlabel('Patients ordered by multimodal benefit score', fontsize=11)
for b in boundaries:
    ax_heat.axvline(b, color='black', linewidth=1.1)
cbar = plt.colorbar(im, ax=ax_heat, fraction=0.022, pad=0.015)
cbar.set_label('Percentile / scaled value', fontsize=9)
legend_patches = [mpatches.Patch(color=mbp_color_map['MBP-I'], label='MBP-I High-benefit'), mpatches.Patch(color=mbp_color_map['MBP-II'], label='MBP-II Intermediate'), mpatches.Patch(color=mbp_color_map['MBP-III'], label='MBP-III Low-benefit')]
ax_anno.legend(handles=legend_patches, loc='upper right', bbox_to_anchor=(1.0, 1.95), ncol=3, frameon=False, fontsize=9)
ax_anno.set_title('Figure 4B. Patient-level multimodal benefit stratification', fontsize=13, pad=30)
plt.tight_layout()
fig4b_path = os.path.join(fig4_out_dir, get_value('science.figures.plot_021', 'plot_021.png'))
plt.savefig(fig4b_path, dpi=600, bbox_inches='tight')
plt.show()



In [ ]:
def plot_km_by_mbp(df, time_col, event_col, title, filename):
    kmf = KaplanMeierFitter()
    fig, ax = plt.subplots(figsize=(5.8, 4.8))
    order = ['MBP-I\nHigh-benefit', 'MBP-II\nIntermediate', 'MBP-III\nLow-benefit']
    for group in order:
        sub = df[df['MBP'] == group].copy()
        if sub.shape[0] == 0:
            continue
        kmf.fit(durations=sub[time_col], event_observed=sub[event_col], label=f"{group.replace(chr(10), ' ')} (n={sub.shape[0]})")
        kmf.plot_survival_function(ax=ax, ci_show=True, linewidth=2)
    lr = multivariate_logrank_test(event_durations=df[time_col], groups=df['MBP'], event_observed=df[event_col])
    p_value = lr.p_value
    ax.text(0.05, 0.08, f'Log-rank P = {p_value:.3g}', transform=ax.transAxes, fontsize=10, bbox=dict(boxstyle='round', facecolor='white', alpha=0.8, edgecolor='gray'))
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('Time')
    ax.set_ylabel('Survival probability')
    ax.grid(alpha=0.25)
    ax.legend(fontsize=8, frameon=False)
    plt.tight_layout()
    path = os.path.join(fig4_out_dir, filename)
    plt.savefig(path, dpi=600, bbox_inches='tight')
    plt.show()
    
    return p_value
p_pfs = plot_km_by_mbp(mbp_df, time_col='PFS_time', event_col='PFS_status', title='Figure 4C. PFS by Multimodal Benefit Phenotype', filename=get_value('science.figures.plot_022', 'plot_022.png'))
p_os = plot_km_by_mbp(mbp_df, time_col='OS_time', event_col='OS_status', title='Figure 4D. OS by Multimodal Benefit Phenotype', filename=get_value('science.figures.plot_023', 'plot_023.png'))
print('PFS log-rank P:', p_pfs)
print('OS log-rank P:', p_os)


In [ ]:
group_order = ['MBP-I\nHigh-benefit', 'MBP-II\nIntermediate', 'MBP-III\nLow-benefit']
event_summary = []
for group in group_order:
    sub = mbp_df[mbp_df['MBP'] == group].copy()
    event_summary.append({'MBP': group, 'n': sub.shape[0], 'Response rate': sub['Response_true'].mean(), 'PFS event-free rate': 1 - sub['PFS_status'].mean(), 'OS event-free rate': 1 - sub['OS_status'].mean()})
event_summary_df = pd.DataFrame(event_summary)
event_summary_path = os.path.join(fig4_out_dir, get_value('science.tables.result_015', 'result_015.csv'))
event_summary_df.to_csv(event_summary_path, index=False, encoding='utf-8-sig')


def chi_square_by_group(df, outcome_col):
    tab = pd.crosstab(df['MBP'], df[outcome_col])
    chi2, p, dof, expected = chi2_contingency(tab)
    return (p, tab)
p_resp, tab_resp = chi_square_by_group(mbp_df, 'Response_true')
p_pfs_event, tab_pfs = chi_square_by_group(mbp_df, 'PFS_status')
p_os_event, tab_os = chi_square_by_group(mbp_df, 'OS_status')
print('Chi-square P values:')
print('Response:', p_resp)
print('PFS event:', p_pfs_event)
print('OS event:', p_os_event)
fig, ax = plt.subplots(figsize=(7.4, 4.8))
x = np.arange(len(group_order))
width = 0.23
outcomes = ['Response rate', 'PFS event-free rate', 'OS event-free rate']
for i, outcome in enumerate(outcomes):
    vals = event_summary_df[outcome].values
    ax.bar(x + (i - 1) * width, vals, width=width, label=outcome, edgecolor='black', linewidth=0.7, alpha=0.88)
    for j, v in enumerate(vals):
        ax.text(x[j] + (i - 1) * width, v + 0.025, f'{v:.2f}', ha='center', va='bottom', fontsize=8)
ax.set_xticks(x)
ax.set_xticklabels(group_order, fontsize=9)
ax.set_ylim(0, 1.15)
ax.set_ylabel('Observed favorable outcome rate')
ax.set_title('Figure 4E. Observed favorable outcomes across MBP groups', fontsize=12)
ax.grid(axis='y', linestyle='--', alpha=0.3)
ax.legend(frameon=False, fontsize=9, loc='upper right')
p_text = f'Response P = {p_resp:.2e}\nPFS event P = {p_pfs_event:.2e}\nOS event P = {p_os_event:.2e}'
ax.text(0.02, 0.97, p_text, transform=ax.transAxes, ha='left', va='top', fontsize=9, bbox=dict(boxstyle='round', facecolor='white', alpha=0.85, edgecolor='gray'))
plt.tight_layout()
fig4e_path = os.path.join(fig4_out_dir, get_value('science.figures.plot_024', 'plot_024.png'))
plt.savefig(fig4e_path, dpi=600, bbox_inches='tight')
plt.show()



In [ ]:
def net_benefit(y_true, y_prob, thresholds):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    n = len(y_true)
    nbs = []
    for pt in thresholds:
        pred_pos = y_prob >= pt
        tp = np.sum((pred_pos == 1) & (y_true == 1))
        fp = np.sum((pred_pos == 1) & (y_true == 0))
        nb = tp / n - fp / n * (pt / (1 - pt))
        nbs.append(nb)
    return np.array(nbs)

def aggregate_model_for_dca(path, model_name):
    pred = read_prediction_file(path, model_name=model_name)
    agg = aggregate_cv_predictions(pred)
    return agg[['sample', 'Response_true', 'Response_pred']].copy()
dca_models = {}
if clinical_pred_path is not None and os.path.exists(clinical_pred_path):
    dca_models['Clinical baseline'] = aggregate_model_for_dca(clinical_pred_path, 'Clinical baseline')
for model_name, path in single_model_pred_paths.items():
    if os.path.exists(path):
        dca_models[model_name] = aggregate_model_for_dca(path, model_name)
print('Models included in DCA:')
print(list(dca_models.keys()))
single_candidates = [m for m in ['Metagenomics', 'Radiomics', 'Blood', 'Urine'] if m in dca_models]
best_single = None
best_auc = -np.inf
for m in single_candidates:
    tmp = dca_models[m]
    try:
        auc = roc_auc_score(tmp['Response_true'], tmp['Response_pred'])
    except:
        auc = np.nan
    if not np.isnan(auc) and auc > best_auc:
        best_auc = auc
        best_single = m
selected_dca_models = {}
if 'Clinical baseline' in dca_models:
    selected_dca_models['Clinical baseline'] = dca_models['Clinical baseline']
if best_single is not None:
    selected_dca_models[f'Best single modality ({best_single})'] = dca_models[best_single]
selected_dca_models['Multimodal'] = dca_models['Multimodal']
print('Selected DCA models:')
print(list(selected_dca_models.keys()))
common_samples = None
for m, df in selected_dca_models.items():
    samples = set(df['sample'])
    if common_samples is None:
        common_samples = samples
    else:
        common_samples = common_samples.intersection(samples)
common_samples = sorted(list(common_samples))
thresholds = np.linspace(0.05, 0.8, 76)
fig, ax = plt.subplots(figsize=(6.4, 4.8))
ref_df = selected_dca_models['Multimodal']
ref_df = ref_df[ref_df['sample'].isin(common_samples)].copy()
ref_df = ref_df.sort_values('sample')
y_true_ref = ref_df['Response_true'].astype(int).values
prevalence = np.mean(y_true_ref)
treat_all_nb = prevalence - (1 - prevalence) * thresholds / (1 - thresholds)
treat_none_nb = np.zeros_like(thresholds)
ax.plot(thresholds, treat_all_nb, linestyle='--', linewidth=1.4, label='Treat all')
ax.plot(thresholds, treat_none_nb, linestyle=':', linewidth=1.8, label='Treat none')
for model_name, df in selected_dca_models.items():
    tmp = df[df['sample'].isin(common_samples)].copy()
    tmp = tmp.sort_values('sample')
    y_true = tmp['Response_true'].astype(int).values
    y_prob = tmp['Response_pred'].astype(float).values
    nb = net_benefit(y_true, y_prob, thresholds)
    ax.plot(thresholds, nb, linewidth=2.2, label=model_name)
ax.set_xlabel('Threshold probability')
ax.set_ylabel('Net benefit')
ax.set_title('Figure 4F. Decision curve analysis for response prediction', fontsize=12)
ax.set_ylim(-0.08, 0.58)
ax.grid(alpha=0.3)
ax.legend(frameon=False, fontsize=8, loc='upper right')
plt.tight_layout()
fig4f_path = os.path.join(fig4_out_dir, get_value('science.figures.plot_025', 'plot_025.png'))
plt.savefig(fig4f_path, dpi=600, bbox_inches='tight')
plt.show()



In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
from scipy.stats import kruskal, chi2_contingency
import matplotlib.pyplot as plt
from lifelines import CoxPHFitter
import statsmodels.api as sm
from statsmodels.tools.sm_exceptions import PerfectSeparationError
stage2_out_dir = str(project_path(get_value('science.paths.directory_019', 'directory_019')))
fig4_out_dir = os.path.join(stage2_out_dir, get_value('science.paths.directory_020', 'directory_020'))
clinical_baseline_path = str(project_path(get_value('science.tables.clinical_001', 'clinical_001.xlsx')))
conf_out_dir = os.path.join(fig4_out_dir, get_value('science.paths.directory_021', 'directory_021'))
os.makedirs(conf_out_dir, exist_ok=True)
try:
    mbp_df
    
except NameError:
    mbp_path = os.path.join(fig4_out_dir, get_value('science.tables.result_014', 'result_014.csv'))
    mbp_df = pd.read_csv(mbp_path, encoding='utf-8-sig')
    print('Loaded mbp_df from:', mbp_path)
print('MBP table shape:', mbp_df.shape)
mbp_df.head()


In [ ]:
def normalize_sample_id(x):
    return str(x).lower().strip()

def read_clinical_baseline(path):
    if path.endswith('.xlsx') or path.endswith('.xls'):
        clin = pd.read_excel(path)
    else:
        clin = pd.read_csv(path, encoding='utf-8-sig')
    sample_candidates = ['sample', 'sample_id', 'Sample', 'ID', 'id']
    sample_col = None
    for c in sample_candidates:
        if c in clin.columns:
            sample_col = c
            break
    if sample_col is None:
        sample_col = clin.columns[0]
    clin = clin.rename(columns={sample_col: 'sample'})
    clin['sample'] = clin['sample'].map(normalize_sample_id)
    clin = clin.drop_duplicates(subset='sample', keep='first')
    clin = clin.set_index('sample')
    return clin

def to_binary_safe(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().lower()
    yes_set = ['1', 'yes', 'y', 'true', 'positive', 'present', 'male', 'm', 'pvtt', 'hbv', 'cirrhosis']
    no_set = ['0', 'no', 'n', 'false', 'negative', 'absent', 'female', 'f', 'none']
    if s in yes_set:
        return 1
    if s in no_set:
        return 0
    try:
        return float(s)
    except:
        return np.nan

def find_first_existing_column(df, candidates):
    lower_map = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    return None
clinical = read_clinical_baseline(clinical_baseline_path)
clinical_col_map = {'Age': ['Age', 'age'], 'Sex': ['Sex', 'sex', 'Gender', 'gender'], 'AFP': ['AFP', 'afp'], 'TumorSize': ['Size', 'Tumor_size', 'tumor_size', 'Tumor size', 'max_diameter', 'Diameter'], 'PVTT': ['PVTT', 'pvtt'], 'ALBI': ['ALBI', 'albi'], 'BCLC': ['BCLC', 'bclc'], 'ChildPugh': ['Child_Pugh', 'ChildPugh', 'Child-Pugh', 'child_pugh'], 'ECOG': ['ECOG', 'ecog'], 'Albumin': ['ALB', 'Albumin', 'albumin'], 'TBIL': ['TBiL', 'TBIL', 'tbil', 'Bilirubin', 'bilirubin'], 'ALT': ['ALT', 'alt'], 'AST': ['AST', 'ast'], 'Creatinine': ['Creatinine', 'Cr', 'creatinine', 'Scr'], 'eGFR': ['eGFR', 'EGFR', 'egfr'], 'HBV': ['HBV', 'HBsAg', 'hbv'], 'Cirrhosis': ['Cirrhosis', 'cirrhosis'], 'Diabetes': ['Diabetes', 'DM', 'diabetes'], 'Hypertension': ['Hypertension', 'HTN', 'hypertension'], 'Antibiotics': ['Antibiotics', 'antibiotics', 'Antibiotic_use', 'antibiotic_use'], 'PPI': ['PPI', 'ppi', 'PPI_use', 'ppi_use']}
clin_use = pd.DataFrame(index=clinical.index)
matched_cols = {}
for new_name, candidates in clinical_col_map.items():
    old_col = find_first_existing_column(clinical, candidates)
    if old_col is None:
        continue
    matched_cols[new_name] = old_col
    if new_name in ['Sex', 'PVTT', 'HBV', 'Cirrhosis', 'Diabetes', 'Hypertension', 'Antibiotics', 'PPI']:
        clin_use[new_name] = clinical[old_col].apply(to_binary_safe)
    elif new_name in ['BCLC', 'ChildPugh']:
        clin_use[new_name] = clinical[old_col].astype(str).str.strip()
        clin_use.loc[clin_use[new_name].isin(['nan', 'None', '']), new_name] = np.nan
    else:
        clin_use[new_name] = pd.to_numeric(clinical[old_col], errors='coerce')
print('Matched clinical columns:')
for k, v in matched_cols.items():
    print(f'{k}  <-  {v}')
print('\nClinical variables available:')
print(clin_use.columns.tolist())


In [ ]:
analysis_df = mbp_df.copy()
analysis_df['sample'] = analysis_df['sample'].map(normalize_sample_id)
analysis_df = analysis_df.set_index('sample')
analysis_df = analysis_df.join(clin_use, how='left')
analysis_df = analysis_df.reset_index()
mbp_ord_map = {'MBP-I': 0, 'MBP-II': 1, 'MBP-III': 2}
if 'MBP_raw' not in analysis_df.columns:
    analysis_df['MBP_raw'] = analysis_df['MBP'].astype(str).str.extract('(MBP-[I]+)')[0]
analysis_df['MBP_risk_ord'] = analysis_df['MBP_raw'].map(mbp_ord_map)
if 'AFP' in analysis_df.columns:
    analysis_df['log_AFP'] = np.log1p(analysis_df['AFP'].clip(lower=0))
if 'TumorSize' in analysis_df.columns:
    analysis_df['log_TumorSize'] = np.log1p(analysis_df['TumorSize'].clip(lower=0))
core_covariates = []
for c in ['log_AFP', 'log_TumorSize', 'PVTT', 'ALBI']:
    if c in analysis_df.columns:
        core_covariates.append(c)
optional_covariates = []
for c in ['Age', 'Sex', 'ECOG', 'eGFR', 'Creatinine', 'HBV', 'Cirrhosis', 'Diabetes', 'Hypertension', 'PPI']:
    if c in analysis_df.columns:
        optional_covariates.append(c)
if 'Antibiotics' in analysis_df.columns:
    optional_covariates.append('Antibiotics')
print('Core covariates:', core_covariates)
print('Optional covariates:', optional_covariates)
analysis_path = os.path.join(conf_out_dir, get_value('science.tables.clinical_002', 'clinical_002.csv'))
analysis_df.to_csv(analysis_path, index=False, encoding='utf-8-sig')




In [ ]:
def median_iqr(x):
    x = pd.to_numeric(x, errors='coerce').dropna()
    if len(x) == 0:
        return 'NA'
    med = np.median(x)
    q1 = np.percentile(x, 25)
    q3 = np.percentile(x, 75)
    return f'{med:.2f} ({q1:.2f}-{q3:.2f})'

def n_percent(x, positive_value=1):
    x = x.dropna()
    if len(x) == 0:
        return 'NA'
    n = np.sum(x == positive_value)
    pct = n / len(x) * 100
    return f'{n}/{len(x)} ({pct:.1f}%)'

def kruskal_p_by_group(df, var, group_col='MBP_raw'):
    groups = []
    for g in ['MBP-I', 'MBP-II', 'MBP-III']:
        x = pd.to_numeric(df.loc[df[group_col] == g, var], errors='coerce').dropna()
        if len(x) > 0:
            groups.append(x)
    if len(groups) < 2:
        return np.nan
    try:
        return kruskal(*groups).pvalue
    except:
        return np.nan

def chi_square_p_by_group(df, var, group_col='MBP_raw'):
    tmp = df[[group_col, var]].dropna().copy()
    if tmp[group_col].nunique() < 2 or tmp[var].nunique() < 2:
        return np.nan
    try:
        tab = pd.crosstab(tmp[group_col], tmp[var])
        chi2, p, dof, expected = chi2_contingency(tab)
        return p
    except:
        return np.nan
continuous_vars = [c for c in ['Age', 'AFP', 'TumorSize', 'ALBI', 'Albumin', 'TBIL', 'ALT', 'AST', 'Creatinine', 'eGFR'] if c in analysis_df.columns]
categorical_vars = [c for c in ['Sex', 'PVTT', 'BCLC', 'ChildPugh', 'ECOG', 'HBV', 'Cirrhosis', 'Diabetes', 'Hypertension', 'Antibiotics', 'PPI'] if c in analysis_df.columns]
baseline_rows = []
for var in continuous_vars:
    row = {'Variable': var, 'Type': 'Continuous', 'MBP-I': median_iqr(analysis_df.loc[analysis_df['MBP_raw'] == 'MBP-I', var]), 'MBP-II': median_iqr(analysis_df.loc[analysis_df['MBP_raw'] == 'MBP-II', var]), 'MBP-III': median_iqr(analysis_df.loc[analysis_df['MBP_raw'] == 'MBP-III', var]), 'P_value': kruskal_p_by_group(analysis_df, var)}
    baseline_rows.append(row)
for var in categorical_vars:
    row = {'Variable': var, 'Type': 'Categorical', 'MBP-I': n_percent(analysis_df.loc[analysis_df['MBP_raw'] == 'MBP-I', var], positive_value=1), 'MBP-II': n_percent(analysis_df.loc[analysis_df['MBP_raw'] == 'MBP-II', var], positive_value=1), 'MBP-III': n_percent(analysis_df.loc[analysis_df['MBP_raw'] == 'MBP-III', var], positive_value=1), 'P_value': chi_square_p_by_group(analysis_df, var)}
    baseline_rows.append(row)
baseline_table = pd.DataFrame(baseline_rows)
baseline_path = os.path.join(conf_out_dir, get_value('science.tables.clinical_003', 'clinical_003.csv'))
baseline_table.to_csv(baseline_path, index=False, encoding='utf-8-sig')


baseline_table


In [ ]:
def prepare_regression_df(df, cols):
    tmp = df[cols].copy()
    for c in tmp.columns:
        if c in ['BCLC', 'ChildPugh']:
            tmp[c] = tmp[c].astype('category')
        else:
            tmp[c] = pd.to_numeric(tmp[c], errors='coerce')
    tmp = pd.get_dummies(tmp, columns=[c for c in tmp.columns if str(tmp[c].dtype) == 'category'], drop_first=True)
    for c in tmp.columns:
        tmp[c] = pd.to_numeric(tmp[c], errors='coerce')
        tmp[c] = tmp[c].fillna(tmp[c].median())
    keep_cols = [c for c in tmp.columns if tmp[c].nunique() > 1]
    tmp = tmp[keep_cols]
    return tmp

def fit_adjusted_logistic(df, outcome_col, exposure_col, covariates, model_name):
    cols = [outcome_col, exposure_col] + covariates
    cols = [c for c in cols if c in df.columns]
    tmp = df[cols].dropna(subset=[outcome_col, exposure_col]).copy()
    if tmp[outcome_col].nunique() < 2:
        print(f'[{model_name}] Outcome has <2 classes. Skipped.')
        return None
    y = tmp[outcome_col].astype(int)
    X = prepare_regression_df(tmp, [exposure_col] + covariates)
    if exposure_col not in X.columns:
        print(f'[{model_name}] Exposure not available after preprocessing. Skipped.')
        return None
    X = sm.add_constant(X, has_constant='add')
    try:
        model = sm.Logit(y, X).fit(disp=0, maxiter=300)
        params = model.params
        conf = model.conf_int()
        pvals = model.pvalues
        res = pd.DataFrame({'model': model_name, 'term': params.index, 'coef': params.values, 'OR': np.exp(params.values), 'CI_lower': np.exp(conf[0].values), 'CI_upper': np.exp(conf[1].values), 'P_value': pvals.values, 'method': 'standard_logit'})
    except Exception as e:
        print(f'[{model_name}] Standard logit failed: {e}')
        print('Trying regularized logistic regression. P-values will be unavailable.')
        try:
            model = sm.GLM(y, X, family=sm.families.Binomial())
            fit = model.fit_regularized(alpha=0.05, L1_wt=0.0)
            params = fit.params
            res = pd.DataFrame({'model': model_name, 'term': params.index, 'coef': params.values, 'OR': np.exp(params.values), 'CI_lower': np.nan, 'CI_upper': np.nan, 'P_value': np.nan, 'method': 'ridge_logistic_no_pvalue'})
        except Exception as e2:
            print(f'[{model_name}] Regularized logistic also failed: {e2}')
            return None
    return res
logit_core = fit_adjusted_logistic(analysis_df, outcome_col='Response_true', exposure_col='MBP_risk_ord', covariates=core_covariates, model_name='Response_core_adjusted')
extended_covariates = core_covariates + optional_covariates
logit_extended = fit_adjusted_logistic(analysis_df, outcome_col='Response_true', exposure_col='MBP_risk_ord', covariates=extended_covariates, model_name='Response_extended_adjusted')
logit_results = pd.concat([x for x in [logit_core, logit_extended] if x is not None], axis=0)
logit_path = os.path.join(conf_out_dir, get_value('science.tables.result_016', 'result_016.csv'))
logit_results.to_csv(logit_path, index=False, encoding='utf-8-sig')


logit_results


In [ ]:
def fit_adjusted_cox(df, time_col, event_col, exposure_col, covariates, model_name, penalizer=0.05):
    cols = [time_col, event_col, exposure_col] + covariates
    cols = [c for c in cols if c in df.columns]
    tmp = df[cols].copy()
    tmp[time_col] = pd.to_numeric(tmp[time_col], errors='coerce')
    tmp[event_col] = pd.to_numeric(tmp[event_col], errors='coerce')
    tmp[exposure_col] = pd.to_numeric(tmp[exposure_col], errors='coerce')
    tmp = tmp.dropna(subset=[time_col, event_col, exposure_col])
    if tmp[event_col].sum() < 5:
        print(f'[{model_name}] Too few events. Skipped.')
        return None
    X = prepare_regression_df(tmp, [exposure_col] + covariates)
    if exposure_col not in X.columns:
        print(f'[{model_name}] Exposure not available after preprocessing. Skipped.')
        return None
    cox_df = pd.concat([tmp[[time_col, event_col]].reset_index(drop=True), X.reset_index(drop=True)], axis=1)
    try:
        cph = CoxPHFitter(penalizer=penalizer)
        cph.fit(cox_df, duration_col=time_col, event_col=event_col, robust=True)
        summ = cph.summary.reset_index().rename(columns={'covariate': 'term'})
        res = summ[['term', 'coef', 'exp(coef)', 'exp(coef) lower 95%', 'exp(coef) upper 95%', 'p']].copy()
        res = res.rename(columns={'exp(coef)': 'HR', 'exp(coef) lower 95%': 'CI_lower', 'exp(coef) upper 95%': 'CI_upper', 'p': 'P_value'})
        res['model'] = model_name
        res['n'] = cox_df.shape[0]
        res['events'] = int(cox_df[event_col].sum())
        res['penalizer'] = penalizer
        return res
    except Exception as e:
        print(f'[{model_name}] Cox model failed: {e}')
        return None
cox_results = []
cox_results.append(fit_adjusted_cox(analysis_df, time_col='PFS_time', event_col='PFS_status', exposure_col='MBP_risk_ord', covariates=core_covariates, model_name='PFS_core_adjusted'))
cox_results.append(fit_adjusted_cox(analysis_df, time_col='OS_time', event_col='OS_status', exposure_col='MBP_risk_ord', covariates=core_covariates, model_name='OS_core_adjusted'))
cox_results.append(fit_adjusted_cox(analysis_df, time_col='PFS_time', event_col='PFS_status', exposure_col='MBP_risk_ord', covariates=extended_covariates, model_name='PFS_extended_adjusted'))
cox_results.append(fit_adjusted_cox(analysis_df, time_col='OS_time', event_col='OS_status', exposure_col='MBP_risk_ord', covariates=extended_covariates, model_name='OS_extended_adjusted'))
cox_results = pd.concat([x for x in cox_results if x is not None], axis=0)
cox_path = os.path.join(conf_out_dir, get_value('science.tables.result_017', 'result_017.csv'))
cox_results.to_csv(cox_path, index=False, encoding='utf-8-sig')


cox_results


In [ ]:
score_results = []
if 'Response_pred' in analysis_df.columns:
    score_results.append(fit_adjusted_logistic(analysis_df, outcome_col='Response_true', exposure_col='Response_pred', covariates=core_covariates, model_name='Response_pred_core_adjusted'))
if 'PFS_risk' in analysis_df.columns:
    score_results.append(fit_adjusted_cox(analysis_df, time_col='PFS_time', event_col='PFS_status', exposure_col='PFS_risk', covariates=core_covariates, model_name='PFS_risk_core_adjusted'))
if 'OS_risk' in analysis_df.columns:
    score_results.append(fit_adjusted_cox(analysis_df, time_col='OS_time', event_col='OS_status', exposure_col='OS_risk', covariates=core_covariates, model_name='OS_risk_core_adjusted'))
score_results = pd.concat([x for x in score_results if x is not None], axis=0)
score_path = os.path.join(conf_out_dir, get_value('science.tables.predictions_014', 'predictions_014.csv'))
score_results.to_csv(score_path, index=False, encoding='utf-8-sig')


score_results


In [ ]:
subgroup_df = analysis_df.copy()
subgroup_specs = []

def add_median_split(var, display_name=None):
    if var not in subgroup_df.columns:
        return
    x = pd.to_numeric(subgroup_df[var], errors='coerce')
    if x.notna().sum() < 20:
        return
    cutoff = x.median()
    new_col = f'{var}_group'
    subgroup_df[new_col] = np.where(x <= cutoff, f'{display_name or var} low', f'{display_name or var} high')
    subgroup_df.loc[x.isna(), new_col] = np.nan
    subgroup_specs.append({'subgroup_var': new_col, 'display': display_name or var, 'type': 'median_split', 'cutoff': cutoff})

def add_binary_subgroup(var, display_name=None, labels=None):
    if var not in subgroup_df.columns:
        return
    x = subgroup_df[var]
    if x.dropna().nunique() != 2:
        return
    new_col = f'{var}_group'
    if labels is None:
        labels = {0: f'{display_name or var} absent/low', 1: f'{display_name or var} present/high'}
    subgroup_df[new_col] = x.map(labels)
    subgroup_specs.append({'subgroup_var': new_col, 'display': display_name or var, 'type': 'binary', 'cutoff': None})
add_median_split('Age', 'Age')
add_median_split('AFP', 'AFP')
add_median_split('TumorSize', 'Tumor size')
add_median_split('ALBI', 'ALBI')
add_binary_subgroup('Sex', 'Sex', labels={0: 'Female', 1: 'Male'})
add_binary_subgroup('PVTT', 'PVTT', labels={0: 'PVTT absent', 1: 'PVTT present'})
add_binary_subgroup('HBV', 'HBV', labels={0: 'HBV negative', 1: 'HBV positive'})
add_binary_subgroup('Cirrhosis', 'Cirrhosis', labels={0: 'No cirrhosis', 1: 'Cirrhosis'})
add_binary_subgroup('Diabetes', 'Diabetes', labels={0: 'No diabetes', 1: 'Diabetes'})
add_binary_subgroup('PPI', 'PPI', labels={0: 'No PPI', 1: 'PPI use'})
add_binary_subgroup('Antibiotics', 'Antibiotics', labels={0: 'No antibiotics', 1: 'Antibiotics'})
print('Subgroup variables:')
for spec in subgroup_specs:
    print(spec)
subgroup_table_path = os.path.join(conf_out_dir, get_value('science.tables.result_018', 'result_018.csv'))
subgroup_df.to_csv(subgroup_table_path, index=False, encoding='utf-8-sig')


In [ ]:
def fit_subgroup_cox(df, time_col, event_col, subgroup_var, subgroup_level, min_n=15, min_events=5):
    sub = df[df[subgroup_var] == subgroup_level].copy()
    sub = sub[[time_col, event_col, 'MBP_risk_ord']].dropna()
    n = sub.shape[0]
    events = int(sub[event_col].sum())
    if n < min_n or events < min_events or sub['MBP_risk_ord'].nunique() < 2:
        return {'n': n, 'events': events, 'HR': np.nan, 'CI_lower': np.nan, 'CI_upper': np.nan, 'P_value': np.nan, 'status': 'skipped'}
    try:
        cph = CoxPHFitter(penalizer=0.05)
        cph.fit(sub, duration_col=time_col, event_col=event_col, robust=True)
        summ = cph.summary.loc['MBP_risk_ord']
        return {'n': n, 'events': events, 'HR': summ['exp(coef)'], 'CI_lower': summ['exp(coef) lower 95%'], 'CI_upper': summ['exp(coef) upper 95%'], 'P_value': summ['p'], 'status': 'ok'}
    except Exception as e:
        return {'n': n, 'events': events, 'HR': np.nan, 'CI_lower': np.nan, 'CI_upper': np.nan, 'P_value': np.nan, 'status': f'failed: {e}'}
subgroup_results = []
for outcome_name, time_col, event_col in [('PFS', 'PFS_time', 'PFS_status'), ('OS', 'OS_time', 'OS_status')]:
    for spec in subgroup_specs:
        subgroup_var = spec['subgroup_var']
        levels = subgroup_df[subgroup_var].dropna().unique().tolist()
        for level in levels:
            res = fit_subgroup_cox(subgroup_df, time_col=time_col, event_col=event_col, subgroup_var=subgroup_var, subgroup_level=level)
            res.update({'outcome': outcome_name, 'time_col': time_col, 'event_col': event_col, 'subgroup_var': subgroup_var, 'subgroup_display': spec['display'], 'subgroup_level': level, 'cutoff': spec['cutoff'], 'effect': 'HR per one-level worsening of MBP'})
            subgroup_results.append(res)
subgroup_results_df = pd.DataFrame(subgroup_results)
subgroup_results_path = os.path.join(conf_out_dir, get_value('science.tables.result_019', 'result_019.csv'))
subgroup_results_df.to_csv(subgroup_results_path, index=False, encoding='utf-8-sig')


subgroup_results_df


In [ ]:
def plot_subgroup_forest(subgroup_results_df, outcome, filename):
    df = subgroup_results_df[(subgroup_results_df['outcome'] == outcome) & (subgroup_results_df['status'] == 'ok')].copy()
    df = df.dropna(subset=['HR', 'CI_lower', 'CI_upper'])
    if df.shape[0] == 0:
        print(f'No valid subgroup results for {outcome}.')
        return
    df['label'] = df['subgroup_display'] + ': ' + df['subgroup_level'].astype(str)
    df = df.sort_values(['subgroup_display', 'subgroup_level']).reset_index(drop=True)
    y = np.arange(df.shape[0])
    fig, ax = plt.subplots(figsize=(7.5, max(4.5, 0.36 * df.shape[0])))
    ax.errorbar(df['HR'], y, xerr=[df['HR'] - df['CI_lower'], df['CI_upper'] - df['HR']], fmt='o', capsize=3, color='black', ecolor='black')
    ax.axvline(1.0, color='gray', linestyle='--', linewidth=1.2)
    ax.set_yticks(y)
    ax.set_yticklabels(df['label'], fontsize=9)
    ax.invert_yaxis()
    ax.set_xscale('log')
    ax.set_xlabel('Hazard ratio per one-level worsening of MBP')
    ax.set_title(f'{outcome} subgroup analysis of MBP', fontsize=12)
    for i, row in df.iterrows():
        txt = f"n={int(row['n'])}, ev={int(row['events'])}, P={row['P_value']:.3g}"
        ax.text(df['CI_upper'].max() * 1.05, i, txt, va='center', fontsize=8)
    plt.tight_layout()
    path = os.path.join(conf_out_dir, filename)
    plt.savefig(path, dpi=600, bbox_inches='tight')
    plt.show()
    
plot_subgroup_forest(subgroup_results_df, outcome='PFS', filename=get_value('science.figures.plot_026', 'plot_026.png'))
plot_subgroup_forest(subgroup_results_df, outcome='OS', filename=get_value('science.figures.plot_027', 'plot_027.png'))


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
base_dir = str(project_path(get_value('science.paths.directory_022', 'directory_022')))
plot_dir = os.path.join(base_dir, get_value('science.paths.directory_023', 'directory_023'))
os.makedirs(plot_dir, exist_ok=True)
logit_path = os.path.join(base_dir, get_value('science.tables.result_016', 'result_016.csv'))
cox_path = os.path.join(base_dir, get_value('science.tables.result_017', 'result_017.csv'))
score_path = os.path.join(base_dir, get_value('science.tables.predictions_014', 'predictions_014.csv'))
subgroup_path = os.path.join(base_dir, get_value('science.tables.result_019', 'result_019.csv'))
interaction_path = os.path.join(base_dir, get_value('science.tables.result_020', 'result_020.csv'))

def draw_forest_plot(df, label_col, effect_col, lower_col, upper_col, p_col=None, n_col=None, title='Forest plot', xlabel='Effect size', ref_line=1.0, figsize=(8, 6), out_path=None, xscale='log', sort=False):
    df = df.copy()
    df = df.dropna(subset=[effect_col, lower_col, upper_col])
    if sort:
        df = df.sort_values(effect_col)
    df = df.reset_index(drop=True)
    y = np.arange(len(df))
    fig, ax = plt.subplots(figsize=figsize)
    ax.errorbar(df[effect_col], y, xerr=[df[effect_col] - df[lower_col], df[upper_col] - df[effect_col]], fmt='o', color='black', ecolor='black', capsize=3, markersize=5)
    ax.axvline(ref_line, color='gray', linestyle='--', linewidth=1.2)
    ax.set_yticks(y)
    ax.set_yticklabels(df[label_col], fontsize=9)
    ax.invert_yaxis()
    if xscale == 'log':
        ax.set_xscale('log')
    ax.set_xlabel(xlabel, fontsize=10)
    ax.set_title(title, fontsize=12, pad=12)
    x_max = np.nanmax(df[upper_col].values)
    text_x1 = x_max * 1.25 if xscale == 'log' else x_max + 0.2
    text_x2 = x_max * 1.9 if xscale == 'log' else x_max + 0.6
    if n_col is not None and n_col in df.columns:
        for i, row in df.iterrows():
            ax.text(text_x1, i, f'n={int(row[n_col])}' if pd.notna(row[n_col]) else '', va='center', fontsize=8)
    if p_col is not None and p_col in df.columns:
        for i, row in df.iterrows():
            ptxt = ''
            if pd.notna(row[p_col]):
                if row[p_col] < 0.001:
                    ptxt = 'P<0.001'
                else:
                    ptxt = f'P={row[p_col]:.3f}'
            ax.text(text_x2, i, ptxt, va='center', fontsize=8)
    ax.grid(axis='x', alpha=0.25)
    plt.tight_layout()
    if out_path is not None:
        plt.savefig(out_path, dpi=600, bbox_inches='tight')
        
    plt.show()
if os.path.exists(logit_path) and os.path.exists(cox_path):
    logit_df = pd.read_csv(logit_path, encoding='utf-8-sig')
    cox_df = pd.read_csv(cox_path, encoding='utf-8-sig')
    logit_main = logit_df[logit_df['term'] == 'MBP_risk_ord'].copy()
    cox_main = cox_df[cox_df['term'] == 'MBP_risk_ord'].copy()
    rows = []
    for _, row in logit_main.iterrows():
        rows.append({'Outcome': row['model'], 'Effect': row['OR'], 'Lower': row['CI_lower'], 'Upper': row['CI_upper'], 'P_value': row['P_value'], 'Type': 'OR'})
    for _, row in cox_main.iterrows():
        rows.append({'Outcome': row['model'], 'Effect': row['HR'], 'Lower': row['CI_lower'], 'Upper': row['CI_upper'], 'P_value': row['P_value'], 'Type': 'HR'})
    main_effect_df = pd.DataFrame(rows)
    rename_map = {'Response_core_adjusted': 'Response (core-adjusted)', 'Response_extended_adjusted': 'Response (extended-adjusted)', 'PFS_core_adjusted': 'PFS (core-adjusted)', 'PFS_extended_adjusted': 'PFS (extended-adjusted)', 'OS_core_adjusted': 'OS (core-adjusted)', 'OS_extended_adjusted': 'OS (extended-adjusted)'}
    main_effect_df['Label'] = main_effect_df['Outcome'].map(rename_map).fillna(main_effect_df['Outcome'])
    draw_forest_plot(main_effect_df, label_col='Label', effect_col='Effect', lower_col='Lower', upper_col='Upper', p_col='P_value', title='Adjusted association of MBP with clinical outcomes', xlabel='OR / HR per one-level worsening of MBP', out_path=os.path.join(plot_dir, get_value('science.figures.plot_028', 'plot_028.png')), figsize=(8, 4.8))
if os.path.exists(score_path):
    score_df = pd.read_csv(score_path, encoding='utf-8-sig')
    rows = []
    for _, row in score_df.iterrows():
        effect_val = row['OR'] if 'OR' in row and pd.notna(row['OR']) else row.get('HR', np.nan)
        rows.append({'Label': row['model'], 'Effect': effect_val, 'Lower': row['CI_lower'], 'Upper': row['CI_upper'], 'P_value': row['P_value']})
    score_plot_df = pd.DataFrame(rows)
    draw_forest_plot(score_plot_df, label_col='Label', effect_col='Effect', lower_col='Lower', upper_col='Upper', p_col='P_value', title='Adjusted association of continuous multimodal prediction scores', xlabel='OR / HR', out_path=os.path.join(plot_dir, get_value('science.figures.plot_029', 'plot_029.png')), figsize=(8, 4.5))
if os.path.exists(subgroup_path):
    subgroup_df = pd.read_csv(subgroup_path, encoding='utf-8-sig')
    for outcome in ['PFS', 'OS']:
        tmp = subgroup_df[(subgroup_df['outcome'] == outcome) & (subgroup_df['status'] == 'ok')].copy()
        if tmp.shape[0] > 0:
            tmp['Label'] = tmp['subgroup_display'] + ' | ' + tmp['subgroup_level'].astype(str)
            draw_forest_plot(tmp, label_col='Label', effect_col='HR', lower_col='CI_lower', upper_col='CI_upper', p_col='P_value', n_col='n', title=f'Subgroup analysis of MBP for {outcome}', xlabel='HR per one-level worsening of MBP', out_path=os.path.join(plot_dir, f'FigureS_subgroup_{outcome}.png'), figsize=(8.2, max(5, 0.35 * tmp.shape[0])))
if os.path.exists(interaction_path):
    inter_df = pd.read_csv(interaction_path, encoding='utf-8-sig')
    for outcome in ['PFS', 'OS']:
        tmp = inter_df[inter_df['outcome'] == outcome].copy()
        if tmp.shape[0] > 0:
            tmp = tmp.sort_values('interaction_P', ascending=True)
            tmp['Label'] = tmp['subgroup_display']
            fig, ax = plt.subplots(figsize=(8, max(4.5, 0.38 * tmp.shape[0])))
            y = np.arange(tmp.shape[0])
            ax.scatter(tmp['interaction_HR'], y, color='black', s=35)
            ax.axvline(1.0, color='gray', linestyle='--', linewidth=1.2)
            ax.set_yticks(y)
            ax.set_yticklabels(tmp['Label'], fontsize=9)
            ax.invert_yaxis()
            ax.set_xscale('log')
            ax.set_xlabel('Interaction HR', fontsize=10)
            ax.set_title(f'Interaction analysis of MBP effect by subgroup ({outcome})', fontsize=12)
            x_max = np.nanmax(tmp['interaction_HR'].replace([np.inf, -np.inf], np.nan).dropna()) if tmp['interaction_HR'].notna().any() else 2
            if not np.isfinite(x_max) or x_max <= 0:
                x_max = 2
            text_x = x_max * 1.5
            for i, row in tmp.iterrows():
                ptxt = ''
                if pd.notna(row['interaction_P']):
                    if row['interaction_P'] < 0.001:
                        ptxt = 'P<0.001'
                    else:
                        ptxt = f"P={row['interaction_P']:.3f}"
                ax.text(text_x, i, ptxt, va='center', fontsize=8)
            ax.grid(axis='x', alpha=0.25)
            plt.tight_layout()
            out_path = os.path.join(plot_dir, f'FigureS_interaction_{outcome}.png')
            plt.savefig(out_path, dpi=600, bbox_inches='tight')
            
            plt.show()


In [ ]:
import os
import re
import textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
stage1_dir = str(project_path(get_value('science.paths.directory_005', 'directory_005')))
golden_feature_path = os.path.join(stage1_dir, get_value('science.tables.result_005', 'result_005.csv'))
top_feature_path = os.path.join(stage1_dir, get_value('science.tables.result_004', 'result_004.csv'))
fig_s3a_out_dir = os.path.join(stage1_dir, get_value('science.paths.directory_024', 'directory_024'))
os.makedirs(fig_s3a_out_dir, exist_ok=True)




In [ ]:
def find_col(df, candidates, required=True):
    pass
    lower_map = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    if required:
        raise ValueError(f'Cannot find any of columns {candidates}. Existing columns: {df.columns.tolist()}')
    return None

def clean_outcome_name(x):
    x = str(x).strip()
    mapper = {'response': 'Response', 'resp': 'Response', 'pfs': 'PFS', 'survival_pfs': 'PFS', 'os': 'OS', 'survival_os': 'OS', 'overall_survival': 'OS'}
    return mapper.get(x.lower(), x)

def clean_modality_name(x):
    x = str(x).strip()
    mapper = {'microbiota': 'Metagenomics', 'microbiome': 'Metagenomics', 'metagenomics': 'Metagenomics', 'radiomics': 'Radiomics', 'blood': 'Blood', 'urine': 'Urine'}
    return mapper.get(x.lower(), x)

def shorten_feature_name(x, max_len=42):
    x = str(x)
    x = x.replace('Metagenomics__', '')
    x = x.replace('Radiomics__', '')
    x = x.replace('Blood__', '')
    x = x.replace('Urine__', '')
    if len(x) <= max_len:
        return x
    return x[:max_len - 3] + '...'

def wrap_label(x, width=24):
    return '\n'.join(textwrap.wrap(str(x), width=width))

def classify_feature_status(outcomes):
    outcomes = sorted(list(set(outcomes)))
    if len(outcomes) == 3:
        return 'Shared by 3 endpoints'
    if len(outcomes) == 2:
        return '+'.join(outcomes)
    if len(outcomes) == 1:
        return f'{outcomes[0]}-specific'
    return 'Unclassified'

def get_shap_col(df):
    candidates = ['mean_abs_shap', 'mean_shap', 'shap_value', 'SHAP', 'shap', 'importance', 'mean_importance', 'avg_abs_shap', 'MeanAbsSHAP']
    col = find_col(df, candidates, required=False)
    if col is not None:
        return col
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    exclude_keywords = ['rank', 'fold', 'repeat', 'seed', 'epoch', 'n_']
    numeric_cols = [c for c in numeric_cols if not any((k in c.lower() for k in exclude_keywords))]
    if len(numeric_cols) == 0:
        raise ValueError('Cannot find SHAP/importance column. Please check column names.')
    return numeric_cols[0]


In [ ]:
top_raw = pd.read_csv(top_feature_path, encoding='utf-8-sig')
golden_raw = pd.read_csv(golden_feature_path, encoding='utf-8-sig')
print('Top feature columns:')
print(top_raw.columns.tolist())
print('\nGolden feature columns:')
print(golden_raw.columns.tolist())
mod_col = find_col(top_raw, ['modality', 'Modal', 'omics', 'data_type'])
outcome_col = find_col(top_raw, ['outcome', 'task', 'endpoint', 'Outcome'])
feature_col = find_col(top_raw, ['feature', 'feature_name', 'Feature', 'name'])
shap_col = get_shap_col(top_raw)
top_df = top_raw.copy()
top_df = top_df.rename(columns={mod_col: 'Modality', outcome_col: 'Outcome', feature_col: 'Feature', shap_col: 'MeanAbsSHAP'})
top_df['Modality'] = top_df['Modality'].apply(clean_modality_name)
top_df['Outcome'] = top_df['Outcome'].apply(clean_outcome_name)
top_df['Feature'] = top_df['Feature'].astype(str)
top_df['MeanAbsSHAP'] = pd.to_numeric(top_df['MeanAbsSHAP'], errors='coerce')
top_df = top_df.dropna(subset=['Modality', 'Outcome', 'Feature', 'MeanAbsSHAP'])
top_df = top_df.groupby(['Modality', 'Outcome', 'Feature'], as_index=False).agg({'MeanAbsSHAP': 'mean'})
top_df['Rank'] = top_df.groupby(['Modality', 'Outcome'])['MeanAbsSHAP'].rank(method='first', ascending=False).astype(int)
top_df = top_df.sort_values(['Modality', 'Outcome', 'Rank']).reset_index(drop=True)
gold_mod_col = find_col(golden_raw, ['modality', 'Modal', 'omics', 'data_type'], required=False)
gold_feature_col = find_col(golden_raw, ['feature', 'feature_name', 'Feature', 'name'], required=False)
gold_df = golden_raw.copy()
if gold_mod_col is not None:
    gold_df = gold_df.rename(columns={gold_mod_col: 'Modality'})
    gold_df['Modality'] = gold_df['Modality'].apply(clean_modality_name)
else:
    gold_df['Modality'] = np.nan
if gold_feature_col is not None:
    gold_df = gold_df.rename(columns={gold_feature_col: 'Feature'})
    gold_df['Feature'] = gold_df['Feature'].astype(str)
else:
    raise ValueError('Cannot find feature column in golden feature table.')
feature_outcome_map = top_df.groupby(['Modality', 'Feature'])['Outcome'].apply(lambda x: sorted(list(set(x)))).reset_index()
feature_outcome_map['FeatureStatus'] = feature_outcome_map['Outcome'].apply(classify_feature_status)
feature_outcome_map['n_outcomes'] = feature_outcome_map['Outcome'].apply(len)
feature_outcome_map['OutcomeSet'] = feature_outcome_map['Outcome'].apply(lambda x: '+'.join(x))
if gold_df['Modality'].notna().any():
    golden_annot = gold_df[['Modality', 'Feature']].drop_duplicates()
    golden_annot = golden_annot.merge(feature_outcome_map, on=['Modality', 'Feature'], how='left')
else:
    golden_annot = feature_outcome_map.copy()
golden_annot['FeatureStatus'] = golden_annot['FeatureStatus'].fillna('Unclassified')
golden_annot['n_outcomes'] = golden_annot['n_outcomes'].fillna(0).astype(int)
golden_annot['OutcomeSet'] = golden_annot['OutcomeSet'].fillna('NA')
print('\nStandardized top_df:')

print('\nGolden annotated:')



In [ ]:
clean_top_path = os.path.join(fig_s3a_out_dir, get_value('science.tables.result_021', 'result_021.csv'))
clean_golden_path = os.path.join(fig_s3a_out_dir, get_value('science.tables.result_022', 'result_022.csv'))
top_df.to_csv(clean_top_path, index=False, encoding='utf-8-sig')
golden_annot.to_csv(clean_golden_path, index=False, encoding='utf-8-sig')





In [ ]:
modality_order = ['Metagenomics', 'Radiomics', 'Blood', 'Urine']
outcome_order = ['Response', 'PFS', 'OS']

def simplify_feature_status(status):
    pass
    status = str(status)
    if status in ['Response-specific', 'PFS-specific', 'OS-specific']:
        return status
    if status == 'Shared by 3 endpoints':
        return 'Shared by 3 endpoints'
    if '+' in status:
        return 'Shared by 2 endpoints'
    return 'Unclassified'
golden_annot['FeatureClassSimple'] = golden_annot['FeatureStatus'].apply(simplify_feature_status)
simple_status_order = ['Response-specific', 'PFS-specific', 'OS-specific', 'Shared by 2 endpoints', 'Shared by 3 endpoints', 'Unclassified']
count_status_df = golden_annot.groupby(['Modality', 'FeatureClassSimple']).size().reset_index(name='Count')
count_status_pivot = count_status_df.pivot(index='Modality', columns='FeatureClassSimple', values='Count').reindex(index=modality_order).fillna(0)
for s in simple_status_order:
    if s not in count_status_pivot.columns:
        count_status_pivot[s] = 0
count_status_pivot = count_status_pivot[simple_status_order]
TOP_N_PER_MODALITY = 5
feature_overall_importance = top_df.groupby(['Modality', 'Feature'], as_index=False).agg({'MeanAbsSHAP': 'mean'})
selected_features = []
for mod in modality_order:
    sub = feature_overall_importance[feature_overall_importance['Modality'] == mod].copy()
    sub = sub.sort_values('MeanAbsSHAP', ascending=False).head(TOP_N_PER_MODALITY)
    selected_features.append(sub)
selected_features = pd.concat(selected_features, axis=0).reset_index(drop=True)
bubble_df = top_df.merge(selected_features[['Modality', 'Feature']], on=['Modality', 'Feature'], how='inner')
grid_rows = []
for _, row in selected_features.iterrows():
    for outcome in outcome_order:
        grid_rows.append({'Modality': row['Modality'], 'Feature': row['Feature'], 'Outcome': outcome})
grid_df = pd.DataFrame(grid_rows)
bubble_df = grid_df.merge(bubble_df[['Modality', 'Feature', 'Outcome', 'MeanAbsSHAP']], on=['Modality', 'Feature', 'Outcome'], how='left')
bubble_df['MeanAbsSHAP'] = bubble_df['MeanAbsSHAP'].fillna(0)
bubble_df['SHAP_norm'] = bubble_df.groupby('Modality')['MeanAbsSHAP'].transform(lambda x: x / x.max() if x.max() > 0 else x)
bubble_df['Feature_short'] = bubble_df['Feature'].apply(lambda x: shorten_feature_name(x, max_len=38))
bubble_df['Feature_label'] = bubble_df['Modality'] + ' | ' + bubble_df['Feature_short']
row_order = []
for mod in modality_order:
    sub = selected_features[selected_features['Modality'] == mod].copy()
    sub = sub.sort_values('MeanAbsSHAP', ascending=True)
    for _, row in sub.iterrows():
        feature_short = shorten_feature_name(row['Feature'], max_len=38)
        row_order.append(f'{mod} | {feature_short}')
row_order = list(dict.fromkeys(row_order))
bubble_df['Feature_label'] = pd.Categorical(bubble_df['Feature_label'], categories=row_order, ordered=True)
bubble_df['Outcome'] = pd.Categorical(bubble_df['Outcome'], categories=outcome_order, ordered=True)
bubble_df = bubble_df.dropna(subset=['Feature_label'])
x_map = {o: i for i, o in enumerate(outcome_order)}
y_map = {f: i for i, f in enumerate(row_order)}
bubble_df['x'] = bubble_df['Outcome'].map(x_map).astype(float)
bubble_df['y'] = bubble_df['Feature_label'].map(y_map).astype(float)
separator_positions = []
current_mod = None
for idx, label in enumerate(row_order):
    mod = label.split(' | ')[0]
    if current_mod is None:
        current_mod = mod
    elif mod != current_mod:
        separator_positions.append(idx - 0.5)
        current_mod = mod
count_status_pivot_path = os.path.join(fig_s3a_out_dir, get_value('science.tables.result_023', 'result_023.csv'))
bubble_data_path = os.path.join(fig_s3a_out_dir, get_value('science.tables.result_024', 'result_024.csv'))
count_status_pivot.to_csv(count_status_pivot_path, encoding='utf-8-sig')
bubble_df.to_csv(bubble_data_path, index=False, encoding='utf-8-sig')
print('Panel A data:')

print('Panel B data:')




plt.rcParams['font.family'] = 'Arial'
plt.rcParams['axes.linewidth'] = 0.8
fig = plt.figure(figsize=(12, 12))
gs = gridspec.GridSpec(2, 1, figure=fig, height_ratios=[0.95, 2.25], hspace=0.32)
ax_a = fig.add_subplot(gs[0, 0])
ax_b = fig.add_subplot(gs[1, 0])
bottom = np.zeros(len(count_status_pivot.index))
x = np.arange(len(count_status_pivot.index))
for status in simple_status_order:
    vals = count_status_pivot[status].values
    if vals.sum() == 0:
        continue
    ax_a.bar(x, vals, bottom=bottom, label=status, edgecolor='black', linewidth=0.45, alpha=0.88)
    bottom += vals
ax_a.set_xticks(x)
ax_a.set_xticklabels(count_status_pivot.index, rotation=20, ha='right', fontsize=10)
ax_a.set_ylabel('Number of selected features', fontsize=10)
ax_a.set_title('A. Endpoint-specific and shared SHAP-selected features', fontsize=12, fontweight='bold', loc='left')
ax_a.grid(axis='y', linestyle='--', alpha=0.3)
for i, total in enumerate(bottom):
    ax_a.text(i, total + 0.4, f'n={int(total)}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax_a.legend(bbox_to_anchor=(1.02, 1.0), loc='upper left', frameon=False, fontsize=8, title='Feature class')
sizes = 45 + bubble_df['SHAP_norm'].values * 260
sc = ax_b.scatter(bubble_df['x'], bubble_df['y'], s=sizes, c=bubble_df['SHAP_norm'], cmap='Reds', vmin=0, vmax=1, alpha=0.84, edgecolor='black', linewidth=0.35)
ax_b.set_xticks(np.arange(len(outcome_order)))
ax_b.set_xticklabels(outcome_order, fontsize=11)
ax_b.set_yticks(np.arange(len(row_order)))
ax_b.set_yticklabels(row_order, fontsize=8)
ax_b.set_xlim(-0.5, len(outcome_order) - 0.5)
ax_b.set_ylim(-0.8, len(row_order) - 0.2)
ax_b.set_title(f'B. Top {TOP_N_PER_MODALITY} SHAP-ranked features per modality across Response, PFS and OS', fontsize=12, fontweight='bold', loc='left')
ax_b.grid(axis='x', linestyle='--', alpha=0.25)
for pos in separator_positions:
    ax_b.axhline(pos, color='black', linewidth=0.8)
cbar_b = plt.colorbar(sc, ax=ax_b, fraction=0.025, pad=0.02)
cbar_b.set_label('Within-modality normalized SHAP', fontsize=9)
for size_val, label in zip([0.25, 0.5, 1.0], ['0.25', '0.50', '1.00']):
    ax_b.scatter([], [], s=45 + size_val * 260, c='white', edgecolor='black', linewidth=0.35, label=label)
size_legend = ax_b.legend(title='Normalized SHAP', bbox_to_anchor=(1.02, 0.55), loc='center left', frameon=False, fontsize=8, title_fontsize=9)
ax_b.add_artist(size_legend)
fig.suptitle('Figure S3A. Composition and representative SHAP-selected features across modalities and endpoints', fontsize=15, fontweight='bold', y=0.995)
plt.tight_layout(rect=[0, 0, 1, 0.975])
fig_path = os.path.join(fig_s3a_out_dir, get_value('science.figures.plot_030', 'plot_030.png'))
pdf_path = fig_path.replace('.png', '.pdf')
plt.savefig(fig_path, dpi=600, bbox_inches='tight')
plt.savefig(pdf_path, bbox_inches='tight')
plt.show()





In [ ]:
fig, ax = plt.subplots(figsize=(6.8, 4.5))
bottom = np.zeros(len(count_status_pivot.index))
x = np.arange(len(count_status_pivot.index))
for status in simple_status_order:
    vals = count_status_pivot[status].values
    if vals.sum() == 0:
        continue
    ax.bar(x, vals, bottom=bottom, label=status, edgecolor='black', linewidth=0.45, alpha=0.88)
    bottom += vals
ax.set_xticks(x)
ax.set_xticklabels(count_status_pivot.index, rotation=20, ha='right', fontsize=10)
ax.set_ylabel('Number of selected features')
ax.set_title('Endpoint-specific and shared SHAP-selected features', fontweight='bold')
ax.grid(axis='y', linestyle='--', alpha=0.3)
for i, total in enumerate(bottom):
    ax.text(i, total + 0.4, f'n={int(total)}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.legend(bbox_to_anchor=(1.02, 1.0), loc='upper left', frameon=False, fontsize=8, title='Feature class')
plt.tight_layout()
panel_a_path = os.path.join(fig_s3a_out_dir, get_value('science.figures.plot_031', 'plot_031.png'))
panel_a_pdf = panel_a_path.replace('.png', '.pdf')
plt.savefig(panel_a_path, dpi=600, bbox_inches='tight')
plt.savefig(panel_a_pdf, bbox_inches='tight')
plt.show()



fig, ax = plt.subplots(figsize=(8.0, 8.5))
sizes = 45 + bubble_df['SHAP_norm'].values * 260
sc = ax.scatter(bubble_df['x'], bubble_df['y'], s=sizes, c=bubble_df['SHAP_norm'], cmap='Reds', vmin=0, vmax=1, alpha=0.84, edgecolor='black', linewidth=0.35)
ax.set_xticks(np.arange(len(outcome_order)))
ax.set_xticklabels(outcome_order, fontsize=11)
ax.set_yticks(np.arange(len(row_order)))
ax.set_yticklabels(row_order, fontsize=8)
ax.set_xlim(-0.5, len(outcome_order) - 0.5)
ax.set_ylim(-0.8, len(row_order) - 0.2)
ax.set_title(f'Top {TOP_N_PER_MODALITY} SHAP-ranked features per modality across Response, PFS and OS', fontweight='bold')
ax.grid(axis='x', linestyle='--', alpha=0.25)
for pos in separator_positions:
    ax.axhline(pos, color='black', linewidth=0.8)
cbar = plt.colorbar(sc, ax=ax, fraction=0.035, pad=0.025)
cbar.set_label('Within-modality normalized SHAP', fontsize=9)
for size_val, label in zip([0.25, 0.5, 1.0], ['0.25', '0.50', '1.00']):
    ax.scatter([], [], s=45 + size_val * 260, c='white', edgecolor='black', linewidth=0.35, label=label)
ax.legend(title='Normalized SHAP', bbox_to_anchor=(1.02, 0.5), loc='center left', frameon=False, fontsize=8, title_fontsize=9)
plt.tight_layout()
panel_b_path = os.path.join(fig_s3a_out_dir, get_value('science.figures.plot_032', 'plot_032.png'))
panel_b_pdf = panel_b_path.replace('.png', '.pdf')
plt.savefig(panel_b_path, dpi=600, bbox_inches='tight')
plt.savefig(panel_b_pdf, bbox_inches='tight')
plt.show()





In [ ]:
import os
import numpy as np
import pandas as pd
stage2_out_dir = str(project_path(get_value('science.paths.directory_019', 'directory_019')))
combo_out_dir = os.path.join(stage2_out_dir, get_value('science.paths.directory_012', 'directory_012'))
possible_summary_paths = [os.path.join(combo_out_dir, get_value('science.tables.metrics_012', 'metrics_012.csv')), os.path.join(combo_out_dir, get_value('science.tables.metrics_007', 'metrics_007.csv')), os.path.join(combo_out_dir, get_value('science.tables.metrics_007', 'metrics_007.csv'))]
table_out_dir = os.path.join(stage2_out_dir, get_value('science.paths.directory_025', 'directory_025'))
os.makedirs(table_out_dir, exist_ok=True)
try:
    all_model_summary_df
    
except NameError:
    loaded = False
    for p in possible_summary_paths:
        if os.path.exists(p):
            all_model_summary_df = pd.read_csv(p, encoding='utf-8-sig')
            print('Loaded summary from:', p)
            loaded = True
            break
    if not loaded:
        raise FileNotFoundError('Cannot find all_model_summary_df in memory or saved summary file. Please check combo_out_dir or load all_model_summary_df first.')
summary_df = all_model_summary_df.copy()
abbr = {'Metagenomics': 'M', 'Radiomics': 'R', 'Blood': 'B', 'Urine': 'U'}
full_modalities = 'Metagenomics+Radiomics+Blood+Urine'

def infer_modalities(row):
    if 'modalities' in row.index and pd.notna(row['modalities']):
        return str(row['modalities'])
    model = str(row['model'])
    if model == 'Multimodal':
        return full_modalities
    if model in ['Metagenomics', 'Radiomics', 'Blood', 'Urine']:
        return model
    return model

def infer_n_modalities(modalities):
    if pd.isna(modalities):
        return np.nan
    return len(str(modalities).split('+'))

def shorten_modalities(x):
    if pd.isna(x):
        return ''
    parts = str(x).split('+')
    short = []
    for p in parts:
        p = p.strip()
        short.append(abbr.get(p, p))
    if len(short) == 4 and set(short) == set(['M', 'R', 'B', 'U']):
        return 'All'
    return '+'.join(short)

def mean_sd_string(mean, sd, digits=3):
    if pd.isna(mean):
        return 'NA'
    if pd.isna(sd):
        return f'{mean:.{digits}f}'
    return f'{mean:.{digits}f} ± {sd:.{digits}f}'
if 'modalities' not in summary_df.columns:
    summary_df['modalities'] = summary_df.apply(infer_modalities, axis=1)
else:
    summary_df['modalities'] = summary_df['modalities'].fillna(summary_df.apply(infer_modalities, axis=1))
if 'n_modalities' not in summary_df.columns:
    summary_df['n_modalities'] = summary_df['modalities'].apply(infer_n_modalities)
else:
    missing_n = summary_df['n_modalities'].isna()
    summary_df.loc[missing_n, 'n_modalities'] = summary_df.loc[missing_n, 'modalities'].apply(infer_n_modalities)
summary_df['n_modalities'] = summary_df['n_modalities'].astype(int)
summary_df['modalities_short'] = summary_df['modalities'].apply(shorten_modalities)
category_map = {1: 'Single modality', 2: 'Two modalities', 3: 'Three modalities', 4: 'Full multimodal'}
summary_df['model_category'] = summary_df['n_modalities'].map(category_map)
summary_df.loc[summary_df['model'] == 'Multimodal', 'modalities'] = full_modalities
summary_df.loc[summary_df['model'] == 'Multimodal', 'modalities_short'] = 'All'
summary_df.loc[summary_df['model'] == 'Multimodal', 'n_modalities'] = 4
summary_df.loc[summary_df['model'] == 'Multimodal', 'model_category'] = 'Full multimodal'
required_cols = ['model', 'modalities', 'modalities_short', 'model_category', 'n_modalities', 'n_samples', 'n_features', 'Response_AUC_mean', 'Response_AUC_sd', 'Response_AUPRC_mean', 'Response_AUPRC_sd', 'PFS_Cindex_mean', 'PFS_Cindex_sd', 'OS_Cindex_mean', 'OS_Cindex_sd', 'best_epoch_median']
for c in required_cols:
    if c not in summary_df.columns:
        summary_df[c] = np.nan
table_s2 = summary_df[required_cols].copy()
table_s2 = table_s2.rename(columns={'model': 'Model', 'modalities': 'Modalities', 'modalities_short': 'Modalities_abbreviation', 'model_category': 'Model_category', 'n_modalities': 'Number_of_modalities', 'n_samples': 'Number_of_samples', 'n_features': 'Number_of_features', 'Response_AUC_mean': 'Response_AUC_mean', 'Response_AUC_sd': 'Response_AUC_SD', 'Response_AUPRC_mean': 'Response_AUPRC_mean', 'Response_AUPRC_sd': 'Response_AUPRC_SD', 'PFS_Cindex_mean': 'PFS_Cindex_mean', 'PFS_Cindex_sd': 'PFS_Cindex_SD', 'OS_Cindex_mean': 'OS_Cindex_mean', 'OS_Cindex_sd': 'OS_Cindex_SD', 'best_epoch_median': 'Best_epoch_median'})
table_s2['Response_AUC'] = table_s2.apply(lambda r: mean_sd_string(r['Response_AUC_mean'], r['Response_AUC_SD']), axis=1)
table_s2['Response_AUPRC'] = table_s2.apply(lambda r: mean_sd_string(r['Response_AUPRC_mean'], r['Response_AUPRC_SD']), axis=1)
table_s2['PFS_Cindex'] = table_s2.apply(lambda r: mean_sd_string(r['PFS_Cindex_mean'], r['PFS_Cindex_SD']), axis=1)
table_s2['OS_Cindex'] = table_s2.apply(lambda r: mean_sd_string(r['OS_Cindex_mean'], r['OS_Cindex_SD']), axis=1)
table_s2['Response_AUC_rank'] = table_s2['Response_AUC_mean'].rank(method='min', ascending=False).astype('Int64')
table_s2['PFS_Cindex_rank'] = table_s2['PFS_Cindex_mean'].rank(method='min', ascending=False).astype('Int64')
table_s2['OS_Cindex_rank'] = table_s2['OS_Cindex_mean'].rank(method='min', ascending=False).astype('Int64')
table_s2 = table_s2.sort_values(['Number_of_modalities', 'Model']).reset_index(drop=True)
table_s2_manuscript = table_s2[['Model', 'Modalities', 'Modalities_abbreviation', 'Model_category', 'Number_of_modalities', 'Number_of_samples', 'Number_of_features', 'Response_AUC', 'Response_AUPRC', 'PFS_Cindex', 'OS_Cindex', 'Best_epoch_median', 'Response_AUC_rank', 'PFS_Cindex_rank', 'OS_Cindex_rank']].copy()
table_s2_numeric = table_s2.copy()
table_s2_csv = os.path.join(table_out_dir, get_value('science.tables.metrics_013', 'metrics_013.csv'))
table_s2_xlsx = os.path.join(table_out_dir, get_value('science.tables.metrics_014', 'metrics_014.xlsx'))
table_s2_manuscript.to_csv(table_s2_csv, index=False, encoding='utf-8-sig')
with pd.ExcelWriter(table_s2_xlsx, engine='openpyxl') as writer:
    table_s2_manuscript.to_excel(writer, sheet_name='Manuscript_table', index=False)
    table_s2_numeric.to_excel(writer, sheet_name='Numeric_values', index=False)

print(table_s2_csv)

table_s2_manuscript.head()


In [ ]:
import os
import numpy as np
import pandas as pd
stage1_dir = str(project_path(get_value('science.paths.directory_005', 'directory_005')))
golden_feature_path = os.path.join(stage1_dir, get_value('science.tables.result_005', 'result_005.csv'))
top_feature_path = os.path.join(stage1_dir, get_value('science.tables.result_004', 'result_004.csv'))
table_out_dir = os.path.join(str(project_path(get_value('science.paths.directory_019', 'directory_019'))), get_value('science.paths.directory_025', 'directory_025'))
os.makedirs(table_out_dir, exist_ok=True)

def find_col(df, candidates, required=True):
    lower_map = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    if required:
        raise ValueError(f'Cannot find columns {candidates}. Existing columns: {df.columns.tolist()}')
    return None

def clean_outcome_name(x):
    x = str(x).strip()
    mapper = {'response': 'Response', 'resp': 'Response', 'pfs': 'PFS', 'survival_pfs': 'PFS', 'os': 'OS', 'survival_os': 'OS', 'overall_survival': 'OS'}
    return mapper.get(x.lower(), x)

def clean_modality_name(x):
    x = str(x).strip()
    mapper = {'microbiota': 'Metagenomics', 'microbiome': 'Metagenomics', 'metagenomics': 'Metagenomics', 'radiomics': 'Radiomics', 'blood': 'Blood', 'urine': 'Urine'}
    return mapper.get(x.lower(), x)

def get_shap_col(df):
    candidates = ['mean_abs_shap', 'mean_shap', 'shap_value', 'SHAP', 'shap', 'importance', 'mean_importance', 'avg_abs_shap', 'MeanAbsSHAP']
    col = find_col(df, candidates, required=False)
    if col is not None:
        return col
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    exclude_keywords = ['rank', 'fold', 'repeat', 'seed', 'epoch', 'n_']
    numeric_cols = [c for c in numeric_cols if not any((k in c.lower() for k in exclude_keywords))]
    if len(numeric_cols) == 0:
        raise ValueError('Cannot find SHAP / importance column.')
    return numeric_cols[0]

def classify_feature(outcomes):
    outcomes = sorted(list(set(outcomes)))
    if len(outcomes) == 1:
        return f'{outcomes[0]}-specific'
    if len(outcomes) == 2:
        return 'Shared by 2 endpoints'
    if len(outcomes) == 3:
        return 'Shared by 3 endpoints'
    return 'Unclassified'

def outcome_set_string(outcomes):
    outcomes = sorted(list(set(outcomes)))
    if len(outcomes) == 0:
        return 'NA'
    return '+'.join(outcomes)
top_raw = pd.read_csv(top_feature_path, encoding='utf-8-sig')
golden_raw = pd.read_csv(golden_feature_path, encoding='utf-8-sig')
print('Top feature file columns:')
print(top_raw.columns.tolist())
print('\nGolden feature file columns:')
print(golden_raw.columns.tolist())
top_mod_col = find_col(top_raw, ['modality', 'Modal', 'omics', 'data_type'])
top_outcome_col = find_col(top_raw, ['outcome', 'task', 'endpoint', 'Outcome'])
top_feature_col = find_col(top_raw, ['feature', 'feature_name', 'Feature', 'name'])
top_shap_col = get_shap_col(top_raw)
top_df = top_raw.copy()
top_df = top_df.rename(columns={top_mod_col: 'Modality', top_outcome_col: 'Outcome', top_feature_col: 'Feature', top_shap_col: 'MeanAbsSHAP'})
top_df['Modality'] = top_df['Modality'].apply(clean_modality_name)
top_df['Outcome'] = top_df['Outcome'].apply(clean_outcome_name)
top_df['Feature'] = top_df['Feature'].astype(str)
top_df['MeanAbsSHAP'] = pd.to_numeric(top_df['MeanAbsSHAP'], errors='coerce')
top_df = top_df.dropna(subset=['Modality', 'Outcome', 'Feature', 'MeanAbsSHAP'])
top_df = top_df.groupby(['Modality', 'Outcome', 'Feature'], as_index=False).agg({'MeanAbsSHAP': 'mean'})
top_df['Rank_within_modality_endpoint'] = top_df.groupby(['Modality', 'Outcome'])['MeanAbsSHAP'].rank(method='first', ascending=False).astype(int)
gold_mod_col = find_col(golden_raw, ['modality', 'Modal', 'omics', 'data_type'], required=False)
gold_feature_col = find_col(golden_raw, ['feature', 'feature_name', 'Feature', 'name'], required=True)
gold_df = golden_raw.copy()
if gold_mod_col is not None:
    gold_df = gold_df.rename(columns={gold_mod_col: 'Modality'})
    gold_df['Modality'] = gold_df['Modality'].apply(clean_modality_name)
else:
    gold_df['Modality'] = np.nan
gold_df = gold_df.rename(columns={gold_feature_col: 'Feature'})
gold_df['Feature'] = gold_df['Feature'].astype(str)
gold_df = gold_df[['Modality', 'Feature']].drop_duplicates()
if gold_df['Modality'].isna().all():
    gold_df = top_df[['Modality', 'Feature']].drop_duplicates()
feature_endpoint = top_df.groupby(['Modality', 'Feature'])['Outcome'].apply(lambda x: sorted(list(set(x)))).reset_index()
feature_endpoint['Outcome_set'] = feature_endpoint['Outcome'].apply(outcome_set_string)
feature_endpoint['Feature_class'] = feature_endpoint['Outcome'].apply(classify_feature)
feature_endpoint['Number_of_endpoints'] = feature_endpoint['Outcome'].apply(lambda x: len(set(x)))
for outcome in ['Response', 'PFS', 'OS']:
    feature_endpoint[f'Selected_for_{outcome}'] = feature_endpoint['Outcome'].apply(lambda x: outcome in set(x))
shap_wide = top_df.pivot_table(index=['Modality', 'Feature'], columns='Outcome', values='MeanAbsSHAP', aggfunc='mean').reset_index()
shap_wide = shap_wide.rename(columns={'Response': 'Response_mean_abs_SHAP', 'PFS': 'PFS_mean_abs_SHAP', 'OS': 'OS_mean_abs_SHAP'})
rank_wide = top_df.pivot_table(index=['Modality', 'Feature'], columns='Outcome', values='Rank_within_modality_endpoint', aggfunc='min').reset_index()
rank_wide = rank_wide.rename(columns={'Response': 'Response_SHAP_rank', 'PFS': 'PFS_SHAP_rank', 'OS': 'OS_SHAP_rank'})
table_s4 = gold_df.merge(feature_endpoint.drop(columns=['Outcome']), on=['Modality', 'Feature'], how='left')
table_s4 = table_s4.merge(shap_wide, on=['Modality', 'Feature'], how='left')
table_s4 = table_s4.merge(rank_wide, on=['Modality', 'Feature'], how='left')
table_s4['Outcome_set'] = table_s4['Outcome_set'].fillna('NA')
table_s4['Feature_class'] = table_s4['Feature_class'].fillna('Unclassified')
table_s4['Number_of_endpoints'] = table_s4['Number_of_endpoints'].fillna(0).astype(int)
for outcome in ['Response', 'PFS', 'OS']:
    col = f'Selected_for_{outcome}'
    if col not in table_s4.columns:
        table_s4[col] = False
    table_s4[col] = table_s4[col].fillna(False).astype(bool)
shap_cols = ['Response_mean_abs_SHAP', 'PFS_mean_abs_SHAP', 'OS_mean_abs_SHAP']
for c in shap_cols:
    if c not in table_s4.columns:
        table_s4[c] = np.nan
table_s4['Mean_abs_SHAP_across_available_endpoints'] = table_s4[shap_cols].mean(axis=1, skipna=True)
table_s4['Overall_SHAP_rank_within_modality'] = table_s4.groupby('Modality')['Mean_abs_SHAP_across_available_endpoints'].rank(method='first', ascending=False).astype('Int64')
modality_order = ['Metagenomics', 'Radiomics', 'Blood', 'Urine']
table_s4['Modality'] = pd.Categorical(table_s4['Modality'], categories=modality_order, ordered=True)
table_s4 = table_s4.sort_values(['Modality', 'Overall_SHAP_rank_within_modality', 'Feature']).reset_index(drop=True)
final_cols = ['Modality', 'Feature', 'Feature_class', 'Outcome_set', 'Number_of_endpoints', 'Selected_for_Response', 'Selected_for_PFS', 'Selected_for_OS', 'Response_mean_abs_SHAP', 'PFS_mean_abs_SHAP', 'OS_mean_abs_SHAP', 'Mean_abs_SHAP_across_available_endpoints', 'Response_SHAP_rank', 'PFS_SHAP_rank', 'OS_SHAP_rank', 'Overall_SHAP_rank_within_modality']
for c in final_cols:
    if c not in table_s4.columns:
        table_s4[c] = np.nan
table_s4 = table_s4[final_cols].copy()
table_s4_summary = table_s4.groupby(['Modality', 'Feature_class']).size().reset_index(name='Number_of_features')
table_s4_csv = os.path.join(table_out_dir, get_value('science.tables.attribution_011', 'attribution_011.csv'))
table_s4_xlsx = os.path.join(table_out_dir, get_value('science.tables.attribution_012', 'attribution_012.xlsx'))
table_s4.to_csv(table_s4_csv, index=False, encoding='utf-8-sig')
with pd.ExcelWriter(table_s4_xlsx, engine='openpyxl') as writer:
    table_s4.to_excel(writer, sheet_name='SHAP_selected_features', index=False)
    table_s4_summary.to_excel(writer, sheet_name='Summary_by_modality', index=False)
    top_df.to_excel(writer, sheet_name='Top_features_by_endpoint', index=False)

print(table_s4_csv)

table_s4.head(20)


In [ ]:
import os
import glob
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import matplotlib.image as mpimg
from scipy.stats import chi2_contingency
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import multivariate_logrank_test
import statsmodels.api as sm
stage2_out_dir = str(project_path(get_value('science.paths.directory_019', 'directory_019')))
clinical_labels_path = str(project_path(get_value('science.tables.outcomes_005', 'outcomes_005.csv')))
clinical_baseline_path = str(project_path(get_value('science.tables.clinical_001', 'clinical_001.xlsx')))
fig4_out_dir = os.path.join(stage2_out_dir, get_value('science.paths.directory_026', 'directory_026'))
os.makedirs(fig4_out_dir, exist_ok=True)




In [ ]:
def normalize_sample_id(x):
    return str(x).strip().lower()

def find_col(df, candidates, required=True):
    lower_map = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    if required:
        raise ValueError(f'Cannot find any of columns {candidates}. Existing columns: {df.columns.tolist()}')
    return None

def percentile_rank(s):
    s = pd.Series(s)
    return s.rank(method='average', pct=True)

def minmax_scale(s):
    s = pd.to_numeric(pd.Series(s), errors='coerce')
    if s.max() == s.min():
        return pd.Series(np.zeros(len(s)), index=s.index)
    return (s - s.min()) / (s.max() - s.min())

def log1p_safe(s):
    s = pd.to_numeric(pd.Series(s), errors='coerce')
    s = s.clip(lower=0)
    return np.log1p(s)

def to_binary_safe(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().lower()
    yes_set = ['1', 'yes', 'y', 'true', 'positive', 'present', 'male', 'm', get_value('science.translation.analysis_001', 'present'), get_value('science.translation.analysis_002', 'yes'), get_value('science.translation.analysis_003', 'positive'), get_value('science.translation.analysis_004', 'multiple')]
    no_set = ['0', 'no', 'n', 'false', 'negative', 'absent', 'female', 'f', get_value('science.translation.analysis_005', 'absent'), get_value('science.translation.analysis_006', 'no'), get_value('science.translation.analysis_007', 'negative'), get_value('science.translation.analysis_008', 'solitary')]
    if s in yes_set:
        return 1
    if s in no_set:
        return 0
    try:
        return float(s)
    except:
        return np.nan

def format_p(p):
    if pd.isna(p):
        return 'P = NA'
    if p < 0.001:
        return 'P < 0.001'
    return f'P = {p:.3f}'

def safe_mode_fill(s, default=0):
    mode_val = s.mode(dropna=True)
    if len(mode_val) > 0:
        return s.fillna(mode_val.iloc[0])
    return s.fillna(default)

def infer_time_unit(time_values):
    pass
    x = pd.to_numeric(pd.Series(time_values), errors='coerce').dropna()
    if x.median() > 60:
        return 'days'
    else:
        return 'months'

def survival_timepoint_value(time_values, months):
    unit = infer_time_unit(time_values)
    if unit == 'days':
        return months * 30.4375
    else:
        return months


In [ ]:
def normalize_modality_name(x):
    x = str(x).strip()
    mapper = {'M': 'Metagenomics', 'R': 'Radiomics', 'B': 'Blood', 'U': 'Urine', 'Microbiota': 'Metagenomics', 'Microbiome': 'Metagenomics', 'Metagenomics': 'Metagenomics', 'Radiomics': 'Radiomics', 'Blood': 'Blood', 'Urine': 'Urine'}
    return mapper.get(x, x)

def modality_set_from_string(x):
    if pd.isna(x):
        return set()
    s = str(x)
    s = s.replace('_plus_', '+')
    s = s.replace('_', '+')
    parts = [p.strip() for p in s.split('+') if p.strip() != '']
    parts = [normalize_modality_name(p) for p in parts]
    return set(parts)

def standardize_prediction_columns(pred):
    pred = pred.copy()
    sample_col = find_col(pred, ['sample', 'sample_id', 'Sample', 'ID', 'id'])
    resp_true_col = find_col(pred, ['Response_true', 'Response', 'y_response', 'y_resp', 'resp_true'])
    resp_pred_col = find_col(pred, ['Response_pred', 'response_pred', 'pred_response', 'resp_pred', 'Response_probability'])
    pfs_time_col = find_col(pred, ['PFS_time', 'pfs_time'])
    pfs_status_col = find_col(pred, ['PFS_status', 'pfs_status', 'PFS_event', 'pfs_event'])
    pfs_risk_col = find_col(pred, ['PFS_risk', 'pfs_risk', 'PFS_pred', 'pfs_pred'])
    os_time_col = find_col(pred, ['OS_time', 'os_time'])
    os_status_col = find_col(pred, ['OS_status', 'os_status', 'OS_event', 'os_event'])
    os_risk_col = find_col(pred, ['OS_risk', 'os_risk', 'OS_pred', 'os_pred'])
    rename_map = {sample_col: 'sample', resp_true_col: 'Response_true', resp_pred_col: 'Response_pred', pfs_time_col: 'PFS_time', pfs_status_col: 'PFS_status', pfs_risk_col: 'PFS_risk', os_time_col: 'OS_time', os_status_col: 'OS_status', os_risk_col: 'OS_risk'}
    pred = pred.rename(columns=rename_map)
    pred['sample'] = pred['sample'].map(normalize_sample_id)
    for c in ['Response_true', 'Response_pred', 'PFS_time', 'PFS_status', 'PFS_risk', 'OS_time', 'OS_status', 'OS_risk']:
        pred[c] = pd.to_numeric(pred[c], errors='coerce')
    keep_cols = ['sample', 'Response_true', 'Response_pred', 'PFS_time', 'PFS_status', 'PFS_risk', 'OS_time', 'OS_status', 'OS_risk']
    extra_cols = [c for c in ['fold_id', 'repeat', 'fold', 'model', 'modalities'] if c in pred.columns]
    return pred[keep_cols + extra_cols].copy()

def find_and_load_mru_predictions(stage2_out_dir):
    mru_set = {'Metagenomics', 'Radiomics', 'Urine'}
    combined_patterns = [os.path.join(stage2_out_dir, '**', get_value('science.tables.predictions_005', 'predictions_005.csv')), os.path.join(stage2_out_dir, '**', '*combination*cv_predictions*.csv')]
    combined_files = []
    for pat in combined_patterns:
        combined_files.extend(glob.glob(pat, recursive=True))
    combined_files = list(dict.fromkeys(combined_files))
    for f in combined_files:
        try:
            tmp = pd.read_csv(f, encoding='utf-8-sig')
        except:
            continue
        if 'modalities' in tmp.columns:
            tmp['_modset'] = tmp['modalities'].apply(modality_set_from_string)
            sub = tmp[tmp['_modset'].apply(lambda x: x == mru_set)].copy()
            if sub.shape[0] > 0:
                
                print(f)
                return standardize_prediction_columns(sub.drop(columns=['_modset']))
    direct_patterns = [os.path.join(stage2_out_dir, '**', '*C3_M_R_U*cv_predictions*.csv'), os.path.join(stage2_out_dir, '**', '*M_R_U*cv_predictions*.csv'), os.path.join(stage2_out_dir, '**', '*Metagenomics*Radiomics*Urine*cv_predictions*.csv'), os.path.join(stage2_out_dir, '**', '*Metagenomics_plus_Radiomics_plus_Urine*cv_predictions*.csv')]
    direct_files = []
    for pat in direct_patterns:
        direct_files.extend(glob.glob(pat, recursive=True))
    direct_files = list(dict.fromkeys(direct_files))
    if len(direct_files) == 0:
        raise FileNotFoundError('Cannot find MRU cv prediction file. Please set mru_pred_path manually.')
    f = direct_files[0]
    tmp = pd.read_csv(f, encoding='utf-8-sig')
    
    print(f)
    return standardize_prediction_columns(tmp)
mru_pred_long = find_and_load_mru_predictions(stage2_out_dir)
print('MRU prediction rows:', mru_pred_long.shape)
display(mru_pred_long.head())


In [ ]:
def aggregate_cv_predictions(pred_long):
    agg = pred_long.groupby('sample', as_index=False).agg({'Response_true': 'first', 'Response_pred': 'mean', 'PFS_time': 'first', 'PFS_status': 'first', 'PFS_risk': 'mean', 'OS_time': 'first', 'OS_status': 'first', 'OS_risk': 'mean'})
    return agg
mru_df = aggregate_cv_predictions(mru_pred_long)
mru_df['Response_benefit'] = percentile_rank(mru_df['Response_pred'])
mru_df['PFS_benefit'] = 1 - percentile_rank(mru_df['PFS_risk'])
mru_df['OS_benefit'] = 1 - percentile_rank(mru_df['OS_risk'])
w_response = 1 / 3
w_pfs = 1 / 3
w_os = 1 / 3
mru_df['MRU_benefit_score'] = w_response * mru_df['Response_benefit'] + w_pfs * mru_df['PFS_benefit'] + w_os * mru_df['OS_benefit']
q_low = mru_df['MRU_benefit_score'].quantile(1 / 3)
q_high = mru_df['MRU_benefit_score'].quantile(2 / 3)

def assign_mbp(score):
    if score >= q_high:
        return 'MBP-I'
    elif score >= q_low:
        return 'MBP-II'
    else:
        return 'MBP-III'
mru_df['MBP_raw'] = mru_df['MRU_benefit_score'].apply(assign_mbp)
mbp_display_map = {'MBP-I': 'MBP-I\nHigh-benefit', 'MBP-II': 'MBP-II\nIntermediate', 'MBP-III': 'MBP-III\nLow-benefit'}
mru_df['MBP'] = mru_df['MBP_raw'].map(mbp_display_map)
mru_df['MBP_risk_ord'] = mru_df['MBP_raw'].map({'MBP-I': 0, 'MBP-II': 1, 'MBP-III': 2})
mru_df = mru_df.sort_values('MRU_benefit_score', ascending=False).reset_index(drop=True)
mru_table_path = os.path.join(fig4_out_dir, get_value('science.tables.result_025', 'result_025.csv'))
mru_df.to_csv(mru_table_path, index=False, encoding='utf-8-sig')


print(mru_df['MBP'].value_counts())



In [ ]:
def read_clinical_table(path):
    if path.endswith('.xlsx') or path.endswith('.xls'):
        df = pd.read_excel(path)
    else:
        df = pd.read_csv(path, encoding='utf-8-sig')
    sample_col = find_col(df, ['sample', 'sample_id', 'Sample', 'ID', 'id', get_column('science.translation.analysis_009', 'patient_name')], required=False)
    if sample_col is None:
        sample_col = df.columns[0]
    df = df.rename(columns={sample_col: 'sample'})
    df['sample'] = df['sample'].map(normalize_sample_id)
    df = df.drop_duplicates(subset='sample', keep='first')
    return df
clinical_raw = read_clinical_table(clinical_baseline_path)
print('Clinical columns:')
print(clinical_raw.columns.tolist())

def extract_clinical_covariates(clinical_raw):
    clin = clinical_raw.copy()
    out = pd.DataFrame()
    out['sample'] = clin['sample']
    afp_col = find_col(clin, ['AFP', 'afp'], required=False)
    if afp_col is not None:
        out['AFP'] = pd.to_numeric(clin[afp_col], errors='coerce')
        out['log_AFP'] = log1p_safe(out['AFP'])
    size_col = find_col(clin, ['Size', 'size', 'Tumor size', 'Tumor_size', get_column('science.translation.analysis_010', 'tumor_size')], required=False)
    if size_col is not None:
        out['Tumor_size'] = pd.to_numeric(clin[size_col], errors='coerce')
        out['log_Tumor_size'] = log1p_safe(out['Tumor_size'])
    pvtt_col = find_col(clin, ['PVTT', 'pvtt', get_column('science.translation.analysis_011', 'tumor_thrombus'), get_column('science.translation.analysis_012', 'portal_vein_tumor_thrombus'), get_column('science.translation.analysis_013', 'vascular_tumor_thrombus'), 'MVI'], required=False)
    if pvtt_col is not None:
        out['PVTT'] = clin[pvtt_col].apply(to_binary_safe)
    albi_col = find_col(clin, ['ALBI', 'albi'], required=False)
    if albi_col is not None:
        out['ALBI'] = pd.to_numeric(clin[albi_col], errors='coerce')
    else:
        alb_col = find_col(clin, [get_column('science.translation.analysis_014', 'albumin'), 'ALB', 'Albumin', 'albumin'], required=False)
        tbil_col = find_col(clin, [get_column('science.translation.analysis_015', 'total_bilirubin'), 'TBIL', 'TBiL', 'Bilirubin', 'bilirubin'], required=False)
        if alb_col is not None and tbil_col is not None:
            albumin = pd.to_numeric(clin[alb_col], errors='coerce')
            tbil = pd.to_numeric(clin[tbil_col], errors='coerce')
            tbil = tbil.clip(lower=0.1)
            out['ALBI'] = np.log10(tbil) * 0.66 + albumin * -0.085
    return out
clinical_core = extract_clinical_covariates(clinical_raw)
analysis_df = mru_df.merge(clinical_core, on='sample', how='left')
core_covariates = [c for c in ['log_AFP', 'log_Tumor_size', 'PVTT', 'ALBI'] if c in analysis_df.columns]
print('Core clinical covariates used:')
print(core_covariates)
analysis_path = os.path.join(fig4_out_dir, get_value('science.tables.clinical_004', 'clinical_004.csv'))
analysis_df.to_csv(analysis_path, index=False, encoding='utf-8-sig')





In [ ]:
heat_df = analysis_df.sort_values('MRU_benefit_score', ascending=False).copy()
heat_df = heat_df.set_index('sample')
heat_rows = pd.DataFrame(index=heat_df.index)
heat_rows['Response probability'] = percentile_rank(heat_df['Response_pred'])
heat_rows['PFS risk percentile'] = percentile_rank(heat_df['PFS_risk'])
heat_rows['OS risk percentile'] = percentile_rank(heat_df['OS_risk'])
heat_rows['Observed response'] = heat_df['Response_true'].astype(float)
if 'AFP' in heat_df.columns:
    afp_log = log1p_safe(heat_df['AFP'])
    heat_rows['AFP'] = percentile_rank(afp_log.fillna(afp_log.median()))
if 'Tumor_size' in heat_df.columns:
    size_log = log1p_safe(heat_df['Tumor_size'])
    heat_rows['Tumor size'] = percentile_rank(size_log.fillna(size_log.median()))
if 'PVTT' in heat_df.columns:
    heat_rows['PVTT'] = safe_mode_fill(heat_df['PVTT'], default=0).astype(float)
if 'ALBI' in heat_df.columns:
    albi = pd.to_numeric(heat_df['ALBI'], errors='coerce')
    heat_rows['ALBI'] = percentile_rank(albi.fillna(albi.median()))
heat_mat = heat_rows.T
mbp_ordered = heat_df['MBP_raw'].values
mbp_color_map = {'MBP-I': '#2C7BB6', 'MBP-II': '#FDB863', 'MBP-III': '#D7191C'}
mbp_colors = [mbp_color_map[x] for x in mbp_ordered]
boundaries = []
for i in range(1, len(mbp_ordered)):
    if mbp_ordered[i] != mbp_ordered[i - 1]:
        boundaries.append(i - 0.5)
group_centers = []
group_labels = []
start = 0
for i in range(1, len(mbp_ordered) + 1):
    if i == len(mbp_ordered) or mbp_ordered[i] != mbp_ordered[start]:
        end = i - 1
        group_centers.append((start + end) / 2)
        group_labels.append(mbp_display_map[mbp_ordered[start]].replace('\n', ' '))
        start = i
fig = plt.figure(figsize=(14.5, 4.9))
gs = gridspec.GridSpec(2, 1, height_ratios=[0.22, 1.0], hspace=0.04)
ax_anno = fig.add_subplot(gs[0, 0])
ax_heat = fig.add_subplot(gs[1, 0])
for i, color in enumerate(mbp_colors):
    ax_anno.add_patch(mpatches.Rectangle((i - 0.5, 0), 1, 1, facecolor=color, edgecolor=color, linewidth=0))
ax_anno.set_xlim(-0.5, len(mbp_colors) - 0.5)
ax_anno.set_ylim(0, 1)
ax_anno.set_xticks([])
ax_anno.set_yticks([])
for spine in ax_anno.spines.values():
    spine.set_visible(False)
for b in boundaries:
    ax_anno.axvline(b, color='black', linewidth=1.0)
for center, label in zip(group_centers, group_labels):
    ax_anno.text(center, 1.18, label, ha='center', va='bottom', fontsize=10, fontweight='bold')
ax_anno.text(-1.0, 0.5, 'MRU-MBP', ha='right', va='center', fontsize=10, fontweight='bold')
legend_patches = [mpatches.Patch(color=mbp_color_map['MBP-I'], label='MBP-I High-benefit'), mpatches.Patch(color=mbp_color_map['MBP-II'], label='MBP-II Intermediate'), mpatches.Patch(color=mbp_color_map['MBP-III'], label='MBP-III Low-benefit')]
ax_anno.legend(handles=legend_patches, loc='upper right', bbox_to_anchor=(1.0, 1.95), ncol=3, frameon=False, fontsize=8)
im = ax_heat.imshow(heat_mat.values, aspect='auto', interpolation='nearest', cmap='RdBu_r', vmin=0, vmax=1)
ax_heat.set_yticks(np.arange(heat_mat.shape[0]))
ax_heat.set_yticklabels(heat_mat.index, fontsize=9)
ax_heat.set_xticks([])
ax_heat.set_xlabel('Patients ordered by MRU-based multimodal benefit score', fontsize=10)
for b in boundaries:
    ax_heat.axvline(b, color='black', linewidth=1.0)
cbar = plt.colorbar(im, ax=ax_heat, fraction=0.022, pad=0.015)
cbar.set_label('Percentile / scaled value', fontsize=9)
ax_anno.set_title('Figure 4A. Patient-level MRU-based multimodal benefit stratification', fontsize=12, pad=28)
plt.tight_layout()
fig4a_path = os.path.join(fig4_out_dir, get_value('science.figures.plot_033', 'plot_033.png'))
plt.savefig(fig4a_path, dpi=600, bbox_inches='tight')
plt.show()



In [ ]:
km_color_map = {'MBP-I\nHigh-benefit': '#2C7BB6', 'MBP-II\nIntermediate': '#FDB863', 'MBP-III\nLow-benefit': '#D7191C'}
group_order = ['MBP-I\nHigh-benefit', 'MBP-II\nIntermediate', 'MBP-III\nLow-benefit']

def plot_km_by_mbp(df, time_col, event_col, title, filename):
    tmp = df[[time_col, event_col, 'MBP']].dropna().copy()
    fig, ax = plt.subplots(figsize=(5.4, 4.4))
    for group in group_order:
        sub = tmp[tmp['MBP'] == group]
        if sub.shape[0] == 0:
            continue
        kmf = KaplanMeierFitter()
        kmf.fit(durations=sub[time_col], event_observed=sub[event_col], label=f"{group.replace(chr(10), ' ')} (n={sub.shape[0]})")
        kmf.plot_survival_function(ax=ax, ci_show=True, color=km_color_map[group], linewidth=2.0)
    lr = multivariate_logrank_test(tmp[time_col], tmp['MBP'], tmp[event_col])
    p = lr.p_value
    ax.text(0.04, 0.08, f'Log-rank {format_p(p)}', transform=ax.transAxes, fontsize=9)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('Time')
    ax.set_ylabel('Survival probability')
    ax.set_ylim(0, 1.05)
    ax.grid(alpha=0.25)
    ax.legend(frameon=False, fontsize=7, loc='upper right')
    plt.tight_layout()
    path = os.path.join(fig4_out_dir, filename)
    plt.savefig(path, dpi=600, bbox_inches='tight')
    plt.show()
    
    return (path, p)
fig4b_path, p_pfs_km = plot_km_by_mbp(analysis_df, time_col='PFS_time', event_col='PFS_status', title='Figure 4B. PFS by MRU-MBP', filename=get_value('science.figures.plot_034', 'plot_034.png'))
fig4c_path, p_os_km = plot_km_by_mbp(analysis_df, time_col='OS_time', event_col='OS_status', title='Figure 4C. OS by MRU-MBP', filename=get_value('science.figures.plot_035', 'plot_035.png'))


In [ ]:
def km_survival_at_time(sub, time_col, event_col, t):
    if sub.shape[0] == 0:
        return np.nan
    kmf = KaplanMeierFitter()
    kmf.fit(sub[time_col], sub[event_col])
    try:
        return float(kmf.survival_function_at_times(t).values[0])
    except:
        return np.nan
pfs_t = survival_timepoint_value(analysis_df['PFS_time'], months=12)
os_t = survival_timepoint_value(analysis_df['OS_time'], months=18)
summary_rows = []
for group in group_order:
    sub = analysis_df[analysis_df['MBP'] == group].copy()
    summary_rows.append({'MBP': group, 'n': sub.shape[0], 'Response rate': sub['Response_true'].mean(), '12-month PFS rate': km_survival_at_time(sub, 'PFS_time', 'PFS_status', pfs_t), '18-month OS rate': km_survival_at_time(sub, 'OS_time', 'OS_status', os_t)})
outcome_summary_df = pd.DataFrame(summary_rows)
outcome_summary_path = os.path.join(fig4_out_dir, get_value('science.tables.metrics_015', 'metrics_015.csv'))
outcome_summary_df.to_csv(outcome_summary_path, index=False, encoding='utf-8-sig')

tab_resp = pd.crosstab(analysis_df['MBP'], analysis_df['Response_true'])
try:
    p_resp = chi2_contingency(tab_resp)[1]
except:
    p_resp = np.nan
fig, ax = plt.subplots(figsize=(6.8, 4.5))
x = np.arange(len(group_order))
width = 0.24
metrics = ['Response rate', '12-month PFS rate', '18-month OS rate']
for i, metric in enumerate(metrics):
    vals = outcome_summary_df[metric].values
    ax.bar(x + (i - 1) * width, vals, width=width, label=metric, edgecolor='black', linewidth=0.7, alpha=0.88)
    for j, v in enumerate(vals):
        ax.text(x[j] + (i - 1) * width, v + 0.025, f'{v:.2f}', ha='center', va='bottom', fontsize=8)
ax.set_xticks(x)
ax.set_xticklabels(group_order, fontsize=8)
ax.set_ylim(0, 1.15)
ax.set_ylabel('Observed favorable outcome rate')
ax.set_title('Figure 4D. Observed outcomes across MRU-MBP groups', fontsize=11)
ax.grid(axis='y', linestyle='--', alpha=0.3)
ax.legend(frameon=False, fontsize=8, loc='upper right')
p_text = f'Response {format_p(p_resp)}\nPFS log-rank {format_p(p_pfs_km)}\nOS log-rank {format_p(p_os_km)}'
ax.text(0.02, 0.97, p_text, transform=ax.transAxes, ha='left', va='top', fontsize=8, bbox=dict(boxstyle='round', facecolor='white', alpha=0.85, edgecolor='gray'))
plt.tight_layout()
fig4d_path = os.path.join(fig4_out_dir, get_value('science.figures.plot_036', 'plot_036.png'))
plt.savefig(fig4d_path, dpi=600, bbox_inches='tight')
plt.show()



In [ ]:
def get_oof_logistic_predictions(df, feature_cols, outcome_col='Response_true', n_splits=5, n_repeats=5, random_state=42):
    tmp = df[['sample', outcome_col] + feature_cols].copy()
    tmp = tmp.dropna(subset=[outcome_col])
    y = tmp[outcome_col].astype(int).values
    X = tmp[feature_cols].copy()
    pred_records = []
    rskf = RepeatedStratifiedKFold(n_splits=n_splits, n_repeats=n_repeats, random_state=random_state)
    for fold_id, (train_idx, test_idx) in enumerate(rskf.split(X, y), start=1):
        model = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler()), ('clf', LogisticRegression(max_iter=1000, class_weight='balanced', solver='liblinear', random_state=random_state))])
        model.fit(X.iloc[train_idx], y[train_idx])
        pred = model.predict_proba(X.iloc[test_idx])[:, 1]
        fold_pred = pd.DataFrame({'sample': tmp.iloc[test_idx]['sample'].values, 'y_true': y[test_idx], 'y_pred': pred, 'fold_id': fold_id})
        pred_records.append(fold_pred)
    pred_long = pd.concat(pred_records, axis=0)
    pred_agg = pred_long.groupby('sample', as_index=False).agg({'y_true': 'first', 'y_pred': 'mean'})
    return (pred_agg, pred_long)

def net_benefit(y_true, y_prob, thresholds):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    n = len(y_true)
    nbs = []
    for pt in thresholds:
        pred_pos = y_prob >= pt
        tp = np.sum((pred_pos == 1) & (y_true == 1))
        fp = np.sum((pred_pos == 1) & (y_true == 0))
        nb = tp / n - fp / n * (pt / (1 - pt))
        nbs.append(nb)
    return np.array(nbs)
dca_df = analysis_df.copy()
clinical_feature_cols = core_covariates.copy()
if len(clinical_feature_cols) == 0:
    raise ValueError('No clinical covariates available for clinical baseline DCA.')
clinical_pred_agg, clinical_pred_long = get_oof_logistic_predictions(dca_df, feature_cols=clinical_feature_cols, outcome_col='Response_true')
clinical_pred_agg = clinical_pred_agg.rename(columns={'y_true': 'Response_true', 'y_pred': 'Clinical_baseline_pred'})
mru_dca = dca_df[['sample', 'Response_true', 'Response_pred']].copy()
mru_dca = mru_dca.rename(columns={'Response_pred': 'MRU_pred'})
dca_merged = mru_dca.merge(clinical_pred_agg[['sample', 'Clinical_baseline_pred']], on='sample', how='inner')
dca_pred_path = os.path.join(fig4_out_dir, get_value('science.tables.predictions_015', 'predictions_015.csv'))
dca_merged.to_csv(dca_pred_path, index=False, encoding='utf-8-sig')
print('DCA samples:', dca_merged.shape[0])
try:
    print('Clinical baseline AUC:', roc_auc_score(dca_merged['Response_true'], dca_merged['Clinical_baseline_pred']))
except:
    print('Clinical baseline AUC could not be calculated.')
try:
    print('MRU AUC:', roc_auc_score(dca_merged['Response_true'], dca_merged['MRU_pred']))
except:
    print('MRU AUC could not be calculated.')
thresholds = np.linspace(0.05, 0.8, 76)
y_true = dca_merged['Response_true'].astype(int).values
prevalence = np.mean(y_true)
treat_all_nb = prevalence - (1 - prevalence) * thresholds / (1 - thresholds)
treat_none_nb = np.zeros_like(thresholds)
fig, ax = plt.subplots(figsize=(6.4, 4.8))
ax.plot(thresholds, treat_all_nb, linestyle='--', linewidth=1.3, label='Treat all')
ax.plot(thresholds, treat_none_nb, linestyle=':', linewidth=1.6, label='Treat none')
dca_models = {'Clinical baseline': dca_merged['Clinical_baseline_pred'].values, 'MRU model': dca_merged['MRU_pred'].values}
for name, prob in dca_models.items():
    nb = net_benefit(y_true, prob, thresholds)
    ax.plot(thresholds, nb, linewidth=2.1, label=name)
ax.set_xlabel('Threshold probability')
ax.set_ylabel('Net benefit')
ax.set_title('Figure 4E. Decision curve analysis for response prediction', fontsize=11)
ax.set_ylim(-0.08, 0.58)
ax.grid(alpha=0.3)
ax.legend(frameon=False, fontsize=8, loc='upper right')
plt.tight_layout()
fig4e_path = os.path.join(fig4_out_dir, get_value('science.figures.plot_037', 'plot_037.png'))
plt.savefig(fig4e_path, dpi=600, bbox_inches='tight')
plt.show()




In [ ]:
def prepare_regression_matrix(df, cols):
    X = df[cols].copy()
    for c in X.columns:
        X[c] = pd.to_numeric(X[c], errors='coerce')
        X[c] = X[c].fillna(X[c].median())
    keep_cols = [c for c in X.columns if X[c].nunique() > 1]
    X = X[keep_cols]
    return X

def fit_adjusted_logistic_for_mbp(df, outcome_col, exposure_col, covariates, model_label):
    cols = [outcome_col, exposure_col] + covariates
    tmp = df[cols].dropna(subset=[outcome_col, exposure_col]).copy()
    y = tmp[outcome_col].astype(int)
    X = prepare_regression_matrix(tmp, [exposure_col] + covariates)
    X = sm.add_constant(X, has_constant='add')
    try:
        fit = sm.Logit(y, X).fit(disp=0, maxiter=300)
        params = fit.params
        conf = fit.conf_int()
        pvals = fit.pvalues
        out = pd.DataFrame({'model': model_label, 'term': params.index, 'coef': params.values, 'effect': np.exp(params.values), 'CI_lower': np.exp(conf[0].values), 'CI_upper': np.exp(conf[1].values), 'P_value': pvals.values, 'effect_type': 'OR'})
        return out
    except Exception as e:
        print(f'Logistic failed for {model_label}: {e}')
        return None

def fit_adjusted_cox_for_mbp(df, time_col, event_col, exposure_col, covariates, model_label, penalizer=0.05):
    cols = [time_col, event_col, exposure_col] + covariates
    tmp = df[cols].copy()
    tmp[time_col] = pd.to_numeric(tmp[time_col], errors='coerce')
    tmp[event_col] = pd.to_numeric(tmp[event_col], errors='coerce')
    tmp[exposure_col] = pd.to_numeric(tmp[exposure_col], errors='coerce')
    tmp = tmp.dropna(subset=[time_col, event_col, exposure_col])
    X = prepare_regression_matrix(tmp, [exposure_col] + covariates)
    cox_df = pd.concat([tmp[[time_col, event_col]].reset_index(drop=True), X.reset_index(drop=True)], axis=1)
    try:
        cph = CoxPHFitter(penalizer=penalizer)
        cph.fit(cox_df, duration_col=time_col, event_col=event_col, robust=True)
        summ = cph.summary.reset_index().rename(columns={'covariate': 'term'})
        out = pd.DataFrame({'model': model_label, 'term': summ['term'], 'coef': summ['coef'], 'effect': summ['exp(coef)'], 'CI_lower': summ['exp(coef) lower 95%'], 'CI_upper': summ['exp(coef) upper 95%'], 'P_value': summ['p'], 'effect_type': 'HR'})
        return out
    except Exception as e:
        print(f'Cox failed for {model_label}: {e}')
        return None
adjust_results = []
adjust_results.append(fit_adjusted_logistic_for_mbp(analysis_df, outcome_col='Response_true', exposure_col='MBP_risk_ord', covariates=core_covariates, model_label='Response'))
adjust_results.append(fit_adjusted_cox_for_mbp(analysis_df, time_col='PFS_time', event_col='PFS_status', exposure_col='MBP_risk_ord', covariates=core_covariates, model_label='PFS'))
adjust_results.append(fit_adjusted_cox_for_mbp(analysis_df, time_col='OS_time', event_col='OS_status', exposure_col='MBP_risk_ord', covariates=core_covariates, model_label='OS'))
adjust_results = pd.concat([x for x in adjust_results if x is not None], axis=0)
adjust_path = os.path.join(fig4_out_dir, get_value('science.tables.result_026', 'result_026.csv'))
adjust_results.to_csv(adjust_path, index=False, encoding='utf-8-sig')



plot_adj = adjust_results[adjust_results['term'] == 'MBP_risk_ord'].copy()
label_map = {'Response': 'Response', 'PFS': 'PFS', 'OS': 'OS'}
plot_adj['label'] = plot_adj['model'].map(label_map)
plot_adj['display_effect'] = plot_adj['effect_type'] + ' per one-level worsening of MRU-MBP'
plot_adj = plot_adj.set_index('model').loc[['Response', 'PFS', 'OS']].reset_index()
fig, ax = plt.subplots(figsize=(6.4, 3.4))
y = np.arange(plot_adj.shape[0])
ax.errorbar(plot_adj['effect'], y, xerr=[plot_adj['effect'] - plot_adj['CI_lower'], plot_adj['CI_upper'] - plot_adj['effect']], fmt='o', color='black', ecolor='black', capsize=3, markersize=5)
ax.axvline(1.0, color='gray', linestyle='--', linewidth=1.2)
ax.set_yticks(y)
ax.set_yticklabels(plot_adj['label'], fontsize=10)
ax.invert_yaxis()
ax.set_xscale('log')
ax.set_xlabel('Adjusted OR / HR per one-level worsening of MRU-MBP')
ax.set_title('Figure 4F. Adjusted association of MRU-MBP with outcomes', fontsize=11)
xmax = plot_adj['CI_upper'].max()
for i, row in plot_adj.iterrows():
    ax.text(xmax * 1.25, i, format_p(row['P_value']), va='center', fontsize=9)
ax.grid(axis='x', alpha=0.25)
plt.tight_layout()
fig4f_path = os.path.join(fig4_out_dir, get_value('science.figures.plot_038', 'plot_038.png'))
plt.savefig(fig4f_path, dpi=600, bbox_inches='tight')
plt.show()



In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from lifelines import CoxPHFitter
try:
    analysis_df
    
except NameError:
    stage2_out_dir = str(project_path(get_value('science.paths.directory_019', 'directory_019')))
    fig4_out_dir = os.path.join(stage2_out_dir, get_value('science.paths.directory_026', 'directory_026'))
    analysis_path = os.path.join(fig4_out_dir, get_value('science.tables.clinical_004', 'clinical_004.csv'))
    analysis_df = pd.read_csv(analysis_path, encoding='utf-8-sig')
    print('Loaded analysis_df from:', analysis_path)
fig_s4_out_dir = os.path.join(fig4_out_dir, get_value('science.paths.directory_027', 'directory_027'))
os.makedirs(fig_s4_out_dir, exist_ok=True)


def format_p(p):
    if pd.isna(p):
        return 'NA'
    if p < 0.001:
        return '<0.001'
    return f'{p:.3f}'

def format_hr_ci(hr, low, high):
    if pd.isna(hr) or pd.isna(low) or pd.isna(high):
        return 'NA'
    return f'{hr:.2f} ({low:.2f}-{high:.2f})'

def add_median_split(df, var, display_name=None, lower_label=None, higher_label=None):
    pass
    if var not in df.columns:
        print(f'[Skip] {var} not found.')
        return None
    x = pd.to_numeric(df[var], errors='coerce')
    if x.notna().sum() < 20 or x.nunique(dropna=True) < 2:
        print(f'[Skip] {var}: insufficient valid values.')
        return None
    cutoff = x.median()
    new_col = f'{var}_subgroup'
    if display_name is None:
        display_name = var
    if lower_label is None:
        lower_label = f'{display_name} low'
    if higher_label is None:
        higher_label = f'{display_name} high'
    df[new_col] = np.nan
    df.loc[x <= cutoff, new_col] = lower_label
    df.loc[x > cutoff, new_col] = higher_label
    return {'var': new_col, 'display': display_name, 'type': 'median_split', 'cutoff': cutoff, 'levels': [lower_label, higher_label]}

def add_binary_split(df, var, display_name=None, labels=None):
    pass
    if var not in df.columns:
        print(f'[Skip] {var} not found.')
        return None
    x = pd.to_numeric(df[var], errors='coerce')
    valid_values = sorted(x.dropna().unique().tolist())
    if len(valid_values) < 2:
        print(f'[Skip] {var}: only one valid category.')
        return None
    new_col = f'{var}_subgroup'
    if display_name is None:
        display_name = var
    if labels is None:
        labels = {0: f'{display_name} absent', 1: f'{display_name} present'}
    df[new_col] = x.map(labels)
    return {'var': new_col, 'display': display_name, 'type': 'binary', 'cutoff': None, 'levels': list(labels.values())}

def fit_subgroup_cox(df, time_col, event_col, subgroup_var, subgroup_level, min_n=10, min_events=3, penalizer=0.05):
    pass
    sub = df[df[subgroup_var] == subgroup_level].copy()
    needed = [time_col, event_col, 'MBP_risk_ord']
    sub = sub[needed].copy()
    for c in needed:
        sub[c] = pd.to_numeric(sub[c], errors='coerce')
    sub = sub.dropna()
    n = sub.shape[0]
    events = int(sub[event_col].sum()) if n > 0 else 0
    n_mbp_levels = sub['MBP_risk_ord'].nunique()
    if n < min_n or events < min_events or n_mbp_levels < 2:
        return {'n': n, 'events': events, 'HR': np.nan, 'CI_lower': np.nan, 'CI_upper': np.nan, 'P_value': np.nan, 'status': 'skipped'}
    try:
        cph = CoxPHFitter(penalizer=penalizer)
        cph.fit(sub, duration_col=time_col, event_col=event_col, robust=True)
        s = cph.summary.loc['MBP_risk_ord']
        return {'n': n, 'events': events, 'HR': s['exp(coef)'], 'CI_lower': s['exp(coef) lower 95%'], 'CI_upper': s['exp(coef) upper 95%'], 'P_value': s['p'], 'status': 'ok'}
    except Exception as e:
        return {'n': n, 'events': events, 'HR': np.nan, 'CI_lower': np.nan, 'CI_upper': np.nan, 'P_value': np.nan, 'status': f'failed: {e}'}
subgroup_df = analysis_df.copy()
subgroup_specs = []
spec = add_median_split(subgroup_df, var='AFP', display_name='AFP', lower_label='AFP low', higher_label='AFP high')
if spec is not None:
    subgroup_specs.append(spec)
spec = add_median_split(subgroup_df, var='Tumor_size', display_name='Tumor size', lower_label='Tumor size low', higher_label='Tumor size high')
if spec is not None:
    subgroup_specs.append(spec)
spec = add_binary_split(subgroup_df, var='PVTT', display_name='PVTT', labels={0: 'PVTT absent', 1: 'PVTT present'})
if spec is not None:
    subgroup_specs.append(spec)
spec = add_median_split(subgroup_df, var='ALBI', display_name='ALBI', lower_label='ALBI low', higher_label='ALBI high')
if spec is not None:
    subgroup_specs.append(spec)
print('Subgroups included:')
for s in subgroup_specs:
    print(s)
subgroup_results = []
for outcome, time_col, event_col in [('PFS', 'PFS_time', 'PFS_status'), ('OS', 'OS_time', 'OS_status')]:
    for spec in subgroup_specs:
        subgroup_var = spec['var']
        for level in spec['levels']:
            if level not in subgroup_df[subgroup_var].dropna().unique():
                continue
            res = fit_subgroup_cox(subgroup_df, time_col=time_col, event_col=event_col, subgroup_var=subgroup_var, subgroup_level=level, min_n=10, min_events=3, penalizer=0.05)
            res.update({'Outcome': outcome, 'Subgroup_variable': spec['display'], 'Subgroup_level': level, 'Subgroup_column': subgroup_var, 'Cutoff': spec['cutoff'], 'Effect': 'HR per one-level worsening of MRU-MBP'})
            subgroup_results.append(res)
subgroup_results_df = pd.DataFrame(subgroup_results)
subgroup_result_path = os.path.join(fig_s4_out_dir, get_value('science.tables.result_027', 'result_027.csv'))
subgroup_results_df.to_csv(subgroup_result_path, index=False, encoding='utf-8-sig')





In [ ]:
def plot_subgroup_forest_table(subgroup_results_df, outcome, filename, title=None):
    plot_df = subgroup_results_df[(subgroup_results_df['Outcome'] == outcome) & (subgroup_results_df['status'] == 'ok')].copy()
    if plot_df.shape[0] == 0:
        print(f'No valid subgroup results for {outcome}.')
        return None
    var_order = {'AFP': 0, 'Tumor size': 1, 'PVTT': 2, 'ALBI': 3}
    level_order = {'AFP low': 0, 'AFP high': 1, 'Tumor size low': 0, 'Tumor size high': 1, 'PVTT absent': 0, 'PVTT present': 1, 'ALBI low': 0, 'ALBI high': 1}
    plot_df['var_order'] = plot_df['Subgroup_variable'].map(var_order)
    plot_df['level_order'] = plot_df['Subgroup_level'].map(level_order)
    plot_df = plot_df.sort_values(['var_order', 'level_order']).reset_index(drop=True)
    plot_df['HR_CI'] = plot_df.apply(lambda r: format_hr_ci(r['HR'], r['CI_lower'], r['CI_upper']), axis=1)
    plot_df['P_text'] = plot_df['P_value'].apply(lambda p: 'P=' + format_p(p))
    y_labels = []
    previous_var = None
    for _, row in plot_df.iterrows():
        var = row['Subgroup_variable']
        level = row['Subgroup_level']
        if var != previous_var:
            y_labels.append(f'{var}\n  {level}')
            previous_var = var
        else:
            y_labels.append(f'  {level}')
    plot_df['Y_label'] = y_labels
    n_rows = plot_df.shape[0]
    fig_height = max(4.8, 0.45 * n_rows + 1.8)
    fig, ax = plt.subplots(figsize=(8.8, fig_height))
    y = np.arange(n_rows)
    for i in range(n_rows):
        if i % 2 == 0:
            ax.axhspan(i - 0.5, i + 0.5, color='#F3F6F4', zorder=0)
    ax.errorbar(plot_df['HR'], y, xerr=[plot_df['HR'] - plot_df['CI_lower'], plot_df['CI_upper'] - plot_df['HR']], fmt='o', color='black', ecolor='black', elinewidth=1.2, capsize=3, markersize=4.5, zorder=3)
    ax.axvline(1.0, color='gray', linestyle='--', linewidth=1.1)
    ax.set_xscale('log')
    xmin = max(0.6, np.nanmin(plot_df['CI_lower']) * 0.75)
    xmax = np.nanmax(plot_df['CI_upper']) * 1.35
    if not np.isfinite(xmin) or xmin <= 0:
        xmin = 0.7
    if not np.isfinite(xmax) or xmax <= 1:
        xmax = 4.0
    ax.set_xlim(xmin, xmax)
    ax.set_yticks(y)
    ax.set_yticklabels(plot_df['Y_label'], fontsize=9)
    ax.invert_yaxis()
    ax.set_xlabel('HR per one-level worsening of MRU-MBP', fontsize=10)
    if title is None:
        title = f'Subgroup analysis of MRU-MBP for {outcome}'
    ax.set_title(title, fontsize=12, fontweight='bold', pad=14)
    ax.grid(axis='x', linestyle='--', alpha=0.25)
    text_x1 = xmax * 1.08
    text_x2 = xmax * 1.55
    text_x3 = xmax * 2.25
    for i, row in plot_df.iterrows():
        ax.text(text_x1, i, f"{int(row['n'])}/{int(row['events'])}", va='center', fontsize=8)
        ax.text(text_x2, i, row['HR_CI'], va='center', fontsize=8)
        ax.text(text_x3, i, row['P_text'], va='center', fontsize=8)
    ax.text(text_x1, -0.9, 'n/events', va='center', fontsize=8, fontweight='bold')
    ax.text(text_x2, -0.9, 'HR (95% CI)', va='center', fontsize=8, fontweight='bold')
    ax.text(text_x3, -0.9, 'P-value', va='center', fontsize=8, fontweight='bold')
    ax.set_xlim(xmin, xmax * 3.2)
    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)
    plt.tight_layout()
    out_path = os.path.join(fig_s4_out_dir, filename)
    pdf_path = out_path.replace('.png', '.pdf')
    plt.savefig(out_path, dpi=600, bbox_inches='tight')
    plt.savefig(pdf_path, bbox_inches='tight')
    plt.show()
    
    
    
    return out_path
fig_s4b_path = plot_subgroup_forest_table(subgroup_results_df, outcome='PFS', filename=get_value('science.figures.plot_039', 'plot_039.png'), title='Figure S4B. Subgroup analysis of MRU-MBP for PFS')
fig_s4c_path = plot_subgroup_forest_table(subgroup_results_df, outcome='OS', filename=get_value('science.figures.plot_040', 'plot_040.png'), title='Figure S4C. Subgroup analysis of MRU-MBP for OS')


In [ ]:
import os
import re
import math
import random
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import shap
import joblib
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.colors import TwoSlopeNorm
from matplotlib.patches import Rectangle, PathPatch
from matplotlib.path import Path

RANDOM_STATE = 42
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
PROJECT_DIR = str(project_path(get_value('science.paths.directory_008', 'directory_008')))
INPUT_DIR = os.path.join(PROJECT_DIR, get_value('science.paths.directory_009', 'directory_009'))
base_out_dir = os.path.join(PROJECT_DIR, get_value('science.paths.directory_010', 'directory_010'))
stage1_dir = os.path.join(base_out_dir, get_value('science.paths.directory_003', 'directory_003'))
mru_out_dir = os.path.join(base_out_dir, get_value('science.paths.directory_028', 'directory_028'))
os.makedirs(mru_out_dir, exist_ok=True)
fig_out_dir = os.path.join(mru_out_dir, get_value('science.paths.directory_029', 'directory_029'))
os.makedirs(fig_out_dir, exist_ok=True)
label_path = os.path.join(INPUT_DIR, get_value('science.tables.outcomes_001', 'outcomes_001.csv'))
golden_feature_path = os.path.join(stage1_dir, get_value('science.tables.result_005', 'result_005.csv'))
matrix_files = {'Metagenomics': os.path.join(INPUT_DIR, get_value('science.tables.features_001', 'features_001.csv.gz')), 'Radiomics': os.path.join(INPUT_DIR, get_value('science.tables.features_002', 'features_002.csv.gz')), 'Urine': os.path.join(INPUT_DIR, get_value('science.tables.features_004', 'features_004.csv.gz'))}
MRU_MODALITIES = ['Metagenomics', 'Radiomics', 'Urine']
TASKS = ['Response', 'PFS', 'OS']
LAMBDA_RESPONSE = 0.4
LAMBDA_PFS = 0.8
LAMBDA_OS = 1.0
MAX_EPOCHS = 500
MIN_EPOCHS = 50
PATIENCE = 40
MIN_DELTA = 0.0001
LR = 0.001
WEIGHT_DECAY = 0.0001
VAL_SIZE = 0.2
SHAP_BACKGROUND_N = 80



for _m, _p in matrix_files.items():
    pass
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['font.family'] = 'Arial'
mpl.rcParams['axes.linewidth'] = 0.8
mpl.rcParams['axes.spines.top'] = False
mpl.rcParams['axes.spines.right'] = False
mpl.rcParams['figure.dpi'] = 120
MODALITY_COLORS = {'Metagenomics': '#2A9D8F', 'Microbiota': '#2A9D8F', 'Radiomics': '#4C78A8', 'Urine': '#8E6BBE'}
OUTCOME_COLORS = {'Response': '#2D6CDF', 'PFS': '#EE854A', 'OS': '#7A4FA3'}
DIVERGE_CMAP = plt.get_cmap('RdBu_r')

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def normalize_sample_id(x):
    return str(x).lower().strip()

def normalize_feature_name(s):
    s = str(s).lower().strip()
    s = re.sub('\\s+', '', s)
    s = s.replace('_', '')
    s = s.replace('-', '')
    s = s.replace('(', '')
    s = s.replace(')', '')
    s = s.replace('[', '')
    s = s.replace(']', '')
    s = s.replace('.', '')
    return s

def sanitize_filename(s):
    s = str(s)
    s = re.sub('[\\\\/:*?\\"<>|]', '_', s)
    s = s.replace(' ', '_')
    return s

def clean_feature_name(s, max_len=42):
    s = str(s)
    if '__' in s:
        s = s.split('__', 1)[1]
    replace_list = ['original_', 'wavelet-', 'wavelet_', 'log-sigma-', 'log_sigma_', 'glcm_', 'glrlm_', 'glszm_', 'gldm_', 'ngtdm_', 'firstorder_', 'shape_', 'POS_', 'NEG_', 'pos_', 'neg_', 'k__', 'p__', 'c__', 'o__', 'f__', 'g__', 's__']
    for r in replace_list:
        s = s.replace(r, '')
    s = s.replace(';', '|')
    s = s.replace('_', ' ')
    s = re.sub('\\s+', ' ', s).strip()
    if len(s) > max_len:
        s = s[:max_len - 2] + '…'
    return s

def minmax_scale(x):
    x = np.asarray(x, dtype=float)
    xmin = np.nanmin(x)
    xmax = np.nanmax(x)
    if xmax - xmin < 1e-12:
        return np.zeros_like(x)
    return (x - xmin) / (xmax - xmin)

def find_column(df, candidates, required=True):
    lower_map = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    if required:
        raise ValueError(f'Cannot find any of {candidates}. Existing columns: {list(df.columns)}')
    return None

def to_binary_response(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x, str):
        s = x.strip().lower()
        if s in ['1', 'yes', 'y', 'responder', 'response', 'r', 'cr', 'pr', 'orr']:
            return 1
        if s in ['0', 'no', 'n', 'nonresponder', 'non-responder', 'nr', 'sd', 'pd']:
            return 0
    return int(float(x))

def standardize_modal_name(x):
    s = str(x).lower()
    if 'micro' in s or 'meta' in s or 'gut' in s:
        return 'Metagenomics'
    if 'radio' in s or 'ct' in s:
        return 'Radiomics'
    if 'urine' in s:
        return 'Urine'
    if 'blood' in s or 'serum' in s or 'plasma' in s:
        return 'Blood'
    return str(x)

def read_labels(path):
    raw = pd.read_csv(path, encoding='utf-8-sig')
    sample_col = find_column(raw, ['Sample ID', 'sample_id', 'sample', 'Sample', 'ID'])
    response_col = find_column(raw, ['Response', 'Response', 'response', 'ORR'])
    pfs_time_col = find_column(raw, ['PFS_time', 'pfs_time', 'PFS', 'PFS_month', 'PFS_months'])
    pfs_status_col = find_column(raw, ['PFS_status', 'pfs_status', 'PFS_event', 'pfs_event'])
    os_time_col = find_column(raw, ['OS_time', 'os_time', 'OS', 'OS_month', 'OS_months'])
    os_status_col = find_column(raw, ['OS_status', 'os_status', 'OS_event', 'os_event'])
    out = pd.DataFrame()
    out['sample'] = raw[sample_col].map(normalize_sample_id)
    out['Response'] = raw[response_col].map(to_binary_response).astype(float)
    out['PFS_time'] = pd.to_numeric(raw[pfs_time_col], errors='coerce')
    out['PFS_status'] = pd.to_numeric(raw[pfs_status_col], errors='coerce')
    out['OS_time'] = pd.to_numeric(raw[os_time_col], errors='coerce')
    out['OS_status'] = pd.to_numeric(raw[os_status_col], errors='coerce')
    out = out.drop_duplicates('sample', keep='first')
    out = out.set_index('sample')
    out = out.dropna(subset=['Response', 'PFS_time', 'PFS_status', 'OS_time', 'OS_status'])
    
    print('  samples:', out.shape[0])
    print('  Response distribution:')
    
    
    
    return out

def read_matrix(path):
    df = pd.read_csv(path, encoding='utf-8-sig')
    if 'sample' in df.columns:
        sample_col = 'sample'
    elif 'sample_id' in df.columns:
        sample_col = 'sample_id'
    else:
        sample_col = df.columns[0]
    df = df.rename(columns={sample_col: 'sample'})
    df['sample'] = df['sample'].map(normalize_sample_id)
    df = df.drop_duplicates('sample', keep='first')
    df = df.set_index('sample')
    df = df.apply(pd.to_numeric, errors='coerce')
    return df

def apply_internal_transform(X, modal_name):
    pass
    return X.copy()

def load_golden_features_for_mru(path):
    gf = pd.read_csv(path, encoding='utf-8-sig')
    modal_col = find_column(gf, ['modal', 'modality', 'Modality', 'data_type', 'source'], required=True)
    feature_col = find_column(gf, ['feature', 'feature_name', 'Feature', 'selected_feature', 'original_feature'], required=True)
    gf['modal_std'] = gf[modal_col].map(standardize_modal_name)
    gf['feature_std'] = gf[feature_col].astype(str)
    gf['feature_std'] = gf['feature_std'].map(lambda x: x.split('__', 1)[1] if '__' in str(x) else str(x))
    features_by_modal = {}
    for modal in MRU_MODALITIES:
        feats = gf.loc[gf['modal_std'] == modal, 'feature_std'].dropna().astype(str).drop_duplicates().tolist()
        features_by_modal[modal] = feats
        print(f'[Golden features] {modal}: {len(feats)}')
    return (features_by_modal, gf)

def align_features_for_modal(X, required_features, modal_name):
    X = X.copy()
    new_X = pd.DataFrame(index=X.index)
    ext_cols = list(X.columns)
    norm_to_cols = {}
    for c in ext_cols:
        nc = normalize_feature_name(c)
        norm_to_cols.setdefault(nc, []).append(c)
    records = []
    for feat in required_features:
        feat_clean = feat.split('__', 1)[1] if '__' in str(feat) else str(feat)
        matched_col = None
        match_type = 'missing'
        if feat_clean in X.columns:
            matched_col = feat_clean
            match_type = 'exact'
        else:
            nfeat = normalize_feature_name(feat_clean)
            if nfeat in norm_to_cols:
                matched_col = norm_to_cols[nfeat][0]
                match_type = 'normalized_exact'
            else:
                candidates = []
                for nc, cols in norm_to_cols.items():
                    if nfeat in nc or nc in nfeat:
                        candidates.extend(cols)
                if len(candidates) > 0:
                    matched_col = candidates[0]
                    match_type = 'contains'
        if matched_col is None:
            new_X[feat_clean] = np.nan
        else:
            new_X[feat_clean] = X[matched_col]
        records.append({'modal': modal_name, 'required_feature': feat_clean, 'matched_matrix_feature': matched_col, 'match_type': match_type})
    match_df = pd.DataFrame(records)
    n_missing = int((match_df['match_type'] == 'missing').sum())
    print(f'[{modal_name}] required={len(required_features)}, missing={n_missing}')
    if n_missing > 0:
        missing_examples = match_df.loc[match_df['match_type'] == 'missing', 'required_feature'].head(10).tolist()
        raise ValueError(f'{modal_name}: Stage-1 retained features do not match the canonical model-ready matrix. Missing={n_missing}; examples={missing_examples}')
    return (new_X, match_df)

def build_mru_feature_matrix():
    labels = read_labels(label_path)
    features_by_modal, gf = load_golden_features_for_mru(golden_feature_path)
    modal_mats = {}
    match_tables = []
    for modal in MRU_MODALITIES:
        X_raw = read_matrix(matrix_files[modal])
        X_raw = apply_internal_transform(X_raw, modal)
        X_sel, match_df = align_features_for_modal(X=X_raw, required_features=features_by_modal[modal], modal_name=modal)
        modal_mats[modal] = X_sel
        match_tables.append(match_df)
    common_samples = labels.index.copy()
    for modal in MRU_MODALITIES:
        common_samples = common_samples.intersection(modal_mats[modal].index)
    labels = labels.loc[common_samples].copy()
    X_parts = []
    feature_metadata = []
    for modal in MRU_MODALITIES:
        X_modal = modal_mats[modal].loc[common_samples].copy()
        renamed_cols = []
        for feat in X_modal.columns:
            new_feat = f'{modal}__{feat}'
            renamed_cols.append(new_feat)
            feature_metadata.append({'feature': new_feat, 'modal': modal, 'original_feature': feat})
        X_modal.columns = renamed_cols
        X_parts.append(X_modal)
    X_mru = pd.concat(X_parts, axis=1)
    feature_metadata = pd.DataFrame(feature_metadata)
    match_all = pd.concat(match_tables, axis=0)
    X_mru.to_csv(os.path.join(mru_out_dir, get_value('science.tables.features_014', 'features_014.csv')), encoding='utf-8-sig')
    labels.to_csv(os.path.join(mru_out_dir, get_value('science.tables.outcomes_006', 'outcomes_006.csv')), encoding='utf-8-sig')
    feature_metadata.to_csv(os.path.join(mru_out_dir, get_value('science.tables.result_028', 'result_028.csv')), index=False, encoding='utf-8-sig')
    match_all.to_csv(os.path.join(mru_out_dir, get_value('science.tables.result_029', 'result_029.csv')), index=False, encoding='utf-8-sig')
    expected_n_features = sum((len(features_by_modal[m]) for m in MRU_MODALITIES))
    if X_mru.shape[1] != expected_n_features:
        raise ValueError(f'MRU feature-count mismatch: matrix has {X_mru.shape[1]} features, but Stage-1 retained-feature union expects {expected_n_features}.')
    print('\nMRU matrix:', X_mru.shape)
    print('MRU labels:', labels.shape)
    print('MRU feature metadata:', feature_metadata.shape)
    print('MRU feature-count check: PASS')
    return (X_mru, labels, feature_metadata)
X_mru_raw, y_mru, feature_metadata = build_mru_feature_matrix()

class MRUMultiTaskNet(nn.Module):

    def __init__(self, input_dim, hidden1=64, hidden2=16, dropout=0.4):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(input_dim, hidden1), nn.BatchNorm1d(hidden1), nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden1, hidden2), nn.BatchNorm1d(hidden2), nn.ReLU(), nn.Dropout(dropout))
        self.response_head = nn.Linear(hidden2, 1)
        self.pfs_head = nn.Linear(hidden2, 1)
        self.os_head = nn.Linear(hidden2, 1)

    def forward(self, x):
        z = self.encoder(x)
        return {'response_logit': self.response_head(z).view(-1), 'pfs_risk': self.pfs_head(z).view(-1), 'os_risk': self.os_head(z).view(-1)}

def get_mru_hyperparams(input_dim):
    pass
    return {'hidden1': min(64, max(16, input_dim // 2)), 'hidden2': 16, 'dropout': 0.45}

def cox_ph_loss(risk, time, event):
    risk = risk.view(-1)
    time = time.view(-1)
    event = event.view(-1)
    event_sum = torch.sum(event)
    if event_sum <= 0:
        return torch.tensor(0.0, device=risk.device)
    order = torch.argsort(time, descending=True)
    risk_sorted = risk[order]
    event_sorted = event[order]
    log_cumsum_exp = torch.logcumsumexp(risk_sorted, dim=0)
    loss = -torch.sum((risk_sorted - log_cumsum_exp) * event_sorted) / (event_sum + 1e-08)
    return loss

def get_response_pos_weight(y):
    y_resp = y['Response'].values.astype(float)
    n_pos = np.sum(y_resp == 1)
    n_neg = np.sum(y_resp == 0)
    if n_pos == 0:
        return torch.tensor(1.0, dtype=torch.float32, device=DEVICE)
    return torch.tensor(n_neg / max(n_pos, 1), dtype=torch.float32, device=DEVICE)

def make_tensors(y):
    tensors = {'response': torch.tensor(y['Response'].values, dtype=torch.float32, device=DEVICE), 'pfs_time': torch.tensor(y['PFS_time'].values, dtype=torch.float32, device=DEVICE), 'pfs_status': torch.tensor(y['PFS_status'].values, dtype=torch.float32, device=DEVICE), 'os_time': torch.tensor(y['OS_time'].values, dtype=torch.float32, device=DEVICE), 'os_status': torch.tensor(y['OS_status'].values, dtype=torch.float32, device=DEVICE)}
    return tensors

def multitask_loss(out, y_tensors, response_pos_weight):
    bce = nn.BCEWithLogitsLoss(pos_weight=response_pos_weight)
    loss_response = bce(out['response_logit'], y_tensors['response'])
    loss_pfs = cox_ph_loss(out['pfs_risk'], y_tensors['pfs_time'], y_tensors['pfs_status'])
    loss_os = cox_ph_loss(out['os_risk'], y_tensors['os_time'], y_tensors['os_status'])
    total = LAMBDA_RESPONSE * loss_response + LAMBDA_PFS * loss_pfs + LAMBDA_OS * loss_os
    return (total, {'loss_response': float(loss_response.detach().cpu()), 'loss_pfs': float(loss_pfs.detach().cpu()), 'loss_os': float(loss_os.detach().cpu()), 'loss_total': float(total.detach().cpu())})

def concordance_index(time, event, risk):
    time = np.asarray(time, dtype=float)
    event = np.asarray(event, dtype=float)
    risk = np.asarray(risk, dtype=float)
    mask = np.isfinite(time) & np.isfinite(event) & np.isfinite(risk)
    time = time[mask]
    event = event[mask]
    risk = risk[mask]
    permissible = 0
    concordant = 0
    ties = 0
    n = len(time)
    for i in range(n):
        for j in range(n):
            if time[i] < time[j] and event[i] == 1:
                permissible += 1
                if risk[i] > risk[j]:
                    concordant += 1
                elif risk[i] == risk[j]:
                    ties += 1
    if permissible == 0:
        return np.nan
    return (concordant + 0.5 * ties) / permissible

def fit_preprocessor(X_train):
    pass
    imputer = SimpleImputer(strategy='median')
    X_processed = imputer.fit_transform(X_train)
    scaler = None
    return (imputer, scaler)

def transform_preprocessor(X, imputer, scaler):
    return imputer.transform(X)

def determine_best_epoch(X_raw, y):
    set_seed(RANDOM_STATE)
    stratify = y['Response'].values if len(np.unique(y['Response'].values)) == 2 else None
    idx_train, idx_val = train_test_split(np.arange(X_raw.shape[0]), test_size=VAL_SIZE, random_state=RANDOM_STATE, stratify=stratify)
    X_train_raw = X_raw.iloc[idx_train]
    X_val_raw = X_raw.iloc[idx_val]
    y_train = y.iloc[idx_train]
    y_val = y.iloc[idx_val]
    imputer, scaler = fit_preprocessor(X_train_raw)
    X_train = transform_preprocessor(X_train_raw, imputer, scaler)
    X_val = transform_preprocessor(X_val_raw, imputer, scaler)
    input_dim = X_train.shape[1]
    hp = get_mru_hyperparams(input_dim)
    model = MRUMultiTaskNet(input_dim=input_dim, hidden1=hp['hidden1'], hidden2=hp['hidden2'], dropout=hp['dropout']).to(DEVICE)
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    X_train_t = torch.tensor(X_train, dtype=torch.float32, device=DEVICE)
    X_val_t = torch.tensor(X_val, dtype=torch.float32, device=DEVICE)
    y_train_t = make_tensors(y_train)
    y_val_t = make_tensors(y_val)
    response_pos_weight = get_response_pos_weight(y_train)
    best_loss = np.inf
    best_epoch = 1
    wait = 0
    history = []
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        optimizer.zero_grad()
        out_train = model(X_train_t)
        loss_train, loss_train_dict = multitask_loss(out_train, y_train_t, response_pos_weight)
        loss_train.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        model.eval()
        with torch.no_grad():
            out_val = model(X_val_t)
            loss_val, loss_val_dict = multitask_loss(out_val, y_val_t, response_pos_weight)
        val_loss = float(loss_val.detach().cpu())
        history.append({'epoch': epoch, 'train_loss': float(loss_train.detach().cpu()), 'val_loss': val_loss, **{f'val_{k}': v for k, v in loss_val_dict.items()}})
        if val_loss < best_loss - MIN_DELTA:
            best_loss = val_loss
            best_epoch = epoch
            wait = 0
        else:
            wait += 1
        if epoch >= MIN_EPOCHS and wait >= PATIENCE:
            break
    history_df = pd.DataFrame(history)
    history_df.to_csv(os.path.join(mru_out_dir, get_value('science.tables.result_030', 'result_030.csv')), index=False, encoding='utf-8-sig')
    
    return best_epoch

def train_final_mru_model(X_raw, y, best_epoch):
    set_seed(RANDOM_STATE)
    imputer, scaler = fit_preprocessor(X_raw)
    X_scaled = transform_preprocessor(X_raw, imputer, scaler)
    input_dim = X_scaled.shape[1]
    hp = get_mru_hyperparams(input_dim)
    model = MRUMultiTaskNet(input_dim=input_dim, hidden1=hp['hidden1'], hidden2=hp['hidden2'], dropout=hp['dropout']).to(DEVICE)
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    X_t = torch.tensor(X_scaled, dtype=torch.float32, device=DEVICE)
    y_t = make_tensors(y)
    response_pos_weight = get_response_pos_weight(y)
    train_history = []
    for epoch in range(1, best_epoch + 1):
        model.train()
        optimizer.zero_grad()
        out = model(X_t)
        loss, loss_dict = multitask_loss(out, y_t, response_pos_weight)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        train_history.append({'epoch': epoch, **loss_dict})
    model.eval()
    X_scaled_df = pd.DataFrame(X_scaled, index=X_raw.index, columns=X_raw.columns)
    joblib.dump(imputer, os.path.join(mru_out_dir, get_value('science.artifacts.transform_004', 'transform_004.joblib')))
    torch.save({'model_state_dict': model.state_dict(), 'input_dim': input_dim, 'feature_names': list(X_raw.columns), 'hyperparams': hp, 'best_epoch': best_epoch}, os.path.join(mru_out_dir, get_value('science.artifacts.transform_005', 'transform_005.pt')))
    X_scaled_df.to_csv(os.path.join(mru_out_dir, get_value('science.tables.features_015', 'features_015.csv')), encoding='utf-8-sig')
    pd.DataFrame(train_history).to_csv(os.path.join(mru_out_dir, get_value('science.tables.result_031', 'result_031.csv')), index=False, encoding='utf-8-sig')
    return (model, X_scaled_df)
best_epoch = determine_best_epoch(X_mru_raw, y_mru)
mru_model, X_mru_scaled = train_final_mru_model(X_mru_raw, y_mru, best_epoch)

def build_mru_prediction_table(model, X_scaled_df, y):
    X_t = torch.tensor(X_scaled_df.values, dtype=torch.float32, device=DEVICE)
    model.eval()
    with torch.no_grad():
        out = model(X_t)
    pred = pd.DataFrame(index=X_scaled_df.index)
    pred['Response_true'] = y.loc[X_scaled_df.index, 'Response'].values
    pred['Response_pred'] = torch.sigmoid(out['response_logit']).detach().cpu().numpy()
    pred['PFS_time'] = y.loc[X_scaled_df.index, 'PFS_time'].values
    pred['PFS_status'] = y.loc[X_scaled_df.index, 'PFS_status'].values
    pred['PFS_risk'] = out['pfs_risk'].detach().cpu().numpy()
    pred['OS_time'] = y.loc[X_scaled_df.index, 'OS_time'].values
    pred['OS_status'] = y.loc[X_scaled_df.index, 'OS_status'].values
    pred['OS_risk'] = out['os_risk'].detach().cpu().numpy()
    try:
        pred['Response_AUC_internal_refit'] = roc_auc_score(pred['Response_true'], pred['Response_pred'])
    except Exception:
        pred['Response_AUC_internal_refit'] = np.nan
    pred.to_csv(os.path.join(mru_out_dir, get_value('science.tables.predictions_016', 'predictions_016.csv')), encoding='utf-8-sig')
    print(f"MRU final refit performance only for reference | Response AUC={roc_auc_score(pred['Response_true'], pred['Response_pred']):.3f}, PFS C={concordance_index(pred['PFS_time'], pred['PFS_status'], pred['PFS_risk']):.3f}, OS C={concordance_index(pred['OS_time'], pred['OS_status'], pred['OS_risk']):.3f}")
    return pred
pred_mru = build_mru_prediction_table(mru_model, X_mru_scaled, y_mru)

class SHAPTaskWrapper(nn.Module):

    def __init__(self, model, task):
        super().__init__()
        self.model = model
        self.task = task

    def forward(self, x):
        out = self.model(x)
        if self.task == 'Response':
            return out['response_logit'].view(-1, 1)
        elif self.task == 'PFS':
            return out['pfs_risk'].view(-1, 1)
        elif self.task == 'OS':
            return out['os_risk'].view(-1, 1)
        else:
            raise ValueError(self.task)

def compute_shap_for_mru(model, X_scaled_df):
    X_np = X_scaled_df.values.astype(np.float32)
    feature_names = list(X_scaled_df.columns)
    rng = np.random.default_rng(RANDOM_STATE)
    bg_n = min(SHAP_BACKGROUND_N, X_np.shape[0])
    bg_idx = rng.choice(X_np.shape[0], size=bg_n, replace=False)
    background = torch.tensor(X_np[bg_idx], dtype=torch.float32, device=DEVICE)
    X_tensor = torch.tensor(X_np, dtype=torch.float32, device=DEVICE)
    shap_signed = {}
    shap_abs = {}
    importance_records = []
    model.eval()
    for task in TASKS:
        
        wrapper = SHAPTaskWrapper(model, task).to(DEVICE)
        wrapper.eval()
        try:
            explainer = shap.DeepExplainer(wrapper, background)
            shap_values = explainer.shap_values(X_tensor)
        except Exception as e:
            print(f'DeepExplainer failed for {task}: {e}')
            print('Fallback to GradientExplainer...')
            explainer = shap.GradientExplainer(wrapper, background)
            shap_values = explainer.shap_values(X_tensor)
        if isinstance(shap_values, list):
            shap_values = shap_values[0]
        shap_values = np.asarray(shap_values)
        if shap_values.ndim == 3:
            shap_values = np.squeeze(shap_values)
        shap_df = pd.DataFrame(shap_values, index=X_scaled_df.index, columns=feature_names)
        abs_df = shap_df.abs()
        shap_signed[task] = shap_df
        shap_abs[task] = abs_df
        shap_df.to_csv(os.path.join(mru_out_dir, f'MRU_final_shap_signed_{task}.csv.gz'), compression='gzip', encoding='utf-8-sig')
        abs_df.to_csv(os.path.join(mru_out_dir, f'MRU_final_shap_abs_{task}.csv.gz'), compression='gzip', encoding='utf-8-sig')
        for feat in feature_names:
            mean_abs = abs_df[feat].mean()
            mean_signed = shap_df[feat].mean()
            importance_records.append({'outcome': task, 'feature': feat, 'mean_abs_shap': mean_abs, 'mean_signed_shap': mean_signed, 'signed_ratio': mean_signed / (mean_abs + 1e-12)})
    importance_df = pd.DataFrame(importance_records)
    importance_df = importance_df.merge(feature_metadata, on='feature', how='left')
    importance_df.to_csv(os.path.join(mru_out_dir, get_value('science.tables.attribution_013', 'attribution_013.csv')), index=False, encoding='utf-8-sig')
    contrib_records = []
    for task in TASKS:
        tmp = importance_df[importance_df['outcome'] == task].copy()
        modal_sum = tmp.groupby('modal')['mean_abs_shap'].sum().reset_index()
        total = modal_sum['mean_abs_shap'].sum()
        modal_sum['fraction'] = modal_sum['mean_abs_shap'] / (total + 1e-12)
        modal_sum['outcome'] = task
        contrib_records.append(modal_sum)
    contrib_df = pd.concat(contrib_records, axis=0)
    contrib_df.to_csv(os.path.join(mru_out_dir, get_value('science.tables.attribution_014', 'attribution_014.csv')), index=False, encoding='utf-8-sig')
    return (shap_signed, shap_abs, importance_df, contrib_df)
shap_signed_mru, shap_abs_mru, imp_mru, contrib_mru = compute_shap_for_mru(mru_model, X_mru_scaled)

def plot_fig5A_modality_contribution(contrib_df):
    df = contrib_df.copy()
    df['modal_display'] = df['modal'].replace({'Metagenomics': 'Microbiota'})
    df['outcome'] = pd.Categorical(df['outcome'], categories=TASKS, ordered=True)
    modal_order = ['Radiomics', 'Metagenomics', 'Urine']
    modal_display_order = ['Radiomics', 'Microbiota', 'Urine']
    pivot = df.pivot_table(index='outcome', columns='modal', values='fraction', aggfunc='sum').reindex(TASKS).fillna(0)
    for m in modal_order:
        if m not in pivot.columns:
            pivot[m] = 0
    pivot = pivot[modal_order]
    fig, ax = plt.subplots(figsize=(6.6, 2.7))
    y = np.arange(len(TASKS))
    left = np.zeros(len(TASKS))
    for modal, label in zip(modal_order, modal_display_order):
        vals = pivot[modal].values
        color = MODALITY_COLORS[modal]
        ax.barh(y, vals, left=left, height=0.58, color=color, edgecolor='white', linewidth=1.0, label=label)
        for i, v in enumerate(vals):
            if v >= 0.07:
                ax.text(left[i] + v / 2, y[i], f'{v * 100:.0f}%', ha='center', va='center', fontsize=8, color='white', fontweight='bold')
        left += vals
    ax.set_yticks(y)
    ax.set_yticklabels(TASKS, fontsize=10, fontweight='bold')
    ax.set_xlim(0, 1)
    ax.invert_yaxis()
    ax.set_xlabel('Fraction of total mean |SHAP|', fontsize=9)
    ax.xaxis.set_major_formatter(mpl.ticker.PercentFormatter(1.0))
    ax.set_title('MRU modality-level contribution', fontsize=12, fontweight='bold', pad=10)
    ax.grid(axis='x', ls=':', lw=0.5, alpha=0.45)
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.25), ncol=3, frameon=False, fontsize=8.5)
    plt.tight_layout()
    out_png = os.path.join(fig_out_dir, get_value('science.figures.plot_041', 'plot_041.png'))
    out_pdf = os.path.join(fig_out_dir, get_value('science.figures.plot_042', 'plot_042.pdf'))
    plt.savefig(out_png, dpi=600, bbox_inches='tight')
    plt.savefig(out_pdf, dpi=600, bbox_inches='tight')
    plt.show()
    
plot_fig5A_modality_contribution(contrib_mru)


In [ ]:
def select_top_mru_features(imp_df, top_per_outcome=8, max_total=22):
    selected = []
    for task in TASKS:
        sub = imp_df[imp_df['outcome'] == task].sort_values('mean_abs_shap', ascending=False).head(top_per_outcome)
        selected.extend(sub['feature'].astype(str).tolist())
    selected = list(dict.fromkeys(selected))
    pivot = imp_df[imp_df['feature'].isin(selected)].pivot_table(index='feature', columns='outcome', values='mean_abs_shap', aggfunc='mean').fillna(0)
    for task in TASKS:
        if task not in pivot.columns:
            pivot[task] = 0
    pivot['total'] = pivot[TASKS].sum(axis=1)
    selected = pivot.sort_values('total', ascending=False).head(max_total).index.tolist()
    meta = feature_metadata[feature_metadata['feature'].isin(selected)].copy()
    modal_order_map = {'Radiomics': 0, 'Metagenomics': 1, 'Urine': 2}
    sort_df = pd.DataFrame({'feature': selected}).merge(meta, on='feature', how='left').merge(pivot[['total']].reset_index(), on='feature', how='left')
    sort_df['modal_order'] = sort_df['modal'].map(modal_order_map).fillna(99)
    sort_df = sort_df.sort_values(['modal_order', 'total'], ascending=[True, False])
    return sort_df['feature'].tolist()

def build_fig5B_dot_data(selected_features, imp_df, shap_signed, shap_abs):
    records = []
    meta_map = feature_metadata.set_index('feature').to_dict('index')
    for feat in selected_features:
        meta = meta_map.get(feat, {})
        modal = meta.get('modal', 'Unknown')
        original_feature = meta.get('original_feature', feat)
        for task in TASKS:
            if feat in shap_abs[task].columns:
                mean_abs = shap_abs[task][feat].mean()
                mean_signed = shap_signed[task][feat].mean()
                signed_ratio = mean_signed / (mean_abs + 1e-12)
            else:
                mean_abs = 0
                signed_ratio = 0
            records.append({'feature': feat, 'feature_label': clean_feature_name(original_feature, max_len=42), 'modal': modal, 'outcome': task, 'mean_abs_shap': mean_abs, 'signed_ratio': signed_ratio})
    df = pd.DataFrame(records)
    df['size_scaled'] = df['mean_abs_shap'] / (df['mean_abs_shap'].max() + 1e-12)
    return df

def plot_fig5B_top_predictor_dot_heatmap(imp_df, shap_signed, shap_abs, top_per_outcome=8, max_total=22):
    selected_features = select_top_mru_features(imp_df=imp_df, top_per_outcome=top_per_outcome, max_total=max_total)
    df = build_fig5B_dot_data(selected_features=selected_features, imp_df=imp_df, shap_signed=shap_signed, shap_abs=shap_abs)
    total_df = df.groupby(['feature', 'feature_label', 'modal'])['mean_abs_shap'].sum().reset_index()
    modal_order_map = {'Radiomics': 0, 'Metagenomics': 1, 'Urine': 2}
    total_df['modal_order'] = total_df['modal'].map(modal_order_map).fillna(99)
    total_df = total_df.sort_values(['modal_order', 'mean_abs_shap'], ascending=[True, False])
    features_order = total_df['feature'].tolist()
    order_reversed = features_order[::-1]
    y_map = {f: i for i, f in enumerate(order_reversed)}
    x_map = {t: i for i, t in enumerate(TASKS)}
    fig, ax = plt.subplots(figsize=(7.1, max(5.2, len(features_order) * 0.28)))
    for _, row in df.iterrows():
        x = x_map[row['outcome']]
        y = y_map[row['feature']]
        ax.scatter(x, y, s=35 + 520 * row['size_scaled'], c=row['signed_ratio'], cmap=DIVERGE_CMAP, norm=TwoSlopeNorm(vmin=-1, vcenter=0, vmax=1), edgecolor='black', linewidth=0.25, alpha=0.92)
    ylabels = []
    ymodals = []
    meta_map = feature_metadata.set_index('feature').to_dict('index')
    for f in order_reversed:
        meta = meta_map.get(f, {})
        ylabels.append(clean_feature_name(meta.get('original_feature', f), max_len=44))
        ymodals.append(meta.get('modal', 'Unknown'))
    ax.set_yticks(np.arange(len(order_reversed)))
    ax.set_yticklabels(ylabels, fontsize=7.6)
    ax.set_xticks(np.arange(len(TASKS)))
    ax.set_xticklabels(TASKS, fontsize=11, fontweight='bold')
    ax.set_xlim(-0.72, len(TASKS) - 0.35)
    ax.set_ylim(-0.8, len(order_reversed) - 0.2)
    ax.grid(axis='x', ls=':', lw=0.5, alpha=0.4)
    ax.set_title('Top MRU predictors across Response, PFS and OS', fontsize=12, fontweight='bold', pad=14)
    for y, modal in enumerate(ymodals):
        ax.add_patch(Rectangle((-0.66, y - 0.38), 0.1, 0.76, color=MODALITY_COLORS.get(modal, '#999999'), clip_on=False, linewidth=0))
    sm = mpl.cm.ScalarMappable(cmap=DIVERGE_CMAP, norm=TwoSlopeNorm(vmin=-1, vcenter=0, vmax=1))
    cbar = plt.colorbar(sm, ax=ax, fraction=0.035, pad=0.02)
    cbar.set_label('Mean signed SHAP / mean |SHAP|', fontsize=8)
    cbar.ax.tick_params(labelsize=7)
    size_handles = []
    size_labels = []
    for v in [0.25, 0.5, 0.9]:
        size_handles.append(ax.scatter([], [], s=35 + 520 * v, c='lightgrey', edgecolor='black', linewidth=0.25))
        size_labels.append(f'{v:.2f}')
    legend_size = ax.legend(size_handles, size_labels, title='Relative\nmean |SHAP|', loc='lower right', bbox_to_anchor=(1.22, 0.0), frameon=False, fontsize=7, title_fontsize=8)
    ax.add_artist(legend_size)
    modal_handles = [mpl.patches.Patch(color=MODALITY_COLORS['Radiomics'], label='Radiomics'), mpl.patches.Patch(color=MODALITY_COLORS['Metagenomics'], label='Microbiota'), mpl.patches.Patch(color=MODALITY_COLORS['Urine'], label='Urine')]
    ax.legend(handles=modal_handles, loc='upper right', bbox_to_anchor=(1.23, 1.0), frameon=False, fontsize=8, title='Modality', title_fontsize=8)
    plt.tight_layout()
    out_png = os.path.join(fig_out_dir, get_value('science.figures.plot_043', 'plot_043.png'))
    out_pdf = os.path.join(fig_out_dir, get_value('science.figures.plot_044', 'plot_044.pdf'))
    plt.savefig(out_png, dpi=600, bbox_inches='tight')
    plt.savefig(out_pdf, dpi=600, bbox_inches='tight')
    plt.show()
    df.to_csv(os.path.join(fig_out_dir, get_value('science.tables.result_032', 'result_032.csv')), index=False, encoding='utf-8-sig')
    
    return (selected_features, df)
selected_mru_features, fig5B_data = plot_fig5B_top_predictor_dot_heatmap(imp_df=imp_mru, shap_signed=shap_signed_mru, shap_abs=shap_abs_mru, top_per_outcome=8, max_total=22)

def draw_curve(ax, x0, y0, x1, y1, color, lw, alpha=0.35, zorder=1):
    verts = [(x0, y0), (x0 + 0.18, y0), (x1 - 0.18, y1), (x1, y1)]
    codes = [Path.MOVETO, Path.CURVE4, Path.CURVE4, Path.CURVE4]
    path = Path(verts, codes)
    patch = PathPatch(path, facecolor='none', edgecolor=color, lw=lw, alpha=alpha, capstyle='round', joinstyle='round', zorder=zorder)
    ax.add_patch(patch)

def plot_fig5C_mru_flow(selected_features, shap_abs, max_features=15):
    meta_map = feature_metadata.set_index('feature').to_dict('index')
    records = []
    for feat in selected_features:
        meta = meta_map.get(feat, {})
        modal = meta.get('modal', 'Unknown')
        original_feature = meta.get('original_feature', feat)
        task_weights = {}
        total = 0
        for task in TASKS:
            if feat in shap_abs[task].columns:
                w = shap_abs[task][feat].mean()
            else:
                w = 0
            task_weights[task] = w
            total += w
        records.append({'feature': feat, 'original_feature': original_feature, 'modal': modal, 'total': total, **task_weights})
    feat_df = pd.DataFrame(records)
    modal_order_map = {'Radiomics': 0, 'Metagenomics': 1, 'Urine': 2}
    feat_df['modal_order'] = feat_df['modal'].map(modal_order_map).fillna(99)
    feat_df = feat_df.sort_values(['modal_order', 'total'], ascending=[True, False]).head(max_features).reset_index(drop=True)
    max_weight = max(feat_df[TASKS].values.max(), feat_df['total'].max())

    def lw_scale(w):
        return 0.7 + 8.0 * (w / (max_weight + 1e-12))
    modal_labels = ['Radiomics', 'Metagenomics', 'Urine']
    endpoint_labels = TASKS
    modal_y = {m: 1 - (i + 1) / (len(modal_labels) + 1) for i, m in enumerate(modal_labels)}
    endpoint_y = {t: 1 - (i + 1) / (len(endpoint_labels) + 1) for i, t in enumerate(endpoint_labels)}
    n_feat = feat_df.shape[0]
    feature_y = {row['feature']: 1 - (i + 1) / (n_feat + 1) for i, row in feat_df.iterrows()}
    fig, ax = plt.subplots(figsize=(10.2, max(5.0, n_feat * 0.34)))
    ax.set_xlim(-0.03, 1.03)
    ax.set_ylim(0, 1)
    ax.axis('off')
    for _, row in feat_df.iterrows():
        feat = row['feature']
        modal = row['modal']
        color = MODALITY_COLORS.get(modal, '#999999')
        y_m = modal_y.get(modal, 0.5)
        y_f = feature_y[feat]
        draw_curve(ax, 0.1, y_m, 0.48, y_f, color=color, lw=lw_scale(row['total']), alpha=0.28, zorder=1)
        for task in TASKS:
            w = row[task]
            if w <= 0:
                continue
            draw_curve(ax, 0.52, y_f, 0.9, endpoint_y[task], color=color, lw=lw_scale(w), alpha=0.38, zorder=2)
    for modal in modal_labels:
        label = 'Microbiota' if modal == 'Metagenomics' else modal
        color = MODALITY_COLORS.get(modal, '#999999')
        y = modal_y[modal]
        ax.scatter(0.08, y, s=520, color=color, edgecolor='white', linewidth=1.0, zorder=5)
        ax.text(0.0, y, label, ha='right', va='center', fontsize=10, fontweight='bold', color='#333333')
    for _, row in feat_df.iterrows():
        feat = row['feature']
        modal = row['modal']
        y = feature_y[feat]
        color = MODALITY_COLORS.get(modal, '#999999')
        ax.scatter(0.5, y, s=115, color=color, edgecolor='white', linewidth=0.8, zorder=6)
        ax.text(0.505, y, clean_feature_name(row['original_feature'], max_len=42), ha='left', va='center', fontsize=7.5, color='#222222')
    for task in endpoint_labels:
        y = endpoint_y[task]
        color = OUTCOME_COLORS[task]
        ax.scatter(0.92, y, s=600, color=color, edgecolor='white', linewidth=1.0, zorder=5)
        ax.text(0.99, y, task, ha='left', va='center', fontsize=11, fontweight='bold', color='#333333')
    ax.text(0.08, 1.02, 'Modality', ha='center', va='bottom', fontsize=12, fontweight='bold')
    ax.text(0.5, 1.02, 'Top MRU predictors', ha='center', va='bottom', fontsize=12, fontweight='bold')
    ax.text(0.92, 1.02, 'Endpoint', ha='center', va='bottom', fontsize=12, fontweight='bold')
    ax.set_title('Modality-to-feature-to-endpoint SHAP flow in the MRU model', fontsize=13, fontweight='bold', pad=20)
    plt.tight_layout()
    out_png = os.path.join(fig_out_dir, get_value('science.figures.plot_045', 'plot_045.png'))
    out_pdf = os.path.join(fig_out_dir, get_value('science.figures.plot_046', 'plot_046.pdf'))
    plt.savefig(out_png, dpi=600, bbox_inches='tight')
    plt.savefig(out_pdf, dpi=600, bbox_inches='tight')
    plt.show()
    feat_df.to_csv(os.path.join(fig_out_dir, get_value('science.tables.result_033', 'result_033.csv')), index=False, encoding='utf-8-sig')
    
    return feat_df
fig5C_source = plot_fig5C_mru_flow(selected_features=selected_mru_features, shap_abs=shap_abs_mru, max_features=15)

def plot_patient_level_mru_shap_heatmap(task, top_n=18):
    sub = imp_mru[imp_mru['outcome'] == task].sort_values('mean_abs_shap', ascending=False).head(top_n)
    features = sub['feature'].astype(str).tolist()
    features = [f for f in features if f in shap_signed_mru[task].columns]
    pred_col_map = {'Response': 'Response_pred', 'PFS': 'PFS_risk', 'OS': 'OS_risk'}
    true_col_map = {'Response': 'Response_true', 'PFS': 'PFS_status', 'OS': 'OS_status'}
    samples_order = pred_mru.sort_values(pred_col_map[task], ascending=True).index.tolist()
    M = shap_signed_mru[task].loc[samples_order, features].T
    M_norm = M.copy()
    for idx in M_norm.index:
        max_abs = np.nanmax(np.abs(M_norm.loc[idx].values))
        if max_abs > 0:
            M_norm.loc[idx] = M_norm.loc[idx] / max_abs
    M_norm = M_norm.clip(-1, 1)
    meta_map = feature_metadata.set_index('feature').to_dict('index')
    feature_labels = []
    feature_modals = []
    for f in features:
        meta = meta_map.get(f, {})
        feature_labels.append(clean_feature_name(meta.get('original_feature', f), max_len=42))
        feature_modals.append(meta.get('modal', 'Unknown'))
    n_features = len(features)
    fig = plt.figure(figsize=(10.8, max(5.0, n_features * 0.31)))
    gs = GridSpec(nrows=4, ncols=3, height_ratios=[0.25, 0.25, 0.12, 5.0], width_ratios=[0.18, 0.2, 6.5], hspace=0.06, wspace=0.04)
    ax_true = fig.add_subplot(gs[0, 2])
    ax_pred = fig.add_subplot(gs[1, 2])
    ax_gap = fig.add_subplot(gs[2, 2])
    ax_heat = fig.add_subplot(gs[3, 2])
    ax_modal = fig.add_subplot(gs[3, 1])
    true_vals = pred_mru.loc[samples_order, true_col_map[task]].values.astype(float).reshape(1, -1)
    ax_true.imshow(true_vals, aspect='auto', cmap='Greys', interpolation='nearest', vmin=0, vmax=1)
    ax_true.set_yticks([0])
    ax_true.set_yticklabels(['Observed'], fontsize=7)
    ax_true.set_xticks([])
    ax_true.tick_params(length=0)
    pred_vals = pred_mru.loc[samples_order, pred_col_map[task]].values.astype(float)
    pred_vals = minmax_scale(pred_vals).reshape(1, -1)
    ax_pred.imshow(pred_vals, aspect='auto', cmap='viridis', interpolation='nearest', vmin=0, vmax=1)
    ax_pred.set_yticks([0])
    ax_pred.set_yticklabels(['Predicted'], fontsize=7)
    ax_pred.set_xticks([])
    ax_pred.tick_params(length=0)
    ax_gap.axis('off')
    im = ax_heat.imshow(M_norm.values, aspect='auto', cmap=DIVERGE_CMAP, interpolation='nearest', vmin=-1, vmax=1)
    ax_heat.set_yticks(np.arange(n_features))
    ax_heat.set_yticklabels(feature_labels, fontsize=7.2)
    ax_heat.set_xticks([])
    ax_heat.set_xlabel(f'Patients ordered by {task} predicted score', fontsize=9)
    ax_heat.set_title(f'{task}: patient-level signed SHAP patterns', fontsize=12, fontweight='bold', pad=10, color=OUTCOME_COLORS.get(task, '#333333'))
    modal_rgb = []
    for modal in feature_modals:
        color = MODALITY_COLORS.get(modal, '#999999')
        modal_rgb.append(mpl.colors.to_rgb(color))
    modal_rgb = np.array(modal_rgb).reshape(n_features, 1, 3)
    ax_modal.imshow(modal_rgb, aspect='auto', interpolation='nearest')
    ax_modal.set_xticks([])
    ax_modal.set_yticks([])
    ax_modal.set_ylabel('Modality', fontsize=8)
    group_df = pd.DataFrame({'modal': feature_modals, 'y': np.arange(n_features)})
    for modal, g in group_df.groupby('modal'):
        y_center = g['y'].mean()
        label = 'Microbiota' if modal == 'Metagenomics' else modal
        ax_modal.text(-0.45, y_center, label, ha='right', va='center', fontsize=7, color=MODALITY_COLORS.get(modal, '#333333'), fontweight='bold', transform=ax_modal.transData)
    cax = fig.add_axes([0.91, 0.18, 0.012, 0.55])
    cbar = fig.colorbar(im, cax=cax)
    cbar.set_label('Row-scaled signed SHAP', fontsize=8)
    cbar.ax.tick_params(labelsize=7)
    for ax in [ax_true, ax_pred, ax_heat, ax_modal]:
        for spine in ax.spines.values():
            spine.set_visible(False)
    panel_letter = {'Response': 'D', 'PFS': 'E', 'OS': 'F'}[task]
    out_png = os.path.join(fig_out_dir, f"{get_value('science.paths.directory_030', 'directory_030')}{panel_letter}_MRU_patient_level_SHAP_heatmap_{task}.png")
    out_pdf = os.path.join(fig_out_dir, f"{get_value('science.paths.directory_030', 'directory_030')}{panel_letter}_MRU_patient_level_SHAP_heatmap_{task}.pdf")
    plt.savefig(out_png, dpi=600, bbox_inches='tight')
    plt.savefig(out_pdf, dpi=600, bbox_inches='tight')
    plt.show()
    source_df = M_norm.copy()
    source_df.to_csv(os.path.join(fig_out_dir, f"{get_value('science.paths.directory_030', 'directory_030')}{panel_letter}_source_row_scaled_SHAP_{task}.csv"), encoding='utf-8-sig')
    
plot_patient_level_mru_shap_heatmap('Response', top_n=18)
plot_patient_level_mru_shap_heatmap('PFS', top_n=18)
plot_patient_level_mru_shap_heatmap('OS', top_n=18)




In [ ]:
import os
import re
import math
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.colors import Normalize
try:
    from scipy.stats import spearmanr
    SCIPY_AVAILABLE = True
except Exception:
    SCIPY_AVAILABLE = False
base_out_dir = str(project_path(get_value('science.paths.directory_002', 'directory_002')))
mru_out_dir = os.path.join(base_out_dir, get_value('science.paths.directory_028', 'directory_028'))
dep_out_dir = os.path.join(mru_out_dir, get_value('science.paths.directory_029', 'directory_029'), get_value('science.paths.directory_031', 'directory_031'))
os.makedirs(dep_out_dir, exist_ok=True)
individual_dir = os.path.join(dep_out_dir, get_value('science.paths.directory_032', 'directory_032'))
os.makedirs(individual_dir, exist_ok=True)
pdf_dir = os.path.join(dep_out_dir, get_value('science.paths.directory_033', 'directory_033'))
os.makedirs(pdf_dir, exist_ok=True)
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['font.family'] = 'Arial'
mpl.rcParams['axes.linewidth'] = 0.8
mpl.rcParams['axes.spines.top'] = False
mpl.rcParams['axes.spines.right'] = False
mpl.rcParams['figure.dpi'] = 120
TASKS = ['Response', 'PFS', 'OS']
MODALITY_COLORS = {'Radiomics': '#4C78A8', 'Metagenomics': '#2A9D8F', 'Microbiota': '#2A9D8F', 'Urine': '#8E6BBE'}
OUTCOME_COLORS = {'Response': '#2D6CDF', 'PFS': '#EE854A', 'OS': '#7A4FA3'}
PRED_COL_MAP = {'Response': 'Response_pred', 'PFS': 'PFS_risk', 'OS': 'OS_risk'}
TRUE_COL_MAP = {'Response': 'Response_true', 'PFS': 'PFS_status', 'OS': 'OS_status'}
TIME_COL_MAP = {'Response': None, 'PFS': 'PFS_time', 'OS': 'OS_time'}
COLOR_MODE = 'observed'
USE_QUANTILE_AXIS_LIMIT = False
Q_LOW = 0.01
Q_HIGH = 0.99
N_COLS = 3
N_ROWS = 4
PLOTS_PER_PAGE = N_COLS * N_ROWS
SAVE_INDIVIDUAL_PNG = True
SAVE_INDIVIDUAL_PDF = True

def normalize_sample_id(x):
    return str(x).lower().strip()

def read_index_csv(path, compression=None):
    df = pd.read_csv(path, index_col=0, encoding='utf-8-sig', compression=compression)
    df.index = df.index.map(normalize_sample_id)
    return df

def sanitize_filename(s):
    s = str(s)
    s = re.sub('[\\\\/:*?\\"<>|]', '_', s)
    s = s.replace(' ', '_')
    s = s.replace(';', '_')
    s = s.replace('|', '_')
    return s[:180]

def clean_feature_name(s, max_len=46):
    s = str(s)
    if '__' in s:
        s = s.split('__', 1)[1]
    replace_list = ['original_', 'wavelet-', 'wavelet_', 'log-sigma-', 'log_sigma_', 'glcm_', 'glrlm_', 'glszm_', 'gldm_', 'ngtdm_', 'firstorder_', 'shape_', 'POS_', 'NEG_', 'pos_', 'neg_', 'k__', 'p__', 'c__', 'o__', 'f__', 'g__', 's__']
    for r in replace_list:
        s = s.replace(r, '')
    s = s.replace(';', '|')
    s = s.replace('_', ' ')
    s = re.sub('\\s+', ' ', s).strip()
    if len(s) > max_len:
        s = s[:max_len - 2] + '…'
    return s

def minmax_scale(x):
    x = np.asarray(x, dtype=float)
    if np.all(~np.isfinite(x)):
        return np.zeros_like(x)
    xmin = np.nanmin(x)
    xmax = np.nanmax(x)
    if xmax - xmin < 1e-12:
        return np.zeros_like(x)
    return (x - xmin) / (xmax - xmin)

def safe_spearman(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    if mask.sum() < 5:
        return (np.nan, np.nan)
    x = x[mask]
    y = y[mask]
    if len(np.unique(x)) < 3 or len(np.unique(y)) < 3:
        return (np.nan, np.nan)
    if SCIPY_AVAILABLE:
        rho, p = spearmanr(x, y)
        return (rho, p)
    rho = pd.Series(x).corr(pd.Series(y), method='spearman')
    return (rho, np.nan)

def get_meta_maps(feature_metadata):
    meta_map = {}
    for _, row in feature_metadata.iterrows():
        feat = str(row['feature'])
        meta_map[feat] = {'modal': str(row.get('modal', 'Unknown')), 'original_feature': str(row.get('original_feature', feat))}
    return meta_map

def get_feature_meta(feature, meta_map):
    if feature in meta_map:
        return meta_map[feature]
    if '__' in feature:
        modal, original = feature.split('__', 1)
        return {'modal': modal, 'original_feature': original}
    return {'modal': 'Unknown', 'original_feature': feature}

def get_color_values(task, pred_df, sample_index, mode='observed'):
    if mode == 'none':
        return (None, None, None)
    if mode == 'prediction':
        col = PRED_COL_MAP[task]
        vals = pred_df.loc[sample_index, col].values.astype(float)
        vals = minmax_scale(vals)
        return (vals, 'viridis', 'Scaled prediction')
    if mode == 'observed':
        col = TRUE_COL_MAP[task]
        vals = pred_df.loc[sample_index, col].values.astype(float)
        if task == 'Response':
            label = 'Observed response'
        else:
            label = f'{task} event'
        return (vals, 'coolwarm', label)
    if mode == 'time':
        if task == 'Response':
            col = TRUE_COL_MAP[task]
            vals = pred_df.loc[sample_index, col].values.astype(float)
            return (vals, 'coolwarm', 'Observed response')
        col = TIME_COL_MAP[task]
        vals = pred_df.loc[sample_index, col].values.astype(float)
        vals = minmax_scale(vals)
        return (vals, 'viridis', f'Scaled {task} time')
    raise ValueError(f'Unknown COLOR_MODE: {mode}')

def load_mru_shap_package(mru_out_dir):
    x_path = os.path.join(mru_out_dir, get_value('science.tables.features_015', 'features_015.csv'))
    pred_path = os.path.join(mru_out_dir, get_value('science.tables.predictions_016', 'predictions_016.csv'))
    imp_path = os.path.join(mru_out_dir, get_value('science.tables.attribution_013', 'attribution_013.csv'))
    meta_path = os.path.join(mru_out_dir, get_value('science.tables.result_028', 'result_028.csv'))
    if not os.path.exists(x_path):
        raise FileNotFoundError(x_path)
    if not os.path.exists(pred_path):
        raise FileNotFoundError(pred_path)
    if not os.path.exists(imp_path):
        raise FileNotFoundError(imp_path)
    if not os.path.exists(meta_path):
        raise FileNotFoundError(meta_path)
    X = read_index_csv(x_path)
    pred = read_index_csv(pred_path)
    imp = pd.read_csv(imp_path, encoding='utf-8-sig')
    feature_metadata = pd.read_csv(meta_path, encoding='utf-8-sig')
    shap_signed = {}
    for task in TASKS:
        shap_path = os.path.join(mru_out_dir, f'MRU_final_shap_signed_{task}.csv.gz')
        if not os.path.exists(shap_path):
            raise FileNotFoundError(shap_path)
        shap_signed[task] = read_index_csv(shap_path, compression='gzip')
    common = X.index.intersection(pred.index)
    for task in TASKS:
        common = common.intersection(shap_signed[task].index)
    X = X.loc[common].copy()
    pred = pred.loc[common].copy()
    for task in TASKS:
        shap_signed[task] = shap_signed[task].loc[common].copy()
    print('MRU X:', X.shape)
    print('MRU pred:', pred.shape)
    print('MRU SHAP importance:', imp.shape)
    print('MRU metadata:', feature_metadata.shape)
    return (X, pred, imp, feature_metadata, shap_signed)
X_mru, pred_mru, imp_mru, feature_metadata, shap_signed_mru = load_mru_shap_package(mru_out_dir)
meta_map = get_meta_maps(feature_metadata)

def plot_one_dependence(ax, task, feature, rank, X, pred, shap_signed, imp_df, meta_map, color_mode='observed', show_colorbar=False):
    meta = get_feature_meta(feature, meta_map)
    modal = meta['modal']
    original_feature = meta['original_feature']
    x = X[feature].values.astype(float)
    y = shap_signed[task][feature].values.astype(float)
    sample_index = X.index
    mask = np.isfinite(x) & np.isfinite(y)
    x_plot = x[mask]
    y_plot = y[mask]
    sample_index_plot = sample_index[mask]
    c_vals, cmap, c_label = get_color_values(task=task, pred_df=pred, sample_index=sample_index_plot, mode=color_mode)
    if color_mode == 'none':
        sc = ax.scatter(x_plot, y_plot, s=24, color='#7F7F7F', alpha=0.82, edgecolor='white', linewidth=0.25)
    else:
        sc = ax.scatter(x_plot, y_plot, c=c_vals, cmap=cmap, s=25, alpha=0.84, edgecolor='white', linewidth=0.25)
    ax.axhline(0, color='#8A8A8A', lw=0.8, ls='--', alpha=0.75)
    if len(x_plot) >= 5 and len(np.unique(x_plot)) >= 3:
        try:
            coef = np.polyfit(x_plot, y_plot, deg=1)
            xx = np.linspace(np.nanmin(x_plot), np.nanmax(x_plot), 100)
            yy = coef[0] * xx + coef[1]
            ax.plot(xx, yy, color='#222222', lw=1.2, alpha=0.78)
        except Exception:
            pass
    rho, pval = safe_spearman(x_plot, y_plot)
    if USE_QUANTILE_AXIS_LIMIT:
        x_low, x_high = np.nanquantile(x_plot, [Q_LOW, Q_HIGH])
        y_low, y_high = np.nanquantile(y_plot, [Q_LOW, Q_HIGH])
        x_pad = (x_high - x_low) * 0.08 + 1e-12
        y_pad = (y_high - y_low) * 0.1 + 1e-12
        ax.set_xlim(x_low - x_pad, x_high + x_pad)
        ax.set_ylim(y_low - y_pad, y_high + y_pad)
    modal_display = 'Microbiota' if modal == 'Metagenomics' else modal
    title_color = OUTCOME_COLORS.get(task, '#333333')
    ax.set_title(f'#{rank} {task} | {modal_display}', fontsize=9.5, fontweight='bold', color=title_color, pad=7)
    ax.set_xlabel(clean_feature_name(original_feature, max_len=36), fontsize=7.6)
    if task == 'Response':
        y_label = 'SHAP value\n(higher response probability)'
    else:
        y_label = 'SHAP value\n(higher predicted risk)'
    ax.set_ylabel(y_label, fontsize=7.6)
    ax.tick_params(labelsize=7)
    ax.grid(ls=':', lw=0.4, alpha=0.35)
    mean_abs_row = imp_df[(imp_df['outcome'] == task) & (imp_df['feature'].astype(str) == str(feature))]
    if mean_abs_row.shape[0] > 0:
        mean_abs = mean_abs_row['mean_abs_shap'].iloc[0]
    else:
        mean_abs = np.nan
    if np.isfinite(rho):
        ann = f'rho={rho:.2f}\nmean|SHAP|={mean_abs:.3g}'
    else:
        ann = f'mean|SHAP|={mean_abs:.3g}'
    ax.text(0.02, 0.98, ann, transform=ax.transAxes, ha='left', va='top', fontsize=6.8, color='#333333', bbox=dict(boxstyle='round,pad=0.20', facecolor='white', edgecolor='none', alpha=0.65))
    return (sc, c_label)

def make_multipage_dependence_pdf(task):
    task_imp = imp_mru[imp_mru['outcome'] == task].copy().sort_values('mean_abs_shap', ascending=False)
    features = [f for f in task_imp['feature'].astype(str).tolist() if f in X_mru.columns and f in shap_signed_mru[task].columns]
    pdf_path = os.path.join(pdf_dir, f'All_MRU_SHAP_dependence_{task}.pdf')
    print(f'\n[{task}] features to plot:', len(features))
    
    with PdfPages(pdf_path) as pdf:
        n_pages = math.ceil(len(features) / PLOTS_PER_PAGE)
        for page in range(n_pages):
            start = page * PLOTS_PER_PAGE
            end = min((page + 1) * PLOTS_PER_PAGE, len(features))
            page_features = features[start:end]
            fig, axes = plt.subplots(N_ROWS, N_COLS, figsize=(12.0, 12.5), squeeze=False)
            last_sc = None
            last_c_label = None
            for i, feat in enumerate(page_features):
                r = i // N_COLS
                c = i % N_COLS
                ax = axes[r][c]
                rank = start + i + 1
                sc, c_label = plot_one_dependence(ax=ax, task=task, feature=feat, rank=rank, X=X_mru, pred=pred_mru, shap_signed=shap_signed_mru, imp_df=imp_mru, meta_map=meta_map, color_mode=COLOR_MODE)
                last_sc = sc
                last_c_label = c_label
            for j in range(len(page_features), PLOTS_PER_PAGE):
                r = j // N_COLS
                c = j % N_COLS
                axes[r][c].axis('off')
            if COLOR_MODE != 'none' and last_sc is not None:
                cbar = fig.colorbar(last_sc, ax=axes.ravel().tolist(), shrink=0.78, pad=0.012)
                cbar.set_label(last_c_label, fontsize=8)
                cbar.ax.tick_params(labelsize=7)
            fig.suptitle(f'All MRU SHAP dependence plots for {task} (page {page + 1}/{n_pages})', fontsize=14, fontweight='bold', y=1.01)
            plt.tight_layout()
            pdf.savefig(fig, bbox_inches='tight')
            plt.close(fig)
    
for task in TASKS:
    make_multipage_dependence_pdf(task)

def make_individual_dependence_plots(task):
    task_dir = os.path.join(individual_dir, task)
    os.makedirs(task_dir, exist_ok=True)
    task_imp = imp_mru[imp_mru['outcome'] == task].copy().sort_values('mean_abs_shap', ascending=False)
    features = [f for f in task_imp['feature'].astype(str).tolist() if f in X_mru.columns and f in shap_signed_mru[task].columns]
    for rank, feat in enumerate(features, start=1):
        meta = get_feature_meta(feat, meta_map)
        modal = meta['modal']
        original = meta['original_feature']
        modal_display = 'Microbiota' if modal == 'Metagenomics' else modal
        fig, ax = plt.subplots(figsize=(4.0, 3.4))
        sc, c_label = plot_one_dependence(ax=ax, task=task, feature=feat, rank=rank, X=X_mru, pred=pred_mru, shap_signed=shap_signed_mru, imp_df=imp_mru, meta_map=meta_map, color_mode=COLOR_MODE)
        if COLOR_MODE != 'none':
            cbar = plt.colorbar(sc, ax=ax, fraction=0.046, pad=0.04)
            cbar.set_label(c_label, fontsize=7.5)
            cbar.ax.tick_params(labelsize=7)
        fig.suptitle(f'{task} | {modal_display}', fontsize=10.5, fontweight='bold', y=1.03)
        plt.tight_layout()
        fname = f'{rank:03d}_{task}_{modal_display}_{sanitize_filename(clean_feature_name(original, max_len=80))}'
        if SAVE_INDIVIDUAL_PNG:
            out_png = os.path.join(task_dir, fname + '.png')
            plt.savefig(out_png, dpi=600, bbox_inches='tight')
        if SAVE_INDIVIDUAL_PDF:
            out_pdf = os.path.join(task_dir, fname + '.pdf')
            plt.savefig(out_pdf, dpi=600, bbox_inches='tight')
        plt.close(fig)
    
for task in TASKS:
    make_individual_dependence_plots(task)
summary_records = []
for task in TASKS:
    task_imp = imp_mru[imp_mru['outcome'] == task].copy().sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)
    for rank, row in task_imp.iterrows():
        feat = str(row['feature'])
        if feat not in X_mru.columns or feat not in shap_signed_mru[task].columns:
            continue
        meta = get_feature_meta(feat, meta_map)
        x = X_mru[feat].values.astype(float)
        y = shap_signed_mru[task][feat].values.astype(float)
        rho, pval = safe_spearman(x, y)
        n_unique = pd.Series(x).nunique(dropna=True)
        missing_rate = pd.Series(x).isna().mean()
        q01, q99 = np.nanquantile(x, [0.01, 0.99])
        q25, q75 = np.nanquantile(x, [0.25, 0.75])
        iqr = q75 - q25
        outlier_like = bool(iqr > 0 and (np.nanmin(x) < q25 - 3 * iqr or np.nanmax(x) > q75 + 3 * iqr))
        summary_records.append({'outcome': task, 'rank_by_mean_abs_shap': rank + 1, 'feature': feat, 'modal': meta['modal'], 'modal_display': 'Microbiota' if meta['modal'] == 'Metagenomics' else meta['modal'], 'original_feature': meta['original_feature'], 'clean_feature_label': clean_feature_name(meta['original_feature'], max_len=80), 'mean_abs_shap': row.get('mean_abs_shap', np.nan), 'mean_signed_shap': row.get('mean_signed_shap', np.nan), 'signed_ratio': row.get('signed_ratio', np.nan), 'spearman_feature_vs_shap': rho, 'spearman_p': pval, 'n_unique_feature_values': n_unique, 'missing_rate_in_visual_matrix': missing_rate, 'outlier_like_distribution': outlier_like})
summary_df = pd.DataFrame(summary_records)
summary_path = os.path.join(dep_out_dir, get_value('science.tables.attribution_015', 'attribution_015.csv'))
summary_df.to_csv(summary_path, index=False, encoding='utf-8-sig')





In [ ]:
import os
import re
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from sklearn.base import clone
try:
    from lifelines import CoxPHFitter
    LIFELINES_AVAILABLE = True
except Exception:
    LIFELINES_AVAILABLE = False
    print('lifelines is not installed. PFS/OS Cox-model stability will be skipped.')
base_out_dir = str(project_path(get_value('science.paths.directory_002', 'directory_002')))
mru_out_dir = os.path.join(base_out_dir, get_value('science.paths.directory_028', 'directory_028'))
stability_out_dir = os.path.join(mru_out_dir, get_value('science.paths.directory_034', 'directory_034'))
os.makedirs(stability_out_dir, exist_ok=True)
fig_dir = os.path.join(stability_out_dir, 'figures')
os.makedirs(fig_dir, exist_ok=True)
RANDOM_STATE = 42
N_BOOTSTRAP = 100
TOP_K = 10
TASKS = ['Response', 'PFS', 'OS']
MODALITY_COLORS = {'Radiomics': '#4C78A8', 'Metagenomics': '#2A9D8F', 'Microbiota': '#2A9D8F', 'Urine': '#8E6BBE'}
OUTCOME_COLORS = {'Response': '#2D6CDF', 'PFS': '#EE854A', 'OS': '#7A4FA3'}
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['font.family'] = 'Arial'
mpl.rcParams['axes.spines.top'] = False
mpl.rcParams['axes.spines.right'] = False

def normalize_sample_id(x):
    return str(x).lower().strip()

def read_index_csv(path, compression=None):
    df = pd.read_csv(path, index_col=0, encoding='utf-8-sig', compression=compression)
    df.index = df.index.map(normalize_sample_id)
    return df

def clean_feature_name(s, max_len=45):
    s = str(s)
    if '__' in s:
        s = s.split('__', 1)[1]
    replace_list = ['original_', 'wavelet-', 'wavelet_', 'log-sigma-', 'log_sigma_', 'glcm_', 'glrlm_', 'glszm_', 'gldm_', 'ngtdm_', 'firstorder_', 'shape_', 'POS_', 'NEG_', 'pos_', 'neg_']
    for r in replace_list:
        s = s.replace(r, '')
    s = s.replace('_', ' ')
    s = s.replace(';', '|')
    s = re.sub('\\s+', ' ', s).strip()
    if len(s) > max_len:
        s = s[:max_len - 2] + '…'
    return s

def minmax01(x):
    x = np.asarray(x, dtype=float)
    if np.all(~np.isfinite(x)):
        return np.zeros_like(x)
    xmin = np.nanmin(x)
    xmax = np.nanmax(x)
    if xmax - xmin < 1e-12:
        return np.zeros_like(x)
    return (x - xmin) / (xmax - xmin)

def normalize_importance(values):
    values = np.asarray(values, dtype=float)
    values = np.nan_to_num(values, nan=0.0, posinf=0.0, neginf=0.0)
    values = np.maximum(values, 0)
    s = values.sum()
    if s <= 0:
        return np.zeros_like(values)
    return values / s

def stratified_bootstrap_indices(y, random_state):
    rng = np.random.default_rng(random_state)
    y = np.asarray(y)
    idx_all = np.arange(len(y))
    idx0 = idx_all[y == 0]
    idx1 = idx_all[y == 1]
    if len(idx0) == 0 or len(idx1) == 0:
        return rng.choice(idx_all, size=len(idx_all), replace=True)
    boot0 = rng.choice(idx0, size=len(idx0), replace=True)
    boot1 = rng.choice(idx1, size=len(idx1), replace=True)
    boot = np.concatenate([boot0, boot1])
    rng.shuffle(boot)
    return boot

def get_rank_and_selection(importance_values, feature_names, top_k=10):
    imp = np.asarray(importance_values, dtype=float)
    imp = np.nan_to_num(imp, nan=0.0, posinf=0.0, neginf=0.0)
    order = np.argsort(-imp)
    ranks = np.empty_like(order)
    ranks[order] = np.arange(1, len(imp) + 1)
    selected = ranks <= top_k
    return pd.DataFrame({'feature': feature_names, 'importance_norm': normalize_importance(imp), 'rank': ranks, 'selected_topk': selected.astype(int)})

def get_feature_meta(feature, meta_df):
    row = meta_df[meta_df['feature'] == feature]
    if row.shape[0] == 0:
        if '__' in feature:
            modal, original = feature.split('__', 1)
        else:
            modal, original = ('Unknown', feature)
        return (modal, original)
    modal = str(row['modal'].iloc[0])
    original = str(row['original_feature'].iloc[0])
    return (modal, original)
X_path = os.path.join(mru_out_dir, get_value('science.tables.features_015', 'features_015.csv'))
y_path = os.path.join(mru_out_dir, get_value('science.tables.outcomes_006', 'outcomes_006.csv'))
meta_path = os.path.join(mru_out_dir, get_value('science.tables.result_028', 'result_028.csv'))
shap_imp_path = os.path.join(mru_out_dir, get_value('science.tables.attribution_013', 'attribution_013.csv'))
X = read_index_csv(X_path)
y = read_index_csv(y_path)
meta_df = pd.read_csv(meta_path, encoding='utf-8-sig')
shap_imp = pd.read_csv(shap_imp_path, encoding='utf-8-sig')
common = X.index.intersection(y.index)
X = X.loc[common].copy()
y = y.loc[common].copy()
feature_names = list(X.columns)
print('MRU feature matrix:', X.shape)
print('Labels:', y.shape)
print('Features:', len(feature_names))
response_models = {'LogisticEN': LogisticRegression(penalty='elasticnet', solver='saga', l1_ratio=0.5, C=0.5, max_iter=5000, class_weight='balanced', random_state=RANDOM_STATE), 'RandomForest': RandomForestClassifier(n_estimators=500, max_features='sqrt', min_samples_leaf=3, class_weight='balanced_subsample', random_state=RANDOM_STATE, n_jobs=-1), 'ExtraTrees': ExtraTreesClassifier(n_estimators=500, max_features='sqrt', min_samples_leaf=3, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1), 'GradientBoosting': GradientBoostingClassifier(n_estimators=200, learning_rate=0.03, max_depth=2, random_state=RANDOM_STATE)}

def get_classifier_importance(model, feature_names):
    if hasattr(model, 'coef_'):
        imp = np.abs(model.coef_).reshape(-1)
    elif hasattr(model, 'feature_importances_'):
        imp = np.asarray(model.feature_importances_).reshape(-1)
    else:
        imp = np.zeros(len(feature_names))
    return normalize_importance(imp)

def run_response_stability():
    records = []
    y_resp = y['Response'].astype(int).values
    X_values = X.values
    for model_name, model_template in response_models.items():
        print(f'\n[Response] Running {model_name}')
        for b in range(N_BOOTSTRAP):
            seed = RANDOM_STATE + b * 17
            boot_idx = stratified_bootstrap_indices(y_resp, random_state=seed)
            X_boot = X_values[boot_idx]
            y_boot = y_resp[boot_idx]
            if len(np.unique(y_boot)) < 2:
                continue
            model = clone(model_template)
            try:
                model.fit(X_boot, y_boot)
                imp = get_classifier_importance(model, feature_names)
                rank_df = get_rank_and_selection(imp, feature_names, top_k=TOP_K)
                rank_df['outcome'] = 'Response'
                rank_df['model'] = model_name
                rank_df['bootstrap'] = b
                records.append(rank_df)
            except Exception as e:
                print(f'[Response] {model_name} bootstrap {b} failed: {e}')
                continue
    if len(records) == 0:
        return pd.DataFrame()
    return pd.concat(records, axis=0, ignore_index=True)
response_long = run_response_stability()

def run_cox_stability_for_task(task):
    if not LIFELINES_AVAILABLE:
        return pd.DataFrame()
    if task == 'PFS':
        time_col = 'PFS_time'
        event_col = 'PFS_status'
    elif task == 'OS':
        time_col = 'OS_time'
        event_col = 'OS_status'
    else:
        raise ValueError(task)
    cox_models = {'CoxRidge': {'penalizer': 0.1, 'l1_ratio': 0.0}, 'CoxElasticNet': {'penalizer': 0.1, 'l1_ratio': 0.5}}
    records = []
    time = y[time_col].astype(float).values
    event = y[event_col].astype(int).values
    valid = np.isfinite(time) & np.isfinite(event)
    X_task = X.loc[valid].copy()
    time_task = time[valid]
    event_task = event[valid]
    print(f'\n[{task}] valid samples:', X_task.shape[0], '| events:', int(event_task.sum()))
    for model_name, params in cox_models.items():
        print(f'[{task}] Running {model_name}')
        for b in range(N_BOOTSTRAP):
            seed = RANDOM_STATE + b * 19
            boot_idx = stratified_bootstrap_indices(event_task, random_state=seed)
            X_boot = X_task.iloc[boot_idx].copy()
            time_boot = time_task[boot_idx]
            event_boot = event_task[boot_idx]
            if event_boot.sum() < 3:
                continue
            df_boot = X_boot.copy()
            df_boot['_time'] = time_boot
            df_boot['_event'] = event_boot
            cph = CoxPHFitter(penalizer=params['penalizer'], l1_ratio=params['l1_ratio'])
            try:
                cph.fit(df_boot, duration_col='_time', event_col='_event', show_progress=False)
                coef = cph.params_.reindex(feature_names).fillna(0).values
                imp = normalize_importance(np.abs(coef))
                rank_df = get_rank_and_selection(imp, feature_names, top_k=TOP_K)
                rank_df['outcome'] = task
                rank_df['model'] = model_name
                rank_df['bootstrap'] = b
                records.append(rank_df)
            except Exception as e:
                if b < 5:
                    print(f'[{task}] {model_name} bootstrap {b} failed: {e}')
                continue
    if len(records) == 0:
        return pd.DataFrame()
    return pd.concat(records, axis=0, ignore_index=True)
pfs_long = run_cox_stability_for_task('PFS')
os_long = run_cox_stability_for_task('OS')
all_long = pd.concat([response_long, pfs_long, os_long], axis=0, ignore_index=True)
all_long_path = os.path.join(stability_out_dir, get_value('science.tables.attribution_016', 'attribution_016.csv'))
all_long.to_csv(all_long_path, index=False, encoding='utf-8-sig')

print('Long table shape:', all_long.shape)

def build_stability_summary(all_long, shap_imp, meta_df):
    if all_long.shape[0] == 0:
        return pd.DataFrame()
    summary = all_long.groupby(['outcome', 'feature']).agg(mean_alt_importance=('importance_norm', 'mean'), median_alt_rank=('rank', 'median'), mean_alt_rank=('rank', 'mean'), selection_frequency_topk=('selected_topk', 'mean'), n_model_bootstrap_runs=('selected_topk', 'size')).reset_index()
    model_freq = all_long.groupby(['outcome', 'feature', 'model'])['selected_topk'].mean().reset_index()
    support = model_freq.assign(model_supported=lambda d: (d['selected_topk'] >= 0.3).astype(int)).groupby(['outcome', 'feature']).agg(n_supported_models=('model_supported', 'sum'), mean_model_selection_frequency=('selected_topk', 'mean')).reset_index()
    summary = summary.merge(support, on=['outcome', 'feature'], how='left')
    shap_tmp = shap_imp.copy()
    shap_tmp['mru_shap_rank'] = shap_tmp.groupby('outcome')['mean_abs_shap'].rank(ascending=False, method='min')
    shap_tmp = shap_tmp[['outcome', 'feature', 'mean_abs_shap', 'mean_signed_shap', 'signed_ratio', 'mru_shap_rank']]
    summary = summary.merge(shap_tmp, on=['outcome', 'feature'], how='left')
    meta_use = meta_df[['feature', 'modal', 'original_feature']].drop_duplicates('feature')
    summary = summary.merge(meta_use, on='feature', how='left')
    summary['modal_display'] = summary['modal'].replace({'Metagenomics': 'Microbiota'})
    summary['feature_label'] = summary['original_feature'].map(clean_feature_name)
    summary = summary.sort_values(['outcome', 'selection_frequency_topk', 'mean_abs_shap'], ascending=[True, False, False])
    return summary
stability_summary = build_stability_summary(all_long=all_long, shap_imp=shap_imp, meta_df=meta_df)
summary_path = os.path.join(stability_out_dir, get_value('science.tables.metrics_016', 'metrics_016.csv'))
stability_summary.to_csv(summary_path, index=False, encoding='utf-8-sig')


model_level = all_long.groupby(['outcome', 'model', 'feature']).agg(mean_importance=('importance_norm', 'mean'), median_rank=('rank', 'median'), selection_frequency_topk=('selected_topk', 'mean')).reset_index()
model_level = model_level.merge(meta_df[['feature', 'modal', 'original_feature']].drop_duplicates('feature'), on='feature', how='left')
model_level['modal_display'] = model_level['modal'].replace({'Metagenomics': 'Microbiota'})
model_level['feature_label'] = model_level['original_feature'].map(clean_feature_name)
model_level_path = os.path.join(stability_out_dir, get_value('science.tables.result_034', 'result_034.csv'))
model_level.to_csv(model_level_path, index=False, encoding='utf-8-sig')


def plot_stability_dot_heatmap(stability_summary, top_n_per_outcome=12):
    selected = []
    for task in TASKS:
        sub = stability_summary[stability_summary['outcome'] == task].sort_values(['selection_frequency_topk', 'mean_abs_shap'], ascending=[False, False]).head(top_n_per_outcome)
        selected.extend(sub['feature'].tolist())
    selected = list(dict.fromkeys(selected))
    plot_df = stability_summary[stability_summary['feature'].isin(selected)].copy()
    total_df = plot_df.groupby(['feature', 'feature_label', 'modal', 'modal_display']).agg(max_freq=('selection_frequency_topk', 'max'), total_shap=('mean_abs_shap', 'sum')).reset_index()
    modal_order = {'Radiomics': 0, 'Metagenomics': 1, 'Urine': 2}
    total_df['modal_order'] = total_df['modal'].map(modal_order).fillna(99)
    total_df = total_df.sort_values(['modal_order', 'max_freq', 'total_shap'], ascending=[True, False, False])
    features_order = total_df['feature'].tolist()
    order_reversed = features_order[::-1]
    y_map = {f: i for i, f in enumerate(order_reversed)}
    x_map = {t: i for i, t in enumerate(TASKS)}
    fig, ax = plt.subplots(figsize=(7.4, max(5.4, len(features_order) * 0.26)))
    max_shap = plot_df['mean_abs_shap'].max()
    for _, row in plot_df.iterrows():
        x = x_map[row['outcome']]
        y_pos = y_map[row['feature']]
        size = 35 + 520 * (row['mean_abs_shap'] / (max_shap + 1e-12))
        ax.scatter(x, y_pos, s=size, c=row['selection_frequency_topk'], cmap='YlOrRd', vmin=0, vmax=1, edgecolor='black', linewidth=0.25, alpha=0.92)
    ylabels = []
    ymodals = []
    meta_map = total_df.set_index('feature')[['feature_label', 'modal', 'modal_display']].to_dict('index')
    for feat in order_reversed:
        ylabels.append(meta_map[feat]['feature_label'])
        ymodals.append(meta_map[feat]['modal'])
    ax.set_yticks(np.arange(len(order_reversed)))
    ax.set_yticklabels(ylabels, fontsize=7.4)
    ax.set_xticks(np.arange(len(TASKS)))
    ax.set_xticklabels(TASKS, fontsize=11, fontweight='bold')
    ax.set_xlim(-0.72, len(TASKS) - 0.35)
    ax.set_ylim(-0.8, len(order_reversed) - 0.2)
    for y_i, modal in enumerate(ymodals):
        ax.add_patch(mpl.patches.Rectangle((-0.66, y_i - 0.38), 0.1, 0.76, color=MODALITY_COLORS.get(modal, '#999999'), clip_on=False, linewidth=0))
    ax.grid(axis='x', ls=':', lw=0.5, alpha=0.35)
    ax.set_title('Stability of MRU predictors across alternative single-task models', fontsize=12.5, fontweight='bold', pad=14)
    sm = mpl.cm.ScalarMappable(cmap='YlOrRd', norm=Normalize(vmin=0, vmax=1))
    cbar = plt.colorbar(sm, ax=ax, fraction=0.035, pad=0.02)
    cbar.set_label(f'Top-{TOP_K} selection frequency', fontsize=8)
    cbar.ax.tick_params(labelsize=7)
    size_handles = []
    size_labels = []
    for v in [0.25, 0.5, 0.9]:
        size_handles.append(ax.scatter([], [], s=35 + 520 * v, c='lightgrey', edgecolor='black', linewidth=0.25))
        size_labels.append(f'{v:.2f}')
    legend_size = ax.legend(size_handles, size_labels, title='Relative\nMRU-SHAP', loc='lower right', bbox_to_anchor=(1.24, 0.0), frameon=False, fontsize=7, title_fontsize=8)
    ax.add_artist(legend_size)
    modal_handles = [mpl.patches.Patch(color=MODALITY_COLORS['Radiomics'], label='Radiomics'), mpl.patches.Patch(color=MODALITY_COLORS['Metagenomics'], label='Microbiota'), mpl.patches.Patch(color=MODALITY_COLORS['Urine'], label='Urine')]
    ax.legend(handles=modal_handles, loc='upper right', bbox_to_anchor=(1.24, 1.0), frameon=False, fontsize=8, title='Modality', title_fontsize=8)
    plt.tight_layout()
    out_png = os.path.join(fig_dir, get_value('science.figures.plot_047', 'plot_047.png'))
    out_pdf = os.path.join(fig_dir, get_value('science.figures.plot_048', 'plot_048.pdf'))
    plt.savefig(out_png, dpi=600, bbox_inches='tight')
    plt.savefig(out_pdf, dpi=600, bbox_inches='tight')
    plt.show()
    
plot_stability_dot_heatmap(stability_summary, top_n_per_outcome=12)

def plot_model_support_heatmap(task, top_n=18):
    sub_sum = stability_summary[stability_summary['outcome'] == task].sort_values(['selection_frequency_topk', 'mean_abs_shap'], ascending=[False, False]).head(top_n)
    features = sub_sum['feature'].tolist()
    sub_model = model_level[(model_level['outcome'] == task) & model_level['feature'].isin(features)].copy()
    heat_df = sub_model.pivot_table(index='feature', columns='model', values='selection_frequency_topk', aggfunc='mean').fillna(0)
    order_map = {f: i for i, f in enumerate(features)}
    heat_df['order'] = [order_map.get(f, 999) for f in heat_df.index]
    heat_df = heat_df.sort_values('order').drop(columns='order')
    label_map = sub_sum.set_index('feature')['feature_label'].to_dict()
    ylabels = [label_map.get(f, f) for f in heat_df.index]
    fig, ax = plt.subplots(figsize=(6.2, max(4.5, len(ylabels) * 0.28)))
    im = ax.imshow(heat_df.values, aspect='auto', cmap='YlOrRd', vmin=0, vmax=1)
    ax.set_xticks(np.arange(heat_df.shape[1]))
    ax.set_xticklabels(heat_df.columns, fontsize=8.5, rotation=35, ha='right')
    ax.set_yticks(np.arange(heat_df.shape[0]))
    ax.set_yticklabels(ylabels, fontsize=7.4)
    for i in range(heat_df.shape[0]):
        for j in range(heat_df.shape[1]):
            val = heat_df.values[i, j]
            ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=6.6, color='black')
    ax.set_title(f'{task}: top-{TOP_K} selection frequency by alternative model', fontsize=11.5, fontweight='bold', color=OUTCOME_COLORS.get(task, '#333333'), pad=12)
    cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label('Selection frequency', fontsize=8)
    cbar.ax.tick_params(labelsize=7)
    plt.tight_layout()
    out_png = os.path.join(fig_dir, f'MRU_model_support_heatmap_{task}.png')
    out_pdf = os.path.join(fig_dir, f'MRU_model_support_heatmap_{task}.pdf')
    plt.savefig(out_png, dpi=600, bbox_inches='tight')
    plt.savefig(out_pdf, dpi=600, bbox_inches='tight')
    plt.show()
    
for task in TASKS:
    plot_model_support_heatmap(task, top_n=18)

def plot_top_stable_features_bar(task, top_n=12):
    sub = stability_summary[stability_summary['outcome'] == task].sort_values(['selection_frequency_topk', 'mean_abs_shap'], ascending=[False, False]).head(top_n).copy()
    sub = sub.iloc[::-1].copy()
    colors = [MODALITY_COLORS.get(m, '#999999') for m in sub['modal']]
    fig, ax = plt.subplots(figsize=(6.4, max(3.8, top_n * 0.32)))
    ax.barh(np.arange(sub.shape[0]), sub['selection_frequency_topk'], color=colors, edgecolor='black', linewidth=0.5, alpha=0.88)
    ax.set_yticks(np.arange(sub.shape[0]))
    ax.set_yticklabels(sub['feature_label'], fontsize=7.8)
    ax.set_xlim(0, 1.02)
    ax.set_xlabel(f'Top-{TOP_K} selection frequency across alternative models', fontsize=9)
    ax.set_title(f'{task}: most stable MRU predictors', fontsize=12, fontweight='bold', color=OUTCOME_COLORS.get(task, '#333333'), pad=12)
    for i, v in enumerate(sub['selection_frequency_topk']):
        ax.text(v + 0.015, i, f'{v:.2f}', ha='left', va='center', fontsize=7)
    ax.grid(axis='x', ls=':', lw=0.5, alpha=0.45)
    modal_handles = [mpl.patches.Patch(color=MODALITY_COLORS['Radiomics'], label='Radiomics'), mpl.patches.Patch(color=MODALITY_COLORS['Metagenomics'], label='Microbiota'), mpl.patches.Patch(color=MODALITY_COLORS['Urine'], label='Urine')]
    ax.legend(handles=modal_handles, frameon=False, fontsize=8, loc='lower right')
    plt.tight_layout()
    out_png = os.path.join(fig_dir, f'MRU_top_stable_features_bar_{task}.png')
    out_pdf = os.path.join(fig_dir, f'MRU_top_stable_features_bar_{task}.pdf')
    plt.savefig(out_png, dpi=600, bbox_inches='tight')
    plt.savefig(out_pdf, dpi=600, bbox_inches='tight')
    plt.show()
    
for task in TASKS:
    plot_top_stable_features_bar(task, top_n=12)




In [ ]:
import os
import re
import random
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize, LinearSegmentedColormap
from matplotlib.patches import Rectangle, Patch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
import torch
import torch.nn as nn
import torch.nn.functional as F
try:
    import shap
    SHAP_AVAILABLE = True
except Exception:
    SHAP_AVAILABLE = False
    print('WARNING: shap is not installed. Please install it with: pip install shap')
SEED = 42
DEVICE = 'cpu'
TASKS = ['Response', 'PFS', 'OS']
MAX_EPOCHS_FOR_EPOCH_SELECTION = 1000
PATIENCE = 80
LEARNING_RATE = 0.001
WEIGHT_DECAY = 0.0001
DROPOUT = 0.25
LOSS_WEIGHTS = {'Response': 1.0, 'PFS': 1.0, 'OS': 1.0}
BACKGROUND_SIZE = 50
PREFER_DEEP_EXPLAINER = True
base_project_dir = str(project_path(''))
module_dir = os.path.join(base_project_dir, get_value('science.paths.directory_035', 'directory_035'))
module_score_path = os.path.join(module_dir, get_value('science.tables.features_016', 'features_016.csv'))
module_meta_path = os.path.join(module_dir, get_value('science.tables.result_035', 'result_035.csv'))
label_path = os.path.join(base_project_dir, get_value('science.tables.outcomes_007', 'outcomes_007.csv'))
out_dir = os.path.join(module_dir, get_value('science.paths.directory_036', 'directory_036'))
fig_dir = os.path.join(out_dir, 'figures')
os.makedirs(out_dir, exist_ok=True)
os.makedirs(fig_dir, exist_ok=True)
print('module_dir:')











os.makedirs(out_dir, exist_ok=True)
os.makedirs(fig_dir, exist_ok=True)
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['font.family'] = 'Arial'
mpl.rcParams['axes.spines.top'] = False
mpl.rcParams['axes.spines.right'] = False
OUTCOME_COLORS = {'Response': '#2D6CDF', 'PFS': '#EE854A', 'OS': '#7A4FA3'}
MODULE_TYPE_COLORS = {'Microbial KO module': '#2A9D8F', 'Metabolite module': '#D55E00', 'Radiomics module': '#4C78A8'}
SIGNED_CMAP = LinearSegmentedColormap.from_list('signed_shap', ['#3C5488', 'white', '#DC0000'])

def set_seed(seed: int=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def normalize_sample_id(x):
    return str(x).strip().lower()

def clean_label(x, max_len=42):
    x = str(x)
    x = re.sub('^KO::', '', x)
    x = re.sub('^Metab::', '', x)
    x = re.sub('^Rad::', '', x)
    x = x.replace('original_', '')
    x = x.replace('wavelet-', '')
    x = x.replace('log-sigma-', '')
    x = x.replace('_', ' ')
    x = re.sub('\\s+', ' ', x).strip()
    if len(x) > max_len:
        x = x[:max_len - 2] + '…'
    return x

def read_matrix_csv(path):
    if not os.path.exists(path):
        raise FileNotFoundError(path)
    df = pd.read_csv(path, index_col=0, encoding='utf-8-sig')
    df.index = df.index.map(normalize_sample_id)
    for c in df.columns:
        df[c] = pd.to_numeric(df[c], errors='coerce')
    return df

def read_labels(path):
    if not os.path.exists(path):
        raise FileNotFoundError(path)
    df = pd.read_csv(path, encoding='utf-8-sig')
    possible_id_cols = ['sample_id', 'SampleID', 'sample', 'ID', 'id', 'patient_id']
    id_col = None
    for c in possible_id_cols:
        if c in df.columns:
            id_col = c
            break
    if id_col is not None:
        df = df.set_index(id_col)
    else:
        first_col = df.columns[0]
        df = df.set_index(first_col)
    df.index = df.index.map(normalize_sample_id)
    col_map = {}
    lower_cols = {c.lower(): c for c in df.columns}

    def find_col(candidates):
        for cand in candidates:
            if cand.lower() in lower_cols:
                return lower_cols[cand.lower()]
        return None
    col_map['Response'] = find_col(['Response', 'response', 'label', 'Responder'])
    col_map['PFS_time'] = find_col(['PFS_time', 'pfs_time', 'PFS', 'pfs'])
    col_map['PFS_status'] = find_col(['PFS_status', 'pfs_status', 'PFS_event', 'pfs_event'])
    col_map['OS_time'] = find_col(['OS_time', 'os_time', 'OS', 'os'])
    col_map['OS_status'] = find_col(['OS_status', 'os_status', 'OS_event', 'os_event'])
    missing = [k for k, v in col_map.items() if v is None]
    if missing:
        raise ValueError(f'Missing label columns: {missing}\nAvailable columns: {list(df.columns)}')
    out = pd.DataFrame(index=df.index)
    out['Response'] = pd.to_numeric(df[col_map['Response']], errors='coerce').astype(float)
    out['PFS_time'] = pd.to_numeric(df[col_map['PFS_time']], errors='coerce').astype(float)
    out['PFS_status'] = pd.to_numeric(df[col_map['PFS_status']], errors='coerce').astype(float)
    out['OS_time'] = pd.to_numeric(df[col_map['OS_time']], errors='coerce').astype(float)
    out['OS_status'] = pd.to_numeric(df[col_map['OS_status']], errors='coerce').astype(float)
    return out

def read_module_metadata(path):
    if not os.path.exists(path):
        raise FileNotFoundError(path)
    meta = pd.read_csv(path, encoding='utf-8-sig')
    required = ['module_id', 'display_label', 'module_type', 'row_color', 'row_order']
    for c in required:
        if c not in meta.columns:
            raise ValueError(f'module metadata missing column: {c}')
    meta['module_id'] = meta['module_id'].astype(str)
    meta['display_label'] = meta['display_label'].astype(str)
    meta['module_type'] = meta['module_type'].astype(str)
    meta['row_color'] = meta['row_color'].astype(str)
    meta['row_order'] = pd.to_numeric(meta['row_order'], errors='coerce')
    return meta

def neg_cox_partial_log_likelihood(risk, time, event):
    pass
    risk = risk.view(-1)
    time = time.view(-1)
    event = event.view(-1)
    valid = torch.isfinite(time) & torch.isfinite(event)
    risk = risk[valid]
    time = time[valid]
    event = event[valid]
    if risk.numel() == 0 or event.sum() < 1:
        return torch.tensor(0.0, device=risk.device)
    order = torch.argsort(time, descending=True)
    risk_sorted = risk[order]
    event_sorted = event[order]
    log_cum_risk = torch.logcumsumexp(risk_sorted, dim=0)
    pll = risk_sorted - log_cum_risk
    loss = -torch.sum(pll * event_sorted) / (torch.sum(event_sorted) + 1e-08)
    return loss

def compute_pos_weight(y_response):
    y = np.asarray(y_response, dtype=float)
    pos = np.sum(y == 1)
    neg = np.sum(y == 0)
    if pos <= 0:
        return 1.0
    return float(neg / pos)
set_seed(SEED)
X_raw = read_matrix_csv(module_score_path)
labels = read_labels(label_path)
module_meta = read_module_metadata(module_meta_path)
available_modules = [m for m in module_meta['module_id'].tolist() if m in X_raw.columns]
if len(available_modules) == 0:
    raise ValueError('No module IDs in metadata match columns of module score matrix.')
module_meta = module_meta[module_meta['module_id'].isin(available_modules)].copy()
module_meta = module_meta.sort_values('row_order')
X_raw = X_raw.loc[:, module_meta['module_id'].tolist()].copy()
common_samples = X_raw.index.intersection(labels.index)
X_raw = X_raw.loc[common_samples].copy()
labels = labels.loc[common_samples].copy()
valid = labels['Response'].notna() & labels['PFS_time'].notna() & labels['PFS_status'].notna() & labels['OS_time'].notna() & labels['OS_status'].notna()
X_raw = X_raw.loc[valid].copy()
labels = labels.loc[valid].copy()
print('Aligned module score matrix:', X_raw.shape)
print('Aligned labels:', labels.shape)
print(labels[['Response', 'PFS_status', 'OS_status']].sum())
X_raw.to_csv(os.path.join(out_dir, get_value('science.tables.features_017', 'features_017.csv')), encoding='utf-8-sig')
labels.to_csv(os.path.join(out_dir, get_value('science.tables.outcomes_008', 'outcomes_008.csv')), encoding='utf-8-sig')
imputer = SimpleImputer(strategy='median')
scaler = StandardScaler()
X_imp = imputer.fit_transform(X_raw.values)
X_scaled = scaler.fit_transform(X_imp)
X_scaled_df = pd.DataFrame(X_scaled, index=X_raw.index, columns=X_raw.columns)
X_scaled_df.to_csv(os.path.join(out_dir, get_value('science.tables.features_018', 'features_018.csv')), encoding='utf-8-sig')
X_tensor_all = torch.tensor(X_scaled, dtype=torch.float32, device=DEVICE)
y_response = torch.tensor(labels['Response'].values, dtype=torch.float32, device=DEVICE)
pfs_time = torch.tensor(labels['PFS_time'].values, dtype=torch.float32, device=DEVICE)
pfs_event = torch.tensor(labels['PFS_status'].values, dtype=torch.float32, device=DEVICE)
os_time = torch.tensor(labels['OS_time'].values, dtype=torch.float32, device=DEVICE)
os_event = torch.tensor(labels['OS_status'].values, dtype=torch.float32, device=DEVICE)
input_dim = X_scaled.shape[1]

class ModuleLevelMultiTaskNet(nn.Module):

    def __init__(self, input_dim, hidden_dims=(64, 32), dropout=0.25):
        super().__init__()
        layers = []
        prev = input_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.BatchNorm1d(h))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            prev = h
        self.shared = nn.Sequential(*layers)
        self.response_head = nn.Linear(prev, 1)
        self.pfs_head = nn.Linear(prev, 1)
        self.os_head = nn.Linear(prev, 1)

    def forward(self, x):
        z = self.shared(x)
        response_logit = self.response_head(z).view(-1)
        pfs_risk = self.pfs_head(z).view(-1)
        os_risk = self.os_head(z).view(-1)
        return (response_logit, pfs_risk, os_risk)

def multitask_loss(model, X, y_resp, pfs_t, pfs_e, os_t, os_e, pos_weight_tensor):
    response_logit, pfs_risk, os_risk = model(X)
    loss_resp = F.binary_cross_entropy_with_logits(response_logit, y_resp, pos_weight=pos_weight_tensor)
    loss_pfs = neg_cox_partial_log_likelihood(pfs_risk, pfs_t, pfs_e)
    loss_os = neg_cox_partial_log_likelihood(os_risk, os_t, os_e)
    total = LOSS_WEIGHTS['Response'] * loss_resp + LOSS_WEIGHTS['PFS'] * loss_pfs + LOSS_WEIGHTS['OS'] * loss_os
    return (total, {'loss_total': float(total.detach().cpu()), 'loss_response': float(loss_resp.detach().cpu()), 'loss_pfs': float(loss_pfs.detach().cpu()), 'loss_os': float(loss_os.detach().cpu())})

def choose_best_epoch():
    idx = np.arange(X_scaled.shape[0])
    y_np = labels['Response'].values.astype(int)
    stratify = y_np if len(np.unique(y_np)) == 2 and min(np.bincount(y_np)) >= 3 else None
    train_idx, val_idx = train_test_split(idx, test_size=0.2, random_state=SEED, stratify=stratify)
    X_train = X_tensor_all[train_idx]
    X_val = X_tensor_all[val_idx]
    y_train = y_response[train_idx]
    y_val = y_response[val_idx]
    pfs_t_train = pfs_time[train_idx]
    pfs_e_train = pfs_event[train_idx]
    pfs_t_val = pfs_time[val_idx]
    pfs_e_val = pfs_event[val_idx]
    os_t_train = os_time[train_idx]
    os_e_train = os_event[train_idx]
    os_t_val = os_time[val_idx]
    os_e_val = os_event[val_idx]
    pos_weight = compute_pos_weight(labels['Response'].values[train_idx])
    pos_weight_tensor = torch.tensor(pos_weight, dtype=torch.float32, device=DEVICE)
    model = ModuleLevelMultiTaskNet(input_dim=input_dim, hidden_dims=(64, 32), dropout=DROPOUT).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    best_epoch = 1
    best_val_loss = np.inf
    best_state = None
    patience_counter = 0
    history = []
    for epoch in range(1, MAX_EPOCHS_FOR_EPOCH_SELECTION + 1):
        model.train()
        optimizer.zero_grad()
        train_loss, train_parts = multitask_loss(model, X_train, y_train, pfs_t_train, pfs_e_train, os_t_train, os_e_train, pos_weight_tensor)
        train_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        model.eval()
        with torch.no_grad():
            val_loss, val_parts = multitask_loss(model, X_val, y_val, pfs_t_val, pfs_e_val, os_t_val, os_e_val, pos_weight_tensor)
        history.append({'epoch': epoch, 'train_loss': train_parts['loss_total'], 'val_loss': val_parts['loss_total'], 'train_response': train_parts['loss_response'], 'train_pfs': train_parts['loss_pfs'], 'train_os': train_parts['loss_os'], 'val_response': val_parts['loss_response'], 'val_pfs': val_parts['loss_pfs'], 'val_os': val_parts['loss_os']})
        if val_parts['loss_total'] < best_val_loss - 1e-05:
            best_val_loss = val_parts['loss_total']
            best_epoch = epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
        if patience_counter >= PATIENCE:
            break
    history_df = pd.DataFrame(history)
    history_df.to_csv(os.path.join(out_dir, get_value('science.tables.result_036', 'result_036.csv')), index=False, encoding='utf-8-sig')
    
    return (best_epoch, best_state, history_df)
best_epoch, best_state, history_df = choose_best_epoch()

def refit_full_model(best_epoch):
    pos_weight = compute_pos_weight(labels['Response'].values)
    pos_weight_tensor = torch.tensor(pos_weight, dtype=torch.float32, device=DEVICE)
    model = ModuleLevelMultiTaskNet(input_dim=input_dim, hidden_dims=(64, 32), dropout=DROPOUT).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    full_history = []
    for epoch in range(1, best_epoch + 1):
        model.train()
        optimizer.zero_grad()
        loss, parts = multitask_loss(model, X_tensor_all, y_response, pfs_time, pfs_event, os_time, os_event, pos_weight_tensor)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        row = {'epoch': epoch}
        row.update(parts)
        full_history.append(row)
    full_history_df = pd.DataFrame(full_history)
    full_history_df.to_csv(os.path.join(out_dir, get_value('science.tables.result_037', 'result_037.csv')), index=False, encoding='utf-8-sig')
    model.eval()
    return (model, full_history_df)
model, full_history_df = refit_full_model(best_epoch)
model_path = os.path.join(out_dir, get_value('science.artifacts.transform_006', 'transform_006.pt'))
torch.save({'model_state_dict': model.state_dict(), 'input_dim': input_dim, 'module_ids': list(X_scaled_df.columns), 'best_epoch': best_epoch, 'loss_weights': LOSS_WEIGHTS}, model_path)

model.eval()
with torch.no_grad():
    resp_logit, pfs_risk, os_risk = model(X_tensor_all)
    pred_df = pd.DataFrame({'Response_true': labels['Response'].values, 'Response_logit': resp_logit.cpu().numpy(), 'Response_pred': torch.sigmoid(resp_logit).cpu().numpy(), 'PFS_time': labels['PFS_time'].values, 'PFS_status': labels['PFS_status'].values, 'PFS_risk': pfs_risk.cpu().numpy(), 'OS_time': labels['OS_time'].values, 'OS_status': labels['OS_status'].values, 'OS_risk': os_risk.cpu().numpy()}, index=X_scaled_df.index)
pred_df.to_csv(os.path.join(out_dir, get_value('science.tables.predictions_017', 'predictions_017.csv')), encoding='utf-8-sig')
if not SHAP_AVAILABLE:
    raise ImportError('Please install shap first: pip install shap')

class TaskWrapper(nn.Module):

    def __init__(self, base_model, task):
        super().__init__()
        self.base_model = base_model
        self.task = task

    def forward(self, x):
        response_logit, pfs_risk, os_risk = self.base_model(x)
        if self.task == 'Response':
            return response_logit.view(-1, 1)
        elif self.task == 'PFS':
            return pfs_risk.view(-1, 1)
        elif self.task == 'OS':
            return os_risk.view(-1, 1)
        else:
            raise ValueError(self.task)

def extract_shap_array(shap_values):
    if isinstance(shap_values, list):
        arr = shap_values[0]
    else:
        arr = shap_values
    if torch.is_tensor(arr):
        arr = arr.detach().cpu().numpy()
    arr = np.asarray(arr)
    if arr.ndim == 3 and arr.shape[-1] == 1:
        arr = arr[:, :, 0]
    return arr

def compute_shap_for_task(task):
    wrapper = TaskWrapper(model, task).to(DEVICE)
    wrapper.eval()
    n = X_tensor_all.shape[0]
    rng = np.random.default_rng(SEED)
    bg_n = min(BACKGROUND_SIZE, n)
    bg_idx = rng.choice(np.arange(n), size=bg_n, replace=False)
    background = X_tensor_all[bg_idx]
    X_explain = X_tensor_all
    shap_values = None
    explainer_used = None
    if PREFER_DEEP_EXPLAINER:
        try:
            explainer = shap.DeepExplainer(wrapper, background)
            shap_values = explainer.shap_values(X_explain)
            explainer_used = 'DeepExplainer'
        except Exception as e:
            print(f'[{task}] DeepExplainer failed: {e}')
    if shap_values is None:
        explainer = shap.GradientExplainer(wrapper, background)
        shap_values = explainer.shap_values(X_explain)
        explainer_used = 'GradientExplainer'
    shap_arr = extract_shap_array(shap_values)
    if shap_arr.shape != X_scaled_df.shape:
        raise ValueError(f'Unexpected SHAP shape for {task}: {shap_arr.shape}, expected {X_scaled_df.shape}')
    signed_df = pd.DataFrame(shap_arr, index=X_scaled_df.index, columns=X_scaled_df.columns)
    abs_df = signed_df.abs()
    signed_path = os.path.join(out_dir, f"{get_value('science.paths.directory_037', 'directory_037')}{task}.csv.gz")
    abs_path = os.path.join(out_dir, f"{get_value('science.paths.directory_038', 'directory_038')}{task}.csv.gz")
    signed_df.to_csv(signed_path, compression='gzip', encoding='utf-8-sig')
    abs_df.to_csv(abs_path, compression='gzip', encoding='utf-8-sig')
    
    
    
    return (signed_df, abs_df, explainer_used)
shap_signed = {}
shap_abs = {}
explainer_log = []
for task in TASKS:
    signed_df, abs_df, explainer_used = compute_shap_for_task(task)
    shap_signed[task] = signed_df
    shap_abs[task] = abs_df
    explainer_log.append({'outcome': task, 'explainer': explainer_used})
pd.DataFrame(explainer_log).to_csv(os.path.join(out_dir, get_value('science.tables.attribution_017', 'attribution_017.csv')), index=False, encoding='utf-8-sig')
records = []
for task in TASKS:
    signed = shap_signed[task]
    absval = shap_abs[task]
    for module_id in X_scaled_df.columns:
        meta_row = module_meta[module_meta['module_id'] == module_id]
        if meta_row.shape[0] == 0:
            display_label = module_id
            module_type = 'Unknown'
            row_color = '#BDBDBD'
            row_order = 999999
        else:
            display_label = str(meta_row['display_label'].iloc[0])
            module_type = str(meta_row['module_type'].iloc[0])
            row_color = str(meta_row['row_color'].iloc[0])
            row_order = float(meta_row['row_order'].iloc[0])
        mean_abs = float(absval[module_id].mean())
        mean_signed = float(signed[module_id].mean())
        records.append({'module_id': module_id, 'display_label': display_label, 'module_type': module_type, 'row_color': row_color, 'row_order': row_order, 'outcome': task, 'mean_abs_shap': mean_abs, 'mean_signed_shap': mean_signed, 'signed_ratio': mean_signed / (mean_abs + 1e-12)})
contrib = pd.DataFrame(records)
contrib['signed_ratio'] = contrib['signed_ratio'].clip(-1, 1)
contrib['module_abs_scaled'] = contrib.groupby('outcome')['mean_abs_shap'].transform(lambda x: x / (x.max() + 1e-12))
contrib['signed_scaled'] = np.sign(contrib['mean_signed_shap']) * contrib['module_abs_scaled']
contrib = contrib.sort_values(['row_order', 'outcome'])
contrib_path = str(project_path(get_value('science.tables.attribution_018', 'attribution_018.csv')))
contrib.to_csv(contrib_path, index=False, encoding='utf-8-sig')


def prepare_plot_data(contrib_df, include_types):
    df = contrib_df[contrib_df['module_type'].isin(include_types)].copy()
    row_df = df[['module_id', 'display_label', 'module_type', 'row_color', 'row_order']].drop_duplicates('module_id').sort_values('row_order')
    return (df, row_df)

def plot_dotmatrix(contrib_df, scope_name, include_types, show_row_names=True):
    df, row_df = prepare_plot_data(contrib_df, include_types)
    if row_df.empty:
        print(f'No rows for {scope_name}')
        return
    modules = row_df['module_id'].tolist()
    size_mat = df.pivot(index='module_id', columns='outcome', values='module_abs_scaled').reindex(index=modules, columns=TASKS).fillna(0)
    ratio_mat = df.pivot(index='module_id', columns='outcome', values='signed_ratio').reindex(index=modules, columns=TASKS).fillna(0).clip(-1, 1)
    n_rows = len(modules)
    n_cols = len(TASKS)
    fig_h = max(3.5, n_rows * 0.22 + 0.8)
    fig_w = 5.8 if show_row_names else 2.4
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    for i in range(n_rows):
        for j in range(n_cols):
            ax.add_patch(Rectangle((j - 0.5, i - 0.5), 1, 1, facecolor='white', edgecolor='#E8E8E8', linewidth=0.5))
    for i, module_id in enumerate(modules):
        for j, task in enumerate(TASKS):
            v = float(size_mat.loc[module_id, task])
            r = float(ratio_mat.loc[module_id, task])
            if v > 0:
                ax.scatter(j, i, s=35 + 520 * np.sqrt(v), c=[r], cmap=SIGNED_CMAP, vmin=-1, vmax=1, edgecolor='#666666', linewidth=0.45, zorder=3)
    for i, (_, row) in enumerate(row_df.iterrows()):
        type_color = MODULE_TYPE_COLORS.get(row['module_type'], '#BDBDBD')
        module_color = row['row_color'] if isinstance(row['row_color'], str) else '#BDBDBD'
        ax.add_patch(Rectangle((-0.78, i - 0.45), 0.12, 0.9, facecolor=type_color, edgecolor='none', clip_on=False))
        ax.add_patch(Rectangle((-0.63, i - 0.45), 0.12, 0.9, facecolor=module_color, edgecolor='none', clip_on=False))
    ax.set_xlim(-0.9, n_cols - 0.35)
    ax.set_ylim(n_rows - 0.5, -0.5)
    ax.set_xticks(range(n_cols))
    ax.set_xticklabels(TASKS, rotation=90, fontsize=8, fontweight='bold')
    ax.set_yticks(range(n_rows))
    if show_row_names:
        ax.set_yticklabels([clean_label(x, 42) for x in row_df['display_label']], fontsize=6.7)
    else:
        ax.set_yticklabels([''] * n_rows)
    ax.tick_params(axis='y', length=0)
    ax.tick_params(axis='x', length=0)
    ax.set_title('Contribution to outcome prediction', fontsize=10, fontweight='bold', pad=10)
    sm = mpl.cm.ScalarMappable(cmap=SIGNED_CMAP, norm=Normalize(vmin=-1, vmax=1))
    cbar = plt.colorbar(sm, ax=ax, fraction=0.045, pad=0.06)
    cbar.set_label('Signed\nSHAP ratio', fontsize=7)
    cbar.set_ticks([-1, 0, 1])
    cbar.set_ticklabels(['Negative', 'Neutral', 'Positive'])
    cbar.ax.tick_params(labelsize=6.5)
    size_handles = []
    size_labels = []
    for v in [0.25, 0.5, 1.0]:
        size_handles.append(ax.scatter([], [], s=35 + 520 * np.sqrt(v), c='#BDBDBD', edgecolor='#666666', linewidth=0.45))
        size_labels.append(f'{v:.2f}')
    ax.legend(size_handles, size_labels, title='Relative\nmodule |SHAP|', loc='lower left', bbox_to_anchor=(1.08, 0.02), frameon=False, fontsize=6.5, title_fontsize=7)
    plt.tight_layout()
    out_png = os.path.join(fig_dir, f"{scope_name}{get_value('science.figures.plot_049', 'plot_049.png')}")
    out_pdf = os.path.join(fig_dir, f"{scope_name}{get_value('science.figures.plot_050', 'plot_050.pdf')}")
    plt.savefig(out_png, dpi=600, bbox_inches='tight')
    plt.savefig(out_pdf, bbox_inches='tight')
    plt.show()
    


In [ ]:
import os
import re
import textwrap
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
base_project_dir = str(project_path(''))
contrib_path = os.path.join(base_project_dir, get_value('science.tables.attribution_019', 'attribution_019.csv'))
meta_path = os.path.join(base_project_dir, get_value('science.tables.attribution_020', 'attribution_020.csv'))
out_dir = os.path.join(base_project_dir, get_value('science.paths.directory_039', 'directory_039'))
os.makedirs(out_dir, exist_ok=True)





mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['font.family'] = 'Arial'
mpl.rcParams['axes.spines.top'] = False
mpl.rcParams['axes.spines.right'] = False
OUTCOMES = ['Response', 'PFS', 'OS']
MODULE_TYPE_ORDER = {'Microbial KO module': 0, 'Metabolite module': 1, 'Radiomics module': 2}
FIG_WIDTH = 10
ROW_HEIGHT = 1.55
FIG_MIN_HEIGHT = 5.8
HSPACE = 1.05
DROP_AXIS_INDEXES = [2]
DROP_AXIS_NAMES = []
USE_ALL_RADIOMICS_FOR_EACH_AXIS = True
USE_SIGNED_VALUE = True
SHOW_LEFT_LAYER_STRIPS = False
SHOW_MODALITY_LEGEND = False
AXIS_LABEL_Y = -0.36
if not os.path.exists(contrib_path):
    raise FileNotFoundError(contrib_path)
if not os.path.exists(meta_path):
    raise FileNotFoundError(meta_path)
contrib = pd.read_csv(contrib_path, encoding='utf-8-sig')
meta = pd.read_csv(meta_path, encoding='utf-8-sig')
print('\nContribution columns:')
print(list(contrib.columns))
print('\nMetadata columns:')
print(list(meta.columns))
if 'module_uid' in contrib.columns:
    contrib['module_uid_for_plot'] = contrib['module_uid'].astype(str)
else:
    contrib['module_uid_for_plot'] = contrib['module_id'].astype(str)
if 'module_uid' in meta.columns:
    meta['module_uid_for_plot'] = meta['module_uid'].astype(str)
else:
    meta['module_uid_for_plot'] = meta['module_id'].astype(str)
if 'axis' not in contrib.columns:
    contrib = contrib.merge(meta[['module_uid_for_plot', 'axis']], on='module_uid_for_plot', how='left')
required_cols = ['module_uid_for_plot', 'module_id', 'display_label', 'module_type', 'row_color', 'row_order', 'outcome']
for c in required_cols:
    if c not in contrib.columns:
        raise ValueError(f'Missing required column in contribution table: {c}')
if USE_SIGNED_VALUE:
    if 'signed_scaled' in contrib.columns:
        VALUE_COL = 'signed_scaled'
    elif 'mean_signed_shap' in contrib.columns:
        VALUE_COL = 'mean_signed_shap'
    else:
        raise ValueError('No signed SHAP column found: signed_scaled or mean_signed_shap')
elif 'module_abs_scaled' in contrib.columns:
    VALUE_COL = 'module_abs_scaled'
elif 'mean_abs_shap' in contrib.columns:
    VALUE_COL = 'mean_abs_shap'
else:
    raise ValueError('No absolute SHAP column found: module_abs_scaled or mean_abs_shap')
print('\nUsing SHAP value column:', VALUE_COL)
for c in ['row_order', VALUE_COL]:
    contrib[c] = pd.to_numeric(contrib[c], errors='coerce')
if 'row_order' in meta.columns:
    meta['row_order'] = pd.to_numeric(meta['row_order'], errors='coerce')
RADIOMICS_MODULE_COLORS = {'Morphology': '#2F5597', 'Low-gray': '#5B9BD5', 'Coarse texture': '#7F95B8', 'Fine heterogeneity': '#6F6FAE'}
for module_name, color in RADIOMICS_MODULE_COLORS.items():
    mask_contrib = (contrib['module_type'] == 'Radiomics module') & (contrib['display_label'].astype(str).str.contains(module_name, case=False, regex=False) | contrib['module_id'].astype(str).str.contains(module_name, case=False, regex=False) | contrib['module_uid_for_plot'].astype(str).str.contains(module_name, case=False, regex=False))
    contrib.loc[mask_contrib, 'row_color'] = color
    mask_meta = (meta['module_type'] == 'Radiomics module') & (meta['display_label'].astype(str).str.contains(module_name, case=False, regex=False) | meta['module_id'].astype(str).str.contains(module_name, case=False, regex=False) | meta['module_uid_for_plot'].astype(str).str.contains(module_name, case=False, regex=False))
    meta.loc[mask_meta, 'row_color'] = color

def valid_color(x, default='#BDBDBD'):
    x = str(x)
    if re.match('^#[0-9A-Fa-f]{6}$', x):
        return x
    return default

def wrap_label(x, width=42):
    x = str(x)
    x = x.replace(' Axis', '\nAxis')
    return '\n'.join(textwrap.wrap(x, width=width))

def clean_axis_name(x):
    return str(x).strip()
axis_order = meta.loc[meta['module_type'].isin(['Microbial KO module', 'Metabolite module']), 'axis'].dropna().astype(str).map(clean_axis_name).tolist()
axis_order = list(dict.fromkeys(axis_order))
axis_order = [x for x in axis_order if x.lower() not in ['radiomics', 'radiomics module', 'nan', '']]
original_axis_order = axis_order.copy()
if len(DROP_AXIS_NAMES) > 0:
    axis_order = [a for a in axis_order if a not in DROP_AXIS_NAMES]
if len(DROP_AXIS_INDEXES) > 0:
    axis_order = [a for i, a in enumerate(axis_order) if i not in DROP_AXIS_INDEXES]
print('\nOriginal axes:')
for i, a in enumerate(original_axis_order):
    pass
print('\nAxes kept for plotting:')
for i, a in enumerate(axis_order):
    pass
if len(axis_order) == 0:
    raise ValueError('No biological axes found after filtering.')
all_radiomics_modules = meta.loc[meta['module_type'] == 'Radiomics module', 'module_uid_for_plot'].dropna().astype(str).tolist()
axis_module_rows = []
for axis_name in axis_order:
    sub_meta = meta[(meta['axis'].astype(str) == axis_name) & meta['module_type'].isin(['Microbial KO module', 'Metabolite module'])].copy()
    module_uids = sub_meta['module_uid_for_plot'].astype(str).tolist()
    if USE_ALL_RADIOMICS_FOR_EACH_AXIS:
        module_uids.extend(all_radiomics_modules)
    module_uids = list(dict.fromkeys(module_uids))
    axis_module_rows.append({'axis': axis_name, 'module_uids': module_uids, 'n_modules': len(module_uids)})
axis_module_df = pd.DataFrame(axis_module_rows)
axis_map_path = os.path.join(out_dir, get_value('science.tables.result_038', 'result_038.csv'))
axis_module_df.to_csv(axis_map_path, index=False, encoding='utf-8-sig')

plot_records = []
for _, row in axis_module_df.iterrows():
    axis_name = row['axis']
    module_uids = row['module_uids']
    sub = contrib[contrib['module_uid_for_plot'].isin(module_uids) & contrib['outcome'].isin(OUTCOMES)].copy()
    if sub.empty:
        print('[Warning] Empty axis:', axis_name)
        continue
    for _, r in sub.iterrows():
        val = r[VALUE_COL]
        if not np.isfinite(val):
            continue
        plot_records.append({'axis': axis_name, 'module_uid_for_plot': r['module_uid_for_plot'], 'module_id': r['module_id'], 'display_label': r['display_label'], 'module_type': r['module_type'], 'row_color': valid_color(r['row_color']), 'row_order': r['row_order'], 'outcome': r['outcome'], 'plot_value_raw': float(val), 'abs_value_raw': abs(float(val))})
plot_df = pd.DataFrame(plot_records)
if plot_df.empty:
    raise ValueError('No plotting data generated.')
plot_df['module_type_rank'] = plot_df['module_type'].map(MODULE_TYPE_ORDER).fillna(99)
bar_total = plot_df.groupby(['axis', 'outcome'])['abs_value_raw'].sum().reset_index(name='total_abs')
global_max_total = bar_total['total_abs'].max()
if not np.isfinite(global_max_total) or global_max_total <= 0:
    global_max_total = 1.0
plot_df['plot_value'] = plot_df['plot_value_raw'] / global_max_total
plot_df['abs_value'] = plot_df['abs_value_raw'] / global_max_total
plot_table_path = os.path.join(out_dir, get_value('science.tables.metrics_017', 'metrics_017.csv'))
plot_df.to_csv(plot_table_path, index=False, encoding='utf-8-sig')

n_axes = len(axis_order)
fig = plt.figure(figsize=(FIG_WIDTH, max(FIG_MIN_HEIGHT, ROW_HEIGHT * n_axes)))
gs = fig.add_gridspec(nrows=n_axes, ncols=1, hspace=HSPACE)
if USE_SIGNED_VALUE:
    axis_outcome_sums = plot_df.groupby(['axis', 'outcome'])['plot_value'].apply(lambda x: max(abs(x[x > 0].sum()) if (x > 0).any() else 0, abs(x[x < 0].sum()) if (x < 0).any() else 0))
    xlim_max = axis_outcome_sums.max()
    if not np.isfinite(xlim_max) or xlim_max <= 0:
        xlim_max = 1.0
else:
    xlim_max = plot_df.groupby(['axis', 'outcome'])['abs_value'].sum().max()
    if not np.isfinite(xlim_max) or xlim_max <= 0:
        xlim_max = 1.0
for i, axis_name in enumerate(axis_order):
    ax = fig.add_subplot(gs[i, 0])
    sub = plot_df[plot_df['axis'] == axis_name].copy()
    y_map = {'Response': 2, 'PFS': 1, 'OS': 0}
    ax.axvline(0, color='black', lw=0.7, zorder=1)
    for y in [0, 1, 2]:
        ax.axhline(y, color='#EFEFEF', lw=0.6, zorder=0)
    for outcome in OUTCOMES:
        y = y_map[outcome]
        sub2 = sub[sub['outcome'] == outcome].copy()
        if sub2.empty:
            continue
        sub2 = sub2.sort_values(['module_type_rank', 'row_order', 'abs_value'], ascending=[True, True, False])
        if USE_SIGNED_VALUE:
            pos_left = 0.0
            pos_sub = sub2[sub2['plot_value'] > 0].copy()
            for _, r in pos_sub.iterrows():
                width = r['plot_value']
                if not np.isfinite(width) or width <= 0:
                    continue
                ax.barh(y, width, left=pos_left, height=0.58, color=r['row_color'], edgecolor='white', linewidth=0.5, zorder=3)
                pos_left += width
            neg_left = 0.0
            neg_sub = sub2[sub2['plot_value'] < 0].copy()
            for _, r in neg_sub.iterrows():
                width = abs(r['plot_value'])
                if not np.isfinite(width) or width <= 0:
                    continue
                ax.barh(y, -width, left=neg_left, height=0.58, color=r['row_color'], edgecolor='white', linewidth=0.5, zorder=3)
                neg_left -= width
        else:
            right = 0.0
            for _, r in sub2.iterrows():
                width = r['abs_value']
                if not np.isfinite(width) or width <= 0:
                    continue
                ax.barh(y, width, left=right, height=0.58, color=r['row_color'], edgecolor='white', linewidth=0.5, zorder=3)
                right += width
    ax.set_yticks([2, 1, 0])
    ax.set_yticklabels(['Response', 'PFS', 'OS'], fontsize=8)
    axis_title = wrap_label(axis_name, width=48)
    ax.text(0.5, AXIS_LABEL_Y, axis_title, transform=ax.transAxes, ha='center', va='top', fontsize=9, fontweight='bold')
    if USE_SIGNED_VALUE:
        ax.set_xlim(-xlim_max * 1.08, xlim_max * 1.08)
    else:
        ax.set_xlim(0, xlim_max * 1.08)
    ax.tick_params(axis='x', labelsize=7)
    ax.tick_params(axis='y', labelsize=8)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_xlabel('')
fig.suptitle('Contribution to outcome prediction', fontsize=12, fontweight='bold', y=0.995)
if USE_SIGNED_VALUE:
    note = 'Bar segments show signed module-level SHAP contributions from the module-score multitask neural network.'
    mode_tag = 'signed'
else:
    note = 'Bar segments show absolute module-level SHAP contributions from the module-score multitask neural network.'
    mode_tag = 'absolute'
fig.text(0.5, 0.008, note, ha='center', va='bottom', fontsize=8)
plt.tight_layout(rect=[0.04, 0.07, 0.98, 0.96])
drop_tag = 'drop' + '_'.join(map(str, DROP_AXIS_INDEXES)) if len(DROP_AXIS_INDEXES) > 0 else 'all_axes'
out_png = os.path.join(out_dir, f"{get_value('science.paths.directory_040', 'directory_040')}{mode_tag}_{drop_tag}.png")
out_pdf = os.path.join(out_dir, f"{get_value('science.paths.directory_040', 'directory_040')}{mode_tag}_{drop_tag}.pdf")
plt.savefig(out_png, dpi=600, bbox_inches='tight')
plt.savefig(out_pdf, bbox_inches='tight')
plt.show()




In [ ]:
import os
import re
import textwrap
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
base_project_dir = str(project_path(''))
module_dir = os.path.join(base_project_dir, get_value('science.paths.directory_035', 'directory_035'))
shap_dir = os.path.join(module_dir, get_value('science.paths.directory_036', 'directory_036'))
meta_path = os.path.join(module_dir, get_value('science.tables.result_035', 'result_035.csv'))
mbp_path = os.path.join(base_project_dir, get_value('science.tables.clinical_005', 'clinical_005.csv'))
out_root = os.path.join(base_project_dir, get_value('science.paths.directory_041', 'directory_041'))
out_dir = os.path.join(out_root, get_value('science.paths.directory_042', 'directory_042'))
fig_dir = os.path.join(out_dir, 'figures')
os.makedirs(out_dir, exist_ok=True)
os.makedirs(fig_dir, exist_ok=True)







mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['font.family'] = 'Arial'
mpl.rcParams['axes.spines.top'] = False
mpl.rcParams['axes.spines.right'] = False
TASKS = ['Response', 'PFS', 'OS']
MBP_GROUPS = ['MBP-I', 'MBP-III']
FLIP_SURVIVAL_RISK_TO_BENEFIT = True
MODULE_TYPE_ORDER = {'Microbial KO module': 0, 'Metabolite module': 1, 'Radiomics module': 2}
USE_ALL_RADIOMICS_FOR_EACH_AXIS = True
FIG_WIDTH_SINGLE = 10.0
FIG_WIDTH_SIDE_BY_SIDE = 12.0
ROW_HEIGHT = 1.55
FIG_MIN_HEIGHT = 5.8
HSPACE = 1.05
AXIS_LABEL_Y = -0.36
DROP_AXIS_INDEXES = []
DROP_AXIS_NAMES = []

def normalize_sample_id(x):
    return str(x).strip().lower()

def valid_color(x, default='#BDBDBD'):
    x = str(x)
    if re.match('^#[0-9A-Fa-f]{6}$', x):
        return x
    return default

def wrap_label(x, width=48):
    x = str(x)
    x = x.replace(' Axis', '\nAxis')
    return '\n'.join(textwrap.wrap(x, width=width))

def clean_axis_name(x):
    return str(x).strip()

def safe_name(x):
    x = str(x)
    x = re.sub('[ /&:;,+()\\[\\]]+', '_', x)
    x = re.sub('_+', '_', x)
    x = re.sub('^_|_$', '', x)
    return x

def read_shap_csv(path):
    if not os.path.exists(path):
        raise FileNotFoundError(path)
    df = pd.read_csv(path, index_col=0, encoding='utf-8-sig')
    df.index = df.index.map(normalize_sample_id)
    for c in df.columns:
        df[c] = pd.to_numeric(df[c], errors='coerce')
    return df

def get_benefit_oriented_signed(signed_df, task):
    out = signed_df.copy()
    if FLIP_SURVIVAL_RISK_TO_BENEFIT and task in ['PFS', 'OS']:
        out = -out
    return out
meta = pd.read_csv(meta_path, encoding='utf-8-sig')
required_meta_cols = ['module_id', 'display_label', 'module_type', 'row_color', 'row_order', 'axis']
for c in required_meta_cols:
    if c not in meta.columns:
        raise ValueError(f'Metadata missing column: {c}')
meta['module_id'] = meta['module_id'].astype(str)
meta['display_label'] = meta['display_label'].astype(str)
meta['module_type'] = meta['module_type'].astype(str)
meta['row_color'] = meta['row_color'].astype(str).map(valid_color)
meta['row_order'] = pd.to_numeric(meta['row_order'], errors='coerce')
meta['axis'] = meta['axis'].astype(str)
RADIOMICS_MODULE_COLORS = {'Morphology': '#2F5597', 'Low-gray': '#5B9BD5', 'Coarse texture': '#7F95B8', 'Fine heterogeneity': '#6F6FAE'}
for module_name, color in RADIOMICS_MODULE_COLORS.items():
    mask_meta = (meta['module_type'] == 'Radiomics module') & (meta['display_label'].astype(str).str.contains(module_name, case=False, regex=False) | meta['module_id'].astype(str).str.contains(module_name, case=False, regex=False))
    meta.loc[mask_meta, 'row_color'] = color
mbp = pd.read_csv(mbp_path, encoding='utf-8-sig')
if 'sample' not in mbp.columns:
    raise ValueError('MBP table must contain column: sample')
if 'MBP_raw' not in mbp.columns:
    if 'MBP' in mbp.columns:
        mbp['MBP_raw'] = mbp['MBP'].astype(str).str.extract('(MBP-[I]+)', expand=False)
    else:
        raise ValueError('MBP table must contain MBP_raw or MBP.')
mbp['sample_norm'] = mbp['sample'].map(normalize_sample_id)
mbp = mbp[mbp['MBP_raw'].isin(MBP_GROUPS)].copy()
print('MBP group counts:')
print(mbp['MBP_raw'].value_counts())
shap_signed_raw = {}
shap_signed_benefit = {}
shap_abs = {}
for task in TASKS:
    signed_path = os.path.join(shap_dir, f"{get_value('science.paths.directory_037', 'directory_037')}{task}.csv.gz")
    abs_path = os.path.join(shap_dir, f"{get_value('science.paths.directory_038', 'directory_038')}{task}.csv.gz")
    signed_df = read_shap_csv(signed_path)
    abs_df = read_shap_csv(abs_path)
    common_modules = [m for m in meta['module_id'].tolist() if m in signed_df.columns]
    signed_df = signed_df[common_modules].copy()
    abs_df = abs_df[common_modules].copy()
    shap_signed_raw[task] = signed_df
    shap_signed_benefit[task] = get_benefit_oriented_signed(signed_df, task)
    shap_abs[task] = abs_df
    print(task, 'signed shape:', signed_df.shape)
all_shap_samples = set(shap_signed_raw['Response'].index)
for task in TASKS:
    all_shap_samples = all_shap_samples.intersection(set(shap_signed_raw[task].index))
mbp_samples = set(mbp['sample_norm'])
common_samples = sorted(list(all_shap_samples.intersection(mbp_samples)))
print('Common samples with SHAP and MBP:', len(common_samples))
if len(common_samples) == 0:
    raise ValueError('No common samples between SHAP matrices and MBP table.')
mbp_use = mbp[mbp['sample_norm'].isin(common_samples)].copy()
group_samples = {g: mbp_use.loc[mbp_use['MBP_raw'] == g, 'sample_norm'].tolist() for g in MBP_GROUPS}
for g in MBP_GROUPS:
    print(g, 'n =', len(group_samples[g]))
records = []
for group in MBP_GROUPS:
    samples = group_samples[group]
    for task in TASKS:
        signed_raw = shap_signed_raw[task].loc[samples]
        signed_benefit = shap_signed_benefit[task].loc[samples]
        absval = shap_abs[task].loc[samples]
        for module_id in signed_raw.columns:
            meta_row = meta[meta['module_id'] == module_id]
            if meta_row.shape[0] == 0:
                display_label = module_id
                module_type = 'Unknown'
                row_color = '#BDBDBD'
                row_order = 999999
                axis = 'Unknown'
            else:
                display_label = str(meta_row['display_label'].iloc[0])
                module_type = str(meta_row['module_type'].iloc[0])
                row_color = valid_color(meta_row['row_color'].iloc[0])
                row_order = float(meta_row['row_order'].iloc[0])
                axis = str(meta_row['axis'].iloc[0])
            mean_abs = float(absval[module_id].mean())
            mean_signed_raw = float(signed_raw[module_id].mean())
            mean_signed_benefit = float(signed_benefit[module_id].mean())
            records.append({'MBP_group': group, 'module_id': module_id, 'display_label': display_label, 'module_type': module_type, 'axis': axis, 'row_color': row_color, 'row_order': row_order, 'outcome': task, 'n_samples': len(samples), 'mean_abs_shap': mean_abs, 'mean_signed_shap_raw': mean_signed_raw, 'mean_signed_shap_benefit_oriented': mean_signed_benefit, 'signed_ratio_raw': mean_signed_raw / (mean_abs + 1e-12), 'signed_ratio_benefit_oriented': mean_signed_benefit / (mean_abs + 1e-12)})
contrib_group = pd.DataFrame(records)
contrib_group['signed_ratio_raw'] = contrib_group['signed_ratio_raw'].clip(-1, 1)
contrib_group['signed_ratio_benefit_oriented'] = contrib_group['signed_ratio_benefit_oriented'].clip(-1, 1)
contrib_group['module_abs_scaled_global'] = contrib_group.groupby('outcome')['mean_abs_shap'].transform(lambda x: x / (x.max() + 1e-12))
contrib_group['signed_scaled_benefit_oriented'] = np.sign(contrib_group['mean_signed_shap_benefit_oriented']) * contrib_group['module_abs_scaled_global']
contrib_group = contrib_group.sort_values(['MBP_group', 'row_order', 'outcome'])
group_out = os.path.join(out_dir, get_value('science.tables.attribution_021', 'attribution_021.csv'))
contrib_group.to_csv(group_out, index=False, encoding='utf-8-sig')

i_df = contrib_group[contrib_group['MBP_group'] == 'MBP-I'].copy()
iii_df = contrib_group[contrib_group['MBP_group'] == 'MBP-III'].copy()
delta = i_df.merge(iii_df, on=['module_id', 'display_label', 'module_type', 'axis', 'row_color', 'row_order', 'outcome'], suffixes=('_MBP_I', '_MBP_III'))
delta['delta_mean_abs_shap_I_minus_III'] = delta['mean_abs_shap_MBP_I'] - delta['mean_abs_shap_MBP_III']
delta['delta_signed_benefit_I_minus_III'] = delta['mean_signed_shap_benefit_oriented_MBP_I'] - delta['mean_signed_shap_benefit_oriented_MBP_III']
delta['delta_signed_scaled_benefit_I_minus_III'] = delta['signed_scaled_benefit_oriented_MBP_I'] - delta['signed_scaled_benefit_oriented_MBP_III']
delta_out = os.path.join(out_dir, get_value('science.tables.attribution_022', 'attribution_022.csv'))
delta.to_csv(delta_out, index=False, encoding='utf-8-sig')

axis_order = meta.loc[meta['module_type'].isin(['Microbial KO module', 'Metabolite module']), 'axis'].dropna().astype(str).map(clean_axis_name).tolist()
axis_order = list(dict.fromkeys(axis_order))
axis_order = [x for x in axis_order if x.lower() not in ['radiomics', 'radiomics module', 'nan', '']]
original_axis_order = axis_order.copy()
if len(DROP_AXIS_NAMES) > 0:
    axis_order = [a for a in axis_order if a not in DROP_AXIS_NAMES]
if len(DROP_AXIS_INDEXES) > 0:
    axis_order = [a for i, a in enumerate(axis_order) if i not in DROP_AXIS_INDEXES]
print('\nOriginal biological axes:')
for i, a in enumerate(original_axis_order):
    pass
print('\nAxes kept for plotting:')
for i, a in enumerate(axis_order):
    pass
if len(axis_order) == 0:
    raise ValueError('No biological axes found after filtering.')
all_radiomics_modules = meta.loc[meta['module_type'] == 'Radiomics module', 'module_id'].dropna().astype(str).tolist()
axis_module_rows = []
for axis_name in axis_order:
    sub_meta = meta[(meta['axis'].astype(str) == axis_name) & meta['module_type'].isin(['Microbial KO module', 'Metabolite module'])].copy()
    module_ids = sub_meta['module_id'].astype(str).tolist()
    if USE_ALL_RADIOMICS_FOR_EACH_AXIS:
        module_ids.extend(all_radiomics_modules)
    module_ids = list(dict.fromkeys(module_ids))
    axis_module_rows.append({'axis': axis_name, 'module_ids': module_ids, 'n_modules': len(module_ids)})
axis_module_df = pd.DataFrame(axis_module_rows)
axis_map_out = os.path.join(out_dir, get_value('science.tables.attribution_023', 'attribution_023.csv'))
axis_module_df.to_csv(axis_map_out, index=False, encoding='utf-8-sig')

plot_records = []
for _, row in axis_module_df.iterrows():
    axis_name = row['axis']
    module_ids = row['module_ids']
    sub = contrib_group[contrib_group['module_id'].isin(module_ids) & contrib_group['outcome'].isin(TASKS) & contrib_group['MBP_group'].isin(MBP_GROUPS)].copy()
    for _, r in sub.iterrows():
        val = r['mean_signed_shap_benefit_oriented']
        abs_val = abs(val)
        if not np.isfinite(val):
            continue
        plot_records.append({'axis': axis_name, 'MBP_group': r['MBP_group'], 'module_id': r['module_id'], 'display_label': r['display_label'], 'module_type': r['module_type'], 'row_color': valid_color(r['row_color']), 'row_order': r['row_order'], 'outcome': r['outcome'], 'plot_value_raw': float(val), 'abs_value_raw': float(abs_val)})
plot_df = pd.DataFrame(plot_records)
if plot_df.empty:
    raise ValueError('No plotting data generated.')
plot_df['module_type_rank'] = plot_df['module_type'].map(MODULE_TYPE_ORDER).fillna(99)
bar_total = plot_df.groupby(['axis', 'MBP_group', 'outcome'])['abs_value_raw'].sum().reset_index(name='total_abs')
global_max_total = bar_total['total_abs'].max()
if not np.isfinite(global_max_total) or global_max_total <= 0:
    global_max_total = 1.0
plot_df['plot_value'] = plot_df['plot_value_raw'] / global_max_total
plot_df['abs_value'] = plot_df['abs_value_raw'] / global_max_total
plot_table_out = os.path.join(out_dir, get_value('science.tables.metrics_018', 'metrics_018.csv'))
plot_df.to_csv(plot_table_out, index=False, encoding='utf-8-sig')


def draw_axis_stacked_bars(ax, sub, axis_name, title=None, show_y=True):
    y_map = {'Response': 2, 'PFS': 1, 'OS': 0}
    ax.axvline(0, color='black', lw=0.7, zorder=1)
    for y in [0, 1, 2]:
        ax.axhline(y, color='#EFEFEF', lw=0.6, zorder=0)
    for outcome in TASKS:
        y = y_map[outcome]
        sub2 = sub[sub['outcome'] == outcome].copy()
        if sub2.empty:
            continue
        sub2 = sub2.sort_values(['module_type_rank', 'row_order', 'abs_value'], ascending=[True, True, False])
        pos_left = 0.0
        pos_sub = sub2[sub2['plot_value'] > 0].copy()
        for _, r in pos_sub.iterrows():
            width = r['plot_value']
            if not np.isfinite(width) or width <= 0:
                continue
            ax.barh(y, width, left=pos_left, height=0.58, color=r['row_color'], edgecolor='white', linewidth=0.5, zorder=3)
            pos_left += width
        neg_left = 0.0
        neg_sub = sub2[sub2['plot_value'] < 0].copy()
        for _, r in neg_sub.iterrows():
            width = abs(r['plot_value'])
            if not np.isfinite(width) or width <= 0:
                continue
            ax.barh(y, -width, left=neg_left, height=0.58, color=r['row_color'], edgecolor='white', linewidth=0.5, zorder=3)
            neg_left -= width
    ax.set_yticks([2, 1, 0])
    if show_y:
        ax.set_yticklabels(['Response', 'PFS', 'OS'], fontsize=8)
    else:
        ax.set_yticklabels(['', '', ''])
    axis_title = wrap_label(axis_name, width=48)
    ax.text(0.5, AXIS_LABEL_Y, axis_title, transform=ax.transAxes, ha='center', va='top', fontsize=9, fontweight='bold')
    ax.tick_params(axis='x', labelsize=7)
    ax.tick_params(axis='y', labelsize=8)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_xlabel('')
    if title is not None:
        ax.set_title(title, fontsize=10, fontweight='bold', pad=5)

def get_xlim_max_for_subset(df):
    tmp = df.groupby(['axis', 'MBP_group', 'outcome'])['plot_value'].apply(lambda x: max(abs(x[x > 0].sum()) if (x > 0).any() else 0, abs(x[x < 0].sum()) if (x < 0).any() else 0))
    xlim_max = tmp.max()
    if not np.isfinite(xlim_max) or xlim_max <= 0:
        xlim_max = 1.0
    return xlim_max

def plot_single_group(group_name):
    sub_all = plot_df[plot_df['MBP_group'] == group_name].copy()
    xlim_max = get_xlim_max_for_subset(sub_all)
    n_axes = len(axis_order)
    fig = plt.figure(figsize=(FIG_WIDTH_SINGLE, max(FIG_MIN_HEIGHT, ROW_HEIGHT * n_axes)))
    gs = fig.add_gridspec(nrows=n_axes, ncols=1, hspace=HSPACE)
    for i, axis_name in enumerate(axis_order):
        ax = fig.add_subplot(gs[i, 0])
        sub = sub_all[sub_all['axis'] == axis_name].copy()
        draw_axis_stacked_bars(ax, sub, axis_name=axis_name, title=None, show_y=True)
        ax.set_xlim(-xlim_max * 1.08, xlim_max * 1.08)
    fig.suptitle(f'{group_name}: benefit-oriented module SHAP contribution', fontsize=12, fontweight='bold', y=0.995)
    note = 'Positive values indicate favorable contribution. PFS/OS SHAP signs were flipped because survival heads predict risk.'
    fig.text(0.5, 0.008, note, ha='center', va='bottom', fontsize=8)
    plt.tight_layout(rect=[0.04, 0.07, 0.98, 0.96])
    tag = safe_name(group_name)
    out_png = os.path.join(fig_dir, f"MBP_Grouped_MECFS_style_axis_summary_{tag}{get_value('science.figures.plot_051', 'plot_051.png')}")
    out_pdf = os.path.join(fig_dir, f"MBP_Grouped_MECFS_style_axis_summary_{tag}{get_value('science.figures.plot_052', 'plot_052.pdf')}")
    plt.savefig(out_png, dpi=600, bbox_inches='tight')
    plt.savefig(out_pdf, bbox_inches='tight')
    plt.show()
    
    

def plot_side_by_side():
    xlim_max = get_xlim_max_for_subset(plot_df)
    n_axes = len(axis_order)
    fig = plt.figure(figsize=(FIG_WIDTH_SIDE_BY_SIDE, max(FIG_MIN_HEIGHT, ROW_HEIGHT * n_axes)))
    gs = fig.add_gridspec(nrows=n_axes, ncols=2, hspace=HSPACE, wspace=0.18)
    for i, axis_name in enumerate(axis_order):
        for j, group_name in enumerate(MBP_GROUPS):
            ax = fig.add_subplot(gs[i, j])
            sub = plot_df[(plot_df['axis'] == axis_name) & (plot_df['MBP_group'] == group_name)].copy()
            draw_axis_stacked_bars(ax, sub, axis_name=axis_name, title=group_name if i == 0 else None, show_y=j == 0)
            ax.set_xlim(-xlim_max * 1.08, xlim_max * 1.08)
    fig.suptitle('MBP-I vs MBP-III: benefit-oriented module SHAP contribution', fontsize=12, fontweight='bold', y=0.995)
    note = 'Bar segments show mean signed module-level SHAP contributions within each MBP group. Positive values indicate favorable contribution; PFS/OS signs were flipped from risk to benefit.'
    fig.text(0.5, 0.008, note, ha='center', va='bottom', fontsize=8)
    plt.tight_layout(rect=[0.04, 0.07, 0.98, 0.96])
    out_png = os.path.join(fig_dir, get_value('science.figures.plot_053', 'plot_053.png'))
    out_pdf = os.path.join(fig_dir, get_value('science.figures.plot_054', 'plot_054.pdf'))
    plt.savefig(out_png, dpi=600, bbox_inches='tight')
    plt.savefig(out_pdf, bbox_inches='tight')
    plt.show()
    
    
plot_single_group('MBP-I')
plot_single_group('MBP-III')
plot_side_by_side()





In [ ]:
import os
import re
import glob
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import matplotlib.image as mpimg
from scipy.stats import chi2_contingency
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score
import statsmodels.api as sm
from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import multivariate_logrank_test
from lifelines.utils import concordance_index
stage2_out_dir = str(project_path(get_value('science.paths.directory_019', 'directory_019')))
clinical_baseline_path = str(project_path(get_value('science.tables.clinical_006', 'clinical_006.xlsx')))
if not os.path.exists(clinical_baseline_path):
    alt_path = str(project_path(get_value('science.tables.clinical_007', 'clinical_007.xlsx')))
    if os.path.exists(alt_path):
        clinical_baseline_path = alt_path
fig4_out_dir = os.path.join(stage2_out_dir, get_value('science.paths.directory_043', 'directory_043'))
os.makedirs(fig4_out_dir, exist_ok=True)
table_out_dir = os.path.join(fig4_out_dir, 'tables')
os.makedirs(table_out_dir, exist_ok=True)
panel_out_dir = os.path.join(fig4_out_dir, 'panels')
os.makedirs(panel_out_dir, exist_ok=True)
supp_out_dir = os.path.join(fig4_out_dir, 'supplementary')
os.makedirs(supp_out_dir, exist_ok=True)
RANDOM_STATE = 42
N_BOOT_STABILITY = 100
STABILITY_THRESHOLD = 0.4
MAX_EXTENDED_VARS = 8
MIN_EXTENDED_VARS = 4
DCA_N_REPEATS = 20





def normalize_sample_id(x):
    return str(x).strip().lower()

def find_col(df, candidates, required=False):
    col_map = {str(c).lower(): c for c in df.columns}
    for cand in candidates:
        key = str(cand).lower()
        if key in col_map:
            return col_map[key]
    for cand in candidates:
        cand_low = str(cand).lower()
        for c in df.columns:
            if cand_low in str(c).lower():
                return c
    if required:
        raise ValueError(f'Cannot find any column from {candidates}. Existing columns: {df.columns.tolist()}')
    return None

def percentile_rank(s):
    s = pd.to_numeric(pd.Series(s), errors='coerce')
    return s.rank(method='average', pct=True)

def log1p_safe(s):
    s = pd.to_numeric(pd.Series(s), errors='coerce')
    s = s.clip(lower=0)
    return np.log1p(s)

def safe_zscore(s):
    s = pd.to_numeric(pd.Series(s), errors='coerce')
    sd = s.std(skipna=True)
    if pd.isna(sd) or sd == 0:
        return pd.Series(np.zeros(len(s)), index=s.index)
    return (s - s.mean(skipna=True)) / sd

def is_binary_series(s):
    x = pd.to_numeric(pd.Series(s), errors='coerce').dropna()
    vals = sorted(x.unique().tolist())
    return len(vals) <= 2 and all((v in [0, 1] for v in vals))

def scale_for_model(s):
    s = pd.to_numeric(pd.Series(s), errors='coerce')
    if is_binary_series(s):
        return s
    return safe_zscore(s)

def format_p(p):
    if pd.isna(p):
        return 'P = NA'
    if p < 0.001:
        return 'P < 0.001'
    return f'P = {p:.3f}'

def format_p_plain(p):
    if pd.isna(p):
        return 'NA'
    if p < 0.001:
        return '<0.001'
    return f'{p:.3f}'

def format_ci_effect(effect, low, high):
    if pd.isna(effect) or pd.isna(low) or pd.isna(high):
        return 'NA'
    return f'{effect:.2f} ({low:.2f}-{high:.2f})'

def infer_time_unit(time_values):
    x = pd.to_numeric(pd.Series(time_values), errors='coerce').dropna()
    if x.shape[0] == 0:
        return 'Time'
    if x.median() > 60:
        return 'Time, days'
    return 'Time, months'

def survival_timepoint_value(time_values, months):
    x = pd.to_numeric(pd.Series(time_values), errors='coerce').dropna()
    if x.shape[0] == 0:
        return months
    if x.median() > 60:
        return months * 30.4375
    return months

def km_survival_at_time(sub, time_col, event_col, t):
    sub = sub[[time_col, event_col]].dropna().copy()
    if sub.shape[0] == 0:
        return np.nan
    kmf = KaplanMeierFitter()
    kmf.fit(sub[time_col], sub[event_col])
    try:
        return float(kmf.survival_function_at_times(t).values[0])
    except:
        return np.nan

def to_binary_safe(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x, (int, float, np.integer, np.floating)):
        if x in [0, 1]:
            return int(x)
    s = str(x).strip().lower()
    s_clean = s.replace(' ', '').replace('-', '').replace('_', '').replace('/', '').replace('\\', '').replace('\u3000', '')
    yes_terms = ['1', 'yes', 'y', 'true', 'positive', 'present', 'male', 'm', get_value('science.translation.analysis_001', 'present'), get_value('science.translation.analysis_002', 'yes'), get_value('science.translation.analysis_003', 'positive'), get_value('science.translation.analysis_004', 'multiple')]
    no_terms = ['0', 'no', 'n', 'false', 'negative', 'absent', 'female', 'f', get_value('science.translation.analysis_005', 'absent'), get_value('science.translation.analysis_006', 'no'), get_value('science.translation.analysis_007', 'negative'), get_value('science.translation.analysis_008', 'solitary')]
    if s_clean in yes_terms:
        return 1
    if s_clean in no_terms:
        return 0
    try:
        v = float(s_clean)
        if v in [0, 1]:
            return int(v)
        return v
    except:
        return np.nan

def child_pugh_to_ordinal(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x, (int, float, np.integer, np.floating)):
        return float(x)
    s = str(x).strip().upper()
    if s in ['A', 'CHILD-PUGH A', 'CHILD A']:
        return 0
    if s in ['B', 'CHILD-PUGH B', 'CHILD B']:
        return 1
    if s in ['C', 'CHILD-PUGH C', 'CHILD C']:
        return 2
    try:
        return float(s)
    except:
        return np.nan

def prepare_regression_matrix(df, cols):
    X = df[cols].copy()
    for c in X.columns:
        X[c] = pd.to_numeric(X[c], errors='coerce')
        if X[c].notna().sum() == 0:
            X[c] = 0
        else:
            X[c] = X[c].fillna(X[c].median())
    keep_cols = [c for c in X.columns if X[c].nunique(dropna=True) > 1]
    X = X[keep_cols]
    return X

def normalize_modality_name(x):
    mapper = {'M': 'Metagenomics', 'R': 'Radiomics', 'B': 'Blood', 'U': 'Urine', 'Microbiota': 'Metagenomics', 'Microbiome': 'Metagenomics', 'Metagenomics': 'Metagenomics', 'Radiomics': 'Radiomics', 'Blood': 'Blood', 'Urine': 'Urine'}
    x = str(x).strip()
    return mapper.get(x, x)

def modality_set_from_string(x):
    if pd.isna(x):
        return set()
    s = str(x)
    s = s.replace('_plus_', '+')
    s = s.replace('_', '+')
    parts = [p.strip() for p in s.split('+') if p.strip()]
    parts = [normalize_modality_name(p) for p in parts]
    return set(parts)

def standardize_prediction_columns(pred):
    pred = pred.copy()
    sample_col = find_col(pred, ['sample', 'sample_id', 'Sample', 'ID', 'id', 'patient_id', 'PatientID'], required=True)
    resp_true_col = find_col(pred, ['Response_true', 'Response', 'y_response', 'y_resp', 'resp_true'], required=True)
    resp_pred_col = find_col(pred, ['Response_pred', 'response_pred', 'pred_response', 'resp_pred', 'Response_probability'], required=True)
    pfs_time_col = find_col(pred, ['PFS_time', 'pfs_time'], required=True)
    pfs_status_col = find_col(pred, ['PFS_status', 'pfs_status', 'PFS_event', 'pfs_event'], required=True)
    pfs_risk_col = find_col(pred, ['PFS_risk', 'pfs_risk', 'PFS_pred', 'pfs_pred'], required=True)
    os_time_col = find_col(pred, ['OS_time', 'os_time'], required=True)
    os_status_col = find_col(pred, ['OS_status', 'os_status', 'OS_event', 'os_event'], required=True)
    os_risk_col = find_col(pred, ['OS_risk', 'os_risk', 'OS_pred', 'os_pred'], required=True)
    pred = pred.rename(columns={sample_col: 'sample', resp_true_col: 'Response_true', resp_pred_col: 'Response_pred', pfs_time_col: 'PFS_time', pfs_status_col: 'PFS_status', pfs_risk_col: 'PFS_risk', os_time_col: 'OS_time', os_status_col: 'OS_status', os_risk_col: 'OS_risk'})
    pred['sample'] = pred['sample'].map(normalize_sample_id)
    for c in ['Response_true', 'Response_pred', 'PFS_time', 'PFS_status', 'PFS_risk', 'OS_time', 'OS_status', 'OS_risk']:
        pred[c] = pd.to_numeric(pred[c], errors='coerce')
    keep_cols = ['sample', 'Response_true', 'Response_pred', 'PFS_time', 'PFS_status', 'PFS_risk', 'OS_time', 'OS_status', 'OS_risk']
    extra_cols = [c for c in ['fold_id', 'repeat', 'fold', 'model', 'modalities'] if c in pred.columns]
    return pred[keep_cols + extra_cols].copy()

def find_and_load_mru_predictions(stage2_out_dir):
    mru_set = {'Metagenomics', 'Radiomics', 'Urine'}
    stage2b_dir = os.path.join(stage2_out_dir, get_value('science.paths.directory_044', 'directory_044'))
    search_root = stage2b_dir if os.path.exists(stage2b_dir) else stage2_out_dir
    combined_patterns = [os.path.join(search_root, '**', get_value('science.tables.predictions_005', 'predictions_005.csv')), os.path.join(search_root, '**', '*combination*cv_predictions*.csv')]
    combined_files = []
    for pat in combined_patterns:
        combined_files.extend(glob.glob(pat, recursive=True))
    combined_files = list(dict.fromkeys(combined_files))
    for f in combined_files:
        try:
            tmp = pd.read_csv(f, encoding='utf-8-sig')
        except:
            continue
        modality_col = find_col(tmp, ['modalities', 'modality', 'combination', 'Model', 'model'], required=False)
        if modality_col is not None:
            tmp['_modset'] = tmp[modality_col].apply(modality_set_from_string)
            sub = tmp[tmp['_modset'].apply(lambda x: x == mru_set)].copy()
            if sub.shape[0] > 0:
                
                print(f)
                return standardize_prediction_columns(sub.drop(columns=['_modset']))
    direct_patterns = [os.path.join(search_root, '**', '*C3_M_R_U*cv_predictions*.csv'), os.path.join(search_root, '**', '*M_R_U*cv_predictions*.csv'), os.path.join(search_root, '**', '*Metagenomics*Radiomics*Urine*cv_predictions*.csv'), os.path.join(search_root, '**', '*Metagenomics_plus_Radiomics_plus_Urine*cv_predictions*.csv')]
    direct_files = []
    for pat in direct_patterns:
        direct_files.extend(glob.glob(pat, recursive=True))
    direct_files = list(dict.fromkeys(direct_files))
    if len(direct_files) > 0:
        f = direct_files[0]
        tmp = pd.read_csv(f, encoding='utf-8-sig')
        
        print(f)
        return standardize_prediction_columns(tmp)
    raise FileNotFoundError('Cannot find MRU cv prediction file.')

def aggregate_cv_predictions(pred_long):
    agg = pred_long.groupby('sample', as_index=False).agg({'Response_true': 'first', 'Response_pred': 'mean', 'PFS_time': 'first', 'PFS_status': 'first', 'PFS_risk': 'mean', 'OS_time': 'first', 'OS_status': 'first', 'OS_risk': 'mean'})
    return agg

def derive_avatar_score(mru_df):
    df = mru_df.copy()
    df['Response_benefit'] = percentile_rank(df['Response_pred'])
    df['PFS_benefit'] = 1 - percentile_rank(df['PFS_risk'])
    df['OS_benefit'] = 1 - percentile_rank(df['OS_risk'])
    df[get_value('science.columns.benefit_001', 'AVATAR_score')] = (df['Response_benefit'] + df['PFS_benefit'] + df['OS_benefit']) / 3
    q_low = df[get_value('science.columns.benefit_001', 'AVATAR_score')].quantile(1 / 3)
    q_high = df[get_value('science.columns.benefit_001', 'AVATAR_score')].quantile(2 / 3)

    def assign_avatar(score):
        if score >= q_high:
            return get_value('science.labels.benefit_001', 'AVATAR-I')
        elif score >= q_low:
            return get_value('science.labels.benefit_002', 'AVATAR-II')
        else:
            return get_value('science.labels.benefit_003', 'AVATAR-III')
    df[get_value('science.columns.benefit_002', 'AVATAR_raw')] = df[get_value('science.columns.benefit_001', 'AVATAR_score')].apply(assign_avatar)
    df[get_value('science.columns.benefit_003', 'AVATAR_group')] = df[get_value('science.columns.benefit_002', 'AVATAR_raw')].map({get_value('science.labels.benefit_001', 'AVATAR-I'): get_value('science.messages.status_001', 'AVATAR-I\nHigh-benefit'), get_value('science.labels.benefit_002', 'AVATAR-II'): get_value('science.messages.status_002', 'AVATAR-II\nIntermediate'), get_value('science.labels.benefit_003', 'AVATAR-III'): get_value('science.messages.status_003', 'AVATAR-III\nLow-benefit')})
    df[get_value('science.columns.benefit_004', 'AVATAR_risk_ord')] = df[get_value('science.columns.benefit_002', 'AVATAR_raw')].map({get_value('science.labels.benefit_001', 'AVATAR-I'): 0, get_value('science.labels.benefit_002', 'AVATAR-II'): 1, get_value('science.labels.benefit_003', 'AVATAR-III'): 2})
    df = df.sort_values(get_value('science.columns.benefit_001', 'AVATAR_score'), ascending=False).reset_index(drop=True)
    return df
try:
    mru_pred_long = find_and_load_mru_predictions(stage2_out_dir)
    mru_df = aggregate_cv_predictions(mru_pred_long)
    mru_df = derive_avatar_score(mru_df)
except Exception as e:
    print('MRU prediction loading failed:')
    print(e)
    print(get_value('science.messages.status_004', 'Trying to load previous Figure 4 MRU/AVATAR patient table...'))
    fallback_paths = [os.path.join(stage2_out_dir, get_value('science.paths.directory_026', 'directory_026'), get_value('science.tables.result_025', 'result_025.csv')), os.path.join(stage2_out_dir, get_value('science.paths.directory_026', 'directory_026'), get_value('science.tables.clinical_004', 'clinical_004.csv'))]
    loaded = False
    for fp in fallback_paths:
        if os.path.exists(fp):
            mru_df = pd.read_csv(fp, encoding='utf-8-sig')
            loaded = True
            print('Loaded fallback table:')
            print(fp)
            break
    if not loaded:
        raise FileNotFoundError('Cannot load MRU predictions or previous Figure 4 table.')
    if 'sample' not in mru_df.columns:
        sample_col = find_col(mru_df, ['sample', 'sample_id', 'ID', 'id'], required=True)
        mru_df = mru_df.rename(columns={sample_col: 'sample'})
    mru_df['sample'] = mru_df['sample'].map(normalize_sample_id)
    if get_value('science.columns.benefit_001', 'AVATAR_score') not in mru_df.columns:
        if 'MRU_benefit_score' in mru_df.columns:
            mru_df[get_value('science.columns.benefit_001', 'AVATAR_score')] = mru_df['MRU_benefit_score']
        else:
            mru_df = derive_avatar_score(mru_df)
    if get_value('science.columns.benefit_002', 'AVATAR_raw') not in mru_df.columns:
        if 'MBP_raw' in mru_df.columns:
            mru_df[get_value('science.columns.benefit_002', 'AVATAR_raw')] = mru_df['MBP_raw'].replace({'MBP-I': get_value('science.labels.benefit_001', 'AVATAR-I'), 'MBP-II': get_value('science.labels.benefit_002', 'AVATAR-II'), 'MBP-III': get_value('science.labels.benefit_003', 'AVATAR-III')})
        else:
            mru_df = derive_avatar_score(mru_df)
    if get_value('science.columns.benefit_003', 'AVATAR_group') not in mru_df.columns:
        mru_df[get_value('science.columns.benefit_003', 'AVATAR_group')] = mru_df[get_value('science.columns.benefit_002', 'AVATAR_raw')].map({get_value('science.labels.benefit_001', 'AVATAR-I'): get_value('science.messages.status_001', 'AVATAR-I\nHigh-benefit'), get_value('science.labels.benefit_002', 'AVATAR-II'): get_value('science.messages.status_002', 'AVATAR-II\nIntermediate'), get_value('science.labels.benefit_003', 'AVATAR-III'): get_value('science.messages.status_003', 'AVATAR-III\nLow-benefit')})
    if get_value('science.columns.benefit_004', 'AVATAR_risk_ord') not in mru_df.columns:
        mru_df[get_value('science.columns.benefit_004', 'AVATAR_risk_ord')] = mru_df[get_value('science.columns.benefit_002', 'AVATAR_raw')].map({get_value('science.labels.benefit_001', 'AVATAR-I'): 0, get_value('science.labels.benefit_002', 'AVATAR-II'): 1, get_value('science.labels.benefit_003', 'AVATAR-III'): 2})
print(get_value('science.messages.status_005', '\nMRU / AVATAR table:'))
print(mru_df.shape)

clinical_raw = pd.read_excel(clinical_baseline_path)
sample_col = find_col(clinical_raw, ['sample', 'sample_id', 'ID', 'id', get_column('science.translation.analysis_009', 'patient_name')], required=True)
clinical_raw = clinical_raw.rename(columns={sample_col: 'sample'})
clinical_raw['sample'] = clinical_raw['sample'].map(normalize_sample_id)
clinical_raw = clinical_raw.drop_duplicates(subset='sample', keep='first')
print('\nClinical baseline table:')
print(clinical_raw.shape)
print(clinical_raw.columns.tolist())

def prepare_clinical_features(clin):
    df = clin.copy()
    if 'Sex' in df.columns:
        df['Sex'] = df['Sex'].apply(to_binary_safe)
    if 'Hepatitis B' in df.columns:
        df['HepatitisB'] = df['Hepatitis B'].apply(to_binary_safe)
    if 'Cirrhosis' in df.columns:
        df['Cirrhosis'] = df['Cirrhosis'].apply(to_binary_safe)
    if 'PVTT' in df.columns:
        df['PVTT'] = df['PVTT'].apply(to_binary_safe)
    if 'Metastasis' in df.columns:
        df['Metastasis'] = df['Metastasis'].apply(to_binary_safe)
    if 'Number' in df.columns:
        df['Number'] = pd.to_numeric(df['Number'], errors='coerce')
    if 'Child-Pugh' in df.columns:
        df['ChildPugh_ord'] = df['Child-Pugh'].apply(child_pugh_to_ordinal)
    for raw_col, new_col in [('AFP', 'log_AFP'), ('PVIKA', 'log_PVIKA'), ('CEA', 'log_CEA'), ('CA199', 'log_CA199'), ('Size', 'log_Size'), ('LDH', 'log_LDH'), ('ALP', 'log_ALP'), ('CRP', 'log_CRP')]:
        if raw_col in df.columns:
            df[new_col] = log1p_safe(df[raw_col])
    if 'FIB-4' in df.columns:
        df['FIB_4'] = pd.to_numeric(df['FIB-4'], errors='coerce')
    numeric_cols = ['Age', 'Ht', 'Wt', 'ECOG', 'ALT', 'AST', 'AFP', 'ALB', 'TBiL', 'PVIKA', 'AG', 'Ca2+', 'Mg2+', 'K+', 'Cl-', 'ALP', 'LDH', 'Cr', 'PT', 'RBC', 'HGB', 'WBC', 'NEU', 'NEU%', 'MONO', 'MONO%', 'BASO', 'BASO%', 'EOS', 'EOS%', 'LYM', 'LYM%', 'PLT', 'Size', 'ALBI', 'BMI', 'PNI', 'NLR', 'PLR', 'SII', 'LMR', 'TBS', 'FIB_4']
    for c in numeric_cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors='coerce')
    return df
clinical_features = prepare_clinical_features(clinical_raw)
clinical_outcome_cols = [c for c in ['Response', 'OS_time', 'OS_status', 'PFS_time', 'PFS_status'] if c in clinical_features.columns]
clinical_feature_only = clinical_features.drop(columns=clinical_outcome_cols)
analysis_df = mru_df.merge(clinical_feature_only, on='sample', how='left')
if 'Tumor_size' not in analysis_df.columns and 'Size' in analysis_df.columns:
    analysis_df['Tumor_size'] = pd.to_numeric(analysis_df['Size'], errors='coerce')
core_covariates = [c for c in ['log_AFP', 'log_Size', 'PVTT', 'ALBI'] if c in analysis_df.columns]
print('\nCore clinical covariates:')
print(core_covariates)
analysis_path = os.path.join(fig4_out_dir, get_value('science.tables.clinical_008', 'clinical_008.csv'))
analysis_df.to_csv(analysis_path, index=False, encoding='utf-8-sig')


candidate_priority = ['log_AFP', 'log_Size', 'PVTT', 'ALBI', 'TBS', 'log_PVIKA', 'Metastasis', 'Number', 'PNI', 'NLR', 'SII', 'PLR', 'LMR', 'FIB_4', 'log_LDH', 'LDH', 'PT', 'Cr', 'ECOG', 'ChildPugh_ord', 'Age', 'Sex', 'BMI', 'HepatitisB', 'Cirrhosis', 'ALT', 'AST', 'log_ALP', 'ALP', 'WBC', 'NEU', 'LYM', 'PLT', 'HGB']
candidate_vars_raw = [c for c in candidate_priority if c in analysis_df.columns]

def filter_candidate_variables(df, vars_in, missing_threshold=0.3, min_class_count=5):
    keep = []
    rows = []
    for v in vars_in:
        s = pd.to_numeric(df[v], errors='coerce')
        missing_rate = s.isna().mean()
        nonmiss = s.dropna()
        n_unique = nonmiss.nunique()
        reason = 'included'
        include = True
        if missing_rate > missing_threshold:
            include = False
            reason = f'missing>{missing_threshold}'
        elif n_unique <= 1:
            include = False
            reason = 'low_variance'
        elif is_binary_series(s):
            vc = nonmiss.value_counts()
            if vc.min() < min_class_count:
                include = False
                reason = 'rare_binary_level'
        rows.append({'Variable': v, 'Missing_rate': missing_rate, 'N_unique': n_unique, 'Is_binary': is_binary_series(s), 'Include_initial': include, 'Filter_reason': reason})
        if include:
            keep.append(v)
    return (keep, pd.DataFrame(rows))
candidate_vars_filtered, candidate_filter_df = filter_candidate_variables(analysis_df, candidate_vars_raw, missing_threshold=0.3, min_class_count=5)

def drop_highly_correlated_variables(df, vars_in, corr_threshold=0.85):
    vars_keep = vars_in.copy()
    dropped = []
    X = df[vars_keep].copy()
    for c in X.columns:
        X[c] = pd.to_numeric(X[c], errors='coerce')
        X[c] = X[c].fillna(X[c].median())
    corr = X.corr(method='spearman').abs()
    for i, v1 in enumerate(vars_keep.copy()):
        if v1 not in vars_keep:
            continue
        for v2 in vars_keep.copy():
            if v1 == v2:
                continue
            if vars_keep.index(v2) <= vars_keep.index(v1):
                continue
            if corr.loc[v1, v2] >= corr_threshold:
                if v2 in vars_keep:
                    vars_keep.remove(v2)
                    dropped.append({'Kept': v1, 'Dropped': v2, 'Spearman_abs_corr': corr.loc[v1, v2]})
    return (vars_keep, pd.DataFrame(dropped))
candidate_vars, corr_drop_df = drop_highly_correlated_variables(analysis_df, candidate_vars_filtered, corr_threshold=0.85)
candidate_filter_path = os.path.join(table_out_dir, get_value('science.tables.result_039', 'result_039.csv'))
candidate_filter_df.to_csv(candidate_filter_path, index=False, encoding='utf-8-sig')
corr_drop_path = os.path.join(table_out_dir, get_value('science.tables.result_040', 'result_040.csv'))
corr_drop_df.to_csv(corr_drop_path, index=False, encoding='utf-8-sig')
print('\nInitial candidate variables:')
print(candidate_vars_raw)
print('\nFiltered candidate variables:')
print(candidate_vars_filtered)
print('\nFinal candidate variables after correlation filtering:')
print(candidate_vars)

def fit_univariable_logistic(df, outcome_col, var):
    tmp = df[[outcome_col, var]].copy()
    tmp[outcome_col] = pd.to_numeric(tmp[outcome_col], errors='coerce')
    tmp[var] = pd.to_numeric(tmp[var], errors='coerce')
    tmp = tmp.dropna()
    if tmp.shape[0] < 20 or tmp[outcome_col].nunique() < 2 or tmp[var].nunique() < 2:
        return None
    x = scale_for_model(tmp[var])
    X = sm.add_constant(x, has_constant='add')
    y = tmp[outcome_col].astype(int)
    try:
        fit = sm.GLM(y, X, family=sm.families.Binomial()).fit()
        coef = fit.params[var]
        ci = fit.conf_int().loc[var]
        p = fit.pvalues[var]
        return {'Effect': np.exp(coef), 'CI_lower': np.exp(ci[0]), 'CI_upper': np.exp(ci[1]), 'P_value': p, 'Effect_type': 'OR'}
    except:
        return None

def fit_univariable_cox(df, time_col, event_col, var, penalizer=0.02):
    tmp = df[[time_col, event_col, var]].copy()
    for c in [time_col, event_col, var]:
        tmp[c] = pd.to_numeric(tmp[c], errors='coerce')
    tmp = tmp.dropna(subset=[time_col, event_col, var])
    if tmp.shape[0] < 20 or tmp[event_col].sum() < 3 or tmp[var].nunique() < 2:
        return None
    tmp[var] = scale_for_model(tmp[var])
    try:
        cph = CoxPHFitter(penalizer=penalizer)
        cph.fit(tmp, duration_col=time_col, event_col=event_col, robust=True)
        s = cph.summary.loc[var]
        return {'Effect': s['exp(coef)'], 'CI_lower': s['exp(coef) lower 95%'], 'CI_upper': s['exp(coef) upper 95%'], 'P_value': s['p'], 'Effect_type': 'HR'}
    except:
        return None
univ_rows = []
for v in candidate_vars:
    row = {'Variable': v}
    res_resp = fit_univariable_logistic(analysis_df, outcome_col='Response_true', var=v)
    if res_resp is not None:
        row.update({'Response_OR': res_resp['Effect'], 'Response_CI_lower': res_resp['CI_lower'], 'Response_CI_upper': res_resp['CI_upper'], 'Response_P': res_resp['P_value']})
    res_pfs = fit_univariable_cox(analysis_df, time_col='PFS_time', event_col='PFS_status', var=v)
    if res_pfs is not None:
        row.update({'PFS_HR': res_pfs['Effect'], 'PFS_CI_lower': res_pfs['CI_lower'], 'PFS_CI_upper': res_pfs['CI_upper'], 'PFS_P': res_pfs['P_value']})
    res_os = fit_univariable_cox(analysis_df, time_col='OS_time', event_col='OS_status', var=v)
    if res_os is not None:
        row.update({'OS_HR': res_os['Effect'], 'OS_CI_lower': res_os['CI_lower'], 'OS_CI_upper': res_os['CI_upper'], 'OS_P': res_os['P_value']})
    univ_rows.append(row)
univ_df = pd.DataFrame(univ_rows)
univ_path = os.path.join(table_out_dir, get_value('science.tables.clinical_009', 'clinical_009.csv'))
univ_df.to_csv(univ_path, index=False, encoding='utf-8-sig')



def stability_select_logistic(df, outcome_col, candidate_vars, n_boot=100, random_state=42, threshold=0.4, max_vars=8, min_vars=4):
    tmp = df[[outcome_col] + candidate_vars].copy()
    tmp[outcome_col] = pd.to_numeric(tmp[outcome_col], errors='coerce')
    tmp = tmp.dropna(subset=[outcome_col]).reset_index(drop=True)
    y = tmp[outcome_col].astype(int).values
    X = tmp[candidate_vars].copy()
    rng = np.random.default_rng(random_state)
    select_count = pd.Series(0, index=candidate_vars, dtype=float)
    valid_boot = 0
    for b in range(n_boot):
        idx = rng.choice(np.arange(tmp.shape[0]), size=tmp.shape[0], replace=True)
        y_b = y[idx]
        if len(np.unique(y_b)) < 2:
            continue
        X_b = X.iloc[idx].copy()
        try:
            pipe = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler()), ('clf', LogisticRegressionCV(Cs=np.logspace(-2, 1, 8), cv=3, penalty='elasticnet', solver='saga', l1_ratios=[0.1, 0.5, 0.9], scoring='roc_auc', class_weight='balanced', max_iter=10000, random_state=random_state + b, n_jobs=-1))])
            pipe.fit(X_b, y_b)
            coef = pipe.named_steps['clf'].coef_.reshape(-1)
            selected = np.abs(coef) > 1e-06
            select_count.loc[np.array(candidate_vars)[selected]] += 1
            valid_boot += 1
        except Exception as e:
            continue
    freq = select_count / max(valid_boot, 1)
    result = pd.DataFrame({'Variable': candidate_vars, 'Selection_frequency': freq.values, 'Outcome': outcome_col, 'Model_type': 'Elastic-net logistic'}).sort_values('Selection_frequency', ascending=False)
    selected_vars = result.loc[result['Selection_frequency'] >= threshold, 'Variable'].tolist()
    if len(selected_vars) < min_vars:
        selected_vars = result.head(min_vars)['Variable'].tolist()
    selected_vars = selected_vars[:max_vars]
    return (result, selected_vars)

def stability_select_cox(df, time_col, event_col, candidate_vars, n_boot=100, random_state=42, threshold=0.4, max_vars=8, min_vars=4, penalizer=0.08, l1_ratio=0.7):
    tmp = df[[time_col, event_col] + candidate_vars].copy()
    for c in [time_col, event_col] + candidate_vars:
        tmp[c] = pd.to_numeric(tmp[c], errors='coerce')
    tmp = tmp.dropna(subset=[time_col, event_col]).reset_index(drop=True)
    rng = np.random.default_rng(random_state)
    select_count = pd.Series(0, index=candidate_vars, dtype=float)
    valid_boot = 0
    for b in range(n_boot):
        idx = rng.choice(np.arange(tmp.shape[0]), size=tmp.shape[0], replace=True)
        boot = tmp.iloc[idx].copy().reset_index(drop=True)
        if boot[event_col].sum() < 3:
            continue
        X = boot[candidate_vars].copy()
        try:
            imputer = SimpleImputer(strategy='median')
            scaler = StandardScaler()
            X_imp = imputer.fit_transform(X)
            X_scaled = scaler.fit_transform(X_imp)
            cox_df = pd.DataFrame(X_scaled, columns=candidate_vars)
            cox_df[time_col] = boot[time_col].values
            cox_df[event_col] = boot[event_col].values
            keep_vars = [c for c in candidate_vars if cox_df[c].nunique(dropna=True) > 1]
            if len(keep_vars) == 0:
                continue
            cox_df = cox_df[[time_col, event_col] + keep_vars]
            cph = CoxPHFitter(penalizer=penalizer, l1_ratio=l1_ratio)
            cph.fit(cox_df, duration_col=time_col, event_col=event_col)
            coefs = cph.params_
            selected = coefs[np.abs(coefs) > 0.0001].index.tolist()
            for v in selected:
                if v in select_count.index:
                    select_count.loc[v] += 1
            valid_boot += 1
        except Exception:
            continue
    freq = select_count / max(valid_boot, 1)
    result = pd.DataFrame({'Variable': candidate_vars, 'Selection_frequency': freq.values, 'Outcome': f'{time_col}/{event_col}', 'Model_type': 'Elastic-net Cox'}).sort_values('Selection_frequency', ascending=False)
    selected_vars = result.loc[result['Selection_frequency'] >= threshold, 'Variable'].tolist()
    if len(selected_vars) < min_vars:
        selected_vars = result.head(min_vars)['Variable'].tolist()
    selected_vars = selected_vars[:max_vars]
    return (result, selected_vars)

stab_resp_df, extended_response_vars = stability_select_logistic(analysis_df, outcome_col='Response_true', candidate_vars=candidate_vars, n_boot=N_BOOT_STABILITY, random_state=RANDOM_STATE, threshold=STABILITY_THRESHOLD, max_vars=MAX_EXTENDED_VARS, min_vars=MIN_EXTENDED_VARS)
stab_pfs_df, extended_pfs_vars = stability_select_cox(analysis_df, time_col='PFS_time', event_col='PFS_status', candidate_vars=candidate_vars, n_boot=N_BOOT_STABILITY, random_state=RANDOM_STATE + 10, threshold=STABILITY_THRESHOLD, max_vars=MAX_EXTENDED_VARS, min_vars=MIN_EXTENDED_VARS)
stab_os_df, extended_os_vars = stability_select_cox(analysis_df, time_col='OS_time', event_col='OS_status', candidate_vars=candidate_vars, n_boot=N_BOOT_STABILITY, random_state=RANDOM_STATE + 20, threshold=STABILITY_THRESHOLD, max_vars=MAX_EXTENDED_VARS, min_vars=MIN_EXTENDED_VARS)
stability_all_df = pd.concat([stab_resp_df.assign(Endpoint='Response'), stab_pfs_df.assign(Endpoint='PFS'), stab_os_df.assign(Endpoint='OS')], axis=0)
stability_path = os.path.join(table_out_dir, get_value('science.tables.result_041', 'result_041.csv'))
stability_all_df.to_csv(stability_path, index=False, encoding='utf-8-sig')
selected_summary_df = pd.DataFrame({'Endpoint': ['Response', 'PFS', 'OS'], 'Selected_extended_variables': ['; '.join(extended_response_vars), '; '.join(extended_pfs_vars), '; '.join(extended_os_vars)]})
selected_summary_path = os.path.join(table_out_dir, get_value('science.tables.clinical_010', 'clinical_010.csv'))
selected_summary_df.to_csv(selected_summary_path, index=False, encoding='utf-8-sig')
print('\nSelected variables for extended clinical models:')
print('Response:', extended_response_vars)
print('PFS:', extended_pfs_vars)
print('OS:', extended_os_vars)



def fit_logistic_avatar(df, exposure_col, covariates, model_label):
    cols = ['Response_true', exposure_col] + covariates
    tmp = df[cols].copy()
    tmp['Response_true'] = pd.to_numeric(tmp['Response_true'], errors='coerce')
    tmp[exposure_col] = pd.to_numeric(tmp[exposure_col], errors='coerce')
    tmp = tmp.dropna(subset=['Response_true', exposure_col])
    X = prepare_regression_matrix(tmp, [exposure_col] + covariates)
    for c in X.columns:
        if c != exposure_col:
            X[c] = scale_for_model(X[c])
    X = sm.add_constant(X, has_constant='add')
    y = tmp['Response_true'].astype(int)
    try:
        fit = sm.GLM(y, X, family=sm.families.Binomial()).fit()
        coef = fit.params[exposure_col]
        ci = fit.conf_int().loc[exposure_col]
        p = fit.pvalues[exposure_col]
        return {'Outcome': 'Response', 'Adjustment': model_label, 'Effect_type': 'OR', 'Effect': np.exp(coef), 'CI_lower': np.exp(ci[0]), 'CI_upper': np.exp(ci[1]), 'P_value': p, 'n': tmp.shape[0], 'events': int(y.sum())}
    except Exception as e:
        return {'Outcome': 'Response', 'Adjustment': model_label, 'Effect_type': 'OR', 'Effect': np.nan, 'CI_lower': np.nan, 'CI_upper': np.nan, 'P_value': np.nan, 'n': tmp.shape[0], 'events': int(y.sum()) if tmp.shape[0] > 0 else 0, 'Status': f'failed: {e}'}

def fit_cox_avatar(df, time_col, event_col, exposure_col, covariates, outcome_label, model_label, penalizer=0.05):
    cols = [time_col, event_col, exposure_col] + covariates
    tmp = df[cols].copy()
    for c in cols:
        tmp[c] = pd.to_numeric(tmp[c], errors='coerce')
    tmp = tmp.dropna(subset=[time_col, event_col, exposure_col])
    X = prepare_regression_matrix(tmp, [exposure_col] + covariates)
    for c in X.columns:
        if c != exposure_col:
            X[c] = scale_for_model(X[c])
    cox_df = pd.concat([tmp[[time_col, event_col]].reset_index(drop=True), X.reset_index(drop=True)], axis=1)
    try:
        cph = CoxPHFitter(penalizer=penalizer)
        cph.fit(cox_df, duration_col=time_col, event_col=event_col, robust=True)
        s = cph.summary.loc[exposure_col]
        return {'Outcome': outcome_label, 'Adjustment': model_label, 'Effect_type': 'HR', 'Effect': s['exp(coef)'], 'CI_lower': s['exp(coef) lower 95%'], 'CI_upper': s['exp(coef) upper 95%'], 'P_value': s['p'], 'n': cox_df.shape[0], 'events': int(cox_df[event_col].sum())}
    except Exception as e:
        return {'Outcome': outcome_label, 'Adjustment': model_label, 'Effect_type': 'HR', 'Effect': np.nan, 'CI_lower': np.nan, 'CI_upper': np.nan, 'P_value': np.nan, 'n': tmp.shape[0], 'events': int(tmp[event_col].sum()) if tmp.shape[0] > 0 else 0, 'Status': f'failed: {e}'}
association_rows = []
association_rows.append(fit_logistic_avatar(analysis_df, exposure_col=get_value('science.columns.benefit_004', 'AVATAR_risk_ord'), covariates=[], model_label='Unadjusted'))
association_rows.append(fit_logistic_avatar(analysis_df, exposure_col=get_value('science.columns.benefit_004', 'AVATAR_risk_ord'), covariates=core_covariates, model_label='Core-adjusted'))
association_rows.append(fit_logistic_avatar(analysis_df, exposure_col=get_value('science.columns.benefit_004', 'AVATAR_risk_ord'), covariates=extended_response_vars, model_label='Extended-adjusted'))
for adjustment, covs in [('Unadjusted', []), ('Core-adjusted', core_covariates), ('Extended-adjusted', extended_pfs_vars)]:
    association_rows.append(fit_cox_avatar(analysis_df, time_col='PFS_time', event_col='PFS_status', exposure_col=get_value('science.columns.benefit_004', 'AVATAR_risk_ord'), covariates=covs, outcome_label='PFS', model_label=adjustment, penalizer=0.05))
for adjustment, covs in [('Unadjusted', []), ('Core-adjusted', core_covariates), ('Extended-adjusted', extended_os_vars)]:
    association_rows.append(fit_cox_avatar(analysis_df, time_col='OS_time', event_col='OS_status', exposure_col=get_value('science.columns.benefit_004', 'AVATAR_risk_ord'), covariates=covs, outcome_label='OS', model_label=adjustment, penalizer=0.05))
association_df = pd.DataFrame(association_rows)
association_path = os.path.join(table_out_dir, get_value('science.tables.result_042', 'result_042.csv'))
association_df.to_csv(association_path, index=False, encoding='utf-8-sig')




def get_oof_logistic_predictions(df, feature_cols, outcome_col='Response_true', n_splits=5, n_repeats=20, random_state=42):
    tmp = df[['sample', outcome_col] + feature_cols].copy()
    tmp[outcome_col] = pd.to_numeric(tmp[outcome_col], errors='coerce')
    tmp = tmp.dropna(subset=[outcome_col]).reset_index(drop=True)
    y = tmp[outcome_col].astype(int).values
    X = tmp[feature_cols].copy()
    if len(np.unique(y)) < 2:
        raise ValueError('Outcome has only one class.')
    event_count = np.min(np.bincount(y))
    n_splits_use = min(n_splits, event_count)
    n_splits_use = max(2, n_splits_use)
    rskf = RepeatedStratifiedKFold(n_splits=n_splits_use, n_repeats=n_repeats, random_state=random_state)
    pred_records = []
    for fold_id, (train_idx, test_idx) in enumerate(rskf.split(X, y), start=1):
        model = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler()), ('clf', LogisticRegression(max_iter=2000, class_weight='balanced', solver='liblinear', random_state=random_state))])
        model.fit(X.iloc[train_idx], y[train_idx])
        pred = model.predict_proba(X.iloc[test_idx])[:, 1]
        pred_records.append(pd.DataFrame({'sample': tmp.iloc[test_idx]['sample'].values, 'y_true': y[test_idx], 'y_pred': pred, 'fold_id': fold_id}))
    pred_long = pd.concat(pred_records, axis=0)
    pred_agg = pred_long.groupby('sample', as_index=False).agg({'y_true': 'first', 'y_pred': 'mean'})
    return (pred_agg, pred_long)

def net_benefit(y_true, y_prob, thresholds):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    n = len(y_true)
    nbs = []
    for pt in thresholds:
        pred_pos = y_prob >= pt
        tp = np.sum((pred_pos == 1) & (y_true == 1))
        fp = np.sum((pred_pos == 1) & (y_true == 0))
        nb = tp / n - fp / n * (pt / (1 - pt))
        nbs.append(nb)
    return np.array(nbs)
dca_df = analysis_df.copy()
core_pred, _ = get_oof_logistic_predictions(dca_df, feature_cols=core_covariates, outcome_col='Response_true', n_repeats=DCA_N_REPEATS, random_state=RANDOM_STATE)
core_pred = core_pred.rename(columns={'y_true': 'Response_true', 'y_pred': 'Core_clinical_pred'})
if len(extended_response_vars) == 0:
    extended_response_vars = core_covariates.copy()
extended_pred, _ = get_oof_logistic_predictions(dca_df, feature_cols=extended_response_vars, outcome_col='Response_true', n_repeats=DCA_N_REPEATS, random_state=RANDOM_STATE + 1)
extended_pred = extended_pred.rename(columns={'y_true': 'Response_true', 'y_pred': 'Extended_clinical_pred'})
avatar_pred = dca_df[['sample', 'Response_true', 'Response_pred']].copy()
avatar_pred = avatar_pred.rename(columns={'Response_pred': get_value('science.columns.benefit_005', 'AVATAR_pred')})
dca_merged = avatar_pred.merge(core_pred[['sample', 'Core_clinical_pred']], on='sample', how='inner').merge(extended_pred[['sample', 'Extended_clinical_pred']], on='sample', how='inner')
dca_pred_path = os.path.join(table_out_dir, get_value('science.tables.predictions_018', 'predictions_018.csv'))
dca_merged.to_csv(dca_pred_path, index=False, encoding='utf-8-sig')


print('\nDCA AUCs:')
for name, col in [('Core clinical', 'Core_clinical_pred'), ('Extended clinical', 'Extended_clinical_pred'), (get_value('science.labels.benefit_004', 'AVATAR'), get_value('science.columns.benefit_005', 'AVATAR_pred'))]:
    try:
        auc = roc_auc_score(dca_merged['Response_true'], dca_merged[col])
        print(f'{name}: {auc:.3f}')
    except:
        print(f'{name}: NA')
heat_df = analysis_df.sort_values(get_value('science.columns.benefit_001', 'AVATAR_score'), ascending=False).copy()
heat_df = heat_df.set_index('sample')
heat_rows = pd.DataFrame(index=heat_df.index)
heat_rows['Response probability'] = percentile_rank(heat_df['Response_pred'])
heat_rows['PFS risk percentile'] = percentile_rank(heat_df['PFS_risk'])
heat_rows['OS risk percentile'] = percentile_rank(heat_df['OS_risk'])
heat_rows['Observed response'] = heat_df['Response_true'].astype(float)
if 'AFP' in heat_df.columns:
    heat_rows['AFP'] = percentile_rank(log1p_safe(heat_df['AFP']).fillna(log1p_safe(heat_df['AFP']).median()))
if 'Tumor_size' in heat_df.columns:
    heat_rows['Tumor size'] = percentile_rank(log1p_safe(heat_df['Tumor_size']).fillna(log1p_safe(heat_df['Tumor_size']).median()))
if 'PVTT' in heat_df.columns:
    heat_rows['PVTT'] = heat_df['PVTT'].fillna(heat_df['PVTT'].mode(dropna=True).iloc[0] if heat_df['PVTT'].notna().sum() > 0 else 0)
if 'ALBI' in heat_df.columns:
    albi = pd.to_numeric(heat_df['ALBI'], errors='coerce')
    heat_rows['ALBI'] = percentile_rank(albi.fillna(albi.median()))
heat_mat = heat_rows.T
avatar_ordered = heat_df[get_value('science.columns.benefit_002', 'AVATAR_raw')].values
avatar_color_map = {get_value('science.labels.benefit_001', 'AVATAR-I'): '#2C7BB6', get_value('science.labels.benefit_002', 'AVATAR-II'): '#FDB863', get_value('science.labels.benefit_003', 'AVATAR-III'): '#D7191C'}
avatar_colors = [avatar_color_map[x] for x in avatar_ordered]
boundaries = []
for i in range(1, len(avatar_ordered)):
    if avatar_ordered[i] != avatar_ordered[i - 1]:
        boundaries.append(i - 0.5)
group_centers = []
group_labels = []
start = 0
for i in range(1, len(avatar_ordered) + 1):
    if i == len(avatar_ordered) or avatar_ordered[i] != avatar_ordered[start]:
        end = i - 1
        group_centers.append((start + end) / 2)
        group_labels.append({get_value('science.labels.benefit_001', 'AVATAR-I'): get_value('science.messages.status_007', 'AVATAR-I High-benefit'), get_value('science.labels.benefit_002', 'AVATAR-II'): get_value('science.messages.status_008', 'AVATAR-II Intermediate'), get_value('science.labels.benefit_003', 'AVATAR-III'): get_value('science.messages.status_009', 'AVATAR-III Low-benefit')}[avatar_ordered[start]])
        start = i
fig = plt.figure(figsize=(14.5, 4.9))
gs = gridspec.GridSpec(2, 1, height_ratios=[0.22, 1.0], hspace=0.04)
ax_anno = fig.add_subplot(gs[0, 0])
ax_heat = fig.add_subplot(gs[1, 0])
for i, color in enumerate(avatar_colors):
    ax_anno.add_patch(mpatches.Rectangle((i - 0.5, 0), 1, 1, facecolor=color, edgecolor=color, linewidth=0))
ax_anno.set_xlim(-0.5, len(avatar_colors) - 0.5)
ax_anno.set_ylim(0, 1)
ax_anno.set_xticks([])
ax_anno.set_yticks([])
for spine in ax_anno.spines.values():
    spine.set_visible(False)
for b in boundaries:
    ax_anno.axvline(b, color='black', linewidth=1.0)
for center, label in zip(group_centers, group_labels):
    ax_anno.text(center, 1.18, label, ha='center', va='bottom', fontsize=10, fontweight='bold')
ax_anno.text(-1.0, 0.5, get_value('science.labels.benefit_004', 'AVATAR'), ha='right', va='center', fontsize=10, fontweight='bold')
legend_patches = [mpatches.Patch(color=avatar_color_map[get_value('science.labels.benefit_001', 'AVATAR-I')], label=get_value('science.messages.status_007', 'AVATAR-I High-benefit')), mpatches.Patch(color=avatar_color_map[get_value('science.labels.benefit_002', 'AVATAR-II')], label=get_value('science.messages.status_008', 'AVATAR-II Intermediate')), mpatches.Patch(color=avatar_color_map[get_value('science.labels.benefit_003', 'AVATAR-III')], label=get_value('science.messages.status_009', 'AVATAR-III Low-benefit'))]
ax_anno.legend(handles=legend_patches, loc='upper right', bbox_to_anchor=(1.0, 1.95), ncol=3, frameon=False, fontsize=8)
im = ax_heat.imshow(heat_mat.values, aspect='auto', interpolation='nearest', cmap='RdBu_r', vmin=0, vmax=1)
ax_heat.set_yticks(np.arange(heat_mat.shape[0]))
ax_heat.set_yticklabels(heat_mat.index, fontsize=9)
ax_heat.set_xticks([])
ax_heat.set_xlabel(get_value('science.messages.status_010', 'Patients ordered by AVATAR score'), fontsize=10)
for b in boundaries:
    ax_heat.axvline(b, color='black', linewidth=1.0)
cbar = plt.colorbar(im, ax=ax_heat, fraction=0.022, pad=0.015)
cbar.set_label('Percentile / scaled value', fontsize=9)
ax_anno.set_title(get_value('science.messages.status_011', 'Figure 4A. Patient-level AVATAR-based multimodal benefit stratification'), fontsize=12, pad=28)
plt.tight_layout()
fig4a_path = os.path.join(panel_out_dir, get_value('science.figures.plot_055', 'plot_055.png'))
plt.savefig(fig4a_path, dpi=600, bbox_inches='tight')
plt.show()

km_color_map = {get_value('science.messages.status_001', 'AVATAR-I\nHigh-benefit'): '#2C7BB6', get_value('science.messages.status_002', 'AVATAR-II\nIntermediate'): '#FDB863', get_value('science.messages.status_003', 'AVATAR-III\nLow-benefit'): '#D7191C'}
group_order = [get_value('science.messages.status_001', 'AVATAR-I\nHigh-benefit'), get_value('science.messages.status_002', 'AVATAR-II\nIntermediate'), get_value('science.messages.status_003', 'AVATAR-III\nLow-benefit')]

def get_avatar_cox_hr_for_plot(df, time_col, event_col):
    tmp = df[[time_col, event_col, get_value('science.columns.benefit_004', 'AVATAR_risk_ord')]].copy()
    for c in [time_col, event_col, get_value('science.columns.benefit_004', 'AVATAR_risk_ord')]:
        tmp[c] = pd.to_numeric(tmp[c], errors='coerce')
    tmp = tmp.dropna()
    try:
        cph = CoxPHFitter(penalizer=0.02)
        cph.fit(tmp, duration_col=time_col, event_col=event_col, robust=True)
        s = cph.summary.loc[get_value('science.columns.benefit_004', 'AVATAR_risk_ord')]
        return {'HR': s['exp(coef)'], 'CI_lower': s['exp(coef) lower 95%'], 'CI_upper': s['exp(coef) upper 95%'], 'P': s['p']}
    except:
        return {'HR': np.nan, 'CI_lower': np.nan, 'CI_upper': np.nan, 'P': np.nan}

def plot_km_by_avatar(df, time_col, event_col, title, filename):
    tmp = df[[time_col, event_col, get_value('science.columns.benefit_003', 'AVATAR_group'), get_value('science.columns.benefit_004', 'AVATAR_risk_ord')]].dropna().copy()
    fig, ax = plt.subplots(figsize=(5.4, 4.4))
    for group in group_order:
        sub = tmp[tmp[get_value('science.columns.benefit_003', 'AVATAR_group')] == group]
        if sub.shape[0] == 0:
            continue
        kmf = KaplanMeierFitter()
        kmf.fit(durations=sub[time_col], event_observed=sub[event_col], label=f"{group.replace(chr(10), ' ')} (n={sub.shape[0]})")
        kmf.plot_survival_function(ax=ax, ci_show=True, color=km_color_map[group], linewidth=2.0)
    lr = multivariate_logrank_test(tmp[time_col], tmp[get_value('science.columns.benefit_003', 'AVATAR_group')], tmp[event_col])
    hr_info = get_avatar_cox_hr_for_plot(tmp, time_col, event_col)
    text = f"Log-rank {format_p(lr.p_value)}\nHR per level worse = {hr_info['HR']:.2f} ({hr_info['CI_lower']:.2f}-{hr_info['CI_upper']:.2f})"
    ax.text(0.04, 0.08, text, transform=ax.transAxes, fontsize=8, ha='left', va='bottom', bbox=dict(boxstyle='round', facecolor='white', alpha=0.86, edgecolor='gray'))
    ax.set_title(title, fontsize=11)
    ax.set_xlabel(infer_time_unit(tmp[time_col]))
    ax.set_ylabel('Survival probability')
    ax.set_ylim(0, 1.05)
    ax.grid(alpha=0.25)
    ax.legend(frameon=False, fontsize=7, loc='upper right')
    plt.tight_layout()
    path = os.path.join(panel_out_dir, filename)
    plt.savefig(path, dpi=600, bbox_inches='tight')
    plt.show()
    
    return (path, lr.p_value)
fig4b_path, p_pfs_km = plot_km_by_avatar(analysis_df, time_col='PFS_time', event_col='PFS_status', title=get_value('science.messages.status_012', 'Figure 4B. PFS by AVATAR'), filename=get_value('science.figures.plot_056', 'plot_056.png'))
fig4c_path, p_os_km = plot_km_by_avatar(analysis_df, time_col='OS_time', event_col='OS_status', title=get_value('science.messages.status_013', 'Figure 4C. OS by AVATAR'), filename=get_value('science.figures.plot_057', 'plot_057.png'))
pfs_t = survival_timepoint_value(analysis_df['PFS_time'], months=12)
os_t = survival_timepoint_value(analysis_df['OS_time'], months=18)
summary_rows = []
for group in group_order:
    sub = analysis_df[analysis_df[get_value('science.columns.benefit_003', 'AVATAR_group')] == group].copy()
    summary_rows.append({get_value('science.columns.benefit_003', 'AVATAR_group'): group, 'n': sub.shape[0], 'Response rate': sub['Response_true'].mean(), '12-month PFS rate': km_survival_at_time(sub, 'PFS_time', 'PFS_status', pfs_t), '18-month OS rate': km_survival_at_time(sub, 'OS_time', 'OS_status', os_t)})
outcome_summary_df = pd.DataFrame(summary_rows)
outcome_summary_path = os.path.join(table_out_dir, get_value('science.tables.metrics_019', 'metrics_019.csv'))
outcome_summary_df.to_csv(outcome_summary_path, index=False, encoding='utf-8-sig')
try:
    tab_resp = pd.crosstab(analysis_df[get_value('science.columns.benefit_003', 'AVATAR_group')], analysis_df['Response_true'])
    p_resp = chi2_contingency(tab_resp)[1]
except:
    p_resp = np.nan
fig, ax = plt.subplots(figsize=(6.8, 4.5))
x = np.arange(len(group_order))
width = 0.24
metrics = ['Response rate', '12-month PFS rate', '18-month OS rate']
for i, metric in enumerate(metrics):
    vals = outcome_summary_df[metric].values
    ax.bar(x + (i - 1) * width, vals, width=width, label=metric, edgecolor='black', linewidth=0.7, alpha=0.88)
    for j, v in enumerate(vals):
        ax.text(x[j] + (i - 1) * width, v + 0.025, f'{v:.2f}', ha='center', va='bottom', fontsize=8)
ax.set_xticks(x)
ax.set_xticklabels([get_value('science.messages.status_001', 'AVATAR-I\nHigh-benefit'), get_value('science.messages.status_002', 'AVATAR-II\nIntermediate'), get_value('science.messages.status_003', 'AVATAR-III\nLow-benefit')], fontsize=8)
ax.set_ylim(0, 1.15)
ax.set_ylabel('Observed favorable outcome rate')
ax.set_title(get_value('science.messages.status_014', 'Figure 4D. Observed outcomes across AVATAR groups'), fontsize=11)
ax.grid(axis='y', linestyle='--', alpha=0.3)
ax.legend(frameon=False, fontsize=8, loc='upper right')
p_text = f'Response {format_p(p_resp)}\nPFS log-rank {format_p(p_pfs_km)}\nOS log-rank {format_p(p_os_km)}'
ax.text(0.02, 0.97, p_text, transform=ax.transAxes, ha='left', va='top', fontsize=8, bbox=dict(boxstyle='round', facecolor='white', alpha=0.85, edgecolor='gray'))
plt.tight_layout()
fig4d_path = os.path.join(panel_out_dir, get_value('science.figures.plot_058', 'plot_058.png'))
plt.savefig(fig4d_path, dpi=600, bbox_inches='tight')
plt.show()

thresholds = np.linspace(0.05, 0.8, 76)
y_true = dca_merged['Response_true'].astype(int).values
prevalence = np.mean(y_true)
treat_all_nb = prevalence - (1 - prevalence) * thresholds / (1 - thresholds)
treat_none_nb = np.zeros_like(thresholds)
fig, ax = plt.subplots(figsize=(6.4, 4.8))
ax.plot(thresholds, treat_all_nb, linestyle='--', linewidth=1.3, label='Treat all')
ax.plot(thresholds, treat_none_nb, linestyle=':', linewidth=1.6, label='Treat none')
dca_models = {'Core clinical': dca_merged['Core_clinical_pred'].values, 'Extended clinical': dca_merged['Extended_clinical_pred'].values, get_value('science.labels.benefit_004', 'AVATAR'): dca_merged[get_value('science.columns.benefit_005', 'AVATAR_pred')].values}
for name, prob in dca_models.items():
    nb = net_benefit(y_true, prob, thresholds)
    ax.plot(thresholds, nb, linewidth=2.0, label=name)
ax.set_xlabel('Threshold probability')
ax.set_ylabel('Net benefit')
ax.set_title('Figure 4E. Decision curve analysis for response prediction', fontsize=11)
ax.set_ylim(-0.08, max(0.55, np.nanmax(treat_all_nb) + 0.1))
ax.grid(alpha=0.3)
ax.legend(frameon=False, fontsize=8, loc='upper right')
plt.tight_layout()
fig4e_path = os.path.join(panel_out_dir, get_value('science.figures.plot_059', 'plot_059.png'))
plt.savefig(fig4e_path, dpi=600, bbox_inches='tight')
plt.show()

plot_df = association_df.copy()
outcome_order = ['Response', 'PFS', 'OS']
adjust_order = ['Unadjusted', 'Core-adjusted', 'Extended-adjusted']
plot_df['Outcome_order'] = plot_df['Outcome'].map({k: i for i, k in enumerate(outcome_order)})
plot_df['Adjustment_order'] = plot_df['Adjustment'].map({k: i for i, k in enumerate(adjust_order)})
plot_df = plot_df.sort_values(['Outcome_order', 'Adjustment_order']).reset_index(drop=True)
plot_df['Label'] = plot_df['Outcome'] + ' | ' + plot_df['Adjustment']
plot_df['Effect_CI'] = plot_df.apply(lambda r: format_ci_effect(r['Effect'], r['CI_lower'], r['CI_upper']), axis=1)
plot_df['P_text'] = plot_df['P_value'].apply(format_p_plain)
fig_height = max(5.2, 0.48 * plot_df.shape[0] + 1.5)
fig, ax = plt.subplots(figsize=(8.4, fig_height))
y = np.arange(plot_df.shape[0])
for i in range(plot_df.shape[0]):
    if i % 2 == 0:
        ax.axhspan(i - 0.5, i + 0.5, color='#F3F6F4', zorder=0)
ax.errorbar(plot_df['Effect'], y, xerr=[plot_df['Effect'] - plot_df['CI_lower'], plot_df['CI_upper'] - plot_df['Effect']], fmt='o', color='black', ecolor='black', elinewidth=1.2, capsize=3, markersize=4.5, zorder=3)
ax.axvline(1.0, color='gray', linestyle='--', linewidth=1.1)
ax.set_xscale('log')
valid_lower = plot_df['CI_lower'].replace([np.inf, -np.inf], np.nan).dropna()
valid_upper = plot_df['CI_upper'].replace([np.inf, -np.inf], np.nan).dropna()
xmin = max(0.005, valid_lower.min() * 0.7) if valid_lower.shape[0] > 0 else 0.01
xmax = valid_upper.max() * 1.4 if valid_upper.shape[0] > 0 else 10
if not np.isfinite(xmin) or xmin <= 0:
    xmin = 0.01
if not np.isfinite(xmax) or xmax <= 1:
    xmax = 10
ax.set_xlim(xmin, xmax)
ax.set_yticks(y)
ax.set_yticklabels(plot_df['Label'], fontsize=9)
ax.invert_yaxis()
ax.set_xlabel(get_value('science.messages.status_015', 'OR / HR per one-level worsening of AVATAR'))
ax.set_title(get_value('science.messages.status_016', 'Figure 4F. Association of AVATAR with outcomes under different adjustment levels'), fontsize=11)
ax.grid(axis='x', linestyle='--', alpha=0.25)
text_x1 = xmax * 1.08
text_x2 = xmax * 1.85
text_x3 = xmax * 2.55
for i, row in plot_df.iterrows():
    ax.text(text_x1, i, row['Effect_CI'], va='center', fontsize=8)
    ax.text(text_x2, i, row['P_text'], va='center', fontsize=8)
    ax.text(text_x3, i, f"{int(row['n'])}/{int(row['events'])}", va='center', fontsize=8)
ax.text(text_x1, -0.9, 'OR/HR (95% CI)', va='center', fontsize=8, fontweight='bold')
ax.text(text_x2, -0.9, 'P', va='center', fontsize=8, fontweight='bold')
ax.text(text_x3, -0.9, 'n/events', va='center', fontsize=8, fontweight='bold')
ax.set_xlim(xmin, xmax * 3.1)
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)
plt.tight_layout()
fig4f_path = os.path.join(panel_out_dir, get_value('science.figures.plot_060', 'plot_060.png'))
plt.savefig(fig4f_path, dpi=600, bbox_inches='tight')
plt.show()


def add_median_split(df, var, display_name=None, lower_label=None, higher_label=None):
    if var not in df.columns:
        return None
    x = pd.to_numeric(df[var], errors='coerce')
    if x.notna().sum() < 20 or x.nunique(dropna=True) < 2:
        return None
    cutoff = x.median()
    new_col = f'{var}_subgroup'
    if display_name is None:
        display_name = var
    if lower_label is None:
        lower_label = f'{display_name} low'
    if higher_label is None:
        higher_label = f'{display_name} high'
    df[new_col] = np.nan
    df.loc[x <= cutoff, new_col] = lower_label
    df.loc[x > cutoff, new_col] = higher_label
    return {'var': new_col, 'display': display_name, 'type': 'median_split', 'cutoff': cutoff, 'levels': [lower_label, higher_label]}

def add_binary_split(df, var, display_name=None, labels=None):
    if var not in df.columns:
        return None
    x = pd.to_numeric(df[var], errors='coerce')
    valid_values = sorted(x.dropna().unique().tolist())
    if len(valid_values) < 2:
        return None
    new_col = f'{var}_subgroup'
    if display_name is None:
        display_name = var
    if labels is None:
        labels = {0: f'{display_name} absent', 1: f'{display_name} present'}
    df[new_col] = x.map(labels)
    return {'var': new_col, 'display': display_name, 'type': 'binary', 'cutoff': None, 'levels': list(labels.values())}

def fit_subgroup_cox(df, time_col, event_col, subgroup_var, subgroup_level, min_n=10, min_events=3, penalizer=0.05):
    sub = df[df[subgroup_var] == subgroup_level].copy()
    needed = [time_col, event_col, get_value('science.columns.benefit_004', 'AVATAR_risk_ord')]
    sub = sub[needed].copy()
    for c in needed:
        sub[c] = pd.to_numeric(sub[c], errors='coerce')
    sub = sub.dropna()
    n = sub.shape[0]
    events = int(sub[event_col].sum()) if n > 0 else 0
    n_levels = sub[get_value('science.columns.benefit_004', 'AVATAR_risk_ord')].nunique()
    if n < min_n or events < min_events or n_levels < 2:
        return {'n': n, 'events': events, 'HR': np.nan, 'CI_lower': np.nan, 'CI_upper': np.nan, 'P_value': np.nan, 'status': 'skipped'}
    try:
        cph = CoxPHFitter(penalizer=penalizer)
        cph.fit(sub, duration_col=time_col, event_col=event_col, robust=True)
        s = cph.summary.loc[get_value('science.columns.benefit_004', 'AVATAR_risk_ord')]
        return {'n': n, 'events': events, 'HR': s['exp(coef)'], 'CI_lower': s['exp(coef) lower 95%'], 'CI_upper': s['exp(coef) upper 95%'], 'P_value': s['p'], 'status': 'ok'}
    except Exception as e:
        return {'n': n, 'events': events, 'HR': np.nan, 'CI_lower': np.nan, 'CI_upper': np.nan, 'P_value': np.nan, 'status': f'failed: {e}'}
subgroup_df = analysis_df.copy()
subgroup_specs = []
spec = add_median_split(subgroup_df, var='AFP', display_name='AFP', lower_label='AFP low', higher_label='AFP high')
if spec is not None:
    subgroup_specs.append(spec)
spec = add_median_split(subgroup_df, var='Tumor_size', display_name='Tumor size', lower_label='Tumor size low', higher_label='Tumor size high')
if spec is not None:
    subgroup_specs.append(spec)
spec = add_binary_split(subgroup_df, var='PVTT', display_name='PVTT', labels={0: 'PVTT absent', 1: 'PVTT present'})
if spec is not None:
    subgroup_specs.append(spec)
spec = add_median_split(subgroup_df, var='ALBI', display_name='ALBI', lower_label='ALBI low', higher_label='ALBI high')
if spec is not None:
    subgroup_specs.append(spec)
extra_candidates = []
for v in extended_pfs_vars + extended_os_vars:
    if v not in ['log_AFP', 'log_Size', 'PVTT', 'ALBI'] and v not in extra_candidates:
        extra_candidates.append(v)
extra_candidates = extra_candidates[:2]
for v in extra_candidates:
    if v in subgroup_df.columns:
        pretty = v.replace('log_', '').replace('_', '-')
        spec = add_median_split(subgroup_df, var=v, display_name=pretty, lower_label=f'{pretty} low', higher_label=f'{pretty} high')
        if spec is not None:
            subgroup_specs.append(spec)
print('\nSubgroup variables:')
for s in subgroup_specs:
    print(s)
subgroup_results = []
for outcome, time_col, event_col in [('PFS', 'PFS_time', 'PFS_status'), ('OS', 'OS_time', 'OS_status')]:
    for spec in subgroup_specs:
        subgroup_var = spec['var']
        for level in spec['levels']:
            if level not in subgroup_df[subgroup_var].dropna().unique():
                continue
            res = fit_subgroup_cox(subgroup_df, time_col=time_col, event_col=event_col, subgroup_var=subgroup_var, subgroup_level=level, min_n=10, min_events=3, penalizer=0.05)
            res.update({'Outcome': outcome, 'Subgroup_variable': spec['display'], 'Subgroup_level': level, 'Subgroup_column': subgroup_var, 'Cutoff': spec['cutoff'], 'Effect': get_value('science.messages.status_017', 'HR per one-level worsening of AVATAR')})
            subgroup_results.append(res)
subgroup_results_df = pd.DataFrame(subgroup_results)
subgroup_result_path = os.path.join(supp_out_dir, get_value('science.tables.result_043', 'result_043.csv'))
subgroup_results_df.to_csv(subgroup_result_path, index=False, encoding='utf-8-sig')



def plot_subgroup_forest(subgroup_results_df, outcome, filename, title=None):
    plot_df = subgroup_results_df[(subgroup_results_df['Outcome'] == outcome) & (subgroup_results_df['status'] == 'ok')].copy()
    if plot_df.shape[0] == 0:
        print(f'No valid subgroup results for {outcome}.')
        return None
    preferred_order = {'AFP': 0, 'Tumor size': 1, 'PVTT': 2, 'ALBI': 3}
    plot_df['var_order'] = plot_df['Subgroup_variable'].map(lambda x: preferred_order.get(x, 10))
    plot_df = plot_df.sort_values(['var_order', 'Subgroup_variable', 'Subgroup_level']).reset_index(drop=True)
    plot_df['HR_CI'] = plot_df.apply(lambda r: format_ci_effect(r['HR'], r['CI_lower'], r['CI_upper']), axis=1)
    plot_df['P_text'] = plot_df['P_value'].apply(format_p_plain)
    y_labels = []
    previous_var = None
    for _, row in plot_df.iterrows():
        var = row['Subgroup_variable']
        level = row['Subgroup_level']
        if var != previous_var:
            y_labels.append(f'{var}\n  {level}')
            previous_var = var
        else:
            y_labels.append(f'  {level}')
    plot_df['Y_label'] = y_labels
    n_rows = plot_df.shape[0]
    fig_height = max(4.8, 0.45 * n_rows + 1.8)
    fig, ax = plt.subplots(figsize=(8.8, fig_height))
    y = np.arange(n_rows)
    for i in range(n_rows):
        if i % 2 == 0:
            ax.axhspan(i - 0.5, i + 0.5, color='#F3F6F4', zorder=0)
    ax.errorbar(plot_df['HR'], y, xerr=[plot_df['HR'] - plot_df['CI_lower'], plot_df['CI_upper'] - plot_df['HR']], fmt='o', color='black', ecolor='black', elinewidth=1.2, capsize=3, markersize=4.5, zorder=3)
    ax.axvline(1.0, color='gray', linestyle='--', linewidth=1.1)
    ax.set_xscale('log')
    xmin = max(0.6, np.nanmin(plot_df['CI_lower']) * 0.75)
    xmax = np.nanmax(plot_df['CI_upper']) * 1.35
    if not np.isfinite(xmin) or xmin <= 0:
        xmin = 0.7
    if not np.isfinite(xmax) or xmax <= 1:
        xmax = 4.0
    ax.set_xlim(xmin, xmax)
    ax.set_yticks(y)
    ax.set_yticklabels(plot_df['Y_label'], fontsize=9)
    ax.invert_yaxis()
    ax.set_xlabel(get_value('science.messages.status_017', 'HR per one-level worsening of AVATAR'), fontsize=10)
    if title is None:
        title = f"{get_value('science.messages.status_018', 'Subgroup analysis of AVATAR for ')}{outcome}"
    ax.set_title(title, fontsize=12, fontweight='bold', pad=14)
    ax.grid(axis='x', linestyle='--', alpha=0.25)
    text_x1 = xmax * 1.08
    text_x2 = xmax * 1.55
    text_x3 = xmax * 2.25
    for i, row in plot_df.iterrows():
        ax.text(text_x1, i, f"{int(row['n'])}/{int(row['events'])}", va='center', fontsize=8)
        ax.text(text_x2, i, row['HR_CI'], va='center', fontsize=8)
        ax.text(text_x3, i, row['P_text'], va='center', fontsize=8)
    ax.text(text_x1, -0.9, 'n/events', va='center', fontsize=8, fontweight='bold')
    ax.text(text_x2, -0.9, 'HR (95% CI)', va='center', fontsize=8, fontweight='bold')
    ax.text(text_x3, -0.9, 'P', va='center', fontsize=8, fontweight='bold')
    ax.set_xlim(xmin, xmax * 3.2)
    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)
    plt.tight_layout()
    out_path = os.path.join(supp_out_dir, filename)
    pdf_path = out_path.replace('.png', '.pdf')
    plt.savefig(out_path, dpi=600, bbox_inches='tight')
    plt.savefig(pdf_path, bbox_inches='tight')
    plt.show()
    
    
    
    return out_path
fig_s4b_path = plot_subgroup_forest(subgroup_results_df, outcome='PFS', filename=get_value('science.figures.plot_061', 'plot_061.png'), title=get_value('science.messages.status_019', 'Figure S4B. Subgroup analysis of AVATAR for PFS'))
fig_s4c_path = plot_subgroup_forest(subgroup_results_df, outcome='OS', filename=get_value('science.figures.plot_062', 'plot_062.png'), title=get_value('science.messages.status_020', 'Figure S4C. Subgroup analysis of AVATAR for OS'))




















In [ ]:
import os
import re
import glob
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Patch
base_project_dir = str(project_path(''))
all_modals_dir = os.path.join(base_project_dir, get_value('science.paths.directory_045', 'directory_045'))
model_out_dir = os.path.join(base_project_dir, get_value('science.paths.directory_010', 'directory_010'))
out_dir = os.path.join(model_out_dir, get_value('science.paths.directory_046', 'directory_046'))
os.makedirs(out_dir, exist_ok=True)
label_candidates = [os.path.join(all_modals_dir, get_value('science.tables.outcomes_009', 'outcomes_009.csv')), os.path.join(all_modals_dir, get_value('science.tables.clinical_011', 'clinical_011.csv')), os.path.join(all_modals_dir, get_value('science.tables.clinical_012', 'clinical_012.csv'))]
label_hits = []
for p in label_candidates:
    if os.path.exists(p):
        label_hits.append(p)
if len(label_hits) == 0:
    label_hits = glob.glob(os.path.join(base_project_dir, '**', get_value('science.tables.outcomes_010', 'outcomes_010.csv')), recursive=True)
if len(label_hits) == 0:
    raise FileNotFoundError(get_value('science.messages.status_021', 'Cannot find input. Please check label path.'))
label_path = label_hits[0]
print('Using label file:')

mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['font.family'] = 'Arial'
mpl.rcParams['axes.spines.top'] = False
mpl.rcParams['axes.spines.right'] = False
MODALITIES = {'Metagenomics': ['microbiota', 'metagenomics', 'Metagenomics'], 'Radiomics': ['radiomics', 'Radiomics'], 'Blood': ['blood', 'Blood'], 'Urine': ['urine', 'Urine']}
COHORT_ORDER = ['Internal', 'External']
MODALITY_COLORS = {'Metagenomics': '#2A9D8F', 'Radiomics': '#4C78A8', 'Blood': '#C44E52', 'Urine': '#E69F00'}
OUTCOME_COLORS = {'Response': '#2D6CDF', 'PFS': '#EE854A', 'OS': '#7A4FA3'}
STATUS_COLORS = {'Positive/Event': '#D95F02', 'Negative/Censored': '#BDBDBD'}

def norm_id(x):
    x = str(x).strip()
    x = re.sub('\\.0$', '', x)
    return x.lower()

def find_col(df, candidates):
    lower_map = {str(c).lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    for c in df.columns:
        cl = str(c).lower()
        for cand in candidates:
            if cand.lower() in cl:
                return c
    return None

def read_csv_any(path):
    return pd.read_csv(path, encoding='utf-8-sig', low_memory=False)

def infer_matrix_orientation_and_counts(path, sample_ids=None):
    pass
    df = pd.read_csv(path, index_col=0, encoding='utf-8-sig', low_memory=False)
    row_ids = set(map(norm_id, df.index))
    col_ids = set(map(norm_id, df.columns))
    if sample_ids is not None and len(sample_ids) > 0:
        sample_ids = set(map(norm_id, sample_ids))
        row_hit = len(row_ids & sample_ids)
        col_hit = len(col_ids & sample_ids)
        if row_hit >= col_hit:
            orientation = 'samples_by_features'
            n_samples = df.shape[0]
            n_features = df.shape[1]
            matched_samples = row_hit
        else:
            orientation = 'features_by_samples'
            n_samples = df.shape[1]
            n_features = df.shape[0]
            matched_samples = col_hit
    elif df.shape[0] <= df.shape[1]:
        orientation = 'samples_by_features'
        n_samples = df.shape[0]
        n_features = df.shape[1]
        matched_samples = np.nan
    else:
        orientation = 'features_by_samples'
        n_samples = df.shape[1]
        n_features = df.shape[0]
        matched_samples = np.nan
    return {'n_samples': int(n_samples), 'n_features': int(n_features), 'orientation': orientation, 'matched_samples': matched_samples, 'shape_rows': df.shape[0], 'shape_cols': df.shape[1]}

def search_modality_matrix_paths():
    pass
    records = []
    search_dirs = [os.path.join(all_modals_dir, get_value('science.paths.directory_047', 'directory_047')), os.path.join(all_modals_dir, get_value('science.paths.directory_048', 'directory_048')), os.path.join(model_out_dir, get_value('science.paths.directory_011', 'directory_011'))]
    for modality_show, stems in MODALITIES.items():
        for cohort in ['internal', 'external']:
            found = []
            for d in search_dirs:
                if not os.path.exists(d):
                    continue
                patterns = []
                for stem in stems:
                    patterns.extend([os.path.join(d, f"{stem}_{cohort}{get_value('science.tables.result_044', 'result_044.csv')}"), os.path.join(d, f'{stem}_{cohort}.csv'), os.path.join(d, f"{stem.capitalize()}_{cohort}{get_value('science.tables.result_044', 'result_044.csv')}")])
                if cohort == 'internal':
                    patterns.append(os.path.join(d, f"{modality_show}{get_value('science.tables.attribution_003', 'attribution_003.csv')}"))
                for pat in patterns:
                    hits = glob.glob(pat)
                    found.extend(hits)
            found = list(dict.fromkeys(found))
            if len(found) > 0:
                records.append({'cohort': cohort.capitalize(), 'modality': modality_show, 'path': found[0], 'all_hits': ' | '.join(found)})
    return pd.DataFrame(records)
labels = read_csv_any(label_path)
sample_col = find_col(labels, ['sample_id', 'sample', 'SampleID', 'ID', 'patient_id', 'PatientID'])
if sample_col is None:
    sample_col = labels.columns[0]
labels[sample_col] = labels[sample_col].map(norm_id)
cohort_col = find_col(labels, ['Cohort', 'cohort', 'dataset', 'set', 'group_set'])
response_col = find_col(labels, ['Response', 'response', 'Responder', 'mRECIST_response', 'response_binary'])
pfs_time_col = find_col(labels, ['PFS_time', 'PFS', 'pfs_time', 'PFS_month', 'PFS_months'])
pfs_status_col = find_col(labels, ['PFS_status', 'pfs_status', 'PFS_event', 'pfs_event', 'PFS_status_binary'])
os_time_col = find_col(labels, ['OS_time', 'OS', 'os_time', 'OS_month', 'OS_months'])
os_status_col = find_col(labels, ['OS_status', 'os_status', 'OS_event', 'os_event', 'OS_status_binary'])
print('\nInferred columns:')
print('sample_col:', sample_col)
print('cohort_col:', cohort_col)
print('response_col:', response_col)
print('pfs_time_col:', pfs_time_col)
print('pfs_status_col:', pfs_status_col)
print('os_time_col:', os_time_col)
print('os_status_col:', os_status_col)
if cohort_col is None:
    labels['Cohort_for_plot'] = 'Internal'
else:
    labels['Cohort_for_plot'] = labels[cohort_col].astype(str).str.capitalize()
for col in [response_col, pfs_status_col, os_status_col]:
    if col is not None:
        labels[col] = pd.to_numeric(labels[col], errors='coerce')
all_sample_ids = labels[sample_col].dropna().astype(str).tolist()
outcome_records = []
for cohort in COHORT_ORDER:
    sub = labels[labels['Cohort_for_plot'].str.lower() == cohort.lower()].copy()
    if sub.empty and cohort == 'Internal':
        sub = labels.copy()
    n_total = len(sub)
    if response_col is not None:
        n_pos = int((sub[response_col] == 1).sum())
        n_neg = int((sub[response_col] == 0).sum())
        outcome_records.append({'cohort': cohort, 'outcome': 'Response', 'status': 'Positive/Event', 'n': n_pos, 'total': n_total})
        outcome_records.append({'cohort': cohort, 'outcome': 'Response', 'status': 'Negative/Censored', 'n': n_neg, 'total': n_total})
    if pfs_status_col is not None:
        n_event = int((sub[pfs_status_col] == 1).sum())
        n_censor = int((sub[pfs_status_col] == 0).sum())
        outcome_records.append({'cohort': cohort, 'outcome': 'PFS', 'status': 'Positive/Event', 'n': n_event, 'total': n_total})
        outcome_records.append({'cohort': cohort, 'outcome': 'PFS', 'status': 'Negative/Censored', 'n': n_censor, 'total': n_total})
    if os_status_col is not None:
        n_event = int((sub[os_status_col] == 1).sum())
        n_censor = int((sub[os_status_col] == 0).sum())
        outcome_records.append({'cohort': cohort, 'outcome': 'OS', 'status': 'Positive/Event', 'n': n_event, 'total': n_total})
        outcome_records.append({'cohort': cohort, 'outcome': 'OS', 'status': 'Negative/Censored', 'n': n_censor, 'total': n_total})
outcome_summary = pd.DataFrame(outcome_records)
outcome_summary = outcome_summary[outcome_summary['total'] > 0].copy()
if not outcome_summary.empty:
    outcome_summary['percentage'] = outcome_summary['n'] / outcome_summary['total'].replace(0, np.nan) * 100
matrix_paths = search_modality_matrix_paths()
matrix_records = []
for _, row in matrix_paths.iterrows():
    try:
        info = infer_matrix_orientation_and_counts(row['path'], sample_ids=all_sample_ids)
        matrix_records.append({'cohort': row['cohort'], 'modality': row['modality'], 'n_samples': info['n_samples'], 'n_features': info['n_features'], 'matched_samples': info['matched_samples'], 'orientation': info['orientation'], 'shape': f"{info['shape_rows']} × {info['shape_cols']}", 'path': row['path']})
    except Exception as e:
        matrix_records.append({'cohort': row['cohort'], 'modality': row['modality'], 'n_samples': np.nan, 'n_features': np.nan, 'matched_samples': np.nan, 'orientation': 'ERROR', 'shape': 'ERROR', 'path': row['path'], 'error': str(e)})
modality_summary = pd.DataFrame(matrix_records)
modality_summary['cohort'] = pd.Categorical(modality_summary['cohort'], categories=COHORT_ORDER, ordered=True)
modality_summary['modality'] = pd.Categorical(modality_summary['modality'], categories=list(MODALITIES.keys()), ordered=True)
modality_summary = modality_summary.sort_values(['cohort', 'modality'])
label_summary_path = os.path.join(out_dir, get_value('science.tables.metrics_020', 'metrics_020.csv'))
modality_summary_path = os.path.join(out_dir, get_value('science.tables.features_019', 'features_019.csv'))
matrix_paths_path = os.path.join(out_dir, get_value('science.tables.features_020', 'features_020.csv'))
outcome_summary.to_csv(label_summary_path, index=False, encoding='utf-8-sig')
modality_summary.to_csv(modality_summary_path, index=False, encoding='utf-8-sig')
matrix_paths.to_csv(matrix_paths_path, index=False, encoding='utf-8-sig')




print('\nModality summary:')

print('\nOutcome summary:')
if outcome_summary.empty:
    print('No outcome summary generated.')
else:
    pass
fig = plt.figure(figsize=(10.5, 5.8))
gs = fig.add_gridspec(nrows=2, ncols=2, width_ratios=[1.1, 1.3], height_ratios=[1.0, 1.0], wspace=0.32, hspace=0.45)
ax1 = fig.add_subplot(gs[:, 0])
heat_df = modality_summary.pivot_table(index='modality', columns='cohort', values='n_samples', aggfunc='first').reindex(index=list(MODALITIES.keys()), columns=COHORT_ORDER)
feat_df = modality_summary.pivot_table(index='modality', columns='cohort', values='n_features', aggfunc='first').reindex(index=list(MODALITIES.keys()), columns=COHORT_ORDER)
max_n = np.nanmax(heat_df.values.astype(float))
if not np.isfinite(max_n) or max_n <= 0:
    max_n = 1
for i, modality in enumerate(heat_df.index):
    for j, cohort in enumerate(heat_df.columns):
        n = heat_df.loc[modality, cohort]
        f = feat_df.loc[modality, cohort]
        if pd.isna(n):
            facecolor = '#F2F2F2'
            text = 'NA'
        else:
            alpha = 0.25 + 0.75 * (float(n) / max_n)
            base_color = mpl.colors.to_rgb(MODALITY_COLORS.get(modality, '#999999'))
            facecolor = tuple((1 - alpha * (1 - c) for c in base_color))
            text = f'n={int(n)}\nfeat={int(f)}' if pd.notna(f) else f'n={int(n)}'
        rect = Rectangle((j, i), 1, 1, facecolor=facecolor, edgecolor='white', linewidth=1.5)
        ax1.add_patch(rect)
        ax1.text(j + 0.5, i + 0.5, text, ha='center', va='center', fontsize=8, color='black')
ax1.set_xlim(0, len(COHORT_ORDER))
ax1.set_ylim(0, len(MODALITIES))
ax1.set_xticks(np.arange(len(COHORT_ORDER)) + 0.5)
ax1.set_xticklabels(COHORT_ORDER, fontsize=9)
ax1.set_yticks(np.arange(len(MODALITIES)) + 0.5)
ax1.set_yticklabels(list(MODALITIES.keys()), fontsize=9)
ax1.invert_yaxis()
ax1.set_title('Cohort and modality availability', fontsize=11, fontweight='bold')
ax1.tick_params(length=0)
for spine in ax1.spines.values():
    spine.set_visible(False)

def plot_endpoint_bars(ax, cohort_name):
    sub = outcome_summary[outcome_summary['cohort'] == cohort_name].copy()
    if sub.empty:
        ax.text(0.5, 0.5, f'No {cohort_name} endpoint labels', ha='center', va='center', fontsize=10)
        ax.axis('off')
        return
    outcomes = ['Response', 'PFS', 'OS']
    y_positions = np.arange(len(outcomes))
    left_values = np.zeros(len(outcomes))
    for status in ['Negative/Censored', 'Positive/Event']:
        vals = []
        labels = []
        for outcome in outcomes:
            row = sub[(sub['outcome'] == outcome) & (sub['status'] == status)]
            if row.empty:
                vals.append(0)
                labels.append('')
            else:
                n = int(row['n'].iloc[0])
                total = int(row['total'].iloc[0])
                pct = 100 * n / total if total > 0 else 0
                vals.append(n)
                labels.append(f'{n}\n{pct:.0f}%')
        bars = ax.barh(y_positions, vals, left=left_values, color=STATUS_COLORS[status], edgecolor='white', linewidth=0.8, label=status)
        for bar, lab, v in zip(bars, labels, vals):
            if v > 0:
                ax.text(bar.get_x() + bar.get_width() / 2, bar.get_y() + bar.get_height() / 2, lab, ha='center', va='center', fontsize=7, color='white' if status == 'Positive/Event' else 'black')
        left_values += np.array(vals)
    ax.set_yticks(y_positions)
    ax.set_yticklabels(outcomes, fontsize=9)
    ax.invert_yaxis()
    ax.set_xlabel('Number of patients', fontsize=9)
    ax.set_title(f'{cohort_name} endpoint distribution', fontsize=11, fontweight='bold')
    ax.tick_params(axis='x', labelsize=8)
    ax.grid(axis='x', color='#EAEAEA', linewidth=0.7)
    ax.set_axisbelow(True)
ax2 = fig.add_subplot(gs[0, 1])
plot_endpoint_bars(ax2, 'Internal')
ax3 = fig.add_subplot(gs[1, 1])
plot_endpoint_bars(ax3, 'External')
handles = [Patch(facecolor=STATUS_COLORS['Negative/Censored'], edgecolor='white', label='NR / censored'), Patch(facecolor=STATUS_COLORS['Positive/Event'], edgecolor='white', label='R / event')]
fig.legend(handles=handles, loc='upper right', bbox_to_anchor=(0.98, 0.98), frameon=False, fontsize=8)
fig.suptitle('Multimodal cohort and outcome overview', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout(rect=[0.02, 0.02, 0.98, 0.95])
out_png = os.path.join(out_dir, get_value('science.figures.plot_063', 'plot_063.png'))
out_pdf = os.path.join(out_dir, get_value('science.figures.plot_064', 'plot_064.pdf'))
plt.savefig(out_png, dpi=600, bbox_inches='tight')
plt.savefig(out_pdf, bbox_inches='tight')
plt.show()





In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from lifelines import CoxPHFitter
from lifelines.utils import concordance_index
stage2_out_dir = str(project_path(get_value('science.paths.directory_019', 'directory_019')))
fig4_out_dir = os.path.join(stage2_out_dir, get_value('science.paths.directory_026', 'directory_026'))
clinical_baseline_path = str(project_path(get_value('science.tables.clinical_001', 'clinical_001.xlsx')))
out_dir = os.path.join(fig4_out_dir, get_value('science.paths.directory_049', 'directory_049'))
os.makedirs(out_dir, exist_ok=True)




In [ ]:
try:
    analysis_df
    
except NameError:
    analysis_path = os.path.join(fig4_out_dir, get_value('science.tables.clinical_004', 'clinical_004.csv'))
    analysis_df = pd.read_csv(analysis_path, encoding='utf-8-sig')
    print('Loaded analysis_df from:')
    print(analysis_path)
analysis_df = analysis_df.copy()

def normalize_sample_id(x):
    return str(x).strip().lower()
analysis_df['sample'] = analysis_df['sample'].map(normalize_sample_id)
required_cols = ['sample', 'OS_time', 'OS_status', 'OS_risk']
missing_cols = [c for c in required_cols if c not in analysis_df.columns]
if len(missing_cols) > 0:
    raise ValueError(f'Missing required columns in analysis_df: {missing_cols}')
for c in ['OS_time', 'OS_status', 'OS_risk']:
    analysis_df[c] = pd.to_numeric(analysis_df[c], errors='coerce')
analysis_df['MRU_OS_risk'] = analysis_df['OS_risk']
print('analysis_df shape:', analysis_df.shape)



In [ ]:
def find_col(df, candidates, required=False):
    col_map = {str(c).lower(): c for c in df.columns}
    for cand in candidates:
        if str(cand).lower() in col_map:
            return col_map[str(cand).lower()]
    for cand in candidates:
        cand_low = str(cand).lower()
        for c in df.columns:
            if cand_low in str(c).lower():
                return c
    if required:
        raise ValueError(f'Cannot find columns from {candidates}. Existing columns: {df.columns.tolist()}')
    return None

def read_clinical_table(path):
    if path.endswith('.xlsx') or path.endswith('.xls'):
        df = pd.read_excel(path)
    else:
        df = pd.read_csv(path, encoding='utf-8-sig')
    sample_col = find_col(df, ['sample', 'sample_id', 'Sample', 'ID', 'id', get_column('science.translation.analysis_009', 'patient_name')], required=False)
    if sample_col is None:
        sample_col = df.columns[0]
    df = df.rename(columns={sample_col: 'sample'})
    df['sample'] = df['sample'].map(normalize_sample_id)
    df = df.drop_duplicates(subset='sample', keep='first')
    return df
clinical_raw = read_clinical_table(clinical_baseline_path)
print('Clinical table shape:', clinical_raw.shape)
print('Clinical columns:')
print(clinical_raw.columns.tolist())
recist_col = find_col(clinical_raw, [get_column('science.translation.analysis_016', 'RECIST1.1'), 'RECIST1.1', 'RECIST 1.1', 'RECIST', 'response_recist', 'RECIST_response'], required=False)
mrecist_col = find_col(clinical_raw, [get_column('science.translation.analysis_017', 'mRECIST'), 'mRECIST', 'mrecist', 'mRECIST response', 'mRECIST_response', 'response_mrecist'], required=False)
response_cols = {}
if recist_col is not None:
    response_cols['RECIST1.1'] = recist_col
if mrecist_col is not None:
    response_cols['mRECIST'] = mrecist_col
if len(response_cols) == 0:
    print('No RECIST/mRECIST column found. Fallback to existing Response_true.')
else:
    print('Detected radiographic response columns:')
    print(response_cols)
    for name, col in response_cols.items():
        
        print(clinical_raw[col].value_counts(dropna=False))


In [ ]:
def map_radiographic_response_to_binary(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x, (int, float, np.integer, np.floating)):
        if x in [0, 1]:
            return int(x)
    s = str(x).strip()
    s_low = s.lower()
    s_clean = s_low.replace(' ', '').replace('-', '').replace('_', '').replace('/', '').replace('\\', '').replace('\u3000', '')
    responder_terms = ['cr', 'pr', 'crpr', 'partialresponse', 'completeresponse', 'response', 'responder', 'orr', get_value('science.translation.analysis_018', 'response'), get_value('science.translation.analysis_019', 'partialresponse'), get_value('science.translation.analysis_020', 'completeresponse'), get_value('science.translation.analysis_021', 'response')]
    nonresponder_terms = ['sd', 'pd', 'sdpd', 'stabledisease', 'progressivedisease', 'nonresponse', 'nonresponder', 'nr', get_value('science.translation.analysis_022', 'stabledisease'), get_value('science.translation.analysis_023', 'progressivedisease'), get_value('science.translation.analysis_024', 'nonresponse')]
    if s_clean in responder_terms:
        return 1
    if s_clean in nonresponder_terms:
        return 0
    if ('cr' in s_clean or 'pr' in s_clean) and 'pd' not in s_clean:
        return 1
    if 'sd' in s_clean or 'pd' in s_clean:
        return 0
    try:
        val = float(s_clean)
        if val in [0, 1]:
            return int(val)
    except:
        pass
    return np.nan
os_df = analysis_df.copy()
if len(response_cols) > 0:
    response_extract = clinical_raw[['sample'] + list(response_cols.values())].copy()
    os_df = os_df.merge(response_extract, on='sample', how='left')
    response_metrics = []
    for metric_name, original_col in response_cols.items():
        safe_name = re.sub('[^A-Za-z0-9]+', '_', metric_name).strip('_')
        bin_col = f'{safe_name}_responder'
        os_df[bin_col] = os_df[original_col].apply(map_radiographic_response_to_binary)
        response_metrics.append({'metric_name': metric_name, 'original_col': original_col, 'bin_col': bin_col})
        print(f'\n{metric_name} binary mapping:')
        print(os_df[bin_col].value_counts(dropna=False))
else:
    os_df['Observed_responder'] = pd.to_numeric(os_df['Response_true'], errors='coerce')
    response_metrics = [{'metric_name': 'Observed response', 'original_col': 'Response_true', 'bin_col': 'Observed_responder'}]
    print('\nUsing Response_true as observed radiographic response.')
    print(os_df['Observed_responder'].value_counts(dropna=False))
merged_path = os.path.join(out_dir, get_value('science.tables.result_045', 'result_045.csv'))
os_df.to_csv(merged_path, index=False, encoding='utf-8-sig')




In [ ]:
def log1p_safe(s):
    s = pd.to_numeric(pd.Series(s), errors='coerce')
    s = s.clip(lower=0)
    return np.log1p(s)
if 'log_AFP' not in os_df.columns and 'AFP' in os_df.columns:
    os_df['log_AFP'] = log1p_safe(os_df['AFP'])
if 'log_Tumor_size' not in os_df.columns and 'Tumor_size' in os_df.columns:
    os_df['log_Tumor_size'] = log1p_safe(os_df['Tumor_size'])
core_covariates = [c for c in ['log_AFP', 'log_Tumor_size', 'PVTT', 'ALBI'] if c in os_df.columns]
print('Core clinical covariates:')
print(core_covariates)
if len(core_covariates) == 0:
    raise ValueError('No clinical covariates found. Check analysis_df clinical columns.')


In [ ]:
def os_cindex_from_risk(df, time_col, event_col, risk_col):
    tmp = df[[time_col, event_col, risk_col]].copy()
    for c in [time_col, event_col, risk_col]:
        tmp[c] = pd.to_numeric(tmp[c], errors='coerce')
    tmp = tmp.dropna()
    if tmp.shape[0] < 5 or tmp[event_col].sum() < 2:
        return np.nan
    return concordance_index(tmp[time_col], -tmp[risk_col], tmp[event_col])

def bootstrap_delta_cindex(df, time_col, event_col, risk_col_a, risk_col_b, n_boot=1000, random_state=42):
    pass
    tmp = df[[time_col, event_col, risk_col_a, risk_col_b]].copy()
    for c in [time_col, event_col, risk_col_a, risk_col_b]:
        tmp[c] = pd.to_numeric(tmp[c], errors='coerce')
    tmp = tmp.dropna().reset_index(drop=True)
    if tmp.shape[0] < 10 or tmp[event_col].sum() < 3:
        return {'delta_mean': np.nan, 'delta_lower': np.nan, 'delta_upper': np.nan, 'p_boot': np.nan, 'n_boot_valid': 0}
    rng = np.random.default_rng(random_state)
    deltas = []
    n = tmp.shape[0]
    for _ in range(n_boot):
        idx = rng.choice(np.arange(n), size=n, replace=True)
        boot = tmp.iloc[idx].copy()
        if boot[event_col].sum() < 2:
            continue
        try:
            ci_a = concordance_index(boot[time_col], -boot[risk_col_a], boot[event_col])
            ci_b = concordance_index(boot[time_col], -boot[risk_col_b], boot[event_col])
            deltas.append(ci_b - ci_a)
        except:
            continue
    deltas = np.array(deltas)
    if len(deltas) == 0:
        return {'delta_mean': np.nan, 'delta_lower': np.nan, 'delta_upper': np.nan, 'p_boot': np.nan, 'n_boot_valid': 0}
    p_boot = 2 * min(np.mean(deltas <= 0), np.mean(deltas >= 0))
    return {'delta_mean': float(np.mean(deltas)), 'delta_lower': float(np.percentile(deltas, 2.5)), 'delta_upper': float(np.percentile(deltas, 97.5)), 'p_boot': float(min(p_boot, 1.0)), 'n_boot_valid': int(len(deltas))}

def format_ci(x, low, high):
    if pd.isna(x) or pd.isna(low) or pd.isna(high):
        return 'NA'
    return f'{x:.3f} ({low:.3f}, {high:.3f})'

def format_p(p):
    if pd.isna(p):
        return 'NA'
    if p < 0.001:
        return '<0.001'
    return f'{p:.3f}'


In [ ]:
def get_oof_cox_risk_predictions(df, time_col, event_col, feature_cols, n_splits=5, n_repeats=20, random_state=42, penalizer=0.05):
    pass
    needed = ['sample', time_col, event_col] + feature_cols
    tmp = df[needed].copy()
    for c in [time_col, event_col] + feature_cols:
        tmp[c] = pd.to_numeric(tmp[c], errors='coerce')
    tmp = tmp.dropna(subset=[time_col, event_col]).reset_index(drop=True)
    X = tmp[feature_cols].copy()
    y_event = tmp[event_col].astype(int).values
    if np.sum(y_event) < n_splits:
        n_splits_use = max(2, int(np.sum(y_event)))
    else:
        n_splits_use = n_splits
    rskf = RepeatedStratifiedKFold(n_splits=n_splits_use, n_repeats=n_repeats, random_state=random_state)
    pred_records = []
    for fold_id, (train_idx, test_idx) in enumerate(rskf.split(X, y_event), start=1):
        train = tmp.iloc[train_idx].copy()
        test = tmp.iloc[test_idx].copy()
        X_train = train[feature_cols].copy()
        X_test = test[feature_cols].copy()
        imputer = SimpleImputer(strategy='median')
        scaler = StandardScaler()
        X_train_imp = imputer.fit_transform(X_train)
        X_test_imp = imputer.transform(X_test)
        X_train_scaled = scaler.fit_transform(X_train_imp)
        X_test_scaled = scaler.transform(X_test_imp)
        train_cox = pd.DataFrame(X_train_scaled, columns=feature_cols, index=train.index)
        train_cox[time_col] = train[time_col].values
        train_cox[event_col] = train[event_col].values
        test_x = pd.DataFrame(X_test_scaled, columns=feature_cols, index=test.index)
        feature_cols_keep = [c for c in feature_cols if train_cox[c].nunique(dropna=True) > 1]
        if len(feature_cols_keep) == 0:
            continue
        train_cox = train_cox[[time_col, event_col] + feature_cols_keep]
        test_x = test_x[feature_cols_keep]
        try:
            cph = CoxPHFitter(penalizer=penalizer)
            cph.fit(train_cox, duration_col=time_col, event_col=event_col)
            risk = cph.predict_partial_hazard(test_x).values.reshape(-1)
            pred_records.append(pd.DataFrame({'sample': test['sample'].values, 'OS_time': test[time_col].values, 'OS_status': test[event_col].values, 'risk': risk, 'fold_id': fold_id}))
        except Exception as e:
            print(f'[Fold {fold_id}] Cox failed: {e}')
            continue
    if len(pred_records) == 0:
        raise RuntimeError('No valid Cox OOF predictions generated.')
    pred_long = pd.concat(pred_records, axis=0)
    pred_agg = pred_long.groupby('sample', as_index=False).agg({'OS_time': 'first', 'OS_status': 'first', 'risk': 'mean'})
    return (pred_agg, pred_long)


In [ ]:
all_summary_rows = []
all_adjusted_rows = []
for spec in response_metrics:
    metric_name = spec['metric_name']
    bin_col = spec['bin_col']
    
    
    
    tmp = os_df.copy()
    needed = ['sample', 'OS_time', 'OS_status', 'MRU_OS_risk', bin_col] + core_covariates
    tmp = tmp[needed].copy()
    for c in ['OS_time', 'OS_status', 'MRU_OS_risk', bin_col] + core_covariates:
        tmp[c] = pd.to_numeric(tmp[c], errors='coerce')
    tmp = tmp.dropna(subset=['OS_time', 'OS_status', 'MRU_OS_risk', bin_col]).reset_index(drop=True)
    tmp[f'{bin_col}_risk'] = 1 - tmp[bin_col]
    print('Analyzable n:', tmp.shape[0])
    print('OS events:', int(tmp['OS_status'].sum()))
    print('Responder counts:')
    print(tmp[bin_col].value_counts(dropna=False))
    ci_recist = os_cindex_from_risk(tmp, time_col='OS_time', event_col='OS_status', risk_col=f'{bin_col}_risk')
    ci_mru = os_cindex_from_risk(tmp, time_col='OS_time', event_col='OS_status', risk_col='MRU_OS_risk')
    delta_mru_vs_recist = bootstrap_delta_cindex(tmp, time_col='OS_time', event_col='OS_status', risk_col_a=f'{bin_col}_risk', risk_col_b='MRU_OS_risk', n_boot=1000, random_state=42)
    clinical_recist_features = core_covariates + [bin_col]
    clinical_recist_mru_features = core_covariates + [bin_col, 'MRU_OS_risk']
    pred_clin_recist, pred_clin_recist_long = get_oof_cox_risk_predictions(tmp, time_col='OS_time', event_col='OS_status', feature_cols=clinical_recist_features, n_splits=5, n_repeats=20, random_state=42, penalizer=0.05)
    pred_clin_recist_mru, pred_clin_recist_mru_long = get_oof_cox_risk_predictions(tmp, time_col='OS_time', event_col='OS_status', feature_cols=clinical_recist_mru_features, n_splits=5, n_repeats=20, random_state=42, penalizer=0.05)
    pred_clin_recist = pred_clin_recist.rename(columns={'risk': 'Clinical_RECIST_risk'})
    pred_clin_recist_mru = pred_clin_recist_mru.rename(columns={'risk': 'Clinical_RECIST_MRU_risk'})
    pred_merged = pred_clin_recist[['sample', 'OS_time', 'OS_status', 'Clinical_RECIST_risk']].merge(pred_clin_recist_mru[['sample', 'Clinical_RECIST_MRU_risk']], on='sample', how='inner')
    ci_clin_recist = os_cindex_from_risk(pred_merged, time_col='OS_time', event_col='OS_status', risk_col='Clinical_RECIST_risk')
    ci_clin_recist_mru = os_cindex_from_risk(pred_merged, time_col='OS_time', event_col='OS_status', risk_col='Clinical_RECIST_MRU_risk')
    delta_increment = bootstrap_delta_cindex(pred_merged, time_col='OS_time', event_col='OS_status', risk_col_a='Clinical_RECIST_risk', risk_col_b='Clinical_RECIST_MRU_risk', n_boot=1000, random_state=42)
    safe_metric = re.sub('[^A-Za-z0-9]+', '_', metric_name).strip('_')
    pred_out_path = os.path.join(out_dir, f'OS_OOF_Cox_predictions_{safe_metric}.csv')
    pred_merged.to_csv(pred_out_path, index=False, encoding='utf-8-sig')
    cox_tmp = tmp.copy()
    cox_tmp['MRU_OS_risk_z'] = (cox_tmp['MRU_OS_risk'] - cox_tmp['MRU_OS_risk'].mean()) / cox_tmp['MRU_OS_risk'].std()
    full_features = core_covariates + [bin_col, 'MRU_OS_risk_z']
    cox_data = cox_tmp[['OS_time', 'OS_status'] + full_features].copy()
    for c in full_features:
        cox_data[c] = pd.to_numeric(cox_data[c], errors='coerce')
        cox_data[c] = cox_data[c].fillna(cox_data[c].median())
    keep_features = [c for c in full_features if cox_data[c].nunique(dropna=True) > 1]
    cox_data = cox_data[['OS_time', 'OS_status'] + keep_features]
    cph = CoxPHFitter(penalizer=0.05)
    cph.fit(cox_data, duration_col='OS_time', event_col='OS_status', robust=True)
    cox_summary = cph.summary.reset_index().rename(columns={'covariate': 'term'})
    cox_summary['Radiographic_metric'] = metric_name
    cox_summary['n'] = cox_data.shape[0]
    cox_summary['events'] = int(cox_data['OS_status'].sum())
    cox_out_path = os.path.join(out_dir, f"OS_adjusted_Cox_clinical_{safe_metric}{get_value('science.tables.result_046', 'result_046.csv')}")
    cox_summary.to_csv(cox_out_path, index=False, encoding='utf-8-sig')
    all_adjusted_rows.append(cox_summary)
    all_summary_rows.extend([{'Radiographic_metric': metric_name, 'Comparison': 'RECIST/mRECIST alone', 'Model': metric_name, 'C_index': ci_recist, 'Delta_reference': np.nan, 'Delta_C_index': np.nan, 'Delta_95CI_lower': np.nan, 'Delta_95CI_upper': np.nan, 'Bootstrap_P': np.nan, 'n': tmp.shape[0], 'events': int(tmp['OS_status'].sum())}, {'Radiographic_metric': metric_name, 'Comparison': 'MRU vs RECIST/mRECIST', 'Model': 'MRU_OS_risk', 'C_index': ci_mru, 'Delta_reference': metric_name, 'Delta_C_index': ci_mru - ci_recist, 'Delta_95CI_lower': delta_mru_vs_recist['delta_lower'], 'Delta_95CI_upper': delta_mru_vs_recist['delta_upper'], 'Bootstrap_P': delta_mru_vs_recist['p_boot'], 'n': tmp.shape[0], 'events': int(tmp['OS_status'].sum())}, {'Radiographic_metric': metric_name, 'Comparison': 'Clinical + RECIST/mRECIST', 'Model': f'Clinical + {metric_name}', 'C_index': ci_clin_recist, 'Delta_reference': np.nan, 'Delta_C_index': np.nan, 'Delta_95CI_lower': np.nan, 'Delta_95CI_upper': np.nan, 'Bootstrap_P': np.nan, 'n': pred_merged.shape[0], 'events': int(pred_merged['OS_status'].sum())}, {'Radiographic_metric': metric_name, 'Comparison': 'Clinical + RECIST/mRECIST + MRU', 'Model': f'Clinical + {metric_name} + MRU', 'C_index': ci_clin_recist_mru, 'Delta_reference': f'Clinical + {metric_name}', 'Delta_C_index': ci_clin_recist_mru - ci_clin_recist, 'Delta_95CI_lower': delta_increment['delta_lower'], 'Delta_95CI_upper': delta_increment['delta_upper'], 'Bootstrap_P': delta_increment['p_boot'], 'n': pred_merged.shape[0], 'events': int(pred_merged['OS_status'].sum())}])
    print('\nC-index summary:')
    print(f'{metric_name}: {ci_recist:.3f}')
    print(f'MRU: {ci_mru:.3f}')
    print(f'Clinical + {metric_name}: {ci_clin_recist:.3f}')
    print(f'Clinical + {metric_name} + MRU: {ci_clin_recist_mru:.3f}')
    
    
    
    
summary_df = pd.DataFrame(all_summary_rows)
summary_path = os.path.join(out_dir, get_value('science.tables.metrics_021', 'metrics_021.csv'))
summary_df.to_csv(summary_path, index=False, encoding='utf-8-sig')





In [ ]:
if len(all_adjusted_rows) > 0:
    adjusted_all_df = pd.concat(all_adjusted_rows, axis=0)
    adjusted_all_path = os.path.join(out_dir, get_value('science.tables.clinical_013', 'clinical_013.csv'))
    adjusted_all_df.to_csv(adjusted_all_path, index=False, encoding='utf-8-sig')
    
    
    


In [ ]:
def plot_os_cindex_comparison(summary_df, out_dir):
    metrics = summary_df['Radiographic_metric'].dropna().unique().tolist()
    for metric in metrics:
        sub = summary_df[summary_df['Radiographic_metric'] == metric].copy()
        plot_order = [metric, 'MRU_OS_risk', f'Clinical + {metric}', f'Clinical + {metric} + MRU']
        plot_df = []
        for model in plot_order:
            row = sub[sub['Model'] == model]
            if row.shape[0] > 0:
                plot_df.append(row.iloc[0])
        plot_df = pd.DataFrame(plot_df)
        if plot_df.shape[0] == 0:
            continue
        fig, ax = plt.subplots(figsize=(7.6, 4.6))
        x = np.arange(plot_df.shape[0])
        vals = plot_df['C_index'].values
        bars = ax.bar(x, vals, edgecolor='black', linewidth=0.8, alpha=0.85)
        ax.axhline(0.5, linestyle='--', linewidth=1.1, color='gray')
        for i, v in enumerate(vals):
            ax.text(i, v + 0.012, f'{v:.3f}', ha='center', va='bottom', fontsize=9)
        row_mru = plot_df[plot_df['Model'] == 'MRU_OS_risk']
        if row_mru.shape[0] > 0:
            r = row_mru.iloc[0]
            ax.text(1, max(vals) + 0.06, f"Δ vs {metric} = {r['Delta_C_index']:.3f}\n95% CI {r['Delta_95CI_lower']:.3f}, {r['Delta_95CI_upper']:.3f}", ha='center', va='bottom', fontsize=8)
        row_inc = plot_df[plot_df['Model'] == f'Clinical + {metric} + MRU']
        if row_inc.shape[0] > 0:
            r = row_inc.iloc[0]
            ax.text(3, max(vals) + 0.06, f"Δ = {r['Delta_C_index']:.3f}\n95% CI {r['Delta_95CI_lower']:.3f}, {r['Delta_95CI_upper']:.3f}", ha='center', va='bottom', fontsize=8)
        labels = [metric, 'MRU', f'Clinical\n+ {metric}', f'Clinical\n+ {metric}\n+ MRU']
        ax.set_xticks(x)
        ax.set_xticklabels(labels, fontsize=9)
        ymin = max(0.45, np.nanmin(vals) - 0.08)
        ymax = min(1.0, np.nanmax(vals) + 0.16)
        ax.set_ylim(ymin, ymax)
        ax.set_ylabel('OS C-index')
        ax.set_title(f'OS survival discrimination beyond {metric}', fontsize=12, fontweight='bold')
        ax.grid(axis='y', linestyle='--', alpha=0.3)
        plt.tight_layout()
        safe_metric = re.sub('[^A-Za-z0-9]+', '_', metric).strip('_')
        fig_path = os.path.join(out_dir, f"FigureS4_OS_MRU_vs_{safe_metric}{get_value('science.figures.plot_065', 'plot_065.png')}")
        pdf_path = fig_path.replace('.png', '.pdf')
        plt.savefig(fig_path, dpi=600, bbox_inches='tight')
        plt.savefig(pdf_path, bbox_inches='tight')
        plt.show()
        
        
        
plot_os_cindex_comparison(summary_df, out_dir)
